<a href="https://colab.research.google.com/github/mukundacity-glitch/football-news-bot/blob/main/FPL_FORAMT_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX NEWSROOM OS v2
Cell 0: Master Control Panel
"""
# @title 🎛️ Vortex Master Controls { display-mode: "form" }

import os
import json

print("🟣 Newsroom OS v2 — Master Control Panel initializing...")

try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

AUDIO_DIR = os.path.join(GD_ROOT, "audio")
os.makedirs(AUDIO_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════
# ★ 1. VIDEO FORMAT & LENGTH SELECTION
# ═══════════════════════════════════════════════════════════════════════
VIDEO_FORMAT_SELECTION = "A: Full GW Preview (17-36 min)" # @param ["Auto: System Decides", "A: Full GW Preview (17-36 min)", "B: Focused Short Update (8-12 min)", "C: Injury Report Only (5-8 min)", "D: Matchday Final Calls (10-15 min)", "E: Custom Build (Edit below)"]

# Translates the dropdown text into the keyword the OS engine needs
format_map = {
    "Auto: System Decides": "auto",
    "A: Full GW Preview (17-36 min)": "full",
    "B: Focused Short Update (8-12 min)": "short",
    "C: Injury Report Only (5-8 min)": "injury_only",
    "D: Matchday Final Calls (10-15 min)": "matchday",
    "E: Custom Build (Edit below)": "custom"
}
VIDEO_FORMAT = format_map[VIDEO_FORMAT_SELECTION]

# ═══════════════════════════════════════════════════════════════════════
# ★ 2. BROADCAST TIER OVERRIDE (Show Urgency & Scope)
# ═══════════════════════════════════════════════════════════════════════
FORCE_TIER_SELECTION = "Tier 4: Detail analysis (Deep Dive)" # @param ["Auto: System Decides", "Tier 1: RAPID (Emergency/Breaking)", "Tier 2: PRIORITY (Important Midweek)", "Tier 3: MAIN_SHOW (Standard Weekly)", "Tier 4: Detail analysis (Deep Dive)"]

tier_map = {
    "Auto: System Decides": None,
    "Tier 1: RAPID (Emergency/Breaking)": "RAPID",
    "Tier 2: PRIORITY (Important Midweek)": "PRIORITY",
    "Tier 3: MAIN_SHOW (Standard Weekly)": "MAIN_SHOW",
    "Tier 4: Detail analysis (Deep Dive)": "Detail analysis"
}
FORCE_TIER = tier_map[FORCE_TIER_SELECTION]

# ═══════════════════════════════════════════════════════════════════════
# ★ 3. DEADLINE PHASE OVERRIDE (Timeline Position)
# ═══════════════════════════════════════════════════════════════════════
FORCE_PHASE_SELECTION = "Phase 1: DISCOVERY (Early Week)" # @param ["Auto: System Decides", "Phase 1: DISCOVERY (Early Week)", "Phase 2: ADJUSTMENT (Midweek Trends)", "Phase 3: FINAL_CHECKS (Press Conferences)", "Phase 4: EXECUTION (Deadline Day)"]

phase_map = {
    "Auto: System Decides": None,
    "Phase 1: DISCOVERY (Early Week)": "DISCOVERY",
    "Phase 2: ADJUSTMENT (Midweek Trends)": "ADJUSTMENT",
    "Phase 3: FINAL_CHECKS (Press Conferences)": "FINAL_CHECKS",
    "Phase 4: EXECUTION (Deadline Day)": "EXECUTION"
}
FORCE_PHASE = phase_map[FORCE_PHASE_SELECTION]

# ═══════════════════════════════════════════════════════════════════════
# ★ 4. CUSTOM MODULES (Only used if Format is "E: Custom Build")
# ═══════════════════════════════════════════════════════════════════════
CUSTOM_MODULES = [
    "intro",
    "injury_report",
    "market_trends",
    "master_plan",
    "outro",
]

# ═══════════════════════════════════════════════════════════════════════
# ★ 5. RENDER QUALITY SETTINGS
# ═══════════════════════════════════════════════════════════════════════
RENDER_QUALITY = "Draft (5 FPS)" # @param ["Draft (5 FPS)", "Final (30 FPS)"]

print(f"✅ Master Controls locked.")
print(f"   Format : {VIDEO_FORMAT}")
print(f"   Tier   : {FORCE_TIER}")
print(f"   Phase  : {FORCE_PHASE}")
print(f"   Quality: {RENDER_QUALITY}")

🟣 Newsroom OS v2 — Master Control Panel initializing...
✅ Master Controls locked.
   Format : full
   Tier   : Detail analysis
   Phase  : DISCOVERY
   Quality: Draft (5 FPS)


In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 1: Environment Setup, Architecture & Global Helpers
"""

import os
import sys
import shutil
import subprocess
import requests
import unicodedata
from io import BytesIO
from PIL import Image
from google.colab import drive

# ── 0. MASTER RENDER CONTROLS ────────────────────────────────────────────
DRAFT_MODE = True  # Set to False for final cinematic render
FPS = 5 if DRAFT_MODE else 24
RESOLUTION = (854, 480) if DRAFT_MODE else (1920, 1080)
print(f"⚙️ PIPELINE MODE: {'DRAFT (Fast)' if DRAFT_MODE else 'FINAL (Cinematic)'}")
print(f"   Resolution: {RESOLUTION[0]}x{RESOLUTION[1]} | FPS: {FPS}\n")

# ── 1. MOUNT GOOGLE DRIVE ────────────────────────────────────────────────
print("Mounting Google Drive...")

# Step 1: Flush existing mount gracefully
try:
    drive.flush_and_unmount()
    print("   Previous mount flushed.")
except Exception:
    pass

# Step 2: Physically wipe the stale mountpoint directory
subprocess.run(["rm", "-rf", "/content/drive"], check=False)
print("   Stale mountpoint cleared.")

# Step 3: Fresh mount
drive.mount('/content/drive')
assert os.path.exists("/content/drive/MyDrive"), "❌ Drive mount failed. Please check permissions."
print("✅ Google Drive mounted successfully.")

# ── 2. INSTALL DEPENDENCIES ──────────────────────────────────────────────
print("\nVerifying and installing required packages...")
!pip install -q Pillow requests edge-tts soundfile nest-asyncio
print("✅ All dependencies installed and ready.")

# ── 3. BUILD DIRECTORY ARCHITECTURE ──────────────────────────────────────
GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

DIRECTORIES = [
    "png",
    "player_cards",
    "audio",
    "video",
    "players",
    "elements",
    "local_player_images"
]

print("\nEstablishing Directory Architecture...")
for folder in DIRECTORIES:
    path = os.path.join(GD_ROOT, folder)
    os.makedirs(path, exist_ok=True)
    print(f"   [Validated] {folder}/ -> {path}")

# ── 4. VERIFY MASTER BACKGROUND ──────────────────────────────────────────
bg_path = os.path.join(GD_ROOT, "elements", "7.png")

if os.path.exists(bg_path):
    print(f"\n✅ Master Background located at: {bg_path}")
else:
    print(f"\n⚠️  WARNING: '7.png' is missing. Please upload it to: {os.path.join(GD_ROOT, 'elements')} before rendering.")

# ── 5. GLOBAL HELPER FUNCTIONS ───────────────────────────────────────────
print("\nInitializing Global Image Helpers...")

# ── CLEAN SLATE PROTOCOL ──
PNG_DIR   = os.path.join(GD_ROOT, "png")
CARDS_DIR = os.path.join(GD_ROOT, "player_cards")
TEMP_DIR  = os.path.join(GD_ROOT, "audio")   # or whichever temp path you intend

# 🔴 FIX: Removed TEMP_DIR (audio) from wipe list to protect intro/outro music.
directories_to_wipe = [PNG_DIR, CARDS_DIR]

for directory in directories_to_wipe:
    if os.path.exists(directory):
        shutil.rmtree(directory)
    os.makedirs(directory, exist_ok=True)

print("🧹 CLEAN SLATE COMPLETE: PNG and Card assets wiped. Audio files preserved.")

# -*- coding: utf-8 -*-
"""
🔬 EDGE-TTS WORDBOUNDARY DIAGNOSTIC
Confirms WordBoundary events are firing after the boundary fix.
"""

import asyncio
import edge_tts
import nest_asyncio
nest_asyncio.apply()

VOICE = "en-GB-RyanNeural"
TEST_TEXT = "Haaland scored a brilliant goal for Manchester City today."

print("="*60)
print(" EDGE-TTS DIAGNOSTIC")
print("="*60)

import edge_tts as _et
print(f"\n[1] edge-tts version: {_et.__version__ if hasattr(_et,'__version__') else 'unknown'}")

async def diagnostic_stream():
    print(f"\n[2] Streaming test sentence: '{TEST_TEXT}'\n")
    comm = edge_tts.Communicate(TEST_TEXT, VOICE, boundary="WordBoundary")
    chunk_types_seen = {}
    word_chunks      = []
    first_word_chunk = None
    async for chunk in comm.stream():
        ctype = chunk.get("type", "UNKNOWN")
        chunk_types_seen[ctype] = chunk_types_seen.get(ctype, 0) + 1
        if ctype == "WordBoundary":
            word_chunks.append(chunk)
            if first_word_chunk is None:
                first_word_chunk = chunk
    print(f"[3] Chunk type counts:")
    for t, c in chunk_types_seen.items():
        print(f"    {t:<20} = {c}")
    print(f"\n[4] WordBoundary events: {len(word_chunks)}")
    if word_chunks:
        print(f"\n[5] First WordBoundary chunk (full dump):")
        for k, v in first_word_chunk.items():
            print(f"    {k!r:<15} = {v!r}")
    else:
        print(f"\n❌ NO WordBoundary events received")

asyncio.run(diagnostic_stream())

print("\n" + "="*60)
print(" DIAGNOSTIC COMPLETE")
print("="*60)

⚙️ PIPELINE MODE: DRAFT (Fast)
   Resolution: 854x480 | FPS: 5

Mounting Google Drive...
Drive not mounted, so nothing to flush and unmount.
   Previous mount flushed.
   Stale mountpoint cleared.
Mounted at /content/drive
✅ Google Drive mounted successfully.

Verifying and installing required packages...
✅ All dependencies installed and ready.

Establishing Directory Architecture...
   [Validated] png/ -> /content/drive/MyDrive/Vortex_Final/png
   [Validated] player_cards/ -> /content/drive/MyDrive/Vortex_Final/player_cards
   [Validated] audio/ -> /content/drive/MyDrive/Vortex_Final/audio
   [Validated] video/ -> /content/drive/MyDrive/Vortex_Final/video
   [Validated] players/ -> /content/drive/MyDrive/Vortex_Final/players
   [Validated] elements/ -> /content/drive/MyDrive/Vortex_Final/elements
   [Validated] local_player_images/ -> /content/drive/MyDrive/Vortex_Final/local_player_images

⚠️  WARNING: '7.png' is missing. Please upload it to: /content/drive/MyDrive/Vortex_Final/eleme

In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX NEWSROOM OS v2
Cell 0: Story Priority Engine — Live event scoring, decay, and saturation
"""

import os, json, math, requests, re
from datetime import datetime, timezone, timedelta

print("🟣 Newsroom OS v2 — Story Priority Engine initializing...")

try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

AUDIO_DIR = os.path.join(GD_ROOT, "audio")
os.makedirs(AUDIO_DIR, exist_ok=True)

EST_TZ     = timezone(timedelta(hours=-5))
NOW_EST    = datetime.now(EST_TZ)
HEADERS    = {"User-Agent": "Mozilla/5.0"}

# ═══════════════════════════════════════════════════════════════════════
# 1. STORY WEIGHT CONSTANTS
# ═══════════════════════════════════════════════════════════════════════
W_OWNERSHIP      = 0.35   # scaled 0–100 (ownership %)
W_DEADLINE       = 0.25   # urgency near deadline
W_NOVELTY        = 0.20   # inverse of story age
W_TEAM_STAKES    = 0.10   # relegation/title relevance
W_SATURATION_PEN = 0.10   # penalty for over-covered topics

# Half-life rules (hours)
HALF_LIFE = {
    "breaking":   1.0,
    "injury":     6.0,
    "press":     24.0,
    "stats":    120.0,
    "fixture":   72.0,
}

# ═══════════════════════════════════════════════════════════════════════
# 2. BOOTSTRAP DATA (requires Cell 2 to have run first)
# ═══════════════════════════════════════════════════════════════════════
try:
    boot  = requests.get("https://fantasy.premierleague.com/api/bootstrap-static/", timeout=15).json()
    fixes = requests.get("https://fantasy.premierleague.com/api/fixtures/", timeout=15).json()

    ELEMENTS  = {p["id"]: p for p in boot["elements"]}
    TEAMS     = {t["id"]: t for t in boot["teams"]}
    EVENTS    = {e["id"]: e for e in boot["events"]}
    CURR_GW   = next((e["id"] for e in boot["events"] if e.get("is_current")), 1)
    NEXT_GW   = next((e["id"] for e in boot["events"] if e.get("is_next")), CURR_GW + 1)
    NEXT_EVENT= next((e for e in boot["events"] if e.get("is_next")), {})

    # Parse deadline
    try:
        DL_UTC = datetime.strptime(NEXT_EVENT["deadline_time"], "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
        DL_EST = DL_UTC.astimezone(EST_TZ)
        HOURS_TO_DEADLINE = max(0.0, (DL_EST - NOW_EST).total_seconds() / 3600)
    except Exception:
        HOURS_TO_DEADLINE = 72.0

    print(f"✅ Bootstrap loaded — GW{NEXT_GW} | Deadline in {HOURS_TO_DEADLINE:.1f}h")
except Exception as e:
    print(f"⚠️ Bootstrap load failed: {e}")
    HOURS_TO_DEADLINE = 72.0
    ELEMENTS, TEAMS, EVENTS = {}, {}, {}
    CURR_GW, NEXT_GW = 1, 2

# ═══════════════════════════════════════════════════════════════════════
# 3. STORY OBJECT BUILDER
# ═══════════════════════════════════════════════════════════════════════
def decay_factor(story_type, age_hours):
    """Exponential decay based on story half-life."""
    hl = HALF_LIFE.get(story_type, 24.0)
    return math.exp(-0.693 * age_hours / hl)

def deadline_urgency(hours_left):
    """Returns 0.0–1.0; peaks at 0h, near-zero at 168h."""
    if hours_left <= 0: return 1.0
    return max(0.0, 1.0 - (hours_left / 168.0) ** 0.5)

def ownership_weight(pct):
    """Non-linear: above 20% ownership gets steep boost."""
    return min(100.0, pct * (1.0 + pct / 40.0))

def score_story(story: dict) -> float:
    """
    story keys: type, player_id, age_hours, custom_base
    Returns float 0–100
    """
    p = ELEMENTS.get(story.get("player_id"), {})
    own_pct  = float(p.get("selected_by_percent", 0) or 0)
    age_h    = float(story.get("age_hours", 0))
    s_type   = story.get("type", "stats")
    custom   = float(story.get("custom_base", 50))
    sat_pen  = float(story.get("saturation_penalty", 0))

    # Component scores
    own_score  = ownership_weight(own_pct)        # 0–100
    dl_score   = deadline_urgency(HOURS_TO_DEADLINE) * 100  # 0–100
    nov_score  = decay_factor(s_type, age_h) * 100  # 0–100
    stake_score = float(story.get("team_stakes", 5)) * 10   # 0–100

    raw = (
        W_OWNERSHIP   * own_score  +
        W_DEADLINE    * dl_score   +
        W_NOVELTY     * nov_score  +
        W_TEAM_STAKES * stake_score +
        (1 - W_OWNERSHIP - W_DEADLINE - W_NOVELTY - W_TEAM_STAKES) * custom
        - W_SATURATION_PEN * sat_pen
    )
    return round(min(100.0, max(0.0, raw)), 2)

# ═══════════════════════════════════════════════════════════════════════
# 4. AUTO-GENERATE LIVE STORIES FROM FPL API
# ═══════════════════════════════════════════════════════════════════════
stories = []

# --- Injury / Suspension flags ---
for pid, p in ELEMENTS.items():
    status  = p.get("status", "a")
    chance  = p.get("chance_of_playing_next_round")
    news    = (p.get("news") or "").strip()
    own_pct = float(p.get("selected_by_percent", 0) or 0)
    if own_pct < 1.0 and status == "a": continue

    if status in ("i", "u") or chance == 0:
        s_type, base, age_h = "injury", 70, 2
    elif status == "s":
        s_type, base, age_h = "breaking", 75, 1
    elif chance is not None and chance <= 50:
        s_type, base, age_h = "injury", 50, 6
    elif news:
        s_type, base, age_h = "press", 30, 12
    else:
        continue

    stories.append({
        "type":       s_type,
        "category":   "injury_suspension",
        "player_id":  pid,
        "player_name": p.get("web_name"),
        "own_pct":    own_pct,
        "news":       news,
        "age_hours":  age_h,
        "custom_base": base,
        "team_stakes": 5,
        "saturation_penalty": 0,
    })

# --- Transfer volume spikes ---
top_in  = sorted(ELEMENTS.values(), key=lambda x: x.get("transfers_in_event", 0), reverse=True)[:5]
top_out = sorted(ELEMENTS.values(), key=lambda x: x.get("transfers_out_event", 0), reverse=True)[:5]

for p in top_in:
    t_vol = p.get("transfers_in_event", 0)
    if t_vol < 10000: continue
    stories.append({
        "type": "stats", "category": "transfer_in",
        "player_id": p["id"], "player_name": p["web_name"],
        "own_pct": float(p.get("selected_by_percent", 0) or 0),
        "age_hours": 4, "custom_base": min(80, t_vol / 5000),
        "team_stakes": 4, "saturation_penalty": 10,
    })

for p in top_out:
    t_vol = p.get("transfers_out_event", 0)
    if t_vol < 10000: continue
    stories.append({
        "type": "stats", "category": "transfer_out",
        "player_id": p["id"], "player_name": p["web_name"],
        "own_pct": float(p.get("selected_by_percent", 0) or 0),
        "age_hours": 4, "custom_base": min(75, t_vol / 5000),
        "team_stakes": 4, "saturation_penalty": 10,
    })

# --- Double / Blank Gameweek detection ---
gw_team_counts = {}
for fx in fixes:
    ev = fx.get("event")
    if ev != NEXT_GW: continue
    for tid in (fx["team_h"], fx["team_a"]):
        gw_team_counts[tid] = gw_team_counts.get(tid, 0) + 1

DGW_TEAMS   = {tid for tid, cnt in gw_team_counts.items() if cnt > 1}
BLANK_TEAMS = set(TEAMS.keys()) - set(gw_team_counts.keys())
IS_DGW      = len(DGW_TEAMS) > 0
IS_BGW      = len(BLANK_TEAMS) > 0

if IS_DGW:
    stories.append({
        "type": "fixture", "category": "dgw",
        "player_id": None, "player_name": None,
        "own_pct": 100, "age_hours": 0,
        "custom_base": 85, "team_stakes": 9,
        "saturation_penalty": 0,
        "meta": {"dgw_teams": list(DGW_TEAMS), "blank_teams": list(BLANK_TEAMS)},
    })

if IS_BGW:
    stories.append({
        "type": "fixture", "category": "bgw",
        "player_id": None, "player_name": None,
        "own_pct": 100, "age_hours": 0,
        "custom_base": 80, "team_stakes": 9,
        "saturation_penalty": 0,
        "meta": {"blank_teams": list(BLANK_TEAMS)},
    })

# ═══════════════════════════════════════════════════════════════════════
# 5. SCORE ALL STORIES
# ═══════════════════════════════════════════════════════════════════════
for s in stories:
    s["score"] = score_story(s)

stories.sort(key=lambda x: x["score"], reverse=True)

# ═══════════════════════════════════════════════════════════════════════
# 6. DETERMINE TOP STORY TYPE + BROADCAST TIER
# ═══════════════════════════════════════════════════════════════════════
top_story_score    = stories[0]["score"] if stories else 0
top_story_category = stories[0]["category"] if stories else "none"
top_story_type     = stories[0]["type"] if stories else "stats"

if top_story_score >= 80:
    BROADCAST_TIER = "RAPID"       # emergency short + full video
elif top_story_score >= 60:
    BROADCAST_TIER = "PRIORITY"          # focused 2–3 module show
elif top_story_score >= 40:
    BROADCAST_TIER = "MAIN_SHOW"        # normal full show
else:
    BROADCAST_TIER = "Detail analysis"       # single long-form in-depth

# Deadline phase
if HOURS_TO_DEADLINE <= 6:
    DEADLINE_PHASE = "EXECUTION"       # captain + transfers only
elif HOURS_TO_DEADLINE <= 24:
    DEADLINE_PHASE = "FINAL_CHECKS"   # team news + decisions
elif HOURS_TO_DEADLINE <= 72:
    DEADLINE_PHASE = "ADJUSTMENT"     # midweek analysis
else:
    DEADLINE_PHASE = "DISCOVERY"      # early preview

globals()["stories"]             = stories
globals()["top_story_score"]     = top_story_score
globals()["top_story_category"]  = top_story_category
globals()["BROADCAST_TIER"]      = BROADCAST_TIER
globals()["DEADLINE_PHASE"]      = DEADLINE_PHASE
globals()["IS_DGW"]              = IS_DGW
globals()["IS_BGW"]              = IS_BGW
globals()["DGW_TEAMS"]           = DGW_TEAMS
globals()["BLANK_TEAMS"]         = BLANK_TEAMS
globals()["HOURS_TO_DEADLINE"]   = HOURS_TO_DEADLINE

print(f"\n{'='*60}")
print(f" 🔴 NEWSROOM OS — STORY ENGINE LOCKED")
print(f"   Stories found      : {len(stories)}")
print(f"   Top story score    : {top_story_score:.1f} / 100")
print(f"   Top story category : {top_story_category}")
print(f"   Broadcast tier     : {BROADCAST_TIER}")
print(f"   Deadline phase     : {DEADLINE_PHASE}")
print(f"   Hours to deadline  : {HOURS_TO_DEADLINE:.1f}h")
print(f"   DGW active         : {IS_DGW}  | BGW active: {IS_BGW}")
print(f"{'='*60}\n")

# Save story log
story_log_path = os.path.join(AUDIO_DIR, "story_scores.json")
with open(story_log_path, "w") as f:
    json.dump({
        "generated_at": NOW_EST.isoformat(),
        "broadcast_tier": BROADCAST_TIER,
        "deadline_phase": DEADLINE_PHASE,
        "hours_to_deadline": HOURS_TO_DEADLINE,
        "is_dgw": IS_DGW,
        "is_bgw": IS_BGW,
        "top_stories": stories[:10],
    }, f, indent=2)

print(f"✅ Story scores saved → {story_log_path}")
print("\n🏆 Top 5 Stories This Gameweek:")
for i, s in enumerate(stories[:5]):
    name = s.get("player_name") or s.get("category", "Global Event")
    print(f"   {i+1}. [{s['score']:.1f}] {name.upper()} — {s['category']}")
    boot_data = boot

🟣 Newsroom OS v2 — Story Priority Engine initializing...
✅ Bootstrap loaded — GW39 | Deadline in 72.0h

 🔴 NEWSROOM OS — STORY ENGINE LOCKED
   Stories found      : 279
   Top story score    : 45.6 / 100
   Top story category : bgw
   Broadcast tier     : MAIN_SHOW
   Deadline phase     : ADJUSTMENT
   Hours to deadline  : 72.0h
   DGW active         : False  | BGW active: True

✅ Story scores saved → /content/drive/MyDrive/Vortex_Final/audio/story_scores.json

🏆 Top 5 Stories This Gameweek:
   1. [45.6] BGW — bgw
   2. [44.2] EKITIKÉ — injury_suspension
   3. [40.4] KUDUS — injury_suspension
   4. [38.7] GREALISH — injury_suspension
   5. [37.6] ROMERO — injury_suspension


In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX NEWSROOM OS v3 - 2026/2027 SKY SPORTS EDITION
Cell 2: Core Mathematical Models, Global CSS Injectors & Asset Optimization Engine
"""
print("Initializing Sky Sports Class Layout Engine & Core Mathematics...")

import math

# ═══════════════════════════════════════════════════════════════════════
# ★ 1. SKY SPORTS PREMIUM COLOR GRAPHICS & CSS PROPERTY INJECTOR
# ═══════════════════════════════════════════════════════════════════════
# High-class dark broadcast aesthetic: Midnight indigo gradients, electric neon accents
SKY_SPORTS_THEME_CSS = """
:root {
    --vortex-bg-top: #0a0118;
    --vortex-bg-bottom: #030006;
    --vortex-panel-bg: rgba(18, 10, 36, 0.85);
    --vortex-panel-border: #00e5ff;

    /* Neon Accents */
    --ss-gold: #ffd700;
    --ss-white: #ffffff;
    --ss-cyan: #00e5ff;
    --ss-green: #00ff87;
    --ss-red: #ff1744;
    --ss-purple: #38003c;

    /* Rows Alternating Benches */
    --ss-row-a: #0e0524;
    --ss-row-b: #160933;

    /* Fonts */
    --ss-font-display: 'Roboto Condensed', 'Arial Black', sans-serif;
    --ss-font-body: 'Roboto', sans-serif;
}

/* Tier Status Badges - Sleek Broadcast Gradients */
.tier-T1 { background: linear-gradient(135deg, #00e676, #ffffff); color: #000000; font-weight: 800; }
.tier-T2 { background: linear-gradient(135deg, #2979ff, #ffffff); color: #000000; font-weight: 800; }
.tier-T3 { background: linear-gradient(135deg, #ffea00, #ffffff); color: #000000; font-weight: 800; }
.tier-T4 { background: linear-gradient(135deg, #9e9e9e, #ffffff); color: #000000; font-weight: 700; }
.tier-T5 { background: linear-gradient(135deg, #f50057, #ffffff); color: #ffffff; font-weight: 700; }

.special-dgw { background: #000000; color: var(--ss-cyan); border: 2px solid var(--ss-cyan); box-shadow: 0 0 15px var(--ss-cyan); }
.special-bgw { background: #000000; color: var(--ss-red); border: 2px solid var(--ss-red); box-shadow: 0 0 15px var(--ss-red); }
"""

TEAM_COLORS = {
    "ARS":"#EF0107","AVL":"#670E36","BOU":"#DA291C","BRE":"#C8102E",
    "BHA":"#0057B8","CHE":"#034694","CRY":"#1B458F","EVE":"#003399",
    "FUL":"#242424","IPS":"#003090","LEI":"#003090","LIV":"#C8102E",
    "MCI":"#6CABDD","MUN":"#DA291C","NEW":"#242424","NFO":"#DD0000",
    "SOU":"#D71920","TOT":"#132257","WHU":"#7A263A","WOL":"#D99000",
    "COV":"#00BFFF","SUN":"#FF0000","HUL":"#FFA500","LEE":"#FFFFFF"
}

# ═══════════════════════════════════════════════════════════════════════
# ★ 2. PARSE AND MAP INCOMING BRIDGE DATASETS
# ═══════════════════════════════════════════════════════════════════════
# Extract historical baselines safely, accounting for planning overrides
if isinstance(fpl_boot, dict) and fpl_boot.get("off_season"):
    # Off-season Planning Mode Active
    CURR_GW = 0
    NEXT_GW = 1
    GLOBAL_T = {
        i+1: {"name": item["team_h_name"], "short": item["team_h_name"][:3].upper(), "code": 100+i,
              "att_h": 1200, "att_a": 1150, "def_h": 1100, "def_a": 1150}
        for i, item in enumerate(fpl_fixes)
    }
    # Append away team elements dynamically to the registry map
    for i, item in enumerate(fpl_fixes):
        GLOBAL_T[20+i] = {"name": item["team_a_name"], "short": item["team_a_name"][:3].upper(), "code": 200+i,
                          "att_h": 1150, "att_a": 1100, "def_h": 1150, "def_a": 1200}
    ELEMENTS = {}
    TARGET_GWS = [1]
else:
    # Live FPL API Production Mode Active
    events = fpl_boot["events"]
    CURR_GW = next((e["id"] for e in events if e.get("is_current")), 1)
    NEXT_GW = next((e["id"] for e in events if e.get("is_next")), CURR_GW + 1)

    GLOBAL_T = {t["id"]: {
        "name": t["name"], "short": t["short_name"], "code": t["code"],
        "att_h": t["strength_attack_home"], "att_a": t["strength_attack_away"],
        "def_h": t["strength_defence_home"], "def_a": t["strength_defence_away"]
    } for t in fpl_boot.get("teams", [])}

    ELEMENTS = {p["id"]: p for p in fpl_boot["elements"]}

    max_season_gw = max((e["id"] for e in events), default=NEXT_GW)
    end_window = min(NEXT_GW + 7, max_season_gw + 1)
    TARGET_GWS = list(range(NEXT_GW, end_window))

print(f"   -> Mode Confirmed. Analysis Locked Onto: Gameweek {NEXT_GW}")

# ═══════════════════════════════════════════════════════════════════════
# ★ 3. ELITE MATHEMATICAL CORE: POISSON EXPECTED GOALS & SIGMOID FDR
# ═══════════════════════════════════════════════════════════════════════
AVG_H = 1.55; AVG_A = 1.25; API_BASE = 1100.0

def poisson_xg(att_h, def_a, att_a, def_h, home=True):
    if home: return AVG_H * (att_h / API_BASE) / (def_a / API_BASE)
    else:    return AVG_A * (att_a / API_BASE) / (def_h / API_BASE)

def cs_pct(xg_opp):
    return math.exp(-xg_opp) * 100

# Calculate Dynamic League Average Expected Goals dynamically over the planning window
_all_xg_samples = []
if isinstance(fpl_boot, dict) and fpl_boot.get("off_season"):
    # Off-season statistical model equalization
    DYNAMIC_LEAGUE_AVG_XG = 1.350
else:
    for fx in fpl_fixes:
        if fx.get("event") in TARGET_GWS and fx["team_h"] in GLOBAL_T and fx["team_a"] in GLOBAL_T:
            hi, ai = fx["team_h"], fx["team_a"]
            _all_xg_samples.append(poisson_xg(GLOBAL_T[hi]["att_h"], GLOBAL_T[ai]["def_a"], GLOBAL_T[ai]["att_a"], GLOBAL_T[hi]["def_h"], True))
            _all_xg_samples.append(poisson_xg(GLOBAL_T[hi]["att_h"], GLOBAL_T[ai]["def_a"], GLOBAL_T[ai]["att_a"], GLOBAL_T[hi]["def_h"], False))
    DYNAMIC_LEAGUE_AVG_XG = sum(_all_xg_samples) / len(_all_xg_samples) if _all_xg_samples else 1.350

print(f"📈 Global Dynamic League Avg xG Locked: {DYNAMIC_LEAGUE_AVG_XG:.3f}")

def compute_fdr(xg_for, league_avg_xg=DYNAMIC_LEAGUE_AVG_XG):
    """Sigmoid-normalized structural FDR calculation matrix: 1.0 (Elite) -> 5.0 (Tough)."""
    ratio = xg_for / max(league_avg_xg, 0.01)
    log_r = math.log(max(ratio, 0.01))
    sigmoid = 1.0 / (1.0 + math.exp(-2.5 * log_r))
    fdr = 5.0 - 4.0 * sigmoid
    return round(max(1.0, min(5.0, fdr)), 1)

def get_vortex_fdr(team_id, opp_id, is_home):
    """Resolves raw numerical parameters into precise custom ratings configurations."""
    t_team = GLOBAL_T.get(team_id)
    o_team = GLOBAL_T.get(opp_id)
    if not t_team or not o_team:
        return 3.0

    if is_home:
        xg_for = poisson_xg(t_team["att_h"], o_team["def_a"], o_team["att_a"], t_team["def_h"], True)
    else:
        xg_for = poisson_xg(o_team["att_h"], t_team["def_a"], t_team["att_a"], o_team["def_h"], False)
    return compute_fdr(xg_for)

# ═══════════════════════════════════════════════════════════════════════
# ★ 4. WEB-CLASS UI TIER TRANSITIONS MAPS
# ═══════════════════════════════════════════════════════════════════════
def get_tier_xg_class(xg):
    if xg >= 1.69: return "tier-T1"
    if xg >= 1.50: return "tier-T2"
    if xg >= 1.30: return "tier-T3"
    if xg >= 1.25: return "tier-T4"
    return "tier-T5"

def get_tier_cs_class(cs_percentage):
    if cs_percentage >= 35: return "tier-T1"
    if cs_percentage >= 30: return "tier-T2"
    if cs_percentage >= 25: return "tier-T3"
    if cs_percentage >= 20: return "tier-T4"
    return "tier-T5"

def get_fdr_color_hex(fdr_score):
    if fdr_score <= 2.0: return "#00ff87" # Safe Green
    if fdr_score <= 2.5: return "#76ff03" # Light Green
    if fdr_score <= 3.5: return "#ffd700" # Warning Yellow
    if fdr_score <= 4.5: return "#ff1744" # High Alert Red
    return "#8b0000"                      # Critical Deep Dark Crimson

print("\n============================================================")
print(" 🟣 CELL 2 COMPLETE: HIGH-CLASS SYSTEM DATA MODULE ONLINE ")
print("============================================================\n")

In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX NEWSROOM OS v2
Cell 0b: Module Planner — Auto Blueprint System
"""

print("🟣 Newsroom OS v2 — Module Planner Engine initializing...")

# ═══════════════════════════════════════════════════════════════════════
# MODULE REGISTRY
# ═══════════════════════════════════════════════════════════════════════
MODULE_REGISTRY = {
    "intro": {
        "slide_key":  "Slide_1_Intro",
        "png_file":   "Slide_1_Intro_Empty.png",
        "slide_type": "agenda_s1",
        "min_dur": 40, "max_dur": 120,
        "always_include": True,
    },
    "market_trends": {
        "slide_key":  "Slide_2_Market_Trends",
        "png_file":   "Slide_2_Empty.png",
        "slide_type": "two_stage",
        "min_dur": 90, "max_dur": 180,
        "triggers": ["transfer_in", "transfer_out", "DISCOVERY", "ADJUSTMENT"],
    },
    "sword_shield": {
        "slide_key":  "Slide_3_Sword_Shield",
        "png_file":   "Slide_3_Empty.png",
        "slide_type": "two_stage",
        "min_dur": 90, "max_dur": 180,
        "triggers": ["DISCOVERY", "ADJUSTMENT"],
    },
    "fdr_map": {
        "slide_key":  "Slide_4_fdr_map",
        "png_file":   "Slide_4_fdr_map.png",
        "slide_type": "none",
        "min_dur": 90, "max_dur": 200,
        "triggers": ["DISCOVERY", "dgw", "bgw"],
    },
    "projected_goals": {
        "slide_key":  "Slide_5_Projected_Goals",
        "png_file":   "Slide_5_Projected_Goals.png",
        "slide_type": "none",
        "min_dur": 90, "max_dur": 200,
        "triggers": ["DISCOVERY", "dgw"],
    },
    "cs_odds": {
        "slide_key":  "Slide_6_CS_Odds",
        "png_file":   "Slide_6_CS_Odds.png",
        "slide_type": "none",
        "min_dur": 60, "max_dur": 160,
        "triggers": ["DISCOVERY", "ADJUSTMENT"],
    },
    "injury_report": {
        "slide_key":  "Slide_7a_Injury_Risk",
        "png_file":   "Slide_7a_Injury_MinuteRisk.png",
        "slide_type": "two_stage",
        "min_dur": 90, "max_dur": 200,
        "triggers": [
            "injury_suspension", "PRIORITY", "RAPID",
            "ADJUSTMENT", "FINAL_CHECKS", "EXECUTION",
        ],
    },
    "all_20_teams": {
        "slide_key":  "Slide_7_All_20_Teams",
        "png_file":   "Slide_7_All_20_Teams.png",
        "slide_type": "none",
        "min_dur": 90, "max_dur": 200,
        "triggers": ["DISCOVERY", "ADJUSTMENT"],
    },
    "targets_avoids": {
        "slide_key":  "Slide_8_Targets_Avoids",
        "png_file":   "Slide_8_Empty.png",
        "slide_type": "two_stage",
        "min_dur": 120, "max_dur": 240,
        "triggers": [
            "ADJUSTMENT", "FINAL_CHECKS", "EXECUTION",
            "transfer_in", "transfer_out",
        ],
    },
    "gw_review": {
        "slide_key":  "Slide_9_Base",
        "png_file":   "Slide_9_Base.png",
        "slide_type": "squad_s9",
        "min_dur": 120, "max_dur": 240,
        "triggers": ["DISCOVERY"],
        "after_match_only": True,
    },
    "master_plan": {
        "slide_key":  "Slide_10_Plan_Base",
        "png_file":   "Slide_10_Plan_Base.png",
        "slide_type": "plan_s10",
        "min_dur": 120, "max_dur": 240,
        "triggers": [
            "FINAL_CHECKS", "EXECUTION",
            "ADJUSTMENT", "dgw",
        ],
    },
    "chip_strategy": {
        "slide_key":  "Slide_11_Chip_Strategy",
        "png_file":   "Slide_11_Chip_Strategy.png",
        "slide_type": "two_stage",  # 🔴 Upgraded to allow animated bullet points
        "min_dur": 60, "max_dur": 120,
        "triggers": ["dgw", "bgw", "DISCOVERY", "ADJUSTMENT"],
    },
    "outro": {
        "slide_key":  "Slide_12_Outro",
        "png_file":   "Slide_12_Outro_Empty.png",
        "slide_type": "two_stage",
        "min_dur": 60, "max_dur": 120,
        "always_include": True,
    },
}

# ═══════════════════════════════════════════════════════════════════════
# FORMAT PRESETS
# ═══════════════════════════════════════════════════════════════════════
FORMAT_PRESETS = {
    "full": [
        "intro", "gw_review", "market_trends", "sword_shield",
        "fdr_map", "projected_goals", "cs_odds", "injury_report",
        "all_20_teams", "targets_avoids", "chip_strategy", "master_plan", "outro"
    ],
    "short": [
        "intro", "market_trends", "injury_report",
        "master_plan", "outro"
    ],
    "injury_only": [
        "intro", "injury_report", "outro"
    ],
    "matchday": [
        "intro", "injury_report", "targets_avoids",
        "master_plan", "outro"
    ],
}

# ═══════════════════════════════════════════════════════════════════════
# DECISION TABLES — AUTO MODE
# ═══════════════════════════════════════════════════════════════════════
TIER_PHASE_MATRIX = {
    ("RAPID", "EXECUTION"):    ["intro", "injury_report", "targets_avoids", "master_plan", "outro"],
    ("RAPID", "FINAL_CHECKS"): ["intro", "injury_report", "targets_avoids", "master_plan", "outro"],
    ("RAPID", "ADJUSTMENT"):   ["intro", "injury_report", "market_trends", "targets_avoids", "master_plan", "outro"],
    ("RAPID", "DISCOVERY"):    ["intro", "injury_report", "market_trends", "sword_shield", "targets_avoids", "outro"],

    ("PRIORITY", "EXECUTION"):       ["intro", "injury_report", "targets_avoids", "master_plan", "outro"],
    ("PRIORITY", "FINAL_CHECKS"):    ["intro", "injury_report", "targets_avoids", "master_plan", "outro"],
    ("PRIORITY", "ADJUSTMENT"):      ["intro", "injury_report", "market_trends", "fdr_map", "targets_avoids", "outro"],
    ("PRIORITY", "DISCOVERY"):       ["intro", "injury_report", "market_trends", "sword_shield", "fdr_map", "outro"],

    ("MAIN_SHOW", "EXECUTION"):     ["intro", "injury_report", "targets_avoids", "master_plan", "outro"],
    ("MAIN_SHOW", "FINAL_CHECKS"):  ["intro", "injury_report", "all_20_teams", "targets_avoids", "master_plan", "outro"],
    ("MAIN_SHOW", "ADJUSTMENT"):    ["intro", "market_trends", "sword_shield", "injury_report", "all_20_teams", "targets_avoids", "outro"],
    ("MAIN_SHOW", "DISCOVERY"):     ["intro", "gw_review", "market_trends", "sword_shield", "fdr_map", "projected_goals", "cs_odds", "injury_report", "all_20_teams", "chip_strategy", "outro"],

    ("Detail analysis", "EXECUTION"):    ["intro", "targets_avoids", "master_plan", "outro"],
    ("Detail analysis", "FINAL_CHECKS"): ["intro", "targets_avoids", "master_plan", "outro"],
    ("Detail analysis", "ADJUSTMENT"):   ["intro", "market_trends", "fdr_map", "cs_odds", "targets_avoids", "outro"],
    ("Detail analysis", "DISCOVERY"):    ["intro", "gw_review", "market_trends", "sword_shield", "fdr_map", "projected_goals", "cs_odds", "injury_report", "all_20_teams", "chip_strategy", "master_plan", "outro"],
}

# ═══════════════════════════════════════════════════════════════════════
# DGW / BGW RULE OVERRIDES
# ═══════════════════════════════════════════════════════════════════════
def _apply_dgw_bgw_rules(module_list):
    """Force-inject chip and projected goals for DGW/BGW contexts."""
    if globals().get('IS_DGW', False):
        if "chip_strategy" not in module_list and "outro" in module_list:
            idx = module_list.index("outro")
            module_list.insert(idx, "chip_strategy")
        if "projected_goals" not in module_list and "fdr_map" in module_list:
            idx = module_list.index("fdr_map") + 1
            module_list.insert(idx, "projected_goals")
    if globals().get('IS_BGW', False):
        if "chip_strategy" not in module_list and "outro" in module_list:
            idx = module_list.index("outro")
            module_list.insert(idx, "chip_strategy")
        if "projected_goals" in module_list:
            module_list.remove("projected_goals")
    return module_list

def _apply_deadline_compression(module_list):
    """Strip long-form modules when deep inside execution window."""
    hours = globals().get('HOURS_TO_DEADLINE', 99)
    if hours <= 6:
        for drop in ["gw_review", "fdr_map", "projected_goals", "cs_odds",
                     "all_20_teams", "chip_strategy"]:
            if drop in module_list:
                module_list.remove(drop)
    elif hours <= 12:
        for drop in ["gw_review", "fdr_map", "projected_goals", "cs_odds"]:
            if drop in module_list:
                module_list.remove(drop)
    return module_list

def _verify_asset_availability(module_list):
    """Asset verification deferred to Cell 0c — PNGs and audio do not exist yet.
    Cell 0c runs after Cells 4–16 and performs the full two-way check."""
    return module_list

# ═══════════════════════════════════════════════════════════════════════
# RESOLVE TIER AND PHASE — RESPECT MANUAL OVERRIDES
# ═══════════════════════════════════════════════════════════════════════
broadcast_tier = FORCE_TIER or globals().get('BROADCAST_TIER', 'STANDARD')
deadline_phase = FORCE_PHASE or globals().get('DEADLINE_PHASE', 'DISCOVERY')
next_gw        = globals().get('NEXT_GW', 'TBC')

# ═══════════════════════════════════════════════════════════════════════
# BUILD FINAL MODULE LIST
# ═══════════════════════════════════════════════════════════════════════
if VIDEO_FORMAT == "custom":
    raw_modules = [m for m in CUSTOM_MODULES if m in MODULE_REGISTRY]
    invalid = [m for m in CUSTOM_MODULES if m not in MODULE_REGISTRY]
    if invalid:
        print(f"⚠️  Unknown modules removed: {invalid}")
    print(f"✅ Custom module list loaded — {len(raw_modules)} modules")

elif VIDEO_FORMAT in FORMAT_PRESETS:
    raw_modules = list(FORMAT_PRESETS[VIDEO_FORMAT])
    raw_modules = _apply_dgw_bgw_rules(raw_modules)
    raw_modules = _apply_deadline_compression(raw_modules)
    raw_modules = _verify_asset_availability(raw_modules)
    print(f"✅ Format preset '{VIDEO_FORMAT}' loaded — {len(raw_modules)} modules")

else:
    # AUTO MODE — full matrix decision
    tier_key     = (broadcast_tier, deadline_phase)
    fallback_key = ("MAIN_SHOW", "DISCOVERY")
    raw_modules  = list(TIER_PHASE_MATRIX.get(tier_key, TIER_PHASE_MATRIX[fallback_key]))
    raw_modules  = _apply_dgw_bgw_rules(raw_modules)
    raw_modules  = _apply_deadline_compression(raw_modules)
    raw_modules  = _verify_asset_availability(raw_modules)
    print(f"✅ Auto mode — tier: {broadcast_tier} / phase: {deadline_phase}")

final_modules = raw_modules

# ═══════════════════════════════════════════════════════════════════════
# BUILD DYNAMIC MANIFEST
# ═══════════════════════════════════════════════════════════════════════
DYNAMIC_MANIFEST = []
for i, mod_name in enumerate(final_modules):
    cfg = MODULE_REGISTRY[mod_name]
    DYNAMIC_MANIFEST.append((
        i + 1,
        cfg["png_file"],
        cfg["slide_key"],
        cfg["slide_type"],
    ))

# ═══════════════════════════════════════════════════════════════════════
# ESTIMATED RUNTIME
# ═══════════════════════════════════════════════════════════════════════
est_min_secs = sum(MODULE_REGISTRY[m]["min_dur"] for m in final_modules)
est_max_secs = sum(MODULE_REGISTRY[m]["max_dur"] for m in final_modules)
est_min_min  = est_min_secs / 60
est_max_min  = est_max_secs / 60

# ═══════════════════════════════════════════════════════════════════════
# VIDEO TYPE LABEL
# ═══════════════════════════════════════════════════════════════════════
if VIDEO_FORMAT == "custom":
    VIDEO_TYPE = f"🎛️ CUSTOM BUILD — GW{next_gw}"
elif VIDEO_FORMAT in FORMAT_PRESETS:
    labels = {
        "full":        f"🔭 GW{next_gw} FULL PREVIEW",
        "short":       f"⚡ GW{next_gw} QUICK UPDATE",
        "injury_only": f"🚨 GW{next_gw} INJURY REPORT",
        "matchday":    f"🎯 GW{next_gw} DEADLINE DAY",
    }
    VIDEO_TYPE = labels.get(VIDEO_FORMAT, f"GW{next_gw} VIDEO")
else:
    if broadcast_tier == "RAPID":
        VIDEO_TYPE = f"🚨 EMERGENCY SHORT — GW{next_gw} BREAKING UPDATE"
    elif deadline_phase == "EXECUTION":
        VIDEO_TYPE = f"⚡ DEADLINE DAY — GW{next_gw} FINAL CALLS"
    elif deadline_phase == "FINAL_CHECKS":
        VIDEO_TYPE = f"🎯 GW{next_gw} FINAL ANALYSIS"
    elif deadline_phase == "ADJUSTMENT":
        VIDEO_TYPE = f"📊 GW{next_gw} MIDWEEK BREAKDOWN"
    else:
        VIDEO_TYPE = f"🔭 GW{next_gw} PREVIEW & MASTER PLAN"

# ═══════════════════════════════════════════════════════════════════════
# LOCK ALL GLOBALS
# ═══════════════════════════════════════════════════════════════════════
globals()["DYNAMIC_MANIFEST"]  = DYNAMIC_MANIFEST
globals()["final_modules"]     = final_modules
globals()["VIDEO_TYPE"]        = VIDEO_TYPE
globals()["est_min_min"]       = est_min_min
globals()["est_max_min"]       = est_max_min

print(f"\n{'='*60}")
print(f" 🟣 MODULE PLANNER LOCKED")
print(f"   Format     : {VIDEO_FORMAT.upper()}")
print(f"   Video type : {VIDEO_TYPE}")
print(f"   Tier/Phase : {broadcast_tier} / {deadline_phase}")
print(f"   Est. length: {est_min_min:.0f}–{est_max_min:.0f} minutes")
print(f"   Modules    : {len(final_modules)}")
for i, m in enumerate(final_modules):
    print(f"     {i+1:>2}. {m}")
print(f"{'='*60}\n")

# Save blueprint
blueprint_path = os.path.join(AUDIO_DIR, "video_blueprint.json")
try:
    with open(blueprint_path, "w") as f:
        json.dump({
            "video_format":      VIDEO_FORMAT,
            "video_type":        VIDEO_TYPE,
            "broadcast_tier":    broadcast_tier,
            "deadline_phase":    deadline_phase,
            "hours_to_deadline": globals().get('HOURS_TO_DEADLINE', 99),
            "is_dgw":            globals().get('IS_DGW', False),
            "is_bgw":            globals().get('IS_BGW', False),
            "final_modules":     final_modules,
            "estimated_minutes": [est_min_min, est_max_min],
            "manifest":          DYNAMIC_MANIFEST,
        }, f, indent=2, default=str)
    print(f"✅ Blueprint saved → {blueprint_path}")
except Exception as e:
    print(f"⚠️ Could not save blueprint: {e}")

🟣 Newsroom OS v2 — Module Planner Engine initializing...
✅ Format preset 'full' loaded — 12 modules

 🟣 MODULE PLANNER LOCKED
   Format     : FULL
   Video type : 🔭 GW39 FULL PREVIEW
   Tier/Phase : Detail analysis / DISCOVERY
   Est. length: 17–37 minutes
   Modules    : 12
      1. intro
      2. gw_review
      3. market_trends
      4. sword_shield
      5. fdr_map
      6. cs_odds
      7. injury_report
      8. all_20_teams
      9. targets_avoids
     10. chip_strategy
     11. master_plan
     12. outro

✅ Blueprint saved → /content/drive/MyDrive/Vortex_Final/audio/video_blueprint.json


In [ ]:

# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 2: Core Data, Unified Math Models, Typography & Asset Engine
*UPDATED: Dynamic 7-Week League Average xG Baseline*
"""

import os
import io
import re
import math
import unicodedata
import requests
from datetime import datetime, timezone, timedelta
from PIL import Image, ImageDraw, ImageFont, ImageFilter

print("Initializing Core Engine & Global Mathematics...")

# ── 1. GLOBAL CONSTANTS & COLORS ─────────────────────────────────────────
W, H = 7680, 4320
HEADERS = {"User-Agent": "Mozilla/5.0"}
EST_TZ = timezone(timedelta(hours=-5))
NOW_STR = datetime.now(EST_TZ).strftime("%Y-%m-%d %I:%M %p EST")

C_BG_TOP = "#0B001A"; C_BG_BOT = "#030005"
C_GOLD   = "#FFD700"; C_WHITE  = "#FFFFFF"
C_BLACK  = "#000000"; C_GREEN  = "#00FF87"
C_L_GREEN= "#76FF03"; C_CYAN   = "#00E5FF"
C_RED    = "#FF1744"; PL_PURPLE= "#38003C"
C_ROW_A  = "#0D0520"; C_ROW_B  = "#150830"

TEAM_COLORS = {
    "ARS":"#EF0107","AVL":"#670E36","BOU":"#DA291C","BRE":"#C8102E",
    "BHA":"#0057B8","CHE":"#034694","CRY":"#1B458F","EVE":"#003399",
    "FUL":"#242424","IPS":"#003090","LEI":"#003090","LIV":"#C8102E",
    "MCI":"#6CABDD","MUN":"#DA291C","NEW":"#242424","NFO":"#DD0000",
    "SOU":"#D71920","TOT":"#132257","WHU":"#7A263A","WOL":"#D99000",
}
# ── UNIFIED TIER SYSTEM (shared by all slide cells) ──────────────────────
TIER_COLORS = {
    "T1": ("#00E676", "#FFFFFF", "#000000"),
    "T2": ("#2979FF", "#FFFFFF", "#000000"),
    "T3": ("#FFEA00", "#FFFFFF", "#000000"),
    "T4": ("#9E9E9E", "#FFFFFF", "#FFFFFF"),
    "T5": ("#F50057", "#FFFFFF", "#FFFFFF"),
}

def make_gradient_fill(width, height, hex_left, hex_right):
    grad = Image.new("RGBA", (width, height), 0)
    pix  = grad.load()
    r1, g1, b1 = int(hex_left[1:3],16),  int(hex_left[3:5],16),  int(hex_left[5:7],16)
    r2, g2, b2 = int(hex_right[1:3],16), int(hex_right[3:5],16), int(hex_right[5:7],16)
    for x in range(width):
        t = x / max(1, width - 1)
        r = int(r1 + (r2 - r1) * t)
        g = int(g1 + (g2 - g1) * t)
        b = int(b1 + (b2 - b1) * t)
        for y in range(height):
            pix[x, y] = (r, g, b, 255)
    return grad

def paint_tier_cell(canvas_img, x, y, w, h, tier_key,
                    value_text=None, value_font=None,
                    sub_text=None, sub_font=None):
    bg_l, bg_r, txt_color = TIER_COLORS[tier_key]
    grad = make_gradient_fill(w, h, bg_l, bg_r)
    mask = Image.new("L", (w, h), 0)
    ImageDraw.Draw(mask).rounded_rectangle([0, 0, w, h], radius=15, fill=255)
    canvas_img.paste(grad, (x, y), mask)
    d = ImageDraw.Draw(canvas_img)
    if sub_text and sub_font:
        sb = d.textbbox((0, 0), sub_text, font=sub_font)
        sx = x + (w - (sb[2] - sb[0])) // 2
        d.text((sx, y + 5), sub_text, font=sub_font, fill="#000000")
    if value_text and value_font:
        vb = d.textbbox((0, 0), value_text, font=value_font)
        vx = x + (w - (vb[2] - vb[0])) // 2
        vy = y + (h - (vb[3] - vb[1])) // 2 + (15 if sub_text else -5)
        d.text((vx, vy), value_text, font=value_font, fill="#000000")

def paint_special_cell(canvas_img, x, y, w, h, label):
    d = ImageDraw.Draw(canvas_img)
    d.rounded_rectangle([x, y, x + w, y + h], radius=15, fill="#000000")
    color = "#FFFFFF" if label == "DGW" else "#FF1744"
    font  = fnt("Bold", 80)
    bbox  = d.textbbox((0, 0), label, font=font)
    tx = x + (w - (bbox[2] - bbox[0])) // 2
    ty = y + (h - (bbox[3] - bbox[1])) // 2 - 5
    d.text((tx, ty), label, font=font, fill=color)

# ── 2. TYPOGRAPHY ENGINE ─────────────────────────────────────────────────
_FONT_CACHE = {}
try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"
ELEMENTS_DIR = os.path.join(GD_ROOT, "elements")

def fnt(style: str, size: int):
    key = (style, size)
    if key not in _FONT_CACHE:
        try:
            fp = os.path.join(ELEMENTS_DIR, f"Roboto-{style}.ttf")
            _FONT_CACHE[key] = ImageFont.truetype(fp, size)
        except:
            _FONT_CACHE[key] = ImageFont.load_default()
    return _FONT_CACHE[key]

# ── 3. LIVE API DATA ACQUISITION ─────────────────────────────────────────

print("Fetching Live Official FPL Data...")
try:
    # 🔴 FIX: Prevent double-fetching if Newsroom OS (Cell 0) already pulled data
    if 'boot_data' not in globals():
        boot_data = requests.get("https://fantasy.premierleague.com/api/bootstrap-static/", timeout=15).json()
    if 'fix_data' not in globals():
        fix_data = requests.get("https://fantasy.premierleague.com/api/fixtures/", timeout=15).json()

    events = boot_data["events"]
    CURR_GW = next((e["id"] for e in events if e.get("is_current")), 1)
    NEXT_GW = next((e["id"] for e in events if e.get("is_next")), CURR_GW + 1)

    print(f"✅ Data Locked. Current GW: {CURR_GW} | Target GW: {NEXT_GW}")
except Exception as e:
    print(f"❌ API Error: {e}")

# ── UNIFIED TIER HELPERS (Shared for Slides 5, 6, 7) ─────────────────────
def get_tier_xg(xg):
    if xg >= 1.69: return "T1"
    if xg >= 1.50: return "T2"
    if xg >= 1.30: return "T3"
    if xg >= 1.25: return "T4"
    return "T5"

def get_tier_cs(cs_pct):
    if cs_pct >= 35: return "T1"
    if cs_pct >= 30: return "T2"
    if cs_pct >= 25: return "T3"
    if cs_pct >= 20: return "T4"
    return "T5"
# ── 4. ELITE MATHEMATICAL MODELS (POISSON & SIGMOID FDR) ─────────────────
AVG_H = 1.55; AVG_A = 1.25; API_BASE = 1100.0

def poisson_xg(att_h, def_a, att_a, def_h, home=True):
    if home: return AVG_H * (att_h/API_BASE) / (def_a/API_BASE)
    else:    return AVG_A * (att_a/API_BASE) / (def_h/API_BASE)

def cs_pct(xg_opp):
    return math.exp(-xg_opp) * 100

# 🔴 NEW: Calculate the LIVE Dynamic League Average xG over the
# remaining gameweeks (capped at season end — never produces
# phantom GW39/GW40 columns).
MAX_GW = max((e["id"] for e in boot_data.get("events", [])), default=NEXT_GW)

def get_remaining_gws(start_gw, window=7):
    """Return up to `window` gameweeks from start_gw, clamped to
    the final gameweek of the current PL season. Future-proof:
    works at any point in the season, including final week."""
    end = min(start_gw + window, MAX_GW + 1)
    return list(range(start_gw, end))

valid_gws  = {e["id"] for e in boot_data.get("events", [])}
target_gws = get_remaining_gws(NEXT_GW, window=7)

# <<< START REPLACEMENT (Cell 2 - Global T Dict) <<<

GLOBAL_T = {t["id"]: {
        "name":  t["name"],  "short": t["short_name"], "code": t["code"],
        "att_h": t["strength_attack_home"],  "att_a": t["strength_attack_away"],
        "def_h": t["strength_defence_home"], "def_a": t["strength_defence_away"]
    } for t in boot_data.get("teams", [])}

T_stats = GLOBAL_T  # Alias for backward compatibility in this cell


_all_xg = []
for fx in fix_data:
    if fx.get("event") in target_gws and fx["team_h"] in T_stats and fx["team_a"] in T_stats:
        hi, ai = fx["team_h"], fx["team_a"]
        _all_xg.append(poisson_xg(T_stats[hi]["att_h"], T_stats[ai]["def_a"], T_stats[ai]["att_a"], T_stats[hi]["def_h"], True))
        _all_xg.append(poisson_xg(T_stats[hi]["att_h"], T_stats[ai]["def_a"], T_stats[ai]["att_a"], T_stats[hi]["def_h"], False))

DYNAMIC_LEAGUE_AVG_XG = sum(_all_xg) / len(_all_xg) if _all_xg else 1.35
print(f"📈 Global Dynamic League Avg xG Locked: {DYNAMIC_LEAGUE_AVG_XG:.3f}")

# 🔴 UPDATED: Uses the dynamic average instead of a hardcoded 1.35
def compute_fdr(xg_for, league_avg_xg=DYNAMIC_LEAGUE_AVG_XG):
    """Sigmoid-normalized FDR: 1.0 (easiest) → 5.0 (hardest)."""
    ratio   = xg_for / max(league_avg_xg, 0.01)
    log_r   = math.log(max(ratio, 0.01))
    sigmoid = 1.0 / (1.0 + math.exp(-2.5 * log_r))
    fdr     = 5.0 - 4.0 * sigmoid
    return round(max(1.0, min(5.0, fdr)), 1)

# GLOBAL UNIFIER: Custom Vortex FDR Generator
def get_vortex_fdr(team_id, opp_id, is_home):
    """Calculates exact decimal FDR on the fly using live Poisson math."""
    att_team = next(t for t in boot_data['teams'] if t['id'] == team_id)
    def_team = next(t for t in boot_data['teams'] if t['id'] == opp_id)

    if is_home:
        xg_for = poisson_xg(att_team["strength_attack_home"], def_team["strength_defence_away"],
                            def_team["strength_attack_away"], att_team["strength_defence_home"], True)
    else:
        xg_for = poisson_xg(def_team["strength_attack_home"], att_team["strength_defence_away"],
                            att_team["strength_attack_away"], def_team["strength_defence_home"], False)

    return compute_fdr(xg_for)

# ── 5. UNIVERSAL ASSET FETCHER (WITH OVERRIDES & GLOBAL CACHE) ───────────
PHOTOS_DIR = os.path.join(GD_ROOT, "players")
_IMAGE_CACHE = {}  # 🔴 FIX: Global in-memory cache to prevent 50+ redundant HTTP calls

def get_player_photo(opta_code, web_name, full_name="", size=(280,340), team_code=None):
    cache_key = f"photo_{opta_code}_{size[0]}x{size[1]}"
    if cache_key in _IMAGE_CACHE:
        return _IMAGE_CACHE[cache_key].copy()

    fb = Image.new("RGBA", size, (0,0,0,0))

    # --- 1. LOCAL CACHE (STRICT ID MATCH ONLY, SUPPORTS PNG/JPG) ---
    for ext in ['.png', '.jpg', '.jpeg']:
        local_path = os.path.join(PHOTOS_DIR, f"{opta_code}{ext}")
        if os.path.exists(local_path):
            try:
                img = Image.open(local_path).convert("RGBA")
                fb.paste(img.resize(size, Image.LANCZOS), (0,0), img.resize(size, Image.LANCZOS))
                _IMAGE_CACHE[cache_key] = fb
                return fb.copy()
            except Exception as e:
                print(f"⚠️ Local cache error for {opta_code}{ext}: {e}")

    # --- 2. OFFICIAL API ---
    try:
        url = f"https://resources.premierleague.com/premierleague/photos/players/250x250/p{opta_code}.png"
        r = requests.get(url, headers=HEADERS, timeout=6)
        if r.status_code == 200:
            if len(r.content) > 3000:
                img = Image.open(io.BytesIO(r.content)).convert("RGBA")
                fb.paste(img.resize(size, Image.LANCZOS), (0,0), img.resize(size, Image.LANCZOS))
                _IMAGE_CACHE[cache_key] = fb
                return fb.copy()
            else:
                print(f"⚠️ API placeholder detected for {opta_code}")
        else:
            print(f"⚠️ API 404 error for {opta_code}: HTTP {r.status_code}")
    except Exception as e:
        print(f"❌ API Request failed for {opta_code}: {e}")

    # --- 3. TEAM BADGE FALLBACK ---
    fallback_team_code = team_code
    if not fallback_team_code and 'boot_data' in globals():
        try:
            p_data = next((p for p in boot_data.get('elements', []) if p.get('code') == opta_code), None)
            if p_data:
                team_id = p_data.get('team')
                team_data = next((t for t in boot_data.get('teams', []) if t['id'] == team_id), None)
                if team_data:
                    fallback_team_code = team_data['code']
        except: pass

    if fallback_team_code:
        try:
            logo = get_team_logo(fallback_team_code, size=(int(size[0]*0.7), int(size[0]*0.7)))
            if logo:
                lx = (size[0] - logo.width) // 2
                ly = (size[1] - logo.height) // 2
                fb.paste(logo, (lx, ly), logo)
                _IMAGE_CACHE[cache_key] = fb
                return fb.copy()
        except Exception as e:
            print(f"⚠️ Team badge fallback failed for {opta_code}: {e}")

    # --- 4. LAST RESORT SILHOUETTE ---
    print(f"⚠️ Using generic silhouette fallback for {opta_code} ({web_name})")
    d = ImageDraw.Draw(fb)
    cx, cy = size[0]//2, size[1]//2+int(size[1]*0.1)
    sw, sh = int(size[0]*0.375), int(size[1]*0.38)
    pts = [(cx-sw*.4,cy-sh*.4),(cx-sw*.2,cy-sh*.5),(cx+sw*.2,cy-sh*.5),
           (cx+sw*.4,cy-sh*.4),(cx+sw*.6,cy-sh*.1),(cx+sw*.4,cy+sh*.1),
           (cx+sw*.35,cy+sh*.5),(cx-sw*.35,cy+sh*.5),
           (cx-sw*.4,cy+sh*.1),(cx-sw*.6,cy-sh*.1)]
    d.polygon(pts, fill="#555555", outline="#000000", width=6)
    _IMAGE_CACHE[cache_key] = fb
    return fb.copy()

def get_team_logo(code, size=(120,120)):
    cache_key = f"logo_{code}_{size[0]}x{size[1]}"
    if cache_key in _IMAGE_CACHE:
        return _IMAGE_CACHE[cache_key].copy()

    url = f"https://resources.premierleague.com/premierleague/badges/70/t{code}.png"
    fb  = Image.new("RGBA", size, (0,0,0,0))
    try:
        r = requests.get(url, headers=HEADERS, timeout=5)
        if r.status_code == 200:
            img = Image.open(io.BytesIO(r.content)).convert("RGBA")
            fb.paste(img.resize(size,Image.LANCZOS),(0,0), img.resize(size,Image.LANCZOS))
            _IMAGE_CACHE[cache_key] = fb
    except: pass

def get_team_logo(code, size=(120,120)):
    url = f"https://resources.premierleague.com/premierleague/badges/70/t{code}.png"
    fb  = Image.new("RGBA", size, (0,0,0,0))
    try:
        r = requests.get(url, headers=HEADERS, timeout=5)
        if r.status_code == 200:
            img = Image.open(io.BytesIO(r.content)).convert("RGBA")
            fb.paste(img.resize(size,Image.LANCZOS),(0,0), img.resize(size,Image.LANCZOS))
    except: pass
    return fb

def get_team_logo(code, size=(120,120)):
    url = f"https://resources.premierleague.com/premierleague/badges/70/t{code}.png"
    fb  = Image.new("RGBA", size, (0,0,0,0))
    try:
        r = requests.get(url, headers=HEADERS, timeout=5)
        if r.status_code == 200:
            img = Image.open(io.BytesIO(r.content)).convert("RGBA")
            fb.paste(img.resize(size,Image.LANCZOS),(0,0), img.resize(size,Image.LANCZOS))
    except: pass
    return fb

def normalize_player_name(web_name):
    name = ''.join(c for c in unicodedata.normalize('NFD', web_name)
                   if unicodedata.category(c) != 'Mn')
    if "." in name:
        name = name.split(".")[-1].strip()
    return name

print("\n" + "=" * 60)
print(" 🟣 CELL 2 COMPLETE: CORE ENGINE & UNIFIED MATH ONLINE ")
print("=" * 60)

Initializing Core Engine & Global Mathematics...
Fetching Live Official FPL Data...
✅ Data Locked. Current GW: 38 | Target GW: 39
📈 Global Dynamic League Avg xG Locked: 1.350

 🟣 CELL 2 COMPLETE: CORE ENGINE & UNIFIED MATH ONLINE 


In [ ]:

# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 3: Complete Visual Engine (Float Bug Fixed + Table Fill)
"""

import os
from PIL import Image, ImageDraw, ImageEnhance

print("Initializing Premium Visual Engine...")

# ── 1. MASTER CANVAS GENERATOR ───────────────────────────────────────────
def create_base_canvas(slide_title, footer_text="Data Source: Official FPL API | Vortex xPTS Model"):
    bg_path = os.path.join(GD_ROOT, "elements", "7.png")

    if os.path.exists(bg_path):
        canvas = Image.open(bg_path).convert("RGBA")
        if canvas.size != (W, H):
            canvas = canvas.resize((W, H), Image.LANCZOS)
    else:
        canvas = Image.new("RGBA", (W, H), C_BG_TOP)

    draw = ImageDraw.Draw(canvas)
    zone_x, zone_y = 7500, 800
    zone_w, zone_h = 3500, 3400

    # Title
    draw.text((zone_x, 300), slide_title, font=fnt("Bold", 180), fill=C_GOLD)

    # Footer
    if footer_text:
        draw.text((zone_x, H - 350), footer_text, font=fnt("Regular", 80), fill=C_GREEN)

    return canvas, draw, (zone_x, zone_y, zone_w, zone_h)

# ── 2. FDR STRIP GENERATOR ───────────────────────────────────────────────
def get_fdr_color(fdr_score):
    if fdr_score <= 2.0: return "#00FF87"
    if fdr_score <= 2.5: return "#76FF03"
    if fdr_score <= 3.5: return "#FFD700"
    if fdr_score <= 4.5: return "#FF1744"
    return "#8B0000"

def draw_fdr_strip(draw, start_x, start_y, width, height, upcoming_fixtures):
    box_w = width // 7
    for i in range(7):
        box_x = start_x + (i * box_w)
        if i < len(upcoming_fixtures):
            fix = upcoming_fixtures[i]
            color = get_fdr_color(fix['fdr'])
            label = f"{fix['opponent_short']} {'(H)' if fix['is_home'] else '(A)'}"
            fdr_num = str(fix['fdr'])
        else:
            color = "#333333"
            label, fdr_num = "BLANK", "-"

        draw.rectangle([box_x, start_y, box_x + box_w - 10, start_y + height], fill=color)
        f_t = fnt("Bold", 45)
        bt = draw.textbbox((0, 0), label, font=f_t)
        draw.text((box_x + (box_w - 10 - (bt[2]-bt[0]))//2, start_y + 30), label, font=f_t, fill=C_BLACK)

        f_n = fnt("Bold", 65)
        bn = draw.textbbox((0, 0), fdr_num, font=f_n)
        draw.text((box_x + (box_w - 10 - (bn[2]-bn[0]))//2, start_y + 100), fdr_num, font=f_n, fill=C_BLACK)


print("✅ Cell 3 (Bug Fixed + Table Filled) Ready.")

Initializing Premium Visual Engine...
✅ Cell 3 (Bug Fixed + Table Filled) Ready.


In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 4: Slide 2 — Market Trends (Empty Base for Animated Reveal)
Cards animate in via Cell 17 — this cell only generates the structural shell.
"""

import os
from PIL import Image, ImageDraw

print("Initializing Slide 2: Market Trends Empty Base...")

# ── 1. PATHS ─────────────────────────────────────────────────────────────
try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

PNG_DIR      = os.path.join(GD_ROOT, "png")
ELEMENTS_DIR = os.path.join(GD_ROOT, "elements")
os.makedirs(PNG_DIR, exist_ok=True)

# ── 2. TEAM NAME EXPANDER ────────────────────────────────────────────────
TEAM_MAP = {
    "Arsenal": "Arsenal", "Aston Villa": "Aston Villa", "Bournemouth": "Bournemouth",
    "Brentford": "Brentford", "Brighton": "Brighton & Hove Albion", "Chelsea": "Chelsea",
    "Crystal Palace": "Crystal Palace", "Everton": "Everton", "Fulham": "Fulham",
    "Ipswich": "Ipswich Town", "Leicester": "Leicester City", "Liverpool": "Liverpool",
    "Man City": "Manchester City", "Man Utd": "Manchester United",
    "Newcastle": "Newcastle United", "Nott'm Forest": "Nottingham Forest",
    "Southampton": "Southampton", "Spurs": "Tottenham Hotspur",
    "West Ham": "West Ham United", "Wolves": "Wolverhampton Wanderers",
    "Sunderland": "Sunderland", "Burnley": "Burnley",
}

# ── 3. LIVE DATA: TOP 5 IN / OUT (locked for Cell 16 narration + Cell 17 animation) ──
elements = boot_data['elements']
teams    = boot_data['teams']

top_in  = sorted(elements, key=lambda x: x['transfers_in_event'],  reverse=True)[:5]
top_out = sorted(elements, key=lambda x: x['transfers_out_event'], reverse=True)[:5]

print(f"✅ Identified {len(top_in)} Targets (IN) and {len(top_out)} Risks (OUT).")

def get_player_team_data(player):
    team_data = next(t for t in teams if t['id'] == player['team']).copy()
    team_data['name'] = TEAM_MAP.get(team_data['name'], team_data['name'])
    return team_data

def lock_player_data(player_list, market_type):
    for p in player_list:
        p['market_type'] = market_type
        team_data = get_player_team_data(p)

        gw_fixes = [f for f in fix_data
                    if (f['team_h'] == team_data['id'] or f['team_a'] == team_data['id'])
                    and f['event'] == NEXT_GW]

        if not gw_fixes:
            p['next_opp']   = "BLANK"
            p['next_fdr']   = 5.0
            p['fdr_color']  = "#333333"
        else:
            opps, fdrs = [], []
            for fix in gw_fixes:
                is_home  = (fix['team_h'] == team_data['id'])
                opp_id   = fix['team_a'] if is_home else fix['team_h']
                opp_team = next(t for t in boot_data['teams'] if t['id'] == opp_id)
                opps.append(f"{opp_team['short_name']} [{'H' if is_home else 'A'}]")
                fdrs.append(get_vortex_fdr(team_data['id'], opp_id, is_home))

            p['next_opp']  = " + ".join(opps)
            p['next_fdr']  = sum(fdrs) / len(fdrs)
            p['fdr_color'] = ("#00FF87" if p['next_fdr'] <= 2.5
                              else "#FFD700" if p['next_fdr'] <= 3.5
                              else "#FF1744")

        transfers          = p['transfers_in_event'] if market_type == "in" else p['transfers_out_event']
        p['transfer_volume'] = transfers
        p['transfer_str']    = f"{'+' if market_type == 'in' else '-'}{int(transfers / 1000)}K"

lock_player_data(top_in,  "in")
lock_player_data(top_out, "out")

# ── 4. BUILD EMPTY STRUCTURAL BASE ───────────────────────────────────────

BG_PATH = os.path.join(ELEMENTS_DIR, "24.png")

if os.path.exists(BG_PATH):
    canvas = Image.open(BG_PATH).convert("RGBA").resize((W, H), Image.LANCZOS)
else:
    print("⚠️ 3.png not found. Falling back to solid canvas.")
    canvas = Image.new("RGBA", (W, H), (10, 5, 25, 255))

draw = ImageDraw.Draw(canvas)

# Column geometry — Matches Slide 3 Standard Layout
col_w  = 2350
col1_x = ((7680 - ((col_w * 2) + 100)) // 2) + 1150
col1_y = 700       # INCREASED from 450 to push both blocks down
col2_x = col1_x + col_w + 100
box_height = 3300  # DECREASED from 3450 to prevent bottom overflow

# Title
title_text = "MARKET TRENDS: IN vs OUT"
title_font = fnt("Bold", 200)
draw.text((col1_x + 15, 165), title_text, font=title_font, fill="#0B001A")
draw.text((col1_x,      150), title_text, font=title_font, fill="#FFFFFF")

# Footer
footer_text = "Data Source: Official FPL API | Vortex xPTS Model"
footer_font = fnt("Regular", 80)
fb          = draw.textbbox((0, 0), footer_text, font=footer_font)
draw.text((W - (fb[2] - fb[0]) - 300, H - 200), footer_text,
          font=footer_font, fill="#00FF87")

# Empty column boxes
# (Boxes removed entirely so animated cards float over the clean cinematic background)


# ── 5. SAVE ──────────────────────────────────────────────────────────────
empty_path = os.path.join(PNG_DIR, "Slide_2_Empty.png")
if os.path.exists(empty_path): os.remove(empty_path)
canvas.convert("RGB").save(empty_path, format="PNG")
print(f"✅ Slide 2 Empty Base Saved: {empty_path}")

print("\n" + "=" * 60)
print(" 🟣 CELL 4 COMPLETE: SLIDE 2 EMPTY BASE READY ")
print("=" * 60)

Initializing Slide 2: Market Trends Empty Base...
✅ Identified 5 Targets (IN) and 5 Risks (OUT).
⚠️ 3.png not found. Falling back to solid canvas.
✅ Slide 2 Empty Base Saved: /content/drive/MyDrive/Vortex_Final/png/Slide_2_Empty.png

 🟣 CELL 4 COMPLETE: SLIDE 2 EMPTY BASE READY 


In [ ]:

# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 5: Slide 3 — Sword & Shield Strategy (Empty Base for Animated Reveal)
Cards animate in via Cell 17 — this cell only generates the structural shell
with column boxes and Sword/Shield icons next to their headers.
"""

import os
from datetime import datetime, timedelta
from PIL import Image, ImageDraw

print("Initializing Slide 3: Sword & Shield Empty Base...")

# ── 1. PATHS ─────────────────────────────────────────────────────────────
try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

PNG_DIR      = os.path.join(GD_ROOT, "png")
ELEMENTS_DIR = os.path.join(GD_ROOT, "elements")
os.makedirs(PNG_DIR, exist_ok=True)

# ── 2. GAMEWEEK + DEADLINE METADATA ──────────────────────────────────────
next_event = next((e for e in boot_data['events'] if e.get('is_next')), None)
if next_event:
    target_gw = next_event['id']
    gw_name   = next_event['name'].upper().replace("GAMEWEEK ", "GW")
    try:
        utc_dt       = datetime.strptime(next_event['deadline_time'], "%Y-%m-%dT%H:%M:%SZ")
        est_dt       = utc_dt - timedelta(hours=4)
        deadline_str = est_dt.strftime("%Y-%m-%d %H:%M EST")
    except Exception:
        deadline_str = next_event['deadline_time']
else:
    target_gw, gw_name, deadline_str = 1, "GW??", "UNKNOWN"

# ── 3. LIVE DATA: SHIELDS + SWORDS (locked for Cell 16 + Cell 17) ────────
elements = boot_data['elements']

teams_with_fixtures = set()
team_fdr_lookup     = {}
team_dgw_lookup     = {}

for f in fix_data:
    if f.get('event') != target_gw:
        continue
    for tid, opp_id, is_home in [
        (f['team_h'], f['team_a'], True),
        (f['team_a'], f['team_h'], False),
    ]:
        teams_with_fixtures.add(tid)
        try:
            fdr = get_vortex_fdr(tid, opp_id, is_home)
        except Exception:
            fdr = f['team_h_difficulty'] if is_home else f['team_a_difficulty']
        team_fdr_lookup.setdefault(tid, []).append(fdr)
        if len(team_fdr_lookup[tid]) > 1:
            team_dgw_lookup[tid] = True

def _team_avg_fdr(team_id):
    fdrs = team_fdr_lookup.get(team_id, [])
    return sum(fdrs) / len(fdrs) if fdrs else 5.0

def _has_dgw(team_id):
    return team_dgw_lookup.get(team_id, False)

def _is_playable(p):
    if p.get('team') not in teams_with_fixtures: return False
    if p.get('status') not in ['a', 'd']:        return False
    if float(p.get('ep_next') or 0) < 1.0:       return False
    return True

# Shields — high ownership + strong outlook
shield_pool = []
for p in elements:
    if not _is_playable(p): continue
    own = float(p.get('selected_by_percent', '0'))
    if own < 20.0: continue

    ep        = float(p.get('ep_next', '0'))
    avg_fdr   = _team_avg_fdr(p['team'])
    dgw_bonus = 1.5 if _has_dgw(p['team']) else 1.0
    fdr_bonus = 2.0 - (avg_fdr / 5.0)

    score = own * ep * dgw_bonus * fdr_bonus
    shield_pool.append((score, p))

shields = [p for _, p in sorted(shield_pool, key=lambda x: x[0], reverse=True)[:6]]

# Swords — low ownership + high upside + good fixture
sword_pool = []
for p in elements:
    if not _is_playable(p): continue
    own = float(p.get('selected_by_percent', '0'))
    if own >= 15.0 or own < 0.5: continue

    ep = float(p.get('ep_next', '0'))
    if ep < 3.0: continue

    avg_fdr   = _team_avg_fdr(p['team'])
    dgw_bonus = 1.5 if _has_dgw(p['team']) else 1.0
    fdr_bonus = 2.0 - (avg_fdr / 5.0)

    score = (1.0 / max(own, 1.0)) * ep * dgw_bonus * fdr_bonus * 100
    sword_pool.append((score, p))

swords = [p for _, p in sorted(sword_pool, key=lambda x: x[0], reverse=True)[:6]]

print(f"✅ Identified {len(shields)} Shields and {len(swords)} Swords.")
print(f"   Shields: {[p['web_name'] for p in shields]}")
print(f"   Swords:  {[p['web_name'] for p in swords]}")

# Lock fixture/FDR/xPts for narration
def lock_strategy_data(player_list, list_type):
    for p in player_list:
        p['strategy_type'] = list_type
        team_id = p['team']

        gw_fixes = [f for f in fix_data
                    if (f['team_h'] == team_id or f['team_a'] == team_id)
                    and f['event'] == target_gw]

        if not gw_fixes:
            p['next_opp']  = "BLANK"
            p['next_fdr']  = 5.0
            p['fdr_color'] = "#333333"
        else:
            opps, fdrs = [], []
            for fix in gw_fixes:
                is_home  = (fix['team_h'] == team_id)
                opp_id   = fix['team_a'] if is_home else fix['team_h']
                opp_team = next(t for t in boot_data['teams'] if t['id'] == opp_id)
                opps.append(f"{opp_team['short_name']} [{'H' if is_home else 'A'}]")
                fdrs.append(get_vortex_fdr(team_id, opp_id, is_home))

            p['next_opp']  = " + ".join(opps)
            p['next_fdr']  = sum(fdrs) / len(fdrs)
            p['fdr_color'] = ("#00FF87" if p['next_fdr'] <= 2.5
                              else "#FFD700" if p['next_fdr'] <= 3.5
                              else "#FF1744")

        xpts = float(p['ep_next'])
        p['locked_xpts'] = xpts

        if list_type == "sword":
            rank_delta       = int(xpts * 12.5)
            p['rank_str']    = f"PROJ CLIMB: +{rank_delta}K"
            p['rank_color']  = "#00FF87"
            p['rank_delta_num'] = rank_delta
        else:
            rank_delta       = int(xpts * 18.2)
            p['rank_str']    = f"NO OWN RISK: -{rank_delta}K"
            p['rank_color']  = "#FF1744"
            p['rank_delta_num'] = rank_delta

lock_strategy_data(shields, "shield")
lock_strategy_data(swords,  "sword")

# ── 4. BUILD EMPTY STRUCTURAL BASE ───────────────────────────────────────
W, H    = 7680, 4320
BG_PATH = os.path.join(ELEMENTS_DIR, "6.png")

if os.path.exists(BG_PATH):
    canvas = Image.open(BG_PATH).convert("RGBA").resize((W, H), Image.LANCZOS)
else:
    print("⚠️ 6.png not found. Falling back to solid canvas.")
    canvas = Image.new("RGBA", (W, H), (10, 5, 25, 255))

draw = ImageDraw.Draw(canvas)

# Column geometry — matches Slide 2 alignment
col_w  = 2150
col1_x = ((7680 - ((col_w * 2) + 100)) // 2) + 1150
col1_y = 450
col2_x = col1_x + col_w + 100
box_h  = 3450

# Title
title_text = "STRATEGY: SWORD & SHIELD"
title_font = fnt("Bold", 200)
draw.text((col1_x + 15, 165), title_text, font=title_font, fill="#0B001A")
draw.text((col1_x,      150), title_text, font=title_font, fill="#FFFFFF")

# Top-right gameweek deadline tag
gw_text   = f"{gw_name} | {deadline_str} | DEADLINE"
gw_font   = fnt("Bold", 90)
gw_bbox   = draw.textbbox((0, 0), gw_text, font=gw_font)
gw_width  = gw_bbox[2] - gw_bbox[0]
draw.text((W - gw_width - 200, 150), gw_text, font=gw_font, fill="#FFFF00")

# Footer
current_year = datetime.now().year
season_str   = f"SEASON {current_year}/{current_year + 1}"
source_str   = "Data Source: Official FPL API | Vortex xPTS Model"
src_bbox     = draw.textbbox((0, 0), source_str, font=fnt("Regular", 80))
sea_bbox     = draw.textbbox((0, 0), season_str, font=fnt("Bold", 80))
draw.text((W - (src_bbox[2] - src_bbox[0]) - 150, H - 320),
          source_str, font=fnt("Regular", 80), fill="#00FF87")
draw.text((W - (sea_bbox[2] - sea_bbox[0]) - 150, H - 200),
          season_str, font=fnt("Bold", 80), fill="#FFFFFF")

# Column boxes — empty containers for animated cards
C_SHIELD_BORDER = "#1E90FF"
C_SWORD_BORDER  = "#FFD700"
draw.rounded_rectangle([col1_x, col1_y, col1_x + col_w, col1_y + box_h],
                       radius=40, fill="#0D0520", outline=C_SHIELD_BORDER, width=12)
draw.rounded_rectangle([col2_x, col1_y, col2_x + col_w, col1_y + box_h],
                       radius=40, fill="#0D0520", outline=C_SWORD_BORDER,  width=12)

# ── 5. HEADERS WITH ICONS ────────────────────────────────────────────────
def draw_header_with_icon(label, icon_filename, box_x, box_w, y, color):
    """Draws an icon + header text centered together inside the column box."""
    icon_path = os.path.join(ELEMENTS_DIR, icon_filename)
    icon_size = 130
    gap       = 40

    # Measure text first
    font = fnt("Bold", 90)
    bbox = draw.textbbox((0, 0), label, font=font)
    txt_w = bbox[2] - bbox[0]
    txt_h = bbox[3] - bbox[1]

    # Total width of icon + gap + text, centered inside the column
    total_w = icon_size + gap + txt_w
    start_x = box_x + (box_w - total_w) // 2

    # Paste icon (vertically centered against text height)
    if os.path.exists(icon_path):
        try:
            icon = Image.open(icon_path).convert("RGBA").resize((icon_size, icon_size), Image.LANCZOS)
            canvas.paste(icon, (start_x, y - 25), icon)
        except Exception as e:
            print(f"   ⚠️ Failed to load {icon_filename}: {e}")

    # Draw text to the right of icon
    text_x = start_x + icon_size + gap
    draw.text((text_x, y), label, font=font, fill=color)

draw_header_with_icon("THE SHIELDS (RANK PROTECTORS)", "Shield.png",
                      col1_x, col_w, col1_y + 50, C_SHIELD_BORDER)
draw_header_with_icon("THE SWORDS (RANK CLIMBERS)",   "Sword.png",
                      col2_x, col_w, col1_y + 50, C_SWORD_BORDER)

# ── 6. SAVE ──────────────────────────────────────────────────────────────
empty_path = os.path.join(PNG_DIR, "Slide_3_Empty.png")
if os.path.exists(empty_path): os.remove(empty_path)
canvas.convert("RGB").save(empty_path, format="PNG")
print(f"✅ Slide 3 Empty Base Saved: {empty_path}")

print("\n" + "=" * 60)
print(" 🟣 CELL 5 COMPLETE: SLIDE 3 EMPTY BASE READY ")
print("=" * 60)

Initializing Slide 3: Sword & Shield Empty Base...
✅ Identified 6 Shields and 6 Swords.
   Shields: ['Haaland', 'Gabriel', 'Raya', 'B.Fernandes', 'Semenyo', 'Guéhi']
   Swords:  ['Dunk', 'Ampadu', 'J.Palhinha', 'Bijol', 'Tel', 'Shaw']
⚠️ 6.png not found. Falling back to solid canvas.
✅ Slide 3 Empty Base Saved: /content/drive/MyDrive/Vortex_Final/png/Slide_3_Empty.png

 🟣 CELL 5 COMPLETE: SLIDE 3 EMPTY BASE READY 


In [ ]:

# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 6: Slide 4 — Fixture Difficulty Map (Unified Tier System + 24.png)

Color System (matches Slides 5 & 6):
  T1 Easiest  FDR ≤ 2.50           → Bright Green  → White gradient
  T2          FDR 2.51 – 2.80      → Bright Blue   → White gradient
  T3          FDR 2.81 – 3.00      → Bright Yellow → White gradient
  T4          FDR 3.01 – 3.40      → Bright Grey   → White gradient
  T5 Hardest  FDR > 3.40           → Bright Pink   → White gradient
  DGW                              → Solid Black,  "DGW" white bold
  BGW                              → Solid Black,  "BGW" red bold
"""

import os, math
from datetime import datetime
from PIL import Image, ImageDraw

print("Initializing Slide 4: Fixture Difficulty Map...")

# ── 1. PATHS ─────────────────────────────────────────────────────────────
try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

PNG_DIR      = os.path.join(GD_ROOT, "png")
ELEMENTS_DIR = os.path.join(GD_ROOT, "elements")
BG_PATH      = os.path.join(ELEMENTS_DIR, "24.png")
os.makedirs(PNG_DIR, exist_ok=True)

# ── 2. UNIFIED 5-TIER GRADIENT SYSTEM ────────────────────────────────────

def get_tier_fdr(fdr):
    if fdr <= 2.50: return "T1"
    if fdr <= 2.80: return "T2"
    if fdr <= 3.00: return "T3"
    if fdr <= 3.40: return "T4"
    return "T5"

# ── 3. CANVAS ────────────────────────────────────────────────────────────

if os.path.exists(BG_PATH):
    canvas = Image.open(BG_PATH).convert("RGBA").resize((W, H), Image.LANCZOS)
else:
    print("⚠️ 24.png not found. Falling back to solid canvas.")
    canvas = Image.new("RGBA", (W, H), (10, 5, 25, 255))

draw = ImageDraw.Draw(canvas)

def ctxt(d, cx, cy, text, font_obj, fill_col):
    bbox = d.textbbox((0, 0), text, font=font_obj)
    d.text((cx - (bbox[2] - bbox[0]) // 2,
            cy - (bbox[3] - bbox[1]) // 2),
           text, font=font_obj, fill=fill_col)

# Title
ctxt(draw, W // 2, 280, "FIXTURE DIFFICULTY MAP", fnt("Bold", 130), "#FFD700")
draw.line([(W // 4, 400), (W * 3 // 4, 400)], fill="#FFD700", width=6)

# ── 4. DATA HARVEST ──────────────────────────────────────────────────────
events     = boot_data["events"]
next_gw    = next((e["id"] for e in events if e.get("is_next")), 1)
target_gws = get_remaining_gws(next_gw, window=7)

# Top-right GW range tag
gw_range_text = f"GW{target_gws[0]} – GW{target_gws[-1]}"
gw_range_font = fnt("Bold", 100)
gw_bbox       = draw.textbbox((0, 0), gw_range_text, font=gw_range_font)
draw.text((W - (gw_bbox[2] - gw_bbox[0]) - 200, 200),
          gw_range_text, font=gw_range_font, fill="#FFFF00")

T = {t["id"]: {
        "name":  t["name"],  "short": t["short_name"], "code": t["code"],
        "att_h": t["strength_attack_home"],  "att_a": t["strength_attack_away"],
        "def_h": t["strength_defence_home"], "def_a": t["strength_defence_away"],
        "gws":   {g: {"matches": []} for g in target_gws},
        "fdr_score": 0.0,
    } for t in boot_data["teams"]}

for fx in fix_data:
    gw = fx.get("event")
    if gw not in target_gws or fx["team_h"] not in T or fx["team_a"] not in T:
        continue
    hi, ai = fx["team_h"], fx["team_a"]
    xgh = poisson_xg(T[hi]["att_h"], T[ai]["def_a"], T[ai]["att_a"], T[hi]["def_h"], True)
    xga = poisson_xg(T[hi]["att_h"], T[ai]["def_a"], T[ai]["att_a"], T[hi]["def_h"], False)
    T[hi]["gws"][gw]["matches"].append({"opp": f"{T[ai]['short']}[H]", "fdr": compute_fdr(xgh)})
    T[ai]["gws"][gw]["matches"].append({"opp": f"{T[hi]['short']}[A]", "fdr": compute_fdr(xga)})

for td in T.values():
    for gw in target_gws:
        m = td["gws"][gw]["matches"]
        if not m:         td["fdr_score"] += 5.0
        elif len(m) == 1: td["fdr_score"] += m[0]["fdr"]
        else:             td["fdr_score"] += (m[0]["fdr"] + m[1]["fdr"]) / 2
    td["avg_fdr"] = td["fdr_score"] / len(target_gws)

ranked     = sorted(T.values(), key=lambda x: x["avg_fdr"])
fdr_ranked = ranked
top10, bot10 = ranked[:10], ranked[-10:]

# ── 5. LAYOUT GEOMETRY ───────────────────────────────────────────────────
draw.line([(W // 2, 580), (W // 2, H - 250)], fill="#FFD700", width=10)

MARGIN   = 200
GAP      = 150
PANEL_W  = (W - 2 * MARGIN - GAP) // 2
LX1, LX2 = MARGIN, MARGIN + PANEL_W
RX1, RX2 = LX2 + GAP, LX2 + GAP + PANEL_W

START_Y  = 750
ROW_H    = (H - START_Y - 350) // 10
TEAM_CW  = 500
TOTAL_CW = 240
BOX_W    = (PANEL_W - TEAM_CW - TOTAL_CW) // len(target_gws)

ctxt(draw, W // 4,    470, "TOP 10 FIXTURES  (EASIEST)",   fnt("Bold", 110), "#00FF87")
ctxt(draw, W * 3 // 4, 470, "BOTTOM 10 FIXTURES  (HARDEST)", fnt("Bold", 110), "#FF1744")

def draw_col_headers(px1, px2, py):
    cy = py - 80
    ctxt(draw, px1 + TEAM_CW // 2, cy, "TEAM", fnt("Bold", 100), "#FFFFFF")
    for i, gw in enumerate(target_gws):
        ctxt(draw, px1 + TEAM_CW + i * BOX_W + BOX_W // 2, cy,
             f"GW{gw}", fnt("Bold", 100), "#FFFFFF")
    ctxt(draw, px2 - TOTAL_CW // 2, cy, "AVG", fnt("Bold", 100), "#FFD700")

draw_col_headers(LX1, LX2, START_Y)
draw_col_headers(RX1, RX2, START_Y)

# ── 6. ROW RENDERING ─────────────────────────────────────────────────────
def fdr_block(canvas_img, x, y, w, h, matches):
    pad_x = 70  # INCREASING this squeezes the sides inward (shrinks width)
    pad_y = 12  # KEEPING this small maintains the box height

    bx, by = x + pad_x, y + pad_y
    bw, bh = w - 2 * pad_x, h - 2 * pad_y

    if not matches:
        paint_special_cell(canvas_img, bx, by, bw, bh, "BGW")
        return

    if len(matches) == 1:
        m  = matches[0]
        tk = get_tier_fdr(m["fdr"])
        paint_tier_cell(canvas_img, bx, by, bw, bh, tk,
                        value_text=f"{m['fdr']:.2f}",
                        value_font=fnt("Bold", 96))
        # Opponent label above the FDR number
        opp_font = fnt("Bold", 80)
        d = ImageDraw.Draw(canvas_img)
        ob = d.textbbox((0, 0), m["opp"], font=opp_font)
        ox = bx + (bw - (ob[2] - ob[0])) // 2
        d.text((ox, by + 15), m["opp"],
               font=opp_font, fill="#000000")  # Forced to black
    else:
            # DGW — split horizontally to show both opponents and FDR scores
            dgw_pad_x = 15  # Expand width to securely fit two boxes side-by-side
            dbx, dby = x + dgw_pad_x, y + pad_y
            dbw, dbh = w - 2 * dgw_pad_x, h - 2 * pad_y

            gap = 15
            half_w = (dbw - gap) // 2
            d = ImageDraw.Draw(canvas_img)

            for idx, m in enumerate(matches[:2]):
                box_x = dbx + idx * (half_w + gap)
                tk = get_tier_fdr(m["fdr"])

                # Draw the tier-colored gradient background and FDR score
                paint_tier_cell(canvas_img, box_x, dby, half_w, dbh, tk,
                                value_text=f"{m['fdr']:.2f}",
                                value_font=fnt("Bold", 75))

                # Draw the opponent label, auto-shrinking to prevent any overlap/bleeding
                opp_font_sz = 60
                opp_font = fnt("Bold", opp_font_sz)
                ob = d.textbbox((0, 0), m["opp"], font=opp_font)

                while (ob[2] - ob[0]) > (half_w - 10) and opp_font_sz > 30:
                    opp_font_sz -= 3
                    opp_font = fnt("Bold", opp_font_sz)
                    ob = d.textbbox((0, 0), m["opp"], font=opp_font)

                ox = box_x + (half_w - (ob[2] - ob[0])) // 2
                d.text((ox, dby + 15), m["opp"], font=opp_font, fill="#000000")

def draw_row(rank, td, px1, px2, py):
    # Alternating row background
    row_overlay = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    rd          = ImageDraw.Draw(row_overlay)
    row_color   = (18, 10, 45, 180) if rank % 2 != 0 else (10, 5, 30, 180)
    rd.rounded_rectangle([px1, py, px2, py + ROW_H - 10], radius=20, fill=row_color)
    canvas.alpha_composite(row_overlay)

    cy = py + ROW_H // 2 - 5
    ctxt(draw, px1 + 70, cy, str(rank), fnt("Bold", 100), "#FFFFFF")

    logo = get_team_logo(td["code"], size=(110, 110))
    canvas.paste(logo, (px1 + 150, cy - 55), logo)
    draw.text((px1 + 290, cy - 50), td["short"].upper(),
              font=fnt("Bold", 95), fill="#FFD700")

    # GW columns
    for i, gw in enumerate(target_gws):
        fdr_block(canvas, px1 + TEAM_CW + i * BOX_W, py,
                  BOX_W, ROW_H - 10, td["gws"][gw]["matches"])

    # AVG column — narrower box, centered in its TOTAL_CW slot
    avg_tier = get_tier_fdr(td["avg_fdr"])
    avg_w    = TOTAL_CW - 25                                # Changed to 20 to make the box wider
    avg_x    = px2 - TOTAL_CW + (TOTAL_CW - avg_w) // 2
    paint_tier_cell(canvas, avg_x, py + 12, avg_w, ROW_H - 34, avg_tier,
                    value_text=f"{td['avg_fdr']:.2f}",
                    value_font=fnt("Bold", 95))

# ── 7. RENDER ────────────────────────────────────────────────────────────
for i, td in enumerate(top10):
    draw_row(i + 1, td, LX1, LX2, START_Y + i * ROW_H)
for i, td in enumerate(bot10):
    draw_row(i + 1, td, RX1, RX2, START_Y + i * ROW_H)

# ── 8. FOOTER ────────────────────────────────────────────────────────────
draw = ImageDraw.Draw(canvas)
footer = f"FPL Vortex  •  Poisson xG Model  •  {datetime.now().strftime('%d %b %Y')}"
ctxt(draw, W // 2, H - 160, footer, fnt("Regular", 80), "#8888AA")

# ── 9. SAVE ──────────────────────────────────────────────────────────────
out_path = os.path.join(PNG_DIR, "Slide_4_fdr_map.png")
canvas.convert("RGB").save(out_path, "PNG", dpi=(300, 300))
print(f"✅ Slide 4 saved → {out_path}")

print("\n" + "=" * 60)
print(" 🟣 CELL 6 COMPLETE: SLIDE 4 FDR MAP READY ")
print("=" * 60)

Initializing Slide 4: Fixture Difficulty Map...
⚠️ 24.png not found. Falling back to solid canvas.
✅ Slide 4 saved → /content/drive/MyDrive/Vortex_Final/png/Slide_4_fdr_map.png

 🟣 CELL 6 COMPLETE: SLIDE 4 FDR MAP READY 


In [ ]:

# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 7: Slide 5 — Projected Goals (Unified Tier System + 24.png)

Color System (matches Slides 4 & 6):
  T1 Strongest  xG ≥ 1.69     → Bright Green  → White gradient
  T2            xG 1.50 – 1.68 → Bright Blue   → White gradient
  T3            xG 1.30 – 1.49 → Bright Yellow → White gradient
  T4            xG 1.25 – 1.29 → Bright Grey   → White gradient
  T5 Weakest    xG ≤ 1.24     → Bright Pink   → White gradient
  DGW                          → Solid Black,  "DGW" white bold
  BGW                          → Solid Black,  "BGW" red bold
"""

import os, math
from datetime import datetime
from PIL import Image, ImageDraw

print("Initializing Slide 5: Projected Goals...")

# ── 1. PATHS ─────────────────────────────────────────────────────────────
try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

PNG_DIR      = os.path.join(GD_ROOT, "png")
ELEMENTS_DIR = os.path.join(GD_ROOT, "elements")
BG_PATH      = os.path.join(ELEMENTS_DIR, "24.png")
os.makedirs(PNG_DIR, exist_ok=True)

# ── 2. UNIFIED 5-TIER GRADIENT SYSTEM ────────────────────────────────────
def get_tier_xg(xg):
    """Higher xG = better attack = lower tier number (T1 strongest)."""
    if xg >= 1.69: return "T1"
    if xg >= 1.50: return "T2"
    if xg >= 1.30: return "T3"
    if xg >= 1.25: return "T4"
    return "T5"


# ── 3. CANVAS ────────────────────────────────────────────────────────────


if os.path.exists(BG_PATH):
    canvas = Image.open(BG_PATH).convert("RGBA").resize((W, H), Image.LANCZOS)
else:
    print("⚠️ 24.png not found. Falling back to solid canvas.")
    canvas = Image.new("RGBA", (W, H), (10, 5, 25, 255))

draw = ImageDraw.Draw(canvas)

def ctxt(d, cx, cy, text, font_obj, fill_col):
    bbox = d.textbbox((0, 0), text, font=font_obj)
    d.text((cx - (bbox[2] - bbox[0]) // 2,
            cy - (bbox[3] - bbox[1]) // 2),
           text, font=font_obj, fill=fill_col)

# Title
ctxt(draw, W // 2, 280, "PROJECTED GOALS MAP", fnt("Bold", 130), "#FFD700")
draw.line([(W // 4, 400), (W * 3 // 4, 400)], fill="#FFD700", width=6)

# ── 4. DATA HARVEST ──────────────────────────────────────────────────────
events     = boot_data["events"]
next_gw    = next((e["id"] for e in events if e.get("is_next")), 1)
target_gws = get_remaining_gws(next_gw, window=7)

# Top-right GW range tag
gw_range_text = f"GW{target_gws[0]} – GW{target_gws[-1]}"
gw_range_font = fnt("Bold", 100)
gw_bbox       = draw.textbbox((0, 0), gw_range_text, font=gw_range_font)
draw.text((W - (gw_bbox[2] - gw_bbox[0]) - 200, 200),
          gw_range_text, font=gw_range_font, fill="#FFFF00")

T = {t["id"]: {
        "short": t["short_name"], "code": t["code"],
        "att_h": t["strength_attack_home"],  "att_a": t["strength_attack_away"],
        "def_h": t["strength_defence_home"], "def_a": t["strength_defence_away"],
        "gws":   {g: {"matches": []} for g in target_gws},
        "total_proj_goals": 0.0,
    } for t in boot_data["teams"]}

for fx in fix_data:
    gw = fx.get("event")
    if gw not in target_gws or fx["team_h"] not in T or fx["team_a"] not in T:
        continue
    hi, ai = fx["team_h"], fx["team_a"]
    xgh = poisson_xg(T[hi]["att_h"], T[ai]["def_a"], T[ai]["att_a"], T[hi]["def_h"], True)
    xga = poisson_xg(T[hi]["att_h"], T[ai]["def_a"], T[ai]["att_a"], T[hi]["def_h"], False)
    T[hi]["gws"][gw]["matches"].append({"opp": T[ai]['short'], "xg": xgh, "home": True})
    T[ai]["gws"][gw]["matches"].append({"opp": T[hi]['short'], "xg": xga, "home": False})
    T[hi]["total_proj_goals"] += xgh
    T[ai]["total_proj_goals"] += xga

for td in T.values():
    td["avg_proj_goals"] = td["total_proj_goals"] / max(1, len(target_gws))

ranked        = sorted(T.values(), key=lambda x: x["avg_proj_goals"], reverse=True)
top10, bot10  = ranked[:10], ranked[-10:]
goals_top10   = top10
goals_bot10   = bot10

# ── 5. LAYOUT GEOMETRY ───────────────────────────────────────────────────
draw.line([(W // 2, 580), (W // 2, H - 250)], fill="#FFD700", width=10)

MARGIN   = 200
GAP      = 150
PANEL_W  = (W - 2 * MARGIN - GAP) // 2
LX1, LX2 = MARGIN, MARGIN + PANEL_W
RX1, RX2 = LX2 + GAP, LX2 + GAP + PANEL_W

START_Y  = 750
ROW_H    = (H - START_Y - 350) // 10
TEAM_CW  = 500
TOTAL_CW = 240
BOX_W    = (PANEL_W - TEAM_CW - TOTAL_CW) // len(target_gws)

ctxt(draw, W // 4,    470, "TOP 10 ATTACKS  (STRONGEST)",   fnt("Bold", 110), "#00FF87")
ctxt(draw, W * 3 // 4, 470, "BOTTOM 10 ATTACKS  (WEAKEST)",  fnt("Bold", 110), "#FF1744")

def draw_col_headers(px1, px2, py):
    cy = py - 80
    ctxt(draw, px1 + TEAM_CW // 2, cy, "TEAM", fnt("Bold", 100), "#FFFFFF")
    for i, gw in enumerate(target_gws):
        ctxt(draw, px1 + TEAM_CW + i * BOX_W + BOX_W // 2, cy,
             f"GW{gw}", fnt("Bold", 100), "#FFFFFF")
    ctxt(draw, px2 - TOTAL_CW // 2, cy, "AVG", fnt("Bold", 100), "#FFD700")

draw_col_headers(LX1, LX2, START_Y)
draw_col_headers(RX1, RX2, START_Y)

# ── 6. ROW RENDERING ─────────────────────────────────────────────────────
def xg_block(canvas_img, x, y, w, h, matches):
    pad = 12
    bx, by = x + pad, y + pad
    bw, bh = w - 2 * pad, h - 2 * pad

    if not matches:
        paint_special_cell(canvas_img, bx, by, bw, bh, "BGW")
        return

    if len(matches) == 1:
        m  = matches[0]
        tk = get_tier_xg(m["xg"])
        opp_label = f"{m['opp']} {'(H)' if m['home'] else '(A)'}"
        paint_tier_cell(canvas_img, bx, by, bw, bh, tk,
                        value_text=f"{m['xg']:.2f}",
                        value_font=fnt("Bold", 95),
                        sub_text=opp_label,
                        sub_font=fnt("Bold", 80))
    else:
        dgw_pad_x = 15
        dbx, dby = x + dgw_pad_x, y + pad
        dbw, dbh = w - 2 * dgw_pad_x, h - 2 * pad

        gap = 15
        half_w = (dbw - gap) // 2
        d = ImageDraw.Draw(canvas_img)

        for idx, m in enumerate(matches[:2]):
            box_x = dbx + idx * (half_w + gap)
            tk = get_tier_xg(m["xg"])
            opp_label = f"{m['opp']} {'(H)' if m['home'] else '(A)'}"

            paint_tier_cell(canvas_img, box_x, dby, half_w, dbh, tk,
                            value_text=f"{m['xg']:.2f}",
                            value_font=fnt("Bold", 75))

            opp_font_sz = 60
            opp_font = fnt("Bold", opp_font_sz)
            ob = d.textbbox((0, 0), opp_label, font=opp_font)

            while (ob[2] - ob[0]) > (half_w - 10) and opp_font_sz > 30:
                opp_font_sz -= 3
                opp_font = fnt("Bold", opp_font_sz)
                ob = d.textbbox((0, 0), opp_label, font=opp_font)

            ox = box_x + (half_w - (ob[2] - ob[0])) // 2
            d.text((ox, dby + 15), opp_label, font=opp_font, fill="#000000")

def draw_row(rank, td, px1, px2, py):
    row_overlay = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    rd          = ImageDraw.Draw(row_overlay)
    row_color   = (18, 10, 45, 180) if rank % 2 != 0 else (10, 5, 30, 180)
    rd.rounded_rectangle([px1, py, px2, py + ROW_H - 10], radius=20, fill=row_color)
    canvas.alpha_composite(row_overlay)

    cy = py + ROW_H // 2 - 5
    ctxt(draw, px1 + 70, cy, str(rank), fnt("Bold", 100), "#FFFFFF")

    logo = get_team_logo(td["code"], size=(110, 110))
    canvas.paste(logo, (px1 + 150, cy - 55), logo)
    draw.text((px1 + 290, cy - 50), td["short"].upper(),
              font=fnt("Bold", 95), fill="#FFD700")

    for i, gw in enumerate(target_gws):
        xg_block(canvas, px1 + TEAM_CW + i * BOX_W, py,
                 BOX_W, ROW_H - 10, td["gws"][gw]["matches"])

    # AVG column — narrower fill, tier driven by per-GW average
    avg_xg   = td["avg_proj_goals"]
    total_tier = get_tier_xg(avg_xg)
    avg_w    = TOTAL_CW - 25
    avg_x    = px2 - TOTAL_CW + (TOTAL_CW - avg_w) // 2
    paint_tier_cell(canvas, avg_x, py + 12, avg_w, ROW_H - 34, total_tier,
                    value_text=f"{avg_xg:.2f}",
                    value_font=fnt("Bold", 95))

# ── 7. RENDER ────────────────────────────────────────────────────────────
for i, td in enumerate(top10):
    draw_row(i + 1, td, LX1, LX2, START_Y + i * ROW_H)
for i, td in enumerate(bot10):
    draw_row(i + 1, td, RX1, RX2, START_Y + i * ROW_H)

# ── 8. FOOTER ────────────────────────────────────────────────────────────
draw = ImageDraw.Draw(canvas)
footer = f"FPL Vortex  •  Poisson xG Distribution  •  {datetime.now().strftime('%d %b %Y')}"
ctxt(draw, W // 2, H - 160, footer, fnt("Regular", 80), "#8888AA")

# ── 9. SAVE ──────────────────────────────────────────────────────────────
out_path = os.path.join(PNG_DIR, "Slide_5_Projected_Goals.png")
canvas.convert("RGB").save(out_path, "PNG", dpi=(300, 300))
print(f"✅ Slide 5 saved → {out_path}")

print("\n" + "=" * 60)
print(" 🟣 CELL 7 COMPLETE: SLIDE 5 PROJECTED GOALS READY ")
print("=" * 60)

Initializing Slide 5: Projected Goals...
⚠️ 24.png not found. Falling back to solid canvas.
✅ Slide 5 saved → /content/drive/MyDrive/Vortex_Final/png/Slide_5_Projected_Goals.png

 🟣 CELL 7 COMPLETE: SLIDE 5 PROJECTED GOALS READY 


In [ ]:

# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 8: Slide 6 — Clean Sheet Odds (Unified Tier System + 24.png)

Color System (matches Slides 4 & 5):
  T1 Easiest  CS% ≥ 35       → Bright Green  → White gradient
  T2          CS% 30 – 34    → Bright Blue   → White gradient
  T3          CS% 25 – 29    → Bright Yellow → White gradient
  T4          CS% 20 – 24    → Bright Grey   → White gradient
  T5 Hardest  CS% < 20       → Bright Pink   → White gradient
  DGW                        → Solid Black,  "DGW" white bold
  BGW                        → Solid Black,  "BGW" red bold

CS% per match = Poisson P(X=0) on opponent xG
GW CS% (for DGW) = 1 - Π(1 - cs_i/100)  → pure "at least one CS" probability
"""

import os, math
from datetime import datetime
from PIL import Image, ImageDraw

print("Initializing Slide 6: Clean Sheet Odds...")

# ── 1. PATHS ─────────────────────────────────────────────────────────────
try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

PNG_DIR      = os.path.join(GD_ROOT, "png")
ELEMENTS_DIR = os.path.join(GD_ROOT, "elements")
BG_PATH      = os.path.join(ELEMENTS_DIR, "1.png")
os.makedirs(PNG_DIR, exist_ok=True)

# ── 2. UNIFIED 5-TIER GRADIENT SYSTEM ────────────────────────────────────

def get_tier_cs(cs_pct):
    """Higher CS% = better defense = lower tier number (T1 strongest)."""
    if cs_pct >= 35: return "T1"
    if cs_pct >= 30: return "T2"
    if cs_pct >= 25: return "T3"
    if cs_pct >= 20: return "T4"
    return "T5"



# ── 3. CANVAS ────────────────────────────────────────────────────────────


if os.path.exists(BG_PATH):
    canvas = Image.open(BG_PATH).convert("RGBA").resize((W, H), Image.LANCZOS)
else:
    print("⚠️ 1.png not found. Falling back to solid canvas.")
    canvas = Image.new("RGBA", (W, H), (10, 5, 25, 255))

draw = ImageDraw.Draw(canvas)

def ctxt(d, cx, cy, text, font_obj, fill_col):
    bbox = d.textbbox((0, 0), text, font=font_obj)
    d.text((cx - (bbox[2] - bbox[0]) // 2,
            cy - (bbox[3] - bbox[1]) // 2),
           text, font=font_obj, fill=fill_col)

# Title
ctxt(draw, W // 2, 280, "DEFENSIVE SOLIDITY & CS ODDS", fnt("Bold", 130), "#FFD700")
draw.line([(W // 4, 400), (W * 3 // 4, 400)], fill="#FFD700", width=6)

# ── 4. DATA HARVEST ──────────────────────────────────────────────────────
events     = boot_data["events"]
next_gw    = next((e["id"] for e in events if e.get("is_next")), 1)
target_gws = get_remaining_gws(next_gw, window=7)

# Top-right GW range tag
gw_range_text = f"GW{target_gws[0]} – GW{target_gws[-1]}"
gw_range_font = fnt("Bold", 100)
gw_bbox       = draw.textbbox((0, 0), gw_range_text, font=gw_range_font)
draw.text((W - (gw_bbox[2] - gw_bbox[0]) - 200, 200),
          gw_range_text, font=gw_range_font, fill="#FFFF00")

T = {t["id"]: {
        "short": t["short_name"], "code": t["code"],
        "att_h": t["strength_attack_home"],  "att_a": t["strength_attack_away"],
        "def_h": t["strength_defence_home"], "def_a": t["strength_defence_away"],
        "gws":   {g: {"matches": []} for g in target_gws},
    } for t in boot_data["teams"]}

for fx in fix_data:
    gw = fx.get("event")
    if gw not in target_gws or fx["team_h"] not in T or fx["team_a"] not in T:
        continue
    hi, ai = fx["team_h"], fx["team_a"]
    xgh = poisson_xg(T[hi]["att_h"], T[ai]["def_a"], T[ai]["att_a"], T[hi]["def_h"], True)
    xga = poisson_xg(T[hi]["att_h"], T[ai]["def_a"], T[ai]["att_a"], T[hi]["def_h"], False)
    cs_h = math.exp(-xga) * 100
    cs_a = math.exp(-xgh) * 100
    T[hi]["gws"][gw]["matches"].append({"opp": T[ai]['short'], "cs": cs_h, "home": True})
    T[ai]["gws"][gw]["matches"].append({"opp": T[hi]['short'], "cs": cs_a, "home": False})

# Compute "≥1 clean sheet" probability per GW (DGW-aware), then 7-week average
for td in T.values():
    total_cs_7wk = 0.0
    for gw in target_gws:
        matches = td["gws"][gw]["matches"]
        if not matches:
            td["gws"][gw]["gw_cs_prob"] = 0.0
            continue
        fail_prob = 1.0
        for m in matches:
            fail_prob *= (1.0 - (m['cs'] / 100.0))
        gw_cs_prob = (1.0 - fail_prob) * 100.0
        td["gws"][gw]["gw_cs_prob"] = gw_cs_prob
        total_cs_7wk += gw_cs_prob
    td["avg_cs"] = total_cs_7wk / len(target_gws)

ranked       = sorted(T.values(), key=lambda x: x["avg_cs"], reverse=True)
top10, bot10 = ranked[:10], ranked[-10:]
cs_top10     = top10
cs_bot10     = bot10

# ── 5. LAYOUT GEOMETRY ───────────────────────────────────────────────────
draw.line([(W // 2, 580), (W // 2, H - 250)], fill="#FFD700", width=10)

MARGIN   = 200
GAP      = 150
PANEL_W  = (W - 2 * MARGIN - GAP) // 2
LX1, LX2 = MARGIN, MARGIN + PANEL_W
RX1, RX2 = LX2 + GAP, LX2 + GAP + PANEL_W

START_Y  = 750
ROW_H    = (H - START_Y - 350) // 10
TEAM_CW  = 500
TOTAL_CW = 240
BOX_W    = (PANEL_W - TEAM_CW - TOTAL_CW) // len(target_gws)

ctxt(draw, W // 4,    470, "TOP 10 DEFENSES  (EASIEST)",    fnt("Bold", 110), "#00FF87")
ctxt(draw, W * 3 // 4, 470, "BOTTOM 10 DEFENSES  (HARDEST)", fnt("Bold", 110), "#FF1744")

def draw_col_headers(px1, px2, py):
    cy = py - 80
    ctxt(draw, px1 + TEAM_CW // 2, cy, "TEAM", fnt("Bold", 100), "#FFFFFF")
    for i, gw in enumerate(target_gws):
        ctxt(draw, px1 + TEAM_CW + i * BOX_W + BOX_W // 2, cy,
             f"GW{gw}", fnt("Bold", 100), "#FFFFFF")
    ctxt(draw, px2 - TOTAL_CW // 2, cy, "AVG CS%", fnt("Bold", 100), "#FFD700")

draw_col_headers(LX1, LX2, START_Y)
draw_col_headers(RX1, RX2, START_Y)

# ── 6. ROW RENDERING ─────────────────────────────────────────────────────
def cs_block(canvas_img, x, y, w, h, matches):
    pad = 12
    bx, by = x + pad, y + pad
    bw, bh = w - 2 * pad, h - 2 * pad

    if not matches:
        paint_special_cell(canvas_img, bx, by, bw, bh, "BGW")
        return

    if len(matches) == 1:
        m  = matches[0]
        tk = get_tier_cs(m["cs"])
        opp_label = f"{m['opp']} {'(H)' if m['home'] else '(A)'}"
        paint_tier_cell(canvas_img, bx, by, bw, bh, tk,
                        value_text=f"{m['cs']:.0f}%",
                        value_font=fnt("Bold", 95),
                        sub_text=opp_label,
                        sub_font=fnt("Bold", 80))
    else:
        dgw_pad_x = 15
        dbx, dby = x + dgw_pad_x, y + pad
        dbw, dbh = w - 2 * dgw_pad_x, h - 2 * pad

        gap = 15
        half_w = (dbw - gap) // 2
        d = ImageDraw.Draw(canvas_img)

        for idx, m in enumerate(matches[:2]):
            box_x = dbx + idx * (half_w + gap)
            tk = get_tier_cs(m["cs"])
            opp_label = f"{m['opp']} {'(H)' if m['home'] else '(A)'}"

            paint_tier_cell(canvas_img, box_x, dby, half_w, dbh, tk,
                            value_text=f"{m['cs']:.0f}%",
                            value_font=fnt("Bold", 75))

            opp_font_sz = 60
            opp_font = fnt("Bold", opp_font_sz)
            ob = d.textbbox((0, 0), opp_label, font=opp_font)

            while (ob[2] - ob[0]) > (half_w - 10) and opp_font_sz > 30:
                opp_font_sz -= 3
                opp_font = fnt("Bold", opp_font_sz)
                ob = d.textbbox((0, 0), opp_label, font=opp_font)

            ox = box_x + (half_w - (ob[2] - ob[0])) // 2
            d.text((ox, dby + 15), opp_label, font=opp_font, fill="#000000")

def draw_row(rank, td, px1, px2, py):
    row_overlay = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    rd          = ImageDraw.Draw(row_overlay)
    row_color   = (18, 10, 45, 180) if rank % 2 != 0 else (10, 5, 30, 180)
    rd.rounded_rectangle([px1, py, px2, py + ROW_H - 10], radius=20, fill=row_color)
    canvas.alpha_composite(row_overlay)

    cy = py + ROW_H // 2 - 5
    ctxt(draw, px1 + 70, cy, str(rank), fnt("Bold", 100), "#FFFFFF")

    logo = get_team_logo(td["code"], size=(110, 110))
    canvas.paste(logo, (px1 + 150, cy - 55), logo)
    draw.text((px1 + 290, cy - 50), td["short"].upper(),
              font=fnt("Bold", 95), fill="#FFD700")

    for i, gw in enumerate(target_gws):
        cs_block(canvas, px1 + TEAM_CW + i * BOX_W, py,
                 BOX_W, ROW_H - 10, td["gws"][gw]["matches"])

    # AVG column — narrower fill, tier driven by per-GW average
    avg_cs     = td["avg_cs"]
    total_tier = get_tier_cs(avg_cs)
    avg_w    = TOTAL_CW - 25
    avg_x    = px2 - TOTAL_CW + (TOTAL_CW - avg_w) // 2
    paint_tier_cell(canvas, avg_x, py + 12, avg_w, ROW_H - 34, total_tier,
                    value_text=f"{avg_cs:.0f}%",
                    value_font=fnt("Bold", 95))

# ── 7. RENDER ────────────────────────────────────────────────────────────
for i, td in enumerate(top10):
    draw_row(i + 1, td, LX1, LX2, START_Y + i * ROW_H)
for i, td in enumerate(bot10):
    draw_row(i + 1, td, RX1, RX2, START_Y + i * ROW_H)

# ── 8. FOOTER ────────────────────────────────────────────────────────────
draw = ImageDraw.Draw(canvas)
footer = f"FPL Vortex  •  Poisson P(X=0)  •  {datetime.now().strftime('%d %b %Y')}"
ctxt(draw, W // 2, H - 160, footer, fnt("Regular", 80), "#8888AA")

# ── 9. SAVE ──────────────────────────────────────────────────────────────
out_path = os.path.join(PNG_DIR, "Slide_6_CS_Odds.png")
canvas.convert("RGB").save(out_path, "PNG", dpi=(300, 300))
print(f"✅ Slide 6 saved → {out_path}")

print("\n" + "=" * 60)
print(" 🟣 CELL 8 COMPLETE: SLIDE 6 CS ODDS READY ")
print("=" * 60)

Initializing Slide 6: Clean Sheet Odds...
✅ Slide 6 saved → /content/drive/MyDrive/Vortex_Final/png/Slide_6_CS_Odds.png

 🟣 CELL 8 COMPLETE: SLIDE 6 CS ODDS READY 


In [ ]:

# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 9: Slide 7 — Single GW Attack vs Defense Projections (All 20 Teams)
Unified Tier System (matches Slides 4, 5, 6) + 24.png background
"""

import os, math
from datetime import datetime
from PIL import Image, ImageDraw

print("Initializing Slide 7: Full 20-Team Target Tables...")

# ── 1. PATHS ─────────────────────────────────────────────────────────────
try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

PNG_DIR      = os.path.join(GD_ROOT, "png")
ELEMENTS_DIR = os.path.join(GD_ROOT, "elements")
BG_PATH      = os.path.join(ELEMENTS_DIR, "2.png")
os.makedirs(PNG_DIR, exist_ok=True)

# ── 2. UNIFIED 5-TIER GRADIENT SYSTEM ────────────────────────────────────

def get_tier_xg(xg):
    if xg >= 1.69: return "T1"
    if xg >= 1.50: return "T2"
    if xg >= 1.30: return "T3"
    if xg >= 1.25: return "T4"
    return "T5"

def get_tier_cs(cs_pct):
    if cs_pct >= 35: return "T1"
    if cs_pct >= 30: return "T2"
    if cs_pct >= 25: return "T3"
    if cs_pct >= 20: return "T4"
    return "T5"


# ── 3. DATA HARVEST (DGW-AWARE PURE PROBABILITY) ─────────────────────────
events = boot_data["events"]
next_event = next((e for e in events if e.get("is_next")), events[0])
target_gw  = next_event["id"]
gw_name    = next_event["name"].upper()

all_teams_data = []

for team in boot_data["teams"]:
    team_dict = {
        "short": team["short_name"], "code": team["code"], "name": team["name"],
        "matches": [], "xg": 0.0, "cs_fail_prob": 1.0, "cs": 0.0,
    }

    gw_matches = [f for f in fix_data
                  if f.get("event") == target_gw
                  and (f["team_h"] == team["id"] or f["team_a"] == team["id"])]

    if not gw_matches:
        team_dict["cs"] = 0.0
    else:
        for f in gw_matches:
            is_home  = (f["team_h"] == team["id"])
            opp_id   = f["team_a"] if is_home else f["team_h"]
            opp_team = next(t for t in boot_data['teams'] if t['id'] == opp_id)

            if is_home:
                xgh = poisson_xg(team["strength_attack_home"], opp_team["strength_defence_away"],
                                 opp_team["strength_attack_away"], team["strength_defence_home"], True)
                xga = poisson_xg(opp_team["strength_attack_away"], team["strength_defence_home"],
                                 team["strength_attack_home"], opp_team["strength_defence_away"], False)
                match_xg = xgh
                match_cs = math.exp(-xga) * 100
            else:
                xga = poisson_xg(opp_team["strength_attack_home"], team["strength_defence_away"],
                                 team["strength_attack_away"], opp_team["strength_defence_home"], False)
                xgh = poisson_xg(team["strength_attack_away"], opp_team["strength_defence_home"],
                                 opp_team["strength_attack_home"], team["strength_defence_away"], True)
                match_xg = xga
                match_cs = math.exp(-xgh) * 100

            team_dict["matches"].append({
                "opp_short": opp_team["short_name"],
                "opp_code":  opp_team["code"],
                "is_home":   is_home,
                "xg":        match_xg,
                "cs":        match_cs,
            })
            team_dict["xg"]           += match_xg
            team_dict["cs_fail_prob"] *= (1.0 - (match_cs / 100.0))

        team_dict["cs"] = (1.0 - team_dict["cs_fail_prob"]) * 100.0

    all_teams_data.append(team_dict)

xg_ranking = sorted(all_teams_data, key=lambda x: x["xg"], reverse=True)
cs_ranking = sorted(all_teams_data, key=lambda x: x["cs"], reverse=True)

# ── 4. CANVAS ────────────────────────────────────────────────────────────

if os.path.exists(BG_PATH):
    canvas = Image.open(BG_PATH).convert("RGBA").resize((W, H), Image.LANCZOS)
else:
    print("⚠️ 24.png not found. Falling back to solid canvas.")
    canvas = Image.new("RGBA", (W, H), (10, 5, 25, 255))

overlay = Image.new("RGBA", (W, H), (0, 0, 0, 0))
ov_draw = ImageDraw.Draw(overlay)
draw    = ImageDraw.Draw(canvas)

def ctxt(d, cx, cy, text, font_obj, fill_col):
    bbox = d.textbbox((0, 0), text, font=font_obj)
    d.text((cx - (bbox[2] - bbox[0]) // 2,
            cy - (bbox[3] - bbox[1]) // 2),
           text, font=font_obj, fill=fill_col)

# ── 5. HEADERS ───────────────────────────────────────────────────────────
START_X_LEFT_TABLE  = 2900
TABLE_W             = 2200
TABLE_GAP           = 100
START_X_RIGHT_TABLE = START_X_LEFT_TABLE + TABLE_W + TABLE_GAP

START_Y = 650
ROW_H   = 160

COL_RANK = 120
COL_TEAM = 600
COL_OPP  = 850
COL_DATA = 320

X_RANK = 0
X_TEAM = X_RANK + COL_RANK
X_OPP  = X_TEAM + COL_TEAM
X_DATA = X_OPP  + COL_OPP

title_x = START_X_LEFT_TABLE + (TABLE_W * 2 + TABLE_GAP) // 2
ctxt(draw, title_x, 280, f"{gw_name} TARGET MATCHUPS", fnt("Bold", 140), "#FFD700")
draw.line([(title_x - 1200, 400), (title_x + 1200, 400)], fill="#FFD700", width=6)

hy = START_Y - 120

# Define individual pixel shifts (increase to move further LEFT)
LEFT_TEAM_SHIFT  = 150
RIGHT_TEAM_SHIFT = 80
LEFT_OPP_SHIFT   = 120   # Adjust left opponent header
RIGHT_OPP_SHIFT  = 110   # Adjust right opponent header

# ── LEFT TABLE HEADERS ──
ctxt(draw, START_X_LEFT_TABLE  + X_TEAM + (COL_TEAM // 2) - LEFT_TEAM_SHIFT, hy, "TEAM",
     fnt("Bold", 90), "#FFFFFF")
ctxt(draw, START_X_LEFT_TABLE  + X_OPP  + (COL_OPP // 2) - LEFT_OPP_SHIFT, hy, "OPPONENT(S)",
     fnt("Bold", 90), "#FFFFFF")
ctxt(draw, START_X_LEFT_TABLE  + X_DATA + COL_DATA // 2, hy, "PROJ. GOALS",
     fnt("Bold", 90), "#FFD700")

# ── RIGHT TABLE HEADERS ──
ctxt(draw, START_X_RIGHT_TABLE + X_TEAM + (COL_TEAM // 2) - RIGHT_TEAM_SHIFT, hy, "TEAM",
     fnt("Bold", 90), "#FFFFFF")
ctxt(draw, START_X_RIGHT_TABLE + X_OPP  + (COL_OPP // 2) - RIGHT_OPP_SHIFT, hy, "OPPONENT(S)",
     fnt("Bold", 90), "#FFFFFF")
ctxt(draw, START_X_RIGHT_TABLE + X_DATA + COL_DATA // 2, hy, "CS %",
     fnt("Bold", 90), "#FFD700")

# ── 6. ROW BACKGROUNDS (alternating bands) ───────────────────────────────
def draw_row_bg(rank, px, py):
    row_color = (25, 20, 45, 180) if rank % 2 != 0 else (15, 10, 30, 180)
    ov_draw.rounded_rectangle([px, py, px + TABLE_W, py + ROW_H - 8],
                              radius=15, fill=row_color)

def draw_opp_band(td, px, py):
    if not td["matches"]:
        return
    match   = td["matches"][0]
    opp_bg  = (26, 26, 46, 255) if match['is_home'] else (10, 10, 31, 255)
    ov_draw.rounded_rectangle(
        [px + X_OPP + 10, py + 12, px + X_OPP + COL_OPP - 10, py + ROW_H - 20],
        radius=12, fill=opp_bg
    )

for i in range(20):
    py = START_Y + i * ROW_H
    draw_row_bg(i + 1, START_X_LEFT_TABLE,  py)
    draw_row_bg(i + 1, START_X_RIGHT_TABLE, py)
    draw_opp_band(xg_ranking[i], START_X_LEFT_TABLE,  py)
    draw_opp_band(cs_ranking[i], START_X_RIGHT_TABLE, py)

canvas = Image.alpha_composite(canvas, overlay)
draw   = ImageDraw.Draw(canvas)

# ── 7. ROW CONTENT (text + logos + tier-colored data cells) ──────────────
def draw_row_content(td, rank, px, py, is_xg_table):
    cy = py + ROW_H // 2 - 5

    # Rank
    ctxt(draw, px + X_RANK + COL_RANK // 2, cy - 1, str(rank),
         fnt("Bold", 70), "#FFD700" if rank <= 5 else "#FFFFFF")

    # Team logo + short name
    try:
        team_logo = get_team_logo(td["code"], size=(100, 100))
        canvas.paste(team_logo, (px + X_TEAM + 10, cy - 50), team_logo)
    except Exception: pass
    draw.text((px + X_TEAM + 130, cy - 40),
              td["short"].upper(), font=fnt("Bold", 75), fill="#FFFFFF")

    # Handle BGW
    if not td["matches"]:
        # Replace the data cell area with a special black BGW cell
        avg_w = COL_DATA - 60
        avg_x = px + X_DATA + (COL_DATA - avg_w) // 2
        paint_special_cell(canvas, avg_x, py + 12, avg_w, ROW_H - 30, "BGW")
        ctxt(draw, px + X_OPP + COL_OPP // 2, cy, "BLANK GAMEWEEK",
             fnt("Bold", 60), "#FF1744")
        return

    # DGW handling
    if len(td["matches"]) > 1:
        # Compress both opponents into the OPP column
        opp_strs = [f"{m['opp_short']} {'(H)' if m['is_home'] else '(A)'}"
                    for m in td["matches"]]
        fixture_text = " + ".join(opp_strs)

        # 🔴 FIX 1: Paste BOTH team logos side by side
        logo_sz = 85
        pad     = 12
        gap     = 20
        name1   = f"{td['matches'][0]['opp_short']} {'(H)' if td['matches'][0]['is_home'] else '(A)'}"
        name2   = f"{td['matches'][1]['opp_short']} {'(H)' if td['matches'][1]['is_home'] else '(A)'}"

        f_sz  = 75
        f_obj = fnt("Bold", f_sz)
        while (logo_sz + pad + draw.textlength(name1, font=f_obj) + gap +
               logo_sz + pad + draw.textlength(name2, font=f_obj)) > (COL_OPP - 30) and f_sz > 40:
            f_sz -= 3
            f_obj = fnt("Bold", f_sz)

        cur_x  = px + X_OPP + 15
        logo_y = cy - logo_sz // 2

        try:
            lg1 = get_team_logo(td["matches"][0]['opp_code'], size=(logo_sz, logo_sz))
            canvas.paste(lg1, (cur_x, logo_y), lg1)
        except Exception: pass
        cur_x += logo_sz + pad
        draw.text((cur_x, cy), name1, font=f_obj, fill="#FFFFFF", anchor="lm")
        cur_x += int(draw.textlength(name1, font=f_obj)) + gap

        try:
            lg2 = get_team_logo(td["matches"][1]['opp_code'], size=(logo_sz, logo_sz))
            canvas.paste(lg2, (cur_x, logo_y), lg2)
        except Exception: pass
        cur_x += logo_sz + pad
        draw.text((cur_x, cy), name2, font=f_obj, fill="#FFFFFF", anchor="lm")

        # 🔴 FIX 2: Print XXXX/XXXX split data instead of "DGW" label
        avg_w = COL_DATA - 60
        avg_x = px + X_DATA + (COL_DATA - avg_w) // 2

        # Draw black background pill
        gap_c  = 8
        half_w = (avg_w - gap_c) // 2
        if is_xg_table:
            v1, v2   = td["matches"][0]["xg"], td["matches"][1]["xg"]
            tier1, tier2 = get_tier_xg(v1), get_tier_xg(v2)
            txt1, txt2   = f"{v1:.2f}", f"{v2:.2f}"
        else:
            v1, v2   = td["matches"][0]["cs"], td["matches"][1]["cs"]
            tier1, tier2 = get_tier_cs(v1), get_tier_cs(v2)
            txt1, txt2   = f"{int(round(v1))}%", f"{int(round(v2))}%"
        paint_tier_cell(canvas, avg_x, py + 12, half_w, ROW_H - 30, tier1,
                        value_text=txt1, value_font=fnt("Bold", 80))
        paint_tier_cell(canvas, avg_x + half_w + gap_c, py + 12, half_w, ROW_H - 30, tier2,
                        value_text=txt2, value_font=fnt("Bold", 80))
        return

    # Single fixture — opponent logo + label
    match = td["matches"][0]
    try:
        opp_logo = get_team_logo(match['opp_code'], size=(100, 100))
        canvas.paste(opp_logo, (px + X_OPP + 30, cy - 50), opp_logo)
    except Exception: pass

    fixture_text = f"{match['opp_short']} {'(H)' if match['is_home'] else '(A)'}"
    draw.text((px + X_OPP + 150, cy - 40),
              fixture_text, font=fnt("Bold", 75), fill="#FFFFFF")

    # Tier-colored data cell
    avg_w = COL_DATA - 60
    avg_x = px + X_DATA + (COL_DATA - avg_w) // 2
    if is_xg_table:
        tier = get_tier_xg(td["xg"])
        paint_tier_cell(canvas, avg_x, py + 12, avg_w, ROW_H - 30, tier,
                        value_text=f"{td['xg']:.2f}",
                        value_font=fnt("Bold", 95))
    else:
        tier = get_tier_cs(td["cs"])
        paint_tier_cell(canvas, avg_x, py + 12, avg_w, ROW_H - 30, tier,
                        value_text=f"{td['cs']:.1f}%",
                        value_font=fnt("Bold", 95))

for i in range(20):
    py = START_Y + i * ROW_H
    draw_row_content(xg_ranking[i], i + 1, START_X_LEFT_TABLE,  py, True)
    draw_row_content(cs_ranking[i], i + 1, START_X_RIGHT_TABLE, py, False)

# ── 8. FOOTER ────────────────────────────────────────────────────────────
draw.text((START_X_LEFT_TABLE, H - 150),
          "METHOD: Poisson Expected Goals & Probability of ≥1 Clean Sheet | Ranked 1–20",
          font=fnt("Regular", 60), fill="#00E5FF")

# ── 9. SAVE ──────────────────────────────────────────────────────────────
out_path = os.path.join(PNG_DIR, "Slide_7_All_20_Teams.png")
canvas.convert("RGB").save(out_path, "PNG")
print(f"✅ Slide 7 saved → {out_path}")

print("\n" + "=" * 60)
print(" 🟣 CELL 9 COMPLETE: SLIDE 7 ALL 20 TEAMS READY ")
print("=" * 60)

Initializing Slide 7: Full 20-Team Target Tables...
⚠️ 24.png not found. Falling back to solid canvas.
✅ Slide 7 saved → /content/drive/MyDrive/Vortex_Final/png/Slide_7_All_20_Teams.png

 🟣 CELL 9 COMPLETE: SLIDE 7 ALL 20 TEAMS READY 


In [ ]:
# <<< START REPLACEMENT (Cell 9a) <<<
# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 9a: Slide 7a — Injury, Suspension & Minute Risk
High-density pro-list format prioritizing highly owned / recently flagged assets.
Adjusted to PERFECTLY fit 1080p Safe Zones (Mapped 4x for 8K Pipeline).
"""

import os
print("Initializing Slide 7a: High-Density Injury & Suspension Report...")

import textwrap
from datetime import datetime as _dt_inj
from PIL import Image, ImageDraw

PNG_DIR      = os.path.join(GD_ROOT, "png")
ELEMENTS_DIR = os.path.join(GD_ROOT, "elements")
BG_PATH      = os.path.join(ELEMENTS_DIR, "1.png")
os.makedirs(PNG_DIR, exist_ok=True)

_nxt = next((e for e in boot_data.get("events", []) if e.get("is_next")), None)
target_gw = _nxt["id"] if _nxt else "??"
gw_str    = f"GW{target_gw}"

# ── UPDATED SEVERITY FUNCTION (includes 75% doubt) ───────────────────────────
def _injury_severity(p):
    status = p.get('status', 'a')
    chance = p.get('chance_of_playing_next_round')
    news   = (p.get('news') or '').strip()

    if status == 's':                       return (1, 'suspended')
    if status in ('i', 'u') or chance == 0: return (2, 'ruled_out')
    if chance is not None and chance <= 25: return (3, 'major_doubt')
    if chance is not None and chance <= 50: return (4, 'real_doubt')
    if chance is not None and chance <= 75: return (5, 'minor_doubt')
    if news and status not in ('a',):       return (6, 'returning')
    return (99, None)

# ── UPDATED POOL LOGIC: FPL Focal Style (Recency First, Then Ownership) ────────
_pool = []
for p in boot_data.get('elements', []):
    prio, tier = _injury_severity(p)
    if tier is None: continue

    # Must have actual news text
    news = (p.get('news') or '').strip()
    if not news: continue

    own  = float(p.get('selected_by_percent', 0) or 0)

    # Filter out absolute ghost players (nobody cares about < 1% ownership)
    if own < 1.0: continue

    # Grab the exact timestamp the FPL server updated this player's news
    # If missing, give it an old date so it falls to the bottom
    news_added = p.get('news_added') or "1970-01-01T00:00:00.000000Z"

    _pool.append({
        'player': p,
        'tier': tier,
        'prio': prio,
        'own': own,
        'news_added': news_added
    })

# Focal's Secret 2.0: Sort PRIMARILY by how recently the news dropped (Newest First),
# and SECONDARILY by Ownership % (Highest First) if the news dropped on the same day.
_pool.sort(key=lambda x: (x['news_added'], x['own']), reverse=True)

# Grab the top 14 most relevant flagged players
top_flags_raw = _pool[:27]

pos_map = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}
display_players = []

for item in top_flags_raw:
    p      = item['player']
    tid    = p['team']
    team   = next((t for t in boot_data['teams'] if t['id'] == tid), None)
    chance = p.get('chance_of_playing_next_round')
    status = p.get('status', 'a')
    severity = 'RED' if (status in ('i', 's', 'u') or chance == 0 or
                         (chance is not None and chance <= 25)) else 'YELLOW'

    display_players.append({
        'code':       p['code'],
        'name':       p['web_name'],
        'team_code':  team['code']         if team else 0,
        'team_name':  team['name'].upper() if team else 'UNKNOWN',
        'status':     status,
        'chance':     chance if chance is not None else 0,
        'news':       (p.get('news') or 'No update.').upper(),
        'severity':   severity,
        'sel':        item['own'],
        'tier':       item['tier'],
        'pos':        pos_map.get(p.get('element_type', 3), "MID"),
        'price':      p.get('now_cost', 0) / 10.0
    })

globals()['s7a_locked_players'] = display_players

print(f"   Locked High-Priority Risks ({len(display_players)} players):")
for d in display_players:
    print(f"      [{d['tier']:<11}] {d['name']:<14} {d['team_name']:<20} "
          f"{d['chance']}%  ({d['sel']:.1f}% own)")

# ── CANVAS SETUP ──────────────────────────────────────────────────────────────
canvas = Image.new("RGBA", (W, H), (15, 5, 20, 255))
if os.path.exists(BG_PATH):
    try:
        bg = Image.open(BG_PATH).convert("RGBA").resize((W, H), Image.LANCZOS)
        canvas.alpha_composite(bg)
    except Exception:
        pass

overlay = Image.new("RGBA", (W, H), (0, 0, 0, 0))
ov_draw = ImageDraw.Draw(overlay)
draw    = ImageDraw.Draw(canvas)

C_RED    = "#FF1744"
C_YELLOW = "#FFEA00"
C_WHITE  = "#FFFFFF"
C_CYAN   = "#00E5FF"

def draw_fpl_triangle(dx, dy, severity):
    color  = C_RED if severity == "RED" else C_YELLOW
    points = [(dx, dy - 30), (dx - 35, dy + 25), (dx + 35, dy + 25)]
    ov_draw.polygon(points, fill=color)
    ex_font = fnt("Bold", 50)
    ex_box  = draw.textbbox((0, 0), "!", font=ex_font)
    ov_draw.text((dx - ((ex_box[2] - ex_box[0]) // 2), dy - 20),
                 "!", font=ex_font, fill="#000000")

def ctxt(d, cx, cy, text, font_obj, fill_col):
    bbox = d.textbbox((0, 0), text, font_obj)
    d.text((cx - (bbox[2] - bbox[0]) // 2,
            cy - (bbox[3] - bbox[1]) // 2),
           text, font=font_obj, fill=fill_col)

# ── BRANDING & HEADERS (Mapped to Safe Heading Area) ─────────────────────────
try:
    logo_img = Image.open(os.path.join(ELEMENTS_DIR, "logo.png")).convert("RGBA").resize((580, 572), Image.LANCZOS)
    canvas.paste(logo_img, (56, 20), logo_img)
except Exception: pass

# 🔴 FPL VORTEX text and Golden/Yellow line have been completely removed
# draw.text((712, 148), "FPL VORTEX", font=fnt("Bold", 200), fill="#FFFFFF")
# draw.line([(0, 528), (6684, 528)], fill="#FFD700", width=12)

has_inj = any(p['status'] != 's' for p in display_players)
has_sus = any(p['status'] == 's' for p in display_players)

if has_inj and has_sus: title_str = "INJURY & SUSPENSION REPORT"
elif has_sus:           title_str = "SUSPENSION REPORT"
else:                   title_str = "INJURY REPORT"

title_x = 1880 + (5120 - 1880) // 2
main_y = 260
sub_y  = 420

# 1. Main Title with RED FILL Background
main_font = fnt("Bold", 140)
main_bbox = draw.textbbox((0,0), title_str, font=main_font)
text_w = main_bbox[2] - main_bbox[0]
text_h = main_bbox[3] - main_bbox[1]

# Draw Red background pill (Position unchanged)
pad_x, pad_y = 80, 45  # Made padding slightly larger to ensure it covers text edges
draw.rounded_rectangle(
    [title_x - text_w // 2 - pad_x, main_y - text_h // 2 - pad_y,
     title_x + text_w // 2 + pad_x, main_y + text_h // 2 + pad_y],
    radius=30, fill=C_RED
)

# 🔴 Shift the text slightly UP inside the box (subtracting 25 pixels)
text_y_shift = 25
ctxt(draw, title_x, main_y - text_y_shift, title_str, main_font, C_WHITE)

# 2. Subtitle
ctxt(draw, title_x, sub_y, "LIVE FLAGS: RECENCY + SEVERITY RANKED", fnt("Bold", 80), C_WHITE)

# Top Right Tag (GW)
draw.text((W - 800, 150), gw_str, font=fnt("Bold", 120), fill=C_WHITE)

import re

# ── PRO LIST LAYOUT (3 Columns x 9 Rows = 27 Players) ────────────────────────
GRID_START_X = 100
GRID_START_Y = 800
BOX_W        = 2350
BOX_H        = 320
GAP_X        = 100
GAP_Y        = 50

def draw_col_hdrs(start_x):
    # Removed the middle STATUS header and shifted UPDATE to the left
    draw.text((start_x + 310,  GRID_START_Y - 90), "PLAYER", font=fnt("Bold", 65), fill="#888888")
    draw.text((start_x + 1250, GRID_START_Y - 90), "UPDATE", font=fnt("Bold", 65), fill="#888888")

# Draw headers for up to 3 columns dynamically
num_cols = min(3, (len(display_players) + 8) // 9)
for c in range(num_cols):
    draw_col_hdrs(GRID_START_X + c * (BOX_W + GAP_X))

for i, p in enumerate(display_players):
    col = i // 9
    row = i % 9
    bx  = GRID_START_X + col * (BOX_W + GAP_X)
    by  = GRID_START_Y + row * (BOX_H + GAP_Y)
    cy  = by + (BOX_H // 2)

    # Box Background
    row_color = (30, 35, 45, 180) if i % 2 == 0 else (20, 25, 35, 180)
    draw.rounded_rectangle([bx, by, bx + BOX_W, by + BOX_H], radius=20, fill=row_color)

    # Player Image
    try:
        photo = get_player_photo(p["code"], p["name"], size=(250, 300))
        canvas.paste(photo, (bx + 20, by + 10), photo)
    except Exception:
        pass

    # ── STATUS TEXT SETUP ──
    stat_color = C_RED if p["severity"] == "RED" else C_YELLOW
    stat_text  = ("SUSPENDED"   if p['status'] == 's'
                  else "RULED OUT" if p['chance'] == 0
                  else f"{p['chance']}%")
    stat_font  = fnt("Bold", 85)
    stat_width = int(draw.textlength(stat_text, font=stat_font))

    # ── PLAYER NAME SETUP (Auto-shrinks to fit Name + Status) ──
    name_str  = p["name"].upper()
    name_font = fnt("Bold", 100)

    # Ensure Name + Gap (40px) + Status fits within the left half of the box (max 900px wide)
    while (draw.textlength(name_str, font=name_font) + 40 + stat_width) > 900 and name_font.size > 50:
        name_font = fnt("Bold", name_font.size - 2)

    # Draw Player Name
    draw.text((bx + 310, by + 40), name_str, font=name_font, fill=C_WHITE)

    # Draw Status perfectly placed 40 pixels after the name
    name_width = int(draw.textlength(name_str, font=name_font))
    draw.text((bx + 310 + name_width + 40, by + 50), stat_text, font=stat_font, fill=stat_color)

    # Team & Price info
    draw.text((bx + 310, by + 190), p["team_name"], font=fnt("Bold", 65), fill=C_YELLOW)
    pos_price = f"{p['pos']} | £{p['price']:.1f}M"
    draw.text((bx + 680, by + 190), pos_price, font=fnt("Regular", 65), fill=C_CYAN)

    # Clean the News Update Text
    raw_news = p["news"].upper()
    clean_news = re.sub(r"\s*-\s*\d+%\s*CHANCE\s*OF\s*PLAYING", "", raw_news)
    clean_news = clean_news.replace("EXPECTED BACK", "EXP BACK")
    if not clean_news.strip():
        clean_news = "UNKNOWN"

    # Shifted to the left (1250) and widened (width=32) to use the new free space
    news_lines  = textwrap.wrap(clean_news, width=32)
    line_h      = 65
    total_news_h = len(news_lines[:2]) * line_h
    start_news_y = cy - (total_news_h // 2) - 10

    for l_idx, line in enumerate(news_lines[:2]):
        draw.text((bx + 1250, start_news_y + (l_idx * line_h)),
                  line, font=fnt("Regular", 60), fill=C_WHITE)

    # Warning Triangle (Far right)
    color  = C_RED if p["severity"] == "RED" else C_YELLOW
    dx, dy = bx + 2200, cy
    points = [(dx, dy - 45), (dx - 50, dy + 35), (dx + 50, dy + 35)]
    ov_draw.polygon(points, fill=color)
    ex_font = fnt("Bold", 70)
    ex_box  = draw.textbbox((0, 0), "!", font=ex_font)
    ov_draw.text((dx - ((ex_box[2] - ex_box[0]) // 2), dy - 35),
                 "!", font=ex_font, fill="#000000")

# ── COMPOSITE & SAVE ──
canvas = Image.alpha_composite(canvas, overlay)
out_path = os.path.join(PNG_DIR, "Slide_7a_Injury_MinuteRisk.png")
if os.path.exists(out_path): os.remove(out_path)
canvas.convert("RGB").save(out_path, "PNG")
print(f"✅ Slide 7a saved → {out_path}")

Initializing Slide 7a: High-Density Injury & Suspension Report...
   Locked High-Priority Risks (24 players):
      [real_doubt ] Alderete       SUNDERLAND           50%  (4.9% own)
      [ruled_out  ] Stach          LEEDS                0%  (1.7% own)
      [minor_doubt] Tonali         NEWCASTLE            75%  (1.0% own)
      [ruled_out  ] Richards       CRYSTAL PALACE       0%  (2.2% own)
      [ruled_out  ] Mitoma         BRIGHTON             0%  (1.9% own)
      [suspended  ] Andersen       FULHAM               0%  (2.8% own)
      [ruled_out  ] L.Miley        NEWCASTLE            0%  (1.7% own)
      [real_doubt ] Šeško          MAN UTD              50%  (4.6% own)
      [minor_doubt] Aina           NOTT'M FOREST        75%  (2.1% own)
      [ruled_out  ] Gudmundsson    LEEDS                0%  (2.5% own)
      [ruled_out  ] Xavi           SPURS                0%  (1.1% own)
      [ruled_out  ] Estêvão        CHELSEA              0%  (1.7% own)
      [ruled_out  ] Livramento    

In [ ]:

# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 10: Slide 8 — Targets & Avoids (FDR-Driven)
Generates empty base and locks exactly 5 targets / 5 avoids.
"""
import os
from PIL import Image, ImageDraw

print("Initializing Slide 8: Targets & Avoids...")

try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

PNG_DIR      = os.path.join(GD_ROOT, "png")
ELEMENTS_DIR = os.path.join(GD_ROOT, "elements")
os.makedirs(PNG_DIR, exist_ok=True)

# ── 1. FDR-DRIVEN PLAYER SELECTION (5 Targets, 5 Avoids) ──────────────
try:
    _pos_map_s8 = {1: 'GK', 2: 'DEF', 3: 'MID', 4: 'FWD'}

    def _s8_avg_fdr(team_id):
        fixes = [f for f in fix_data if f.get('event') == NEXT_GW and
                 (f['team_h'] == team_id or f['team_a'] == team_id)]
        if not fixes:
            return 5.0, []
        fdrs, opps = [], []
        for fx in fixes:
            ih = fx['team_h'] == team_id
            oid = fx['team_a'] if ih else fx['team_h']
            fdr_val = get_vortex_fdr(team_id, oid, ih)
            fdrs.append(fdr_val)
            opp = next((t for t in boot_data['teams'] if t['id'] == oid), None)
            if opp:
                # 🔴 FIX: Added FDR number to string directly (e.g., BUR[H]-2)
                opps.append(f"{opp['short_name']} [{'H' if ih else 'A'}]-{int(round(fdr_val))}")
        return sum(fdrs) / len(fdrs), opps, len(fixes) > 1

    _tgt_pool  = {k: [] for k in ('GK', 'DEF', 'MID', 'FWD')}
    _avd_pool  = {k: [] for k in ('GK', 'DEF', 'MID', 'FWD')}

    for _el in boot_data['elements']:
        if _el.get('status') not in ('a', 'd'):
            continue
        _ep  = float(_el.get('ep_next') or 0)
        _own = float(_el.get('selected_by_percent') or 0)
        _tid = _el['team']
        _pos = _pos_map_s8.get(_el.get('element_type', 3), 'MID')

        _gw_fixes = [f for f in fix_data if f.get('event') == NEXT_GW and
                     (f['team_h'] == _tid or f['team_a'] == _tid)]
        _is_bgw = len(_gw_fixes) == 0
        _is_dgw = len(_gw_fixes) > 1

        if _is_bgw:
            _avg_fdr_val, _opps = 5.0, ['BLANK']
        else:
            _fdr_result = _s8_avg_fdr(_tid)
            _avg_fdr_val = _fdr_result[0]
            _opps        = _fdr_result[1]

        # 🔴 FIX: Assign FDR Color string for animation cell to use as box fill
        if _avg_fdr_val <= 2.5: fdr_col = "#00FF87"
        elif _avg_fdr_val <= 3.5: fdr_col = "#FFD700"
        else: fdr_col = "#FF1744"

        _base = {
            'code':        _el['code'],
            'name':        _el['web_name'],
            'web_name':    _el['web_name'],
            'sel':         _own,
            'ep_next':     _ep,
            'now_cost':    _el.get('now_cost', 0),
            'element_type': _el.get('element_type', 3),
            'locked_xpts': round(_ep * (1.8 if _is_dgw else 1.0), 2),
            'next_fdr':    round(_avg_fdr_val, 2),
            'next_opp':    ' + '.join(_opps),
            'fdr_color':   fdr_col,
            'is_dgw':      _is_dgw,
            'is_bgw':      _is_bgw,
            'pos':         _pos,
            'team_id':     _tid,
            'team':        _tid,
        }

        if not _is_bgw and _ep >= 1.0:
            _tgt_score = _ep * (1.8 if _is_dgw else 1.0) * (2.0 - _avg_fdr_val / 5.0)
            _tgt_pool[_pos].append(dict(_base, score=_tgt_score))

        if (_is_bgw or _avg_fdr_val >= 3.5) and _own >= 3.0:
            _avd_score = _avg_fdr_val * _own / max(_ep, 0.1) * (3.0 if _is_bgw else 1.0)
            _avd_pool[_pos].append(dict(_base, score=_avd_score))

    for _p in (_tgt_pool, _avd_pool):
        for _k in _p:
            _p[_k].sort(key=lambda x: x['score'], reverse=True)

    # 🔴 FIX: Exactly 5 Target Players to prevent bottom overflow
    target_players = (
        _tgt_pool['GK'][:1]  +
        _tgt_pool['DEF'][:1] +
        _tgt_pool['MID'][:2] +
        _tgt_pool['FWD'][:1]
    )
    # 🔴 FIX: Exactly 5 Avoid Players
    avoid_players = (
        _avd_pool['DEF'][:2] +
        _avd_pool['MID'][:2] +
        _avd_pool['FWD'][:1]
    )

    globals()['target_players'] = target_players
    globals()['avoid_players']  = avoid_players
    print(f"✅ Slide 8 player selection locked — "
          f"{len(target_players)} targets, {len(avoid_players)} avoids")
except Exception as _e_s8_sel:
    print(f"⚠️ Slide 8 player selection failed: {_e_s8_sel}")
    target_players = globals().get('top_in', [])[:5]
    avoid_players  = globals().get('top_out', [])[:5]

# ── 2. BUILD EMPTY STRUCTURAL BASE ───────────────────────────────────────
W, H    = 7680, 4320
BG_PATH = os.path.join(ELEMENTS_DIR, "1.png")

if os.path.exists(BG_PATH):
    canvas = Image.open(BG_PATH).convert("RGBA").resize((W, H), Image.LANCZOS)
else:
    canvas = Image.new("RGBA", (W, H), (10, 5, 25, 255))

draw = ImageDraw.Draw(canvas)

# Title
draw.text((3000, 150), "TARGETS & AVOIDS", font=fnt("Bold", 200), fill="#FFFFFF")

# 🔴 FIX: Top-right gameweek deadline tag
try:
    gw_name = f"GW{NEXT_GW}"
except NameError:
    gw_name = "GW??"
gw_font = fnt("Bold", 120)
gw_bbox = draw.textbbox((0, 0), gw_name, font=gw_font)
draw.text((W - (gw_bbox[2] - gw_bbox[0]) - 200, 200), gw_name, font=gw_font, fill="#FFFF00")

# 🔴 FIX: Footer
footer_text = "Data Source: Official FPL API | Vortex xPTS Model"
footer_font = fnt("Regular", 80)
fb = draw.textbbox((0, 0), footer_text, font=footer_font)
draw.text((W - (fb[2] - fb[0]) - 300, H - 200), footer_text, font=footer_font, fill="#00FF87")

empty_path = os.path.join(PNG_DIR, "Slide_8_Empty.png")
if os.path.exists(empty_path): os.remove(empty_path)
canvas.convert("RGB").save(empty_path, format="PNG")
print(f"✅ Slide 8 Empty Base Saved: {empty_path}")

Initializing Slide 8: Targets & Avoids...
✅ Slide 8 player selection locked — 0 targets, 5 avoids
✅ Slide 8 Empty Base Saved: /content/drive/MyDrive/Vortex_Final/png/Slide_8_Empty.png


In [ ]:
# <<< START REPLACEMENT (Cell 11+12 - Squad Review & Plan) <<<
# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 11+12: Slide 9 (GW Review) + Slide 10 (Plan) Squad Data
Saves cutouts to player_cards/squad_frames + plan_frames
Generates EMPTY base PNGs that Cell 17 animates cards onto.
"""

import os, re, io, requests, unicodedata, shutil
from PIL import Image, ImageDraw, ImageFont

print("🟣 Cell 11+12 — Squad Review (Slide 9) + Plan (Slide 10)")

if 'boot_data' not in globals() or 'fix_data' not in globals():
    raise RuntimeError("Run Cell 2 first.")
if 'CURR_GW' not in globals() or 'NEXT_GW' not in globals():
    raise RuntimeError("Run Cell 2 first.")

try:    GD_ROOT
except: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

CARDS_DIR    = os.path.join(GD_ROOT, "player_cards")
SQUAD_FRAMES = os.path.join(CARDS_DIR, "squad_frames")
PLAN_FRAMES  = os.path.join(CARDS_DIR, "plan_frames")
PHOTOS_DIR   = os.path.join(GD_ROOT, "players")
ELEM_DIR     = os.path.join(GD_ROOT, "elements")
PNG_DIR      = os.path.join(GD_ROOT, "png")

import shutil
# 🔴 FIX: Force clear old cached images before recreating folders to stop "ghosting"
if os.path.exists(SQUAD_FRAMES): shutil.rmtree(SQUAD_FRAMES)
if os.path.exists(PLAN_FRAMES): shutil.rmtree(PLAN_FRAMES)

os.makedirs(SQUAD_FRAMES, exist_ok=True)
os.makedirs(PLAN_FRAMES,  exist_ok=True)
os.makedirs(PNG_DIR,      exist_ok=True)

HEADERS    = {"User-Agent": "Mozilla/5.0"}
MANAGER_ID = globals().get('MANAGER_ID', 664987)

# ── HELPERS ───────────────────────────────────────────────────────────────

def _strip_accents(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ASCII", "ignore").decode()

def _next_opp(team_id, gw):
    fx = [f for f in fix_data if f.get('event') == gw
          and (f['team_h'] == team_id or f['team_a'] == team_id)]
    if not fx:
        return "BLANK"

    labels = []
    for f in fx:
        is_h   = (f['team_h'] == team_id)
        opp_id = f['team_a'] if is_h else f['team_h']
        opp    = next((t for t in boot_data['teams'] if t['id'] == opp_id), None)
        if opp:
            labels.append(f"{opp['name']} {'(H)' if is_h else '(A)'}")
    return " / ".join(labels)

def _avg_fdr(team_id, start_gw, window=5):
    vals = []
    for gw in range(start_gw, start_gw + window):
        for f in fix_data:
            if f.get('event') != gw:
                continue
            if f['team_h'] == team_id:
                vals.append(f['team_h_difficulty'])
            elif f['team_a'] == team_id:
                vals.append(f['team_a_difficulty'])
    return round(sum(vals) / len(vals), 2) if vals else 3.0

def _build_dict(el, pick_position, is_cap=False, is_vc=False, pts=0, multiplier=1):
    team = next(t for t in boot_data['teams'] if t['id'] == el['team'])
    return {
        'id':            el['id'],
        'code':          el['code'],
        'name':          el['web_name'],
        'web_name':      el['web_name'],
        'full_name':     f"{el.get('first_name','')} {el.get('second_name','')}".strip(),
        'first_name':    el.get('first_name', ''),
        'second_name':   el.get('second_name', ''),
        'pos':           el['element_type'],
        'element_type':  el['element_type'],
        'team_id':       el['team'],
        'team_short':    team['name'],
        'team_code':     team['code'],
        'cost':          el['now_cost'] / 10.0,
        'ep_next':       float(el.get('ep_next') or 0.0),
        'next_opp':      _next_opp(el['team'], NEXT_GW),
        'avg_fdr':       _avg_fdr(el['team'], NEXT_GW, window=5),
        'is_cap':        is_cap,
        'is_vc':         is_vc,
        'pick_position': pick_position,
        'pts':           pts,
        'multiplier':    multiplier,
    }

def _fetch_picks(gw):
    url = f"https://fantasy.premierleague.com/api/entry/{MANAGER_ID}/event/{gw}/picks/"
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return None

def _picks_to_squad(payload, ref_gw, auto_sub=False):
    _el_by_id = {el['id']: el for el in boot_data['elements']}

    # ── LIVE CHIP DETECTION FROM API PAYLOAD ─────────────────────────
    active_chip = payload.get('active_chip')  # e.g. '3xc', 'freehit', 'wildcard', 'bboost'

    # ── LIVE POINTS MAP FROM API (multiplier included) ────────────────
    pts_map = {}
    mult_map = {}
    for pick in payload.get('picks', []):
        el = _el_by_id.get(pick['element'])
        if el:
            # API returns correct multiplier: 1 normal, 2 captain, 3 TC captain
            pts_map[pick['element']]  = int(el.get('event_points') or 0)
            mult_map[pick['element']] = pick.get('multiplier', 1)

    squad = []
    for pick in payload.get('picks', []):
        el = _el_by_id.get(pick['element'])
        if el is None:
            continue
        squad.append(_build_dict(
            el,
            pick_position = pick['position'],
            is_cap        = pick.get('is_captain', False),
            is_vc         = pick.get('is_vice_captain', False),
            pts           = pts_map.get(pick['element'], 0),
            multiplier    = mult_map.get(pick['element'], 1),
        ))

    # ── GW REVIEW ENGINE (live data, no hardcoding) ───────────────────
    if not auto_sub:
        starters = sorted(
            [p for p in squad if p['pick_position'] <= 11],
            key=lambda x: (x['element_type'], -x['ep_next'])
        )
        bench = sorted(
            [p for p in squad if p['pick_position'] >= 12],
            key=lambda x: x['pick_position']
        )
        return starters, bench

    # ── GW PLAN ENGINE (mathematical optimization) ─────────────────────
    gks = [p for p in squad if p['element_type'] == 1]
    out = sorted([p for p in squad if p['element_type'] != 1],
                 key=lambda x: -x['ep_next'])

    gks.sort(key=lambda x: (x['avg_fdr'], -x['ep_next']))
    starters_list = [gks[0]] if gks else []
    bench_list    = gks[1:] if len(gks) > 1 else []

    defs = [p for p in out if p['element_type'] == 2]
    fwds = [p for p in out if p['element_type'] == 4]

    starting_outfield = []
    starting_outfield.extend(defs[:3])
    starting_outfield.extend(fwds[:1])

    rem_out = [p for p in out if p not in starting_outfield]
    starting_outfield.extend(rem_out[:6])
    bench_outfield = rem_out[6:]

    starters_list.extend(starting_outfield)
    bench_list.extend(bench_outfield)

    def _cap_score(p):
        ep  = float(p.get('ep_next', 0) or 0)
        own = float(p.get('selected_by_percent', 0) or 0)
        return ep * (1.0 + (own / 100.0))

    starters_list.sort(key=lambda x: -_cap_score(x))

    for i, p in enumerate(starters_list):
        p['is_cap'] = (i == 0)
        p['is_vc']  = (i == 1)

    for p in bench_list:
        p['is_cap'] = False
        p['is_vc']  = False

    starters = sorted(starters_list, key=lambda x: (x['element_type'], -x['ep_next']))
    bench    = sorted(bench_list,    key=lambda x: (0 if x['element_type'] == 1 else 1, -x['ep_next']))

    return starters, bench

# ── FONT + PHOTO + CUTOUT ─────────────────────────────────────────────────

def _get_font(size, style="Bold"):
    try:
        return ImageFont.truetype(os.path.join(ELEM_DIR, f"Roboto-{style}.ttf"), size)
    except Exception:
        return ImageFont.load_default()

# 🔴 FIX: Deleted redundant _fetch_photo. Routing through Cell 2 Global Cache.

# <<< START REPLACEMENT (Cell 11+12: _build_cutout) <<<
def _build_cutout(player, badge=None, show_pts=False):
    W, H = 1800, 2400
    img = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    d = ImageDraw.Draw(img)

    # 🔴 FIX: Route player photo through cache
    ph = get_player_photo(player['code'], player['web_name'], player.get('full_name', ''), size=(1400, 1700))
    img.paste(ph, ((W - 1400) // 2, 50), ph)

    try:
        t_code = player.get('team_code', 0)
        if t_code:
            # 🔴 FIX: Route team logo through cache
            logo_img = get_team_logo(t_code, size=(350, 350))
            img.paste(logo_img, (1350, 150), logo_img)
    except: pass

    name_sz = 420
    fnt_name = _get_font(name_sz, "Bold")
    name_str = player['web_name'].upper()
    name_str = re.sub(r'^[A-Z]\.\s*', '', name_str)

    nb = d.textbbox((0, 0), name_str, font=fnt_name)
    while (nb[2] - nb[0]) > (W - 150) and name_sz > 200:
        name_sz -= 20
        fnt_name = _get_font(name_sz, "Bold")
        nb = d.textbbox((0, 0), name_str, font=fnt_name)

    name_y = 1620
    nw, nh = (nb[2] - nb[0]) + 180, (nb[3] - nb[1]) + 90
    d.rounded_rectangle([(W-nw)//2, name_y-nh//2, (W+nw)//2, name_y+nh//2], radius=50, fill=(15, 10, 30, 240), outline="#00E5FF", width=8)
    d.text((W // 2, name_y), name_str, font=fnt_name, fill="white", anchor="mm")

    pill_y = 2100
    if show_pts:
        pts      = player.get('pts', 0)
        mult     = player.get('multiplier', 1)
        pill_txt = f"{pts*mult} PTS  ({pts}×{mult})" if mult > 1 else f"{pts} PTS"
        pill_col = ("#FFD700" if (pts * mult) >= 12 else "#00FF87" if (pts * mult) >= 6  else "#FF1744")
    else:
        pill_txt = f"xP {player.get('ep_next', 0):.1f}   FDR {player.get('avg_fdr', 3)}"
        pill_col = "#00FF87"

    stat_sz = 350
    fnt_stat = _get_font(stat_sz, "Bold")
    bb = d.textbbox((0, 0), pill_txt, font=fnt_stat)

    while (bb[2] - bb[0]) > (W - 100) and stat_sz > 120:
        stat_sz -= 15
        fnt_stat = _get_font(stat_sz, "Bold")
        bb = d.textbbox((0, 0), pill_txt, font=fnt_stat)

    pw, ph_ = (bb[2] - bb[0]) + 250, (bb[3] - bb[1]) + 120
    d.rounded_rectangle([(W-pw)//2, pill_y-ph_//2, (W+pw)//2, pill_y+ph_//2], radius=50, fill=(0, 0, 0, 255), outline=pill_col, width=10)
    d.text((W // 2, pill_y), pill_txt, font=fnt_stat, fill=pill_col, anchor="mm")

    if badge:
        # 🔴 PRO FIX: Array of case-sensitive fallbacks to defeat Colab/Linux strict casing
        _badge_file_map = {
            'C': ['c.png', 'C.png', 'c.PNG', 'C.PNG'],
            'V': ['vc.png', 'VC.png', 'vc.PNG', 'VC.PNG']
        }
        _badge_candidates = _badge_file_map.get(badge, [])
        _badge_size       = 380          # visible at pitch-card scale

        if _badge_candidates:
            _badge_full = None
            for cand in _badge_candidates:
                test_path = os.path.join(ELEM_DIR, cand)
                if os.path.exists(test_path):
                    _badge_full = test_path
                    break

            if _badge_full:
                try:
                    _bi = Image.open(_badge_full).convert("RGBA")
                    _bi = _bi.resize((_badge_size, _badge_size), Image.LANCZOS)
                    img.paste(_bi, (25, 25), _bi)   # left side of player image
                except Exception as _be:
                    print(f"⚠️ Badge load failed ({_badge_full}): {_be}")
                    _cfg = {'C': ("#FFD700","C","black"), 'V': ("#00E5FF","V","black")}
                    _bg, _lt, _fg = _cfg.get(badge, ("#FFFFFF","?","black"))
                    d.ellipse([25, 25, 405, 405], fill=_bg, outline="white", width=14)
                    d.text((215, 215), _lt, font=_get_font(500,"Bold"), fill=_fg, anchor="mm")
            else:
                print(f"⚠️ c.png / vc.png not found in elements — using letter fallback.")
                _cfg = {'C': ("#FFD700","C","black"), 'V': ("#00E5FF","V","black")}
                _bg, _lt, _fg = _cfg.get(badge, ("#FFFFFF","?","black"))
                d.ellipse([25, 25, 405, 405], fill=_bg, outline="white", width=14)
                d.text((215, 215), _lt, font=_get_font(500,"Bold"), fill=_fg, anchor="mm")
        else:
            # 'W' wildcard badge kept as letter
            _cfg = {'W': ("#B366FF","W","white")}
            _bg, _lt, _fg = _cfg.get(badge, ("#FFFFFF","?","black"))
            d.ellipse([25, 25, 405, 405], fill=_bg, outline="white", width=14)
            d.text((215, 215), _lt, font=_get_font(500,"Bold"), fill=_fg, anchor="mm")

    return img
# >>> END REPLACEMENT >>>

def _safe_fn(web_name):
    s = _strip_accents(web_name).strip()
    s = re.sub(r'^[A-Z]\.\s*', '', s)
    return s.replace(" ", "_").replace("/", "_")

def _clear_folder(folder):
    for f in os.listdir(folder):
        if f.endswith(".png"):
            try: os.remove(os.path.join(folder, f))
            except: pass

# ═══════════════════════════════════════════════════════════════════
# PART A — SLIDE 9 (CURR_GW REVIEW)
# ═══════════════════════════════════════════════════════════════════
print(f"\n  SLIDE 9 — GW{CURR_GW} REVIEW")

s9_payload = _fetch_picks(CURR_GW)
if s9_payload is None: raise RuntimeError(f"Cannot fetch picks for GW{CURR_GW}.")
globals()['s9_payload'] = s9_payload

s9_starters, s9_bench = _picks_to_squad(s9_payload, CURR_GW, auto_sub=False)
_clear_folder(SQUAD_FRAMES)
_idx = 0
for p in s9_starters:
    badge = 'C' if p['is_cap'] else 'V' if p['is_vc'] else None
    _build_cutout(p, badge=badge, show_pts=True).save(os.path.join(SQUAD_FRAMES, f"squad_{_idx:02d}_{_safe_fn(p['web_name'])}.png"))
    _idx += 1
for p in s9_bench:
    _build_cutout(p, show_pts=True).save(os.path.join(SQUAD_FRAMES, f"squad_{_idx:02d}_{_safe_fn(p['web_name'])}.png"))
    _idx += 1
print(f"  Saved {_idx} squad frames")

globals()['s9_starters']   = s9_starters
globals()['s9_bench']      = s9_bench
globals()['s9_squad_pool'] = s9_starters + s9_bench

# ═══════════════════════════════════════════════════════════════════
# PART B — SLIDE 10 (NEXT_GW PLAN)
# ═══════════════════════════════════════════════════════════════════
print(f"\n  SLIDE 10 — GW{NEXT_GW} PLAN")

s10_payload = None
source_gw   = None
for _gw in range(CURR_GW, 0, -1):
    payload = _fetch_picks(_gw)
    if not payload: continue
    if payload.get('active_chip') in ('freehit', 'wildcard'): continue
    s10_payload = payload
    source_gw   = _gw
    break

if s10_payload is None:
    s10_payload = s9_payload
    source_gw   = CURR_GW

s10_starters, s10_bench = _picks_to_squad(s10_payload, source_gw, auto_sub=True)
s10_squad_pool = s10_starters + s10_bench

_owned     = {p['code'] for p in s10_squad_pool}
_sorted_tr = sorted(boot_data['elements'], key=lambda e: int(e.get('transfers_in_event') or 0), reverse=True)
top_targets = []
for el in _sorted_tr:
    if el['code'] in _owned: continue
    if el.get('status') in ('u', 'i'): continue
    if int(el.get('transfers_in_event') or 0) <= 0: continue
    top_targets.append(_build_dict(el, pick_position=16 + len(top_targets)))
    if len(top_targets) >= 4: break

_clear_folder(PLAN_FRAMES)
_idx = 0
for p in s10_starters:
    badge = 'C' if p['is_cap'] else 'V' if p['is_vc'] else None
    _build_cutout(p, badge=badge, show_pts=False).save(os.path.join(PLAN_FRAMES, f"plan_{_idx:02d}_{_safe_fn(p['web_name'])}.png"))
    _idx += 1
for p in s10_bench:
    _build_cutout(p, show_pts=False).save(os.path.join(PLAN_FRAMES, f"plan_{_idx:02d}_{_safe_fn(p['web_name'])}.png"))
    _idx += 1
for p in top_targets:
    _build_cutout(p, badge=None, show_pts=False).save(os.path.join(PLAN_FRAMES, f"plan_{_idx:02d}_{_safe_fn(p['web_name'])}.png"))
    _idx += 1
print(f"  Saved {_idx} plan frames")

globals()['s10_starters']   = s10_starters
globals()['s10_bench']      = s10_bench
globals()['s10_squad_pool'] = s10_squad_pool
globals()['top_targets']    = top_targets

globals()['current_gw_pts'] = sum(p['pts'] * p['multiplier'] for p in s9_starters + s9_bench if p['multiplier'] > 0)
globals()['overall_pts']  = s9_payload.get('entry_history', {}).get('total_points', '---')
globals()['current_rank'] = s9_payload.get('entry_history', {}).get('overall_rank', 0)
globals()['prev_rank']    = s9_payload.get('entry_history', {}).get('rank_sort', globals()['current_rank'])

# <<< START REPLACEMENT (Bottom of Cell 11+12) <<<
# ═══════════════════════════════════════════════════════════════════
# EMPTY BASE PNGs — pitch backgrounds for Cell 17 animation
# ═══════════════════════════════════════════════════════════════════
W_VID, H_VID = 1920, 1080

def _make_pitch_base(title_text, out_path, subtitle=None):
    bg_squad = os.path.join(ELEM_DIR, "squad.png")
    if os.path.exists(bg_squad):
        base = Image.open(bg_squad).convert("RGBA").resize((W_VID, H_VID), Image.LANCZOS)
    else:
        base = Image.new("RGBA", (W_VID, H_VID), (10, 5, 25, 255))

    d = ImageDraw.Draw(base)
    title_font = _get_font(52, "Bold")

    if "PLAN" in title_text:
        d.text((1850, 90), title_text, font=title_font, fill="#FFD700", anchor="rt")

        # ── MOVED DOWNWARD FOR BETTER SPACING BALANCE ──────────────────
        TRANSFER_PLAN_Y = 350
        d.rectangle([60, TRANSFER_PLAN_Y, 600, TRANSFER_PLAN_Y + 80], fill="#FF1744")
        d.text((330, TRANSFER_PLAN_Y + 40), "TRANSFER PLAN", font=_get_font(45, "Bold"), fill="white", anchor="mm")
    else:
        d.text((1850, 50), title_text, font=title_font, fill="#FFD700", anchor="rt")

    if subtitle:
        sub_font = _get_font(45, "Bold")
        # ── MOVED UPWARD FOR BETTER ALIGNMENT ──────────────────
        d.text((160, 580), subtitle, font=sub_font, fill="#00FF87", anchor="ls")
        d.text((160, 650), "WATCH LIST", font=_get_font(65, "Bold"), fill="#FFD700", anchor="ls")

    base.convert("RGB").save(out_path, "PNG")

# 🔴 FIX: RESTORED EXECUTION CALLS TO GENERATE THE FILES
_make_pitch_base(f"GW{CURR_GW} REVIEW", os.path.join(PNG_DIR, "Slide_9_Base.png"))
_make_pitch_base(f"GW{NEXT_GW} MASTER PLAN", os.path.join(PNG_DIR, "Slide_10_Plan_Base.png"), subtitle=None)

print(f"\n✅ Cell 11+12 Complete")
print(f"   GW{CURR_GW} review: {len(s9_starters)+len(s9_bench)} squad frames")
print(f"   GW{NEXT_GW} plan:   {len(s10_starters)+len(s10_bench)+len(top_targets)} plan frames")

# >>> END REPLACEMENT >>>

🟣 Cell 11+12 — Squad Review (Slide 9) + Plan (Slide 10)

  SLIDE 9 — GW38 REVIEW
⚠️ c.png / vc.png not found in elements — using letter fallback.
⚠️ API 404 error for 118748: HTTP 403
⚠️ c.png / vc.png not found in elements — using letter fallback.
⚠️ API 404 error for 224117: HTTP 403
  Saved 15 squad frames

  SLIDE 10 — GW39 PLAN
⚠️ c.png / vc.png not found in elements — using letter fallback.
⚠️ c.png / vc.png not found in elements — using letter fallback.
  Saved 15 plan frames

✅ Cell 11+12 Complete
   GW38 review: 15 squad frames
   GW39 plan:   15 plan frames


In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 13: Slide 11 — Chip Strategy (Dynamic Global Usage & 2x Rule)
"""

import os
from PIL import Image, ImageDraw, ImageFont

print("Initializing Slide 11: Chip Strategy (Dynamic Global Usage)...")

# ── 1. PATHS ─────────────────────────────────────────────────────────────
try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

PNG_DIR      = os.path.join(GD_ROOT, "png")
ELEMENTS_DIR = os.path.join(GD_ROOT, "elements")
BG_PATH      = os.path.join(ELEMENTS_DIR, "24.png")
os.makedirs(PNG_DIR, exist_ok=True)

AGENDA_DIR = os.path.join(PNG_DIR, "slide11_agenda")
os.makedirs(AGENDA_DIR, exist_ok=True)
for f in os.listdir(AGENDA_DIR): os.remove(os.path.join(AGENDA_DIR, f))

# ── 2. CANVAS CONSTANTS ──────────────────────────────────────────────────
C_BG_PANEL = (15, 10, 30, 220)
C_GOLD, C_CYAN, C_RED, C_GREEN = "#FFD700", "#00E5FF", "#FF1744", "#00FF87"
C_WHITE, C_GREY = "#FFFFFF", "#888888"

if os.path.exists(BG_PATH):
    canvas = Image.open(BG_PATH).convert("RGBA").resize((W, H), Image.LANCZOS)
else:
    canvas = Image.new("RGBA", (W, H), (10, 5, 20, 255))

overlay = Image.new("RGBA", (W, H), (0, 0, 0, 0))
ov_draw = ImageDraw.Draw(overlay)

def draw_dynamic_centered(d, text, max_w, max_size, color, cx, cy, weight="Bold"):
    text = str(text).upper()
    f = max_size; fnt_o = fnt(weight, f)
    bb = d.textbbox((cx, cy), text, font=fnt_o, anchor="mm")
    while (bb[2]-bb[0]) > max_w and f > 30:
        f -= 5; fnt_o = fnt(weight, f)
        bb = d.textbbox((cx, cy), text, font=fnt_o, anchor="mm")
    d.text((cx, cy), text, font=fnt_o, fill=color, anchor="mm")

def draw_dynamic_left(d, text, max_w, max_size, color, x, y, weight="Bold"):
    text = str(text).upper()
    f = max_size; fnt_o = fnt(weight, f)
    bb = d.textbbox((x, y), text, font=fnt_o, anchor="lm")
    while (bb[2]-bb[0]) > max_w and f > 30:
        f -= 5; fnt_o = fnt(weight, f)
        bb = d.textbbox((x, y), text, font=fnt_o, anchor="lm")
    d.text((x, y), text, font=fnt_o, fill=color, anchor="lm")

# ── 3. DATA EXTRACTION (H1 VS H2 CHIPS) ──────────────────────────────────
CURR_GW = globals().get('CURR_GW', 38)

h1_chips = {"wildcard": 0, "freehit": 0, "bboost": 0, "3xc": 0}
h2_chips = {"wildcard": 0, "freehit": 0, "bboost": 0, "3xc": 0}

for ev in boot_data.get('events', []):
    gw = ev['id']
    for cp in ev.get('chip_plays', []):
        cname = cp['chip_name']
        if cname in h1_chips:
            if gw <= 19: h1_chips[cname] += cp['num_played']
            else:        h2_chips[cname] += cp['num_played']

total_mgrs = boot_data.get('total_players', 10000000)
if total_mgrs == 0: total_mgrs = 10000000

def get_pct(num): return (num / total_mgrs) * 100

h1_tot = sum(h1_chips.values())
h2_tot = sum(h2_chips.values())

# ── 4. HEADERS ───────────────────────────────────────────────────────────
draw_dynamic_centered(ov_draw, "CHIP STRATEGY: GLOBAL DEPLOYMENT", W - 400, 200, C_GOLD, W // 2, 220)
draw_dynamic_centered(ov_draw, "NEW RULE: 2X CHIPS PER SEASON (HALF 1 & HALF 2)", W - 400, 110, C_CYAN, W // 2, 410)

# ── 5. PANELS ────────────────────────────────────────────────────────────
P1_X1, P1_X2 = 150, 2400
P2_X1, P2_X2 = 2500, 4800
P3_X1, P3_X2 = 4900, 7530
P_Y1, P_Y2   = 780, 4100

# PANEL 1: FIRST HALF
ov_draw.rounded_rectangle([P1_X1, P_Y1, P1_X2, P_Y2], radius=40, fill=C_BG_PANEL, outline=C_CYAN, width=8)
ov_draw.rectangle([P1_X1 + 40, P_Y1 + 40, P1_X2 - 40, P_Y1 + 220], fill=C_CYAN)
draw_dynamic_centered(ov_draw, "FIRST HALF (GW1 - GW19)", P1_X2 - P1_X1 - 100, 110, "#000000", (P1_X1+P1_X2)//2, P_Y1 + 130)

# PANEL 2: SECOND HALF
ov_draw.rounded_rectangle([P2_X1, P_Y1, P2_X2, P_Y2], radius=40, fill=C_BG_PANEL, outline=C_GREEN, width=8)
ov_draw.rectangle([P2_X1 + 40, P_Y1 + 40, P2_X2 - 40, P_Y1 + 220], fill=C_GREEN)
draw_dynamic_centered(ov_draw, "SECOND HALF (GW20 - GW38)", P2_X2 - P2_X1 - 100, 110, "#000000", (P2_X1+P2_X2)//2, P_Y1 + 130)

# PANEL 3: STRATEGY (DYNAMIC)
if CURR_GW <= 19:
    p3_title = "FIRST HALF LENS"
    strat_text = [
        "1. PRESERVE CHIPS FOR LATER DGWS.",
        "2. PLAY WC1 TO FIX STRUCTURE.",
        "3. BUILD SQUAD VALUE EARLY."
    ]
elif CURR_GW < 38:
    p3_title = "SECOND HALF LENS"
    comp_txt = "H2 CHIP DEPLOYMENT OUTPACES H1." if h2_tot > h1_tot else "H1 USAGE STILL LEADS H2."
    strat_text = [
        comp_txt,
        "1. ALIGN FREE HIT WITH BLANKS.",
        "2. SAVE BENCH BOOST FOR DGW.",
        "3. PATIENCE IS RANK EQUITY."
    ]
else:
    p3_title = "SEASON REVIEW"
    comp_txt = "H2 CHIP DEPLOYMENT DOMINATED." if h2_tot > h1_tot else "EARLY H1 USAGE DOMINATED."
    strat_text = [
        comp_txt,
        "PATIENCE REWARDED IN LATE DGWS.",
        "FINAL DAY MOVES DICTATE RANK."
    ]

ov_draw.rounded_rectangle([P3_X1, P_Y1, P3_X2, P_Y2], radius=40, fill=C_BG_PANEL, outline=C_GOLD, width=8)
ov_draw.rectangle([P3_X1 + 40, P_Y1 + 40, P3_X2 - 40, P_Y1 + 220], fill=C_GOLD)
draw_dynamic_centered(ov_draw, p3_title, P3_X2 - P3_X1 - 100, 110, "#000000", (P3_X1+P3_X2)//2, P_Y1 + 130)

chips_def = [
    ("WC.png", "WILDCARD", "wildcard", C_WHITE),
    ("BB.png", "BENCH BOOST", "bboost", C_CYAN),
    ("FH.png", "FREE HIT", "freehit", C_RED),
    ("TC.png", "TRIPLE CAPTAIN", "3xc", C_GOLD)
]

for i, (fn, title, ckey, col) in enumerate(chips_def):
    y_pos = P_Y1 + 450 + (i * 700)
    path = os.path.join(ELEMENTS_DIR, fn)
    img_c = None
    if os.path.exists(path):
        img_c = Image.open(path).convert("RGBA").resize((400, 400), Image.LANCZOS)

    p1_pct = get_pct(h1_chips[ckey])
    if img_c: overlay.paste(img_c, (P1_X1 + 150, y_pos - 100), img_c)
    ov_draw.text((P1_X1 + 650, y_pos + 40), title, font=fnt("Bold", 100), fill=col, anchor="ls")
    ov_draw.text((P1_X1 + 650, y_pos + 160), f"PLAYED: {p1_pct:.1f}%", font=fnt("Bold", 100), fill=C_WHITE, anchor="ls")

    p2_pct = get_pct(h2_chips[ckey])
    if img_c: overlay.paste(img_c, (P2_X1 + 150, y_pos - 100), img_c)
    ov_draw.text((P2_X1 + 650, y_pos + 40), title, font=fnt("Bold", 100), fill=col, anchor="ls")
    ov_draw.text((P2_X1 + 650, y_pos + 160), f"PLAYED: {p2_pct:.1f}%", font=fnt("Bold", 100), fill=C_WHITE, anchor="ls")

# ── 6. COMPOSITE & SAVE BASE ─────────────────────────────────────────────
canvas = Image.alpha_composite(canvas, overlay)
output_path = os.path.join(PNG_DIR, "Slide_11_Chip_Strategy.png")
canvas.convert("RGB").save(output_path, "PNG")

# ── 7. GENERATE TRANSPARENT OVERLAYS FOR FFmpeg ANIMATION ────────────────
sy = P_Y1 + 550
for idx, line in enumerate(strat_text):
    if not line.strip(): continue
    ovr = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    odraw = ImageDraw.Draw(ovr)

    cx, cy = P3_X1 + 150, sy
    r = 30
    color = C_CYAN if (CURR_GW > 19 and idx == 0) else C_WHITE

    odraw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=color)
    draw_dynamic_left(odraw, line, P3_X2 - cx - 150, 95, color, cx + 80, cy)

    ovr.save(os.path.join(AGENDA_DIR, f"chip_{idx:02d}.png"), "PNG")
    sy += 300

print(f"✅ Slide 11 Generated with Dynamic GW{CURR_GW} Logic")

Initializing Slide 11: Chip Strategy (Dynamic Global Usage)...
✅ Slide 11 Generated with Dynamic GW38 Logic


In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 14: Slide 1 — Intro (Empty Base + Per-Item Agenda Overlays for Animation)
"""

import os
from datetime import datetime, timedelta
from PIL import Image, ImageDraw, ImageFont

print("Initializing Slide 1: Intro Empty Base + Agenda Overlays...")

try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

PNG_DIR             = os.path.join(GD_ROOT, "png")
ELEMENTS_DIR        = os.path.join(GD_ROOT, "elements")
AGENDA_OVERLAYS_DIR = os.path.join(PNG_DIR, "slide1_agenda")
os.makedirs(PNG_DIR, exist_ok=True)
os.makedirs(AGENDA_OVERLAYS_DIR, exist_ok=True)

BG_PATH     = os.path.join(ELEMENTS_DIR, "1.png")
FALLBACK_BG = os.path.join(ELEMENTS_DIR, "7.png")


C_GOLD, C_WHITE, C_CYAN, C_GREEN, C_BG_PANEL = "#FFD700", "#FFFFFF", "#00E5FF", "#00FF87", (15, 10, 30, 220)
C_MAGENTA, C_ORANGE = "#FF00FF", "#FF4500"

def get_font(weight="Bold", size=50):
    try:    return ImageFont.truetype(os.path.join(ELEMENTS_DIR, f"Roboto-{weight}.ttf"), size)
    except: return ImageFont.load_default()

def draw_dynamic_centered_text(d, text, max_w, max_size, color, cx, cy, weight="Bold"):
    text = str(text).upper()
    f = max_size
    fnt_o = get_font(weight, f)
    bb = d.textbbox((cx, cy), text, font=fnt_o, anchor="mm")
    while (bb[2]-bb[0]) > max_w and f > 30:
        f -= 5; fnt_o = get_font(weight, f)
        bb = d.textbbox((cx, cy), text, font=fnt_o, anchor="mm")
    d.text((cx, cy), text, font=fnt_o, fill=color, anchor="mm")

def draw_dynamic_left_aligned_text(d, text, max_w, max_size, color, x, y, weight="Bold"):
    text = str(text).upper()
    f = max_size
    fnt_o = get_font(weight, f)
    bb = d.textbbox((x, y), text, font=fnt_o, anchor="lm")
    while (bb[2]-bb[0]) > max_w and f > 30:
        f -= 5; fnt_o = get_font(weight, f)
        bb = d.textbbox((x, y), text, font=fnt_o, anchor="lm")
    d.text((x, y), text, font=fnt_o, fill=color, anchor="lm")

# Gameweek info
try:
    _nxt = next((e for e in boot_data.get("events", []) if e.get("is_next")), None)
except NameError:
    _nxt = None
if _nxt:
    target_gw = _nxt["id"]
    gw_name   = _nxt["name"].upper().replace("GAMEWEEK ", "GW")
    try:
        _utc   = datetime.strptime(_nxt["deadline_time"], "%Y-%m-%dT%H:%M:%SZ")
        est_dt = _utc - timedelta(hours=4)
        deadline_str = est_dt.strftime("%A, %B %d | %H:%M EST").upper()
    except Exception:
        deadline_str = _nxt["deadline_time"].upper()
else:
    target_gw, gw_name, deadline_str = 1, "GW??", "UNKNOWN DEADLINE"

_agenda_mapping = {
    "market_trends":   ("MARKET TRENDS",       C_CYAN),
    "sword_shield":    ("SWORD & SHIELD",      C_GOLD),
    "fdr_map":         ("FIXTURE TICKER",      C_CYAN),
    "projected_goals": ("ATTACKING XG",        C_MAGENTA),
    "cs_odds":         ("CLEAN SHEET ODDS",    C_GREEN),
    "injury_report":   ("INJURY ALERT",        C_ORANGE),
    "all_20_teams":    ("20 TEAM PROJECTIONS", C_CYAN),
    "targets_avoids":  ("TARGETS & AVOIDS",    C_MAGENTA),
    "gw_review":       ("SQUAD REVIEW",        C_CYAN),
    "master_plan":     ("TEAM REVEAL",         C_GREEN),
    "chip_strategy":   ("CHIP STRATEGY",       C_GOLD),
}

agenda_items = []
# 🔴 FIX: Dynamically read the modules chosen by the Newsroom OS Planner
for mod in globals().get('final_modules', []):
    if mod in _agenda_mapping:
        agenda_items.append(_agenda_mapping[mod])
        if len(agenda_items) >= 5: break # Cap visually at 5 items

if not agenda_items:
    agenda_items = [("GW PREVIEW", C_CYAN), ("TEAM REVEAL", C_GREEN)]

globals()['agenda_items_s1'] = agenda_items

# Canvas
if os.path.exists(BG_PATH):
    canvas = Image.open(BG_PATH).convert("RGBA").resize((W, H), Image.Resampling.LANCZOS)
elif os.path.exists(FALLBACK_BG):
    canvas = Image.open(FALLBACK_BG).convert("RGBA").resize((W, H), Image.Resampling.LANCZOS)
else:
    canvas = Image.new("RGBA", (W, H), (10, 5, 25, 255))

draw = ImageDraw.Draw(canvas)

PANEL_W, PANEL_H = 3400, 3200
PANEL_CX = W - (PANEL_W // 2) - 300
PANEL_CY = (H // 2) + 80

# Explicit boundaries let us shrink the box without moving the text
box_left = PANEL_CX - PANEL_W//2
box_right = PANEL_CX + PANEL_W//2
box_top = PANEL_CY - PANEL_H//2
box_bottom = (PANEL_CY + PANEL_H//2) - 200  # <--- Increase 400 to move the bottom line further UP

draw.rounded_rectangle(
    [box_left, box_top, box_right, box_bottom],
    radius=60, fill=C_BG_PANEL, outline=C_GOLD, width=10
)

Y_TITLE        = PANEL_CY - 1100
Y_LINE         = PANEL_CY - 800
Y_SUBTITLE     = PANEL_CY - 550
Y_BULLET_START = PANEL_CY - 200

draw_dynamic_centered_text(draw, "FPL VORTEX", PANEL_W - 400, 380, C_WHITE, PANEL_CX, Y_TITLE)
draw.line([(PANEL_CX - 1300, Y_LINE), (PANEL_CX + 1300, Y_LINE)], fill=C_GOLD, width=15)
draw_dynamic_centered_text(draw, f"{gw_name} PREVIEW & MASTER PLAN",
                           PANEL_W - 400, 180, C_GOLD, PANEL_CX, Y_SUBTITLE)

FOOTER_H = 450
draw.rectangle([0, H - FOOTER_H, W, H], fill=(15, 10, 30, 255))
draw.line([(0, H - FOOTER_H), (W, H - FOOTER_H)], fill=C_GREEN, width=12)

# Move deadline text slightly up to the top half of the footer
draw_dynamic_left_aligned_text(draw, f"DEADLINE: {deadline_str}", W - 400, 160, C_WHITE, 300, H - 280)


# Move "Powered By" text to the right corner, increase font to 130, and switch back to "Bold"
draw_dynamic_centered_text(draw, "POWERED BY POISSON XG & CUSTOM FDR MATHEMATICS",
                           3000, 130, C_CYAN, W - 1800, H - (FOOTER_H // 2), weight="Bold")
# ── LEFT SIDE: RECORDING DATE / TIME / DEADLINE INFO BLOCK ───────────────
from datetime import datetime as _dt_s1
_now_s1        = _dt_s1.now()
_rec_date_s1   = _now_s1.strftime("%B %d, %Y").upper()
_rec_time_s1   = _now_s1.strftime("%I:%M %p EST").upper()

_LB_X     = 300    # left anchor (8K)
_LB_Y     = 1600   # vertical start
_LB_W     = 2100   # pill width
_LABEL_H  = 185    # dark label strip height
_VALUE_H  = 215    # colored value strip height
_ITEM_H   = _LABEL_H + _VALUE_H
_LB_GAP   = 65     # gap between items

_info_data = [
    ("RECORDED DATE", _rec_date_s1, "#00E5FF", "#000000"),
    ("RECORDED TIME", _rec_time_s1, "#00FF87", "#000000"),
    ("GW DEADLINE",   deadline_str, "#FF1744", "#FFFFFF"),
]

_cy = _LB_Y
for _lbl, _val, _bg, _fg in _info_data:
    # Dark label strip
    draw.rounded_rectangle(
        [_LB_X, _cy, _LB_X + _LB_W, _cy + _LABEL_H],
        radius=35, fill=(22, 14, 42, 215)
    )
    draw.text(
    (_LB_X + 70, _cy + _LABEL_H // 2), _lbl,
    font=get_font("Bold", 105), fill=_bg, anchor="lm"
)
    # Colored value pill
    draw.rounded_rectangle(
        [_LB_X, _cy + _LABEL_H, _LB_X + _LB_W, _cy + _ITEM_H],
        radius=35, fill=_bg
    )
    # Auto-shrink text to fit pill width
    _vf = get_font("Bold", 145)
    _vb = draw.textbbox((0, 0), _val, font=_vf)
    while (_vb[2] - _vb[0]) > (_LB_W - 90) and _vf.size > 60:
        _vf = get_font("Bold", _vf.size - 5)
        _vb = draw.textbbox((0, 0), _val, font=_vf)
    draw.text(
        (_LB_X + _LB_W // 2, _cy + _LABEL_H + _VALUE_H // 2),
        _val, font=_vf, fill=_fg, anchor="mm"
    )
    _cy += _ITEM_H + _LB_GAP

empty_path = os.path.join(PNG_DIR, "Slide_1_Intro_Empty.png")
if os.path.exists(empty_path): os.remove(empty_path)
canvas.copy().convert("RGB").save(empty_path, format="PNG")
print(f"✅ Slide 1 Empty Base: {empty_path}")

# Per-item agenda overlays
for _f in os.listdir(AGENDA_OVERLAYS_DIR):
    if _f.lower().endswith(".png"):
        try: os.remove(os.path.join(AGENDA_OVERLAYS_DIR, _f))
        except: pass

bullet_x_start = PANEL_CX - 1000
current_y = Y_BULLET_START
circle_r  = 45

for idx, (item_text, circle_color) in enumerate(agenda_items):
    overlay = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    odraw = ImageDraw.Draw(overlay)
    odraw.ellipse(
        [bullet_x_start, current_y - circle_r,
         bullet_x_start + (circle_r * 2), current_y + circle_r],
        fill=circle_color
    )
    text_x = bullet_x_start + (circle_r * 2) + 70
    draw_dynamic_left_aligned_text(odraw, item_text, PANEL_W - 800, 140,
                                   C_WHITE, text_x, current_y)
    overlay_path = os.path.join(AGENDA_OVERLAYS_DIR, f"agenda_{idx:02d}.png")
    overlay.save(overlay_path, "PNG")
    print(f"   ↳ agenda_{idx:02d}.png ({item_text})")
    current_y += 240

print(f"\n✅ Cell 14 Complete: empty intro + {len(agenda_items)} agenda overlays")

Initializing Slide 1: Intro Empty Base + Agenda Overlays...
✅ Slide 1 Empty Base: /content/drive/MyDrive/Vortex_Final/png/Slide_1_Intro_Empty.png
   ↳ agenda_00.png (SQUAD REVIEW)
   ↳ agenda_01.png (MARKET TRENDS)
   ↳ agenda_02.png (SWORD & SHIELD)
   ↳ agenda_03.png (FIXTURE TICKER)
   ↳ agenda_04.png (CLEAN SHEET ODDS)

✅ Cell 14 Complete: empty intro + 5 agenda overlays


In [ ]:
# <<< START REPLACEMENT (Cell 15 - Slide 12 Outro) <<<
# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX MASTER PIPELINE
Cell 15: Slide 12 — Outro (Empty Base + Captain Card Animation)
+ Final DATA LOCK for all locked slide data
"""

import os, re, copy, json
from datetime import datetime
from PIL import Image, ImageDraw, ImageFont

print("Initializing Slide 12: Outro Empty Base...")

# ── 1. PATHS ─────────────────────────────────────────────────────────────
try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

PNG_DIR      = os.path.join(GD_ROOT, "png")
ELEMENTS_DIR = os.path.join(GD_ROOT, "elements")
AUDIO_DIR    = os.path.join(GD_ROOT, "audio")
os.makedirs(PNG_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)

BG_PATH     = os.path.join(ELEMENTS_DIR, "14.png")
FALLBACK_BG = os.path.join(ELEMENTS_DIR, "12.png")

# ── 2. CANVAS CONSTANTS ──────────────────────────────────────────────────

C_GOLD     = "#FFD700"
C_WHITE    = "#FFFFFF"
C_CYAN     = "#00E5FF"
C_GREEN    = "#00FF87"
C_RED      = "#FF1744"
C_MAGENTA  = "#FF00FF"
C_ORANGE   = "#FF4500"
C_BG_PANEL = (15, 10, 30, 230)

# ── 3. FONT + TEXT HELPERS ───────────────────────────────────────────────
def get_font(weight="Bold", size=50):
    try:
        return ImageFont.truetype(os.path.join(ELEMENTS_DIR, f"Roboto-{weight}.ttf"), size)
    except Exception:
        return ImageFont.load_default()

def draw_dynamic_centered(d, text, max_w, max_size, color, cx, cy, weight="Bold"):
    text = str(text).upper()
    f = max_size; fnt_o = get_font(weight, f)
    bb = d.textbbox((cx, cy), text, font=fnt_o, anchor="mm")
    while (bb[2] - bb[0]) > max_w and f > 30:
        f -= 5; fnt_o = get_font(weight, f)
        bb = d.textbbox((cx, cy), text, font=fnt_o, anchor="mm")
    d.text((cx, cy), text, font=fnt_o, fill=color, anchor="mm")

def draw_dynamic_left_aligned(d, text, max_w, max_size, color, x, y, weight="Bold"):
    text = str(text).upper()
    f = max_size; fnt_o = get_font(weight, f)
    bb = d.textbbox((x, y), text, font=fnt_o, anchor="lm")
    while (bb[2] - bb[0]) > max_w and f > 30:
        f -= 5; fnt_o = get_font(weight, f)
        bb = d.textbbox((x, y), text, font=fnt_o, anchor="lm")
    d.text((x, y), text, font=fnt_o, fill=color, anchor="lm")

# ── 4. DYNAMIC GAMEWEEK + MARKET DATA ────────────────────────────────────
events     = boot_data.get("events", [])
next_event = next((e for e in events if e.get("is_next")), events[0] if events else {})
target_gw  = next_event.get("id", "??")

try:
    _players = [p for p in boot_data.get("elements", []) if p.get("status") != "u"]
    _top_in  = sorted(_players, key=lambda x: x.get("transfers_in_event", 0)
                                              - x.get("transfers_out_event", 0),
                      reverse=True)[0]
    _top_out = sorted(_players, key=lambda x: x.get("transfers_out_event", 0)
                                              - x.get("transfers_in_event", 0),
                      reverse=True)[0]
    dyn_in   = _top_in['web_name'].upper()
    dyn_out  = _top_out['web_name'].upper()
except Exception:
    dyn_in, dyn_out = "PREMIUM TARGETS", "RED-FLAGGED ASSETS"

# Right-panel summary points (4 items)
summary_points = [
    (f"ATTACK HIGH xG MATCHUPS IN GW{target_gw}", C_CYAN),
    (f"ESSENTIAL ACQUISITION: {dyn_in}",          C_MAGENTA),
    (f"MITIGATE RISK: MOVE OFF {dyn_out}",        C_ORANGE),
    ("ROLL FREE TRANSFERS IF STRUCTURE IS SOUND", C_GREEN),
]

# ── 5. CANVAS LOAD ───────────────────────────────────────────────────────
if os.path.exists(BG_PATH):
    canvas = Image.open(BG_PATH).convert("RGBA").resize((W, H), Image.LANCZOS)
elif os.path.exists(FALLBACK_BG):
    canvas = Image.open(FALLBACK_BG).convert("RGBA").resize((W, H), Image.LANCZOS)
else:
    canvas = Image.new("RGBA", (W, H), (10, 5, 25, 255))

draw = ImageDraw.Draw(canvas)

# ── 6. TOP-RIGHT GW TAG ──────────────────────────────────────────────────
gw_text  = f"NEXT: GW{target_gw}"
gw_font  = get_font("Bold", 140)
gw_bb    = draw.textbbox((0, 0), gw_text, font=gw_font)
draw.text((W - (gw_bb[2] - gw_bb[0]) - 250, 150),
          gw_text, font=gw_font, fill=C_GOLD)

# ── 7. LEFT-SIDE BRANDING (outro/sub/like icons) ─────────────────────────
OUTRO_Y, BUTTONS_Y = 600, 3200
icons_to_paste = [
    ("outro.png", 600, OUTRO_Y,    2600, 2600),
    ("sub.png",   700, BUTTONS_Y,  1100, 1100),
    ("like.png",  2000, BUTTONS_Y, 1100, 1100),
]
for fname, x, y, w_, h_ in icons_to_paste:
    p = os.path.join(ELEMENTS_DIR, fname)
    if os.path.exists(p):
        try:
            im = Image.open(p).convert("RGBA").resize((w_, h_), Image.LANCZOS)
            canvas.paste(im, (x, y), im)
        except Exception:
            pass

# ── 8. RIGHT-SIDE SUMMARY PANEL ──────────────────────────────────────────
PANEL_W, PANEL_H = 3400, 3200
PANEL_CX = W - (PANEL_W // 2) - 300
PANEL_CY = (H // 2) + 50

draw.rounded_rectangle(
    [PANEL_CX - PANEL_W // 2, PANEL_CY - PANEL_H // 2,
     PANEL_CX + PANEL_W // 2, PANEL_CY + PANEL_H // 2],
    radius=60, fill=C_BG_PANEL, outline=C_CYAN, width=10
)

Y_TITLE         = PANEL_CY - 1100
Y_LINE          = PANEL_CY - 800
Y_BULLET_START  = PANEL_CY - 500
Y_CAPTAIN_HEAD  = PANEL_CY + 950

# Panel title
draw_dynamic_centered(draw, f"FINAL TAKEAWAYS: GW{target_gw}",
                      PANEL_W - 400, 200, C_WHITE, PANEL_CX, Y_TITLE)

# Cyan divider line
draw.line([(PANEL_CX - 1300, Y_LINE), (PANEL_CX + 1300, Y_LINE)],
          fill=C_CYAN, width=15)

# 4 takeaway bullets
bullet_x_start = PANEL_CX - 1300
current_y = Y_BULLET_START
circle_r  = 45

OUTRO_AGENDA_DIR = os.path.join(PNG_DIR, "slide12_agenda")
os.makedirs(OUTRO_AGENDA_DIR, exist_ok=True)

for idx, (item_text, circle_color) in enumerate(summary_points):
    overlay = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    odraw   = ImageDraw.Draw(overlay)

    _dot_x1 = bullet_x_start
    _dot_y1 = current_y - circle_r
    _dot_x2 = bullet_x_start + circle_r * 2
    _dot_y2 = current_y + circle_r
    odraw.ellipse([_dot_x1, _dot_y1, _dot_x2, _dot_y2], fill=circle_color)

    _clean_item = re.sub(r'[^\x00-\x7F★▲▼•·]', '', item_text)
    _clean_item = re.sub(r'[★▲▼•·]', '', _clean_item).strip()

    text_x = bullet_x_start + (circle_r * 2) + 90
    draw_dynamic_left_aligned(odraw, _clean_item, PANEL_W - 600, 130, C_WHITE, text_x, current_y)

    overlay_path = os.path.join(OUTRO_AGENDA_DIR, f"outro_{idx:02d}.png")
    overlay.save(overlay_path, "PNG")
    current_y += 300

# Captain section header
_clk_font  = get_font("Bold", 130)
_clk_text  = "CAPTAINCY LOCK"
_clk_x     = PANEL_CX - 1100
_clk_y     = Y_CAPTAIN_HEAD
_clk_dot_r = 22
_clk_gap   = 30
_clk_bb    = draw.textbbox((0, 0), _clk_text, font=_clk_font)
_clk_txt_w = _clk_bb[2] - _clk_bb[0]
_clk_txt_h = _clk_bb[3] - _clk_bb[1]

# Left gold dot
draw.ellipse([
    _clk_x, _clk_y - _clk_dot_r, _clk_x + _clk_dot_r * 2, _clk_y + _clk_dot_r,
], fill=C_GOLD)

# Text
draw.text(
    (_clk_x + _clk_dot_r * 2 + _clk_gap, _clk_y - _clk_txt_h // 2),
    _clk_text, font=_clk_font, fill=C_GOLD
)

# Right gold dot
_clk_right_x = _clk_x + _clk_dot_r * 2 + _clk_gap + _clk_txt_w + _clk_gap
draw.ellipse([
    _clk_right_x, _clk_y - _clk_dot_r, _clk_right_x + _clk_dot_r * 2, _clk_y + _clk_dot_r,
], fill=C_GOLD)

# ── 9. SAVE EMPTY BASE ───────────────────────────────────────────────────
empty_path = os.path.join(PNG_DIR, "Slide_12_Outro_Empty.png")
if os.path.exists(empty_path):
    os.remove(empty_path)
canvas.convert("RGB").save(empty_path, format="PNG")
print(f"✅ Slide 12 Empty Base saved → {empty_path}")


# ── 10. CAPTAINCY EXTRACTION & DATA POPULATION FOR ANIMATION ─────────────
def _clean_name_s12(raw):
    s = str(raw).strip()
    s = re.sub(r"^[A-Z]\.\s*", "", s)
    s = re.sub(r"\s+[A-Z]\.?$", "", s)
    return s.strip()

try:
    # Safely retrieve the mathematical plan established in Cell 11+12
    _s12_pool  = globals().get('s10_starters', []) + globals().get('s10_bench', [])
    _cap_data  = next((p for p in _s12_pool if p.get("is_cap")), None)
    _vc_data   = next((p for p in _s12_pool if p.get("is_vc")),  None)

    c_name  = _clean_name_s12(_cap_data['web_name']) if _cap_data else "our premium captain"
    vc_name = _clean_name_s12(_vc_data['web_name'])  if _vc_data  else "our vice captain"

    # Prepend full club name for natural broadcast phrasing
    if _cap_data:
        _c_team = next((t['name'] for t in boot_data['teams'] if t['id'] == _cap_data.get('team', 0)), "")
        if _c_team: c_name = f"{_c_team}'s {c_name}"

    if _vc_data:
        _vc_team = next((t['name'] for t in boot_data['teams'] if t['id'] == _vc_data.get('team', 0)), "")
        if _vc_team: vc_name = f"{_vc_team}'s {vc_name}"

    # 🔴 CORE FIX: Dynamically populate tc_options for pop-up cards
    tc_options = []

    def _build_outro_card(p_data, strategy_label):
        if not p_data: return None
        c = p_data.copy()
        c['strategy_type'] = strategy_label

        # Apply exact multiplier logic to expected points
        raw_xp = float(c.get('ep_next', 0.0))
        mult   = 2.0 if strategy_label == "captain" else 1.0
        c['locked_xpts'] = round(raw_xp * mult, 2)
        c['multiplier']  = int(mult)

        # Cross-check upcoming fixtures to display exact FDR data
        tid = c.get('team_id', c.get('team', 0))
        gw_fixes = [f for f in fix_data if f.get('event') == target_gw and (f['team_h'] == tid or f['team_a'] == tid)]

        if not gw_fixes:
            c['next_opp']  = "BLANK"
            c['next_fdr']  = 5.0
            c['fdr_color'] = "#333333"
        else:
            opps, fdrs = [], []
            for fix in gw_fixes:
                is_home = (fix['team_h'] == tid)
                opp_id = fix['team_a'] if is_home else fix['team_h']
                opp_team = next((t for t in boot_data['teams'] if t['id'] == opp_id), None)
                if opp_team:
                    opps.append(f"{opp_team['short_name']} [{'H' if is_home else 'A'}]")
                fdrs.append(get_vortex_fdr(tid, opp_id, is_home))
            c['next_opp'] = " + ".join(opps)
            c['next_fdr'] = round(sum(fdrs) / len(fdrs), 2)
            c['fdr_color'] = ("#00FF87" if c['next_fdr'] <= 2.5 else "#FFD700" if c['next_fdr'] <= 3.5 else "#FF1744")

        # Format the rank string for the UI Card
        c['rank_str']   = f"PROJ XP: {c['locked_xpts']} PTS"
        c['rank_color'] = C_GOLD if strategy_label == "captain" else C_CYAN

        return c

    # Build and lock the cards to globals so Cell 17 can animate them
    if _cap_data: tc_options.append(_build_outro_card(_cap_data, "captain"))
    if _vc_data:  tc_options.append(_build_outro_card(_vc_data, "vice_captain"))
    globals()['tc_options'] = tc_options

except Exception as _cap_err:
    print(f"   ⚠️ Captaincy extraction fallback triggered: {_cap_err}")
    c_name  = "our premium captain"
    vc_name = "our vice captain"
    globals()['tc_options'] = []

# ── 11. ASSEMBLE OUTRO NARRATION ───────────────────────────────────────
s12 = (
    f"That concludes the analytical blueprint for Gameweek {target_gw}. "
    f"Let's lock in the absolute non-negotiables. "
    f"First, aggressively attack the high expected goals matchups this round. "
    f"Second, execute your essential acquisitions without hesitation. "
    f"Prioritizing players like {dyn_in} allows you to capitalize on elite underlying metrics "
    f"before the rest of the market catches up. "
    f"Third, mitigate your structural risk by moving off vulnerable assets like {dyn_out} "
    f"before their inevitable price drop compounds your losses. "
    f"Fourth, if your core structure is already optimally aligned, simply roll your free transfer. "
    f"Protecting that extra move going into a swing gameweek is genuine rank equity. "
    f"Finally, the captaincy is mathematically locked on {c_name}, "
    f"with {vc_name} providing the ultimate safety net if anything changes before deadline. "
    f"If this breakdown sharpened your strategy, hit Subscribe, drop a Like, "
    f"and join the Vortex elite. "
    f"If you are debating a specific transfer or have a squad dilemma, "
    f"drop it in the comments below. "
    f"I will personally go through them and reply in the chat before the deadline, "
    f"or address them directly in the next video. "
    f"Remove emotion, trust the model, execute the optimal path, "
    f"and let the data dictate your rank. Good luck."
)

if 'script_data' not in globals():
    script_data = {}
script_data['Slide_12_Outro'] = s12
print(f"✅ Slide 12 — Cap: {c_name} | VC: {vc_name}")


# ═══════════════════════════════════════════════════════════════════════
# 🔒 FINAL DATA LOCK — snapshots all slide globals for downstream cells
# ═══════════════════════════════════════════════════════════════════════
print("\n🔒 Locking all slide data...")

def safe_lock(name, fb=None):
    if fb is None: fb = []
    return copy.deepcopy(globals()[name]) if name in globals() else fb

_locked_keys = [
    'top_in', 'top_out',
    'shields', 'swords',
    'fdr_ranked',
    'goals_top10', 'goals_bot10',
    'cs_top10', 'cs_bot10',
    'xg_ranking', 'cs_ranking',
    'target_players', 'avoid_players',
    's7a_locked_players',
    's9_starters', 's9_bench',
    's10_starters', 's10_bench', 's10_squad_pool',
    'top_targets',
    'blank_teams', 'fh_targets',
    'my_plan', 'tc_options',
    'target_dgw_gw', 'target_bgw_gw',
]

_lock_status = {}
for key in _locked_keys:
    val = safe_lock(key)
    _lock_status[key] = "✓" if (val and key in globals()) else "·"

# Print compact lock report
print("   Locked globals:")
for i in range(0, len(_locked_keys), 4):
    chunk = _locked_keys[i:i+4]
    line = "   "
    for k in chunk:
        line += f"  {_lock_status[k]} {k:<22}"
    print(line)

# Persist lock metadata to JSON for resilience across Colab disconnects
lock_path = os.path.join(AUDIO_DIR, "data_lock.json")
try:
    with open(lock_path, 'w') as f:
        json.dump({
            "NEXT_GW":   globals().get('NEXT_GW', 0),
            "CURR_GW":   globals().get('CURR_GW', 0),
            "locked_at": datetime.now().isoformat(),
            "globals_present": [k for k in _locked_keys if k in globals()],
        }, f, indent=2)
    print(f"\n✅ Data lock metadata saved → {lock_path}")
except Exception as e:
    print(f"⚠️ Could not write data_lock.json: {e}")

print("\n" + "=" * 60)
print(" 🟣 CELL 15 COMPLETE: SLIDE 12 EMPTY BASE + DATA LOCK ")
print("=" * 60)
# >>> END REPLACEMENT >>>

Initializing Slide 12: Outro Empty Base...
✅ Slide 12 Empty Base saved → /content/drive/MyDrive/Vortex_Final/png/Slide_12_Outro_Empty.png
✅ Slide 12 — Cap: Gabriel | VC: Calvert-Lewin

🔒 Locking all slide data...
   Locked globals:
     ✓ top_in                  ✓ top_out                 ✓ shields                 ✓ swords                
     ✓ fdr_ranked              ✓ goals_top10             ✓ goals_bot10             ✓ cs_top10              
     ✓ cs_bot10                ✓ xg_ranking              ✓ cs_ranking              · target_players        
     ✓ avoid_players           ✓ s7a_locked_players      ✓ s9_starters             ✓ s9_bench              
     ✓ s10_starters            ✓ s10_bench               ✓ s10_squad_pool          · top_targets           
     · blank_teams             · fh_targets              · my_plan                 ✓ tc_options            
     · target_dgw_gw           · target_bgw_gw         

✅ Data lock metadata saved → /content/drive/MyDrive/Vortex_Fina

In [ ]:


# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX — Cell 16 Rewrite
Elite broadcast narration. Short. Sharp. No repetition.
Target: 18-22 minutes total.
"""

import os, re, json, asyncio
import edge_tts, nest_asyncio
nest_asyncio.apply()

try:    GD_ROOT
except: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

AUDIO_DIR = os.path.join(GD_ROOT, "audio")
os.makedirs(AUDIO_DIR, exist_ok=True)

# ── AUTO DURATION + CLEAN OLD FILES ──────────────────────────────────
import subprocess, glob

# Step 1 — calculate duration from existing files FIRST
_secs = 0
for _f in os.listdir(AUDIO_DIR):
    if not _f.endswith('.mp3'): continue
    if _f in ['intro.mp3','mid.mp3','outro.mp3']: continue
    _r = subprocess.run(
        ["ffprobe","-v","error","-show_entries","format=duration",
         "-of","default=noprint_wrappers=1:nokey=1",
         os.path.join(AUDIO_DIR,_f)],
        capture_output=True, text=True)
    try: _secs += float(_r.stdout.strip())
    except: pass

_mins = int(_secs/60)
_dur_str = (
    "around ten minutes"          if _mins<=10 else
    "around fifteen minutes"      if _mins<=15 else
    "around twenty minutes"       if _mins<=20 else
    "around twenty five minutes"  if _mins<=25 else
    "around thirty minutes"       if _mins<=30 else
    "around thirty five minutes"
)
print(f"✅ Previous duration: ~{_mins} min → '{_dur_str}'")

# Step 2 — NOW delete old files safely
KEEP = ['intro.mp3','mid.mp3','outro.mp3']
old_files = (
    glob.glob(os.path.join(AUDIO_DIR, "*.mp3")) +
    glob.glob(os.path.join(AUDIO_DIR, "*.vtt")) +
    glob.glob(os.path.join(AUDIO_DIR, "*.json"))
)
for f in old_files:
    if os.path.basename(f) in KEEP:
        continue
    try: os.remove(f)
    except: pass
print("✅ Old audio files cleared")
# ─────────────────────────────────────────────────────────────────────

# ────────────────────────────────────────────────────────────────────

# ── VOICE CONFIGURATION ──────────────────────────────────────────────────
# Ryan brings a grounded, professional Premier League broadcast vibe.
# +8% Rate keeps the pace snappy; -5Hz Pitch adds depth and removes the AI hum.
VOICE = "en-GB-RyanNeural"
RATE = "+12%"
PITCH = "-5Hz"

# ── TEAM EXPANDER ──────────────────────────────────────────────────────────
TEAM_EXPANDER = {
    "ARS": "Arsenal",
    "AVL": "Aston Villa",
    "BOU": "Bournemouth",
    "BRE": "Brentford",
    "BHA": "Brighton",
    "CHE": "Chelsea",
    "CRY": "Crystal Palace",
    "EVE": "Everton",
    "FUL": "Fulham",
    "IPS": "Ipswich",
    "LEI": "Leicester",
    "LIV": "Liverpool",
    "MCI": "Manchester City",
    "MUN": "Man United",
    "NEW": "Newcastle",
    "NFO": "Nottingham Forest",
    "SOU": "Southampton",
    "TOT": "Tottenham",
    "WHU": "West Ham",
    "WOL": "Wolves",
    "SUN": "Sunderland",
    "BUR": "Burnley",
}

def _num(val, d=1):
    try:
        f = float(str(val).replace('%','').strip())
        if f == int(f): return str(int(f))
        return f"{f:.{d}f}"
    except: return str(val)

def expand(text):
    if not isinstance(text, str): return str(text)
    def fix_decimal(m):
        w, d = m.group(1), m.group(2)
        return w if d == '0' else f"{w} point {' '.join(d)}"
    text = re.sub(r'(\d+)\.(\d+)', fix_decimal, text)
    text = re.sub(r'\b[xX][gG]\b', 'expected goals', text)
    text = re.sub(r'\b[xX][aA]\b', 'expected assists', text)
    text = re.sub(r'\b[xX][pP]ts\b', 'expected points', text)
    text = re.sub(r'\b[fF][dD][rR]\b', 'fixture difficulty rating', text)
    text = re.sub(r'\bGW(\d+)', r'Gameweek \1', text)
    text = re.sub(r'\bGW\b', 'Gameweek', text)
    text = re.sub(r'\bDGW(\d+)', r'double gameweek \1', text)
    text = re.sub(r'\bDGW\b', 'double gameweek', text)
    text = re.sub(r'\bBGW(\d+)', r'blank gameweek \1', text)
    text = re.sub(r'\bBGW\b', 'blank gameweek', text)
    text = re.sub(r'\bEO%\b', 'effective ownership', text)
    text = re.sub(r'\bCS%\b', 'clean sheet probability', text)
    for s, f in TEAM_EXPANDER.items():
        text = text.replace(f"{s}[H]", f"{f} at home")
        text = text.replace(f"{s}[A]", f"{f} away")
        text = re.sub(rf'\b{s}\b', f, text)
    text = text.replace("[H]", "at home").replace("[A]", "away")
    text = text.replace(" / ", " and ").replace(" vs ", " versus ")
    text = re.sub(r'\b[A-Z]\.\s+([A-Z][a-z])', r'\1', text)

    # 🔴 NEW: Phonetic pronunciation overrides for Edge-TTS
    text = re.sub(r'\bWan-?Bissaka\b', 'Wonn Bissaka', text, flags=re.IGNORECASE)
    text = re.sub(r'\bBruno Fernandes\b', 'Broo no Fer NAN desh', text, flags=re.IGNORECASE)

    return text.strip()


# ── DYNAMIC WEEK-COUNT HELPER ─────────────────────────────────────────────
def _num_to_words(n):
    """Convert small integers to words for natural narration."""
    table = {
        1:"one", 2:"two", 3:"three", 4:"four", 5:"five",
        6:"six", 7:"seven", 8:"eight", 9:"nine", 10:"ten"
    }
    return table.get(int(n), str(n))

try:
    REMAINING_WEEKS = len(target_gws)
    if REMAINING_WEEKS <= 0:
        REMAINING_WEEKS = 1
except Exception:
    REMAINING_WEEKS = 7

WEEKS_WORD   = _num_to_words(REMAINING_WEEKS)
WEEKS_NOUN   = "week" if REMAINING_WEEKS == 1 else "weeks"
WEEKS_PHRASE = f"{WEEKS_WORD} {WEEKS_NOUN}"

def check_flag(p):
    chance = p.get('chance_of_playing_next_round')
    if p.get('status','a') != 'a' or (chance is not None and chance < 100):
        return f" He carries a fitness flag, so mandate a firm hold until the Friday press conferences confirm his exact status. "
    return ""

def get_team_name(team_id):
    t = next((t for t in boot_data['teams'] if t['id']==team_id), None)
    if not t: return "their club"
    m = {
        "Brighton & Hove Albion": "Brighton",
        "Manchester United": "Manchester United",
        "Manchester City": "Manchester City",
        "Tottenham Hotspur": "Tottenham Hotspur",
        "Nottingham Forest": "Nottingham Forest"
    }
    return m.get(t['name'], t['name'])

def get_fixture(team_id):
    fixes = [f for f in fix_data
             if (f['team_h']==team_id or f['team_a']==team_id)
             and f['event']==NEXT_GW]
    if not fixes: return "no fixture"
    parts = []
    for fx in fixes:
        ih = fx['team_h']==team_id
        oid = fx['team_a'] if ih else fx['team_h']
        opp = next((t for t in boot_data['teams'] if t['id']==oid), None)
        if opp: parts.append(f"{opp['name']} {'at home' if ih else 'away'}")
    return " and ".join(parts)

def get_pos(p):
    return {1:"goalkeeper",2:"defender",3:"midfielder",4:"forward"}.get(
        p.get('element_type',3),"player")


# ══════════════════════════════════════════════════════════════════════════
# SLIDE SCRIPTS
# ══════════════════════════════════════════════════════════════════════════
if 'script_data' not in globals():
    script_data = {}

def _get_transition(current_mod):
    """Dynamically fetches the broadcast transition for the actual NEXT slide."""
    mods = globals().get('final_modules', [])
    try:
        current_idx = mods.index(current_mod)
        if current_idx + 1 < len(mods):
            next_mod = mods[current_idx + 1]
        else:
            return ""
    except ValueError:
        return ""

    transitions = {
        "market_trends":   "Let's shift our focus to the live market trends and see where the sharp money is heading. ",
        "sword_shield":    "Next, we move into the Sword and Shield view, separating the must-haves from the higher-upside differentials. ",
        "fdr_map":         "Now let's go straight to the fixture difficulty map and see exactly where the schedule starts to open up. ",
        "projected_goals": "With that in mind, let's turn to the expected goals map and identify the strongest attacking ceilings. ",
        "cs_odds":         "Switching to the defensive side, let's break down the clean sheet probabilities. ",
        "injury_report":   "Before any structural moves, we need to clear up the latest injury and suspension picture. ",
        "all_20_teams":    "Let's pull everything together and review the full twenty-team projections for the upcoming round. ",
        "targets_avoids":  "Now let's use the data to isolate the best transfer targets and the assets to stay away from. ",
        "gw_review":       "Let's look back quickly at the squad review and see how those structural decisions actually played out. ",
        "master_plan":     "With the analysis complete, it's time to reveal the master plan and the locked starting eleven. ",
        "chip_strategy":   "Let's zoom out for a moment and assess the wider chip strategy based on the current landscape. ",
        "outro":           "Let's bring it all together with the final tactical takeaways before the deadline. "
    }
    return transitions.get(next_mod, "")

# Inject video type into Slide 1 intro context
_video_type = globals().get("VIDEO_TYPE", f"GW{NEXT_GW} PREVIEW & MASTER PLAN")
_is_dgw     = globals().get("IS_DGW", False)
_is_bgw     = globals().get("IS_BGW", False)
_dl_phase   = globals().get("DEADLINE_PHASE", "DISCOVERY")
_top_cat    = globals().get("top_story_category", "none")
_top_score  = globals().get("top_story_score", 0)


# ── SLIDE 1: INTRO ────────────────────────────────────────────────────────
try:
    _intro_base = (
        f"Welcome back to FPL Vortex. {_video_type}. "
      + (f"There have been major developments ahead of the deadline, and we will break down exactly what managers need to know this week. " if _top_score >= 80 else "")
      + (f"This is a double gameweek edition, with rotation risk and chip strategy becoming increasingly important. We are covering everything you need to know today. " if _is_dgw else "")
      + (f"With multiple blanks confirmed this round, the focus naturally shifts toward squad structure, long-term planning, and chip management. " if _is_bgw else "")
      + (f"The analytical blueprint for Gameweek {NEXT_GW} is now locked in. " if _top_score < 80 else "")
      + f"This analysis is built on Poisson expected goals modelling and a custom fixture difficulty rating system, giving every recommendation a strong data foundation. "
      + f"We are running for around {_dur_str} today. "
    )

    _agenda_scripts = {
        "market_trends":   "We begin with the market trends, identifying where the underlying numbers are pointing right now. ",
        "sword_shield":    "Next, we break down the Sword and Shield strategy, separating the key foundation picks from the higher-upside differentials. ",
        "fdr_map":         "We then move to the fixture ticker to understand where the schedule becomes favourable and where caution may be needed over the coming weeks. ",
        "projected_goals": "After that, we analyze the expected goals map to identify the strongest attacking ceilings. ",
        "cs_odds":         "We also cover the clean sheet probabilities, highlighting the most reliable defensive opportunities. ",
        "injury_report":   "Crucially, we review the injury and availability picture, because team news can quickly reshape transfer priorities. ",
        "all_20_teams":    "From there, we look across all twenty team projections to identify the key matchups for the round ahead. ",
        "targets_avoids":  "We then isolate the strongest transfer targets and the players managers may want to move away from. ",
        "gw_review":       "After that, we take a quick look back at the squad review to assess how the previous structure performed. ",
        "master_plan":     "And we finish with the master plan and team reveal, showing exactly how the model is approaching the deadline. ",
        "chip_strategy":   "We will also touch on the chip strategy best suited to the current state of the season. ",
    }

    _added_count = 0
    # 🔴 FIX: Dynamically stitch the script together based on chosen modules
    for mod in globals().get('final_modules', []):
        if mod in _agenda_scripts:
            _intro_base += _agenda_scripts[mod]
            _added_count += 1
            if _added_count >= 5: break

    _intro_base += "Let's get into it."

    script_data["Slide_1_Intro"] = _intro_base
    print(f"✅ Slide 1 (Dynamic Agenda Script Built — {_added_count} items)")
except Exception as e:
    print(f"❌ Slide 1: {e}")
# >>> END REPLACEMENT >>>


# ── SLIDE 2: MARKET TRENDS (NATURAL FLOW + AUTO-ANALYTICS) ────────────────
try:
    import re

    def _get_pos(p):
        return {1: "goalkeeper", 2: "defender", 3: "midfielder", 4: "forward"}.get(
            p.get('element_type', 3), "player"
        )

    def _get_club(p):
        tid = p.get('team', 0)
        team = next((t for t in boot_data['teams'] if t['id'] == tid), None)
        if not team:
            return "their club"
        m = {
            "Brighton & Hove Albion": "Brighton",
            "Manchester United": "Man United",
            "Manchester City": "Man City",
            "Tottenham Hotspur": "Spurs",
            "Nottingham Forest": "Nottingham Forest"
        }
        return m.get(team['name'], team['name'])

    def _flag(p):
        chance = p.get('chance_of_playing_next_round')
        if p.get('status', 'a') != 'a' or (chance is not None and chance < 100):
            return " He is carrying a fitness flag, so mandate a firm hold until the Friday press conferences confirm his exact status. "
        return ""

    TEAM_EXPANDER = globals().get('TEAM_EXPANDER', {})

    # ── DYNAMIC TIMESTAMP ────────────────────────────────────────────────
    try:
        _now_dt = datetime.now(EST_TZ)
        _day_name = _now_dt.strftime("%A")          # e.g. Saturday
        _date_str = _now_dt.strftime("%-d %B %Y")   # e.g. 10 May 2025
        _time_str = _now_dt.strftime("%-I:%M %p")    # e.g. 3:45 PM
        _timestamp_line = (
            f"At the time of recording, on {_day_name} {_date_str} at {_time_str} Eastern, "
            f"this is the live state of the Gameweek {NEXT_GW} transfer market. "
            f"These numbers are pulled directly from the official FPL API in real time, "
            f"giving us the most up-to-date picture of where the market is moving. "
        )
    except Exception:
        _timestamp_line = (
            f"At the time of recording, this is the live state of the Gameweek {NEXT_GW} transfer market. "
            f"These numbers are pulled directly from the official FPL API in real time. "
        )

    s2 = (
        f"{_timestamp_line}"
        + (
            f"We are still in the early stages of the season, which means there is plenty of time for rankings to shift and strategies to evolve. "
            if NEXT_GW in [1, 2, 3] else ""
        )
        + (
            f"We are now around the halfway point of the season, where every transfer decision starts carrying a little more weight. "
            if NEXT_GW in [18, 19] else ""
        )
        + (
            f"We are firmly into the closing stretch of the season, where small decisions can create major swings in rank. "
            if NEXT_GW in [36, 37] else ""
        )
        + (
            f"This is the final gameweek of the season, so every move now becomes decisive. "
            if NEXT_GW == 38 else ""
        )
        + "Raw transfer volume can tell us a lot, but the real edge comes from separating market noise from genuinely smart moves. "
        + "Let's break down the three biggest names coming in this week and understand exactly why experienced managers are moving for them right now. "
    )

    for i, p in enumerate(top_in[:3]):
        name = p['web_name']
        vol = int(p['transfers_in_event'] / 1000)
        raw_fix = p.get('next_opp', 'upcoming fixture')
        fix = raw_fix.replace('[H]', ' at home').replace('[A]', ' away').replace(' + ', ' and ')
        for _s, _f in TEAM_EXPANDER.items():
            fix = re.sub(rf'\b{_s}\b', _f, fix)

        own = p.get('selected_by_percent', '0')
        xpts = float(p.get('ep_next', 0) or 0)
        form = float(p.get('form', 0) or 0)
        role = _get_pos(p)
        club = _get_club(p)

        if form >= 5.0 and xpts >= 4.0:
            in_reason = "the sharp money backing elite recent form into an highly exploitable matchup"
        elif form >= 5.0 and xpts < 4.0:
            in_reason = "the broader market chasing past points despite a capping fixture, a trap that top managers typically avoid"
        elif form < 5.0 and xpts >= 4.0:
            in_reason = "elite managers ignoring average form to aggressively attack a genuine fixture swing"
        else:
            in_reason = "a highly speculative punt, likely driven by casual transfers rather than the underlying data model"

        if i == 0:
            s2 += (
                f"The biggest mover this week is {name}, the {role} from {club}. "
                f"{vol} thousand managers have brought him in, and against {fix} the model projects {xpts} expected points. "
                f"That level of movement is no coincidence, and what we are seeing here is {in_reason}. "
            )
        elif i == 1:
            s2 += (
                f"Second in the incomings is {name} from {club}. "
                f"The {role} has attracted {vol} thousand transfers this week, with a projected return of {xpts} expected points against {fix}. "
                f"This looks like a well-backed move, with the wider market clearly responding to {in_reason}. "
            )
        else:
            s2 += (
                f"And completing the top three is {name} from {club}. "
                f"The {role} is now owned by {own} percent of squads after {vol} thousand new transfers this week. "
                f"Against {fix}, the projection sits at {xpts} expected points, with this trend being driven largely by {in_reason}. "
            )

        s2 += _flag(p)

    s2 += (
        "Now for the other side of the market. The sell data can be just as revealing as the buys, because it often shows where confidence is beginning to weaken. "
        "Here are the three players managers are moving away from most heavily this week. "
    )

    for i, p in enumerate(top_out[:3]):
        name = p['web_name']
        vol = int(p['transfers_out_event'] / 1000)
        raw_fix = p.get('next_opp', 'upcoming fixture')
        fix = raw_fix.replace('[H]', ' at home').replace('[A]', ' away').replace(' + ', ' and ')
        for _s, _f in TEAM_EXPANDER.items():
            fix = re.sub(rf'\b{_s}\b', _f, fix)

        role = _get_pos(p)
        club = _get_club(p)
        status = p.get('status', 'a')
        news = p.get('news', '').strip().lower()
        chance = p.get('chance_of_playing_next_round')
        form = float(p.get('form', 0.0) or 0)
        is_blank = 'blank' in raw_fix.lower() or 'none' in raw_fix.lower()

        if i == 0:
            if is_blank:
                out_reason = "With a blank gameweek coming up, the market is moving quickly to avoid dead spots in the squad."
            elif status in ['i', 's', 'd'] or chance == 0:
                out_reason = f"Already flagged with {news if news else 'an injury'}, this has become a straightforward market exit."
            elif form < 2.5:
                out_reason = f"Returning just {form} points recently, managers are moving him on to free up funds for stronger options."
            else:
                out_reason = f"With {fix} on the horizon, the market is reacting to a difficult run of fixtures."

            s2 += (
                f"The heaviest sell this week is {name} from {club}. "
                f"The {role} is being moved on by {vol} thousand managers, and the reason is fairly clear. "
                f"{out_reason} "
            )

        elif i == 1:
            if is_blank:
                out_reason = "Managers are preparing for the upcoming blank and making sure they are not left short of a full lineup."
            elif status in ['i', 's', 'd'] or chance == 0:
                out_reason = f"Out with {news if news else 'an injury'}, owners are shifting him out for a more reliable minutes profile."
            elif form < 2.5:
                out_reason = f"His output has fallen to {form} points per game, which is enough for a wave of exits."
            else:
                out_reason = f"With {fix} next, this is a protective move to avoid being caught by a tough fixture."

            s2 += (
                f"Next is {name} from {club}. The {role} is seeing {vol} thousand transfers out this week. "
                f"{out_reason} "
            )

        else:
            if is_blank:
                out_reason = "holding a non-playing asset right now is a poor use of budget."
            elif status in ['i', 's', 'd'] or chance == 0:
                out_reason = "carrying a red-flagged player keeps valuable money tied up."
            elif form < 2.5:
                out_reason = "his recent returns have made him a fading asset."
            else:
                out_reason = f"holding him into the {fix} fixture is a risk most managers are not willing to take."

            s2 += (
                f"And {name} from {club} completes the outgoings picture. "
                f"Over {vol} thousand teams have already made the move, and the logic is straightforward. "
                f"{out_reason} "
            )

        s2 += _flag(p)

    s2 += (
        "Looking at these transfer trends helps us separate short-term noise from the moves that actually matter. "
        "The market reacts fast, but the real edge comes from spotting where the structure is strongest and where managers are beginning to create value. "
        "Step back from the individual names and the wider pattern is pretty clear—most managers are leaning toward better fixtures, stronger projected returns, and safer minutes, while moving away from tougher schedules, weaker form, or players who are not available. "
        f"{_get_transition('market_trends')}"
    )

    script_data["Slide_2_Market_Trends"] = s2
    print(f"✅ Slide 2 — Timestamp injected: {_timestamp_line[:80]}...")
except Exception as e:
    print(f"❌ Slide 2: {e}")

# ── SLIDE 3: SWORD & SHIELD ───────────────────────────────────────────────
# Reads from Cell 5's `shields` and `swords` globals (single source of truth)
try:
    import re
    # Use Cell 5's already-picked players
    _shields = shields[:3]
    _swords  = swords[:3]

    # Helper: clean name based on standard rules (strip leading initials)
    def _get_name(p):
        web_raw = p.get('web_name', '').strip()
        return re.sub(r'^[A-Z]\.\s*', '', web_raw)

    s3 = (
        "We move now to the Sword and Shield strategy. The idea here is simple. Before chasing aggressive differentials, you first need to secure the high-ownership players capable of damaging your rank if you go without them. "
        "The shields come first. These are the reliable, heavily backed assets who provide stability and protection across the gameweek. "
        "We begin with the Shields, the trusted core picks carrying both strong ownership and strong projections heading into favorable fixtures. "
    )

    shield_templates = [
        "{name} from {club} headlines the shield group. The {role} is currently backed by {own} percent of managers, and that ownership reflects both consistency and fixture quality. Against {fix}, the model projects {xpts} expected points. This is less about chasing explosive upside and more about protecting your position from a highly owned return. ",

        "Next up is {name}, the {club} {role}. At {own} percent ownership, going without him already introduces risk before a ball is even kicked. The projection against {fix} sits at {xpts} expected points, which gives managers a dependable foundation going into the gameweek. Fading a player at this level of ownership usually needs a very strong reason. ",

        "Completing the shield group is {name} from {club}. The matchup against {fix} carries a projected return of {xpts} points, while {own} percent of managers already have him in place. This is the type of player you settle into the squad first before looking for opportunities to gain ground elsewhere. "
    ]

    for i, p in enumerate(_shields):
        own  = _num(p.get('selected_by_percent', 0))
        xpts = _num(p.get('ep_next', 0))
        fix  = get_fixture(p.get('team', 0))
        role = get_pos(p)
        club = get_team_name(p.get('team', 0))
        name = _get_name(p)

        s3 += shield_templates[i % 3].format(
            name=name,
            club=club,
            own=own,
            xpts=xpts,
            fix=fix,
            role=role
        )

        s3 += check_flag(p)

    s3 += (
        "With the core structure in place, attention turns toward the differential side of the strategy. "
        "The swords are the lower-owned players the model believes can outperform expectations this gameweek. "
        "These are not safe picks by definition, but the combination of projected output, fixture quality, and ownership profile creates genuine upside for managers looking to climb rank. "
    )

    sword_templates = [
        "The standout differential right now is {name}. Sitting at just {own} percent ownership, the {club} {role} offers real upside potential this week. The underlying projections point toward an expected {xpts} return against {fix}. This is the type of calculated gamble that can create meaningful rank gains if the fixture opens up. ",

        "Then we have {name}, operating well below the wider market radar at only {own} percent ownership. What stands out is the projected output against {fix}, where the model currently forecasts {xpts} expected points. Profiles like this often become the difference between maintaining rank and making a major jump. ",

        "Finally, keep a close eye on {name}. The minutes look secure, yet ownership remains low at just {own} percent. Against {fix}, the projected return sits at {xpts} expected points, suggesting the market may still be undervaluing the {club} {role}. Identifying these opportunities early is often what separates aggressive elite play from template management. "
    ]

    for i, p in enumerate(_swords):
        own  = _num(p.get('selected_by_percent', 0))
        xpts = _num(p.get('ep_next', 0))
        fix  = get_fixture(p.get('team', 0))
        role = get_pos(p)
        club = get_team_name(p.get('team', 0))
        name = _get_name(p)
        s3 += sword_templates[i % 3].format(name=name, club=club, own=own, xpts=xpts, fix=fix, role=role)
        s3 += check_flag(p)

    s3 += (
        "Hold the shields, consider the swords if they fit your squad, and do not overthink the middle ground. "
        f"{_get_transition('sword_shield')}"
    )

    script_data["Slide_3_Sword_Shield"] = s3
    print(f"✅ Slide 3   Shields: {[_get_name(p) for p in _shields]}   Swords: {[_get_name(p) for p in _swords]}")
except Exception as e:
    print(f"❌ Slide 3: {e}")

# ── SLIDE 4: FDR MAP ──────────────────────────────────────────────────────
try:
    import re
    TEAM_EXPANDER = globals().get('TEAM_EXPANDER', {})

    def _format_fdr(val):
        # Converts 2.45 to "2 point 5" for flawless Edge-TTS pronunciation
        return str(round(val, 1)).replace(".", " point ")

    def _expand_fixtures(fix_list):
        if not fix_list: return "their upcoming matches"

        # Strip out any [H] or (A) tags to prevent audio clutter
        clean_fixes = [re.sub(r'\[.*?\]|\(.*?\)', '', f).strip() for f in fix_list]

        if len(clean_fixes) > 1:
            base_str = ", ".join(clean_fixes[:-1]) + " and " + clean_fixes[-1]
        else:
            base_str = clean_fixes[0]

        # Expand short codes to full club names
        for _s, _f in TEAM_EXPANDER.items():
            base_str = re.sub(rf'\b{_s}\b', _f, base_str)
        return base_str

    s4 = (
        f"Here is the fixture outlook across the next {WEEKS_PHRASE}. "
        f"Instead of a basic opponent list, this is built from projected expected goals, which gives a clearer read on where the real attacking and defensive value sits. "
        f"Let's begin with the three most favorable schedules. "
    )

    for i, t in enumerate(fdr_ranked[:3]):
        # Extract matches, limit to 4 to prevent TTS run-on sentences
        raw_fixes = [m["opp"] for gw in target_gws for m in t["gws"][gw]["matches"]]
        fix_str = _expand_fixtures(raw_fixes[:4])
        fdr_spoken = _format_fdr(t['avg_fdr'])

        if i == 0:
            s4 += (f"{t['name']} sit at the top of the fixture list with an average difficulty score of {fdr_spoken}. "
                   f"Their run includes {fix_str}, and the underlying attacking projections support that view. If you are mapping out squad direction over the next few weeks, this is a strong place to begin. ")
        elif i == 1:
            s4 += (f"Just behind them, {t['name']} carry a {fdr_spoken} average. "
                   f"Matches against {fix_str} give their key players a real scoring window, and the numbers back that up. ")
        else:
            s4 += (f"And {t['name']} also land in a favorable zone, sitting at {fdr_spoken} across this stretch. "
                   f"Fixtures like {fix_str} keep their assets on the radar, especially if you already own them and are weighing up a hold or sale. ")

    s4 += "At the other end of the scale, these are the three schedules where the data makes a clear case for moving players out. "

    for i, t in enumerate(fdr_ranked[-3:]):
        raw_fixes = [m["opp"] for gw in target_gws for m in t["gws"][gw]["matches"]]
        fix_str = _expand_fixtures(raw_fixes[:4])
        fdr_spoken = _format_fdr(t['avg_fdr'])

        if i == 0:
            s4 += (f"The toughest schedule belongs to {t['name']}, at {fdr_spoken} on average. "
                   f"They face {fix_str} over this run, which makes a strong case for moving their players on sooner rather than later. ")
        elif i == 1:
            s4 += (f"{t['name']} also enter a difficult window with a {fdr_spoken} rating. "
                   f"Games against {fix_str} reduce what their attacking players can realistically produce during this stretch. ")
        else:
            s4 += (f"And {t['name']} sit near the hardest end at {fdr_spoken}, with {fix_str} coming up. "
                   f"The score alone does not rule them out, but once you add the fixture set, the argument for selling is there. ")

    s4 += (
        "That is the wider picture. The next step is narrowing it down to which players inside these schedules are actually worth targeting on a week by week basis. "
        f"{_get_transition('fdr_map')}"
    )

    script_data["Slide_4_fdr_map"] = s4
    print("✅ Slide 4 (Refined for Broadcast Flow & TTS Limits)")
except Exception as e:
    print(f"❌ Slide 4: {e}")

# ── SLIDE 5: PROJECTED GOALS ──────────────────────────────────────────────
try:
    def _get_full_team_s5(short_name):
        try:
            return next((t['name'] for t in boot_data['teams'] if t['short_name'] == short_name), short_name)
        except:
            return short_name

    def _format_goals(val):
        # Converts 4.5 to "4 point 5" for flawless Edge-TTS pronunciation
        return str(round(float(val), 1)).replace(".", " point ")

    s5 = (
        f"The projected goals map covers the next {WEEKS_PHRASE} of attacking output across all twenty clubs. "
        f"This is not based on recent form. It is built on projected shot volume against specific opponents, which gives a more stable picture of where the returns are likely to come from. "
    )

    if len(goals_top10) >= 3:
        t1, t2, t3 = goals_top10[0], goals_top10[1], goals_top10[2]
        c1 = _get_full_team_s5(t1.get('short', ''))
        c2 = _get_full_team_s5(t2.get('short', ''))
        c3 = _get_full_team_s5(t3.get('short', ''))
        v1 = _format_goals(t1.get('total_proj_goals', 0))
        v2 = _format_goals(t2.get('total_proj_goals', 0))
        v3 = _format_goals(t3.get('total_proj_goals', 0))

    s5 += (
            f'Leading the attacking projections is {c1}, with {v1} expected goals across this window. '
            f'Their structure is consistently creating strong chances, and the underlying numbers support that. If you are not holding their key attackers, the fixture run gives you a solid reason to consider them. '
            f'In second place is {c2}, projected for {v2} expected goals. '
            f'The chance creation is reliable here, and the schedule backs it up. Their forwards and attacking midfielders deserve attention if the pricing works. '
            f'And {c3} complete the top three at {v3} expected goals. '
            f'The shot volume and fixture profile combine well here, which makes their attacking assets a sensible hold through this stretch. '
        )

    if len(goals_bot10) >= 3:
        b1, b2, b3 = goals_bot10[-3], goals_bot10[-2], goals_bot10[-1]
        cb1 = _get_full_team_s5(b1.get('short', ''))
        cb2 = _get_full_team_s5(b2.get('short', ''))
        cb3 = _get_full_team_s5(b3.get('short', ''))
        vb1 = _format_goals(b1.get('total_proj_goals', 0))
        vb2 = _format_goals(b2.get('total_proj_goals', 0))
        vb3 = _format_goals(b3.get('total_proj_goals', 0))

    s5 += (
            'The other side of the story matters just as much. These are the attacks where the numbers are not offering much encouragement right now. '
            f'{cb1} sit near the bottom with a projection of {vb1} expected goals. Their setup is not producing the kind of volume you want from FPL assets over this run. '
            f'Similarly, {cb2} are projected at {vb2} expected goals. The output in the final third has been limited, and the fixtures do not give them much help either. '
            f'And at the bottom of the table, {cb3} come in at {vb3} expected goals. '
            f'Holding their attackers as regular starters through this window is a tough case to make when stronger options are available. '
        )

    s5 += (
        'The attacking picture is fairly clear. Focus on the squads at the top of the table, and be cautious with the ones sitting at the bottom. '
        f"{_get_transition('projected_goals')}"
    )

    script_data['Slide_5_Projected_Goals'] = s5
    print('✅ Slide 5 (Refined for Broadcast Flow & TTS Limits)')
except Exception as e:
    print(f'❌ Slide 5: {e}')

# ═══════════════════════════════════════════════════════════════════
# SLIDE 6 — CLEAN SHEET ODDS (7-WEEK AVERAGE)
# ═══════════════════════════════════════════════════════════════════
try:
    def _get_full_team_s6(short_name):
        try:
            return next((t['name'] for t in boot_data['teams'] if t['short_name'] == short_name), short_name)
        except:
            return short_name

    def _format_pct(val):
        # Converts 45.2 to "45 point 2" so Edge-TTS reads decimals perfectly
        return str(round(float(val), 1)).replace(".", " point ")

    s6 = (
        f"The clean sheet map covers the next {WEEKS_PHRASE} of defensive probability across all clubs. "
        f"This uses opponent expected goals to calculate the likelihood of a shutout, which is a more accurate baseline than just looking at recent defensive form. "
    )

    if len(cs_top10) >= 3:
        t1, t2, t3 = cs_top10[0], cs_top10[1], cs_top10[2]
        c1 = _get_full_team_s6(t1.get('short', ''))
        c2 = _get_full_team_s6(t2.get('short', ''))
        c3 = _get_full_team_s6(t3.get('short', ''))
        v1 = _format_pct(t1.get('avg_cs', 0))
        v2 = _format_pct(t2.get('avg_cs', 0))
        v3 = _format_pct(t3.get('avg_cs', 0))

    s6 += (
            f"The strongest clean sheet probability over this stretch belongs to {c1}, with an average of {v1} percent. "
            f"The defensive structure is consistently restricting chances, and the data backs that up. Their defenders and goalkeeper look like dependable starting options. "
            f"Just behind them, {c2} project at {v2} percent. "
            f"The defensive numbers are solid here, and the fixture run supports that view. Their back line is worth holding or targeting if the schedule fits your plan. "
            f"And {c3} complete the top three at {v3} percent. "
            f"All three squads offer a fairly stable defensive floor over the next few weeks. "
        )

    if len(cs_bot10) >= 3:
        b1, b2, b3 = cs_bot10[-3], cs_bot10[-2], cs_bot10[-1]
        cb1 = _get_full_team_s6(b1.get('short', ''))
        cb2 = _get_full_team_s6(b2.get('short', ''))
        cb3 = _get_full_team_s6(b3.get('short', ''))
        vb1 = _format_pct(b1.get('avg_cs', 0))
        vb2 = _format_pct(b2.get('avg_cs', 0))
        vb3 = _format_pct(b3.get('avg_cs', 0))

    s6 += (
            "The other side of the table is where you start thinking seriously about reducing exposure. "
            f"At the bottom, {cb1} project at just {vb1} percent clean sheet probability. Their back line is giving up chances at a rate that makes a shutout look unlikely over this stretch. "
            f"Similarly, {cb2} come in at {vb2} percent. Their defensive output has been limited, and the upcoming fixtures do not offer much relief either. "
            f"And {cb3} sit at the bottom of the clean sheet table with {vb3} percent. "
            f"If you own defenders from any of these three clubs, it is worth asking whether the points floor is strong enough to justify the starting spot. "
        )

    s6 += (
        "The defensive picture is fairly clear. Clean sheet probability tracks the fixture map closely. "
        f"{_get_transition('cs_odds')}"
    )

    script_data['Slide_6_CS_Odds'] = s6
    print('✅ Slide 6 (Refined for Broadcast Flow & TTS Sync)')
except Exception as e:
    print(f'❌ Slide 6: {e}')

# ── SLIDE 7a: INJURY, SUSPENSION & MINUTE RISK (synced with Cell 9a) ─────
try:
    import re
    _locked = globals().get('s7a_locked_players', [])
    if not _locked:
        raise RuntimeError("s7a_locked_players is empty — run Cell 9a first")

    top_flags = []
    for d in _locked:
        p_full = next((p for p in boot_data['elements'] if p['code'] == d['code']), None)
        if not p_full:
            continue
        top_flags.append({
            'player': p_full,
            'tier': d.get('tier', 'minor_doubt'),
            'own': d.get('sel', 0),
            'chance': d.get('chance') or None,
            'status': d.get('status', 'u'),
            'news': (p_full.get('news') or '').strip(),
        })

    def _player_role(p):
        return {1: 'goalkeeper', 2: 'defender', 3: 'midfielder', 4: 'forward'}.get(p.get('element_type', 3), 'asset')

    def _player_club(p):
        tid = p.get('team', 0)
        team = next((t for t in boot_data['teams'] if t['id'] == tid), None)
        m = {
            "Brighton & Hove Albion": "Brighton",
            "Manchester United": "Man United",
            "Manchester City": "Man City",
            "Tottenham Hotspur": "Spurs",
            "Nottingham Forest": "Nottingham Forest",
        }
        if team:
            return m.get(team['name'], team['name'])
        return 'their club'

    def _news_source(news_text):
        nl = (news_text or '').lower()
        if any(k in nl for k in ('press conference', 'manager confirmed', 'said')):
            return "the latest press embargoes"
        if any(k in nl for k in ('scan', 'medical', 'specialist')):
            return "recent medical assessments"
        if any(k in nl for k in ('match', 'fixture', 'game')):
            return "match-day reporting"
        if 'season' in nl or 'long-term' in nl:
            return "official club statements"
        return "the live FPL data feed"

    def _format_pct(val):
        return str(round(float(val), 1)).replace(".", " point ")

    s7a = (
        "Now to the injury and availability picture. This is often where gameweek planning starts to unravel, so it is important to separate the serious flags from the precautionary ones. "
        "We are focusing only on the highest-priority cases, ranked by ownership and the likelihood of missing minutes. "
    )

    for idx, item in enumerate(top_flags[:5]):
        p, tier, chance, news = item['player'], item['tier'], item['chance'], item['news']
        name = p.get('web_name', '')
        role = _player_role(p)
        club = _player_club(p)
        own_spoken = _format_pct(item['own'])
        src = _news_source(news)

        clean_news = re.sub(r'\s*-\s*\d+% chance of playing.*', '', news, flags=re.IGNORECASE)
        clean_news = clean_news.rstrip('.').strip()
        if clean_news and len(clean_news) > 100:
            clean_news = clean_news[:100].rsplit(' ', 1)[0] + "..."

        if idx == 0:
            s7a += f"The most important case to flag this week is {name} from {club}. "
        elif idx == 1:
            s7a += f"The next concern is {name}, the {role} from {club}. "
        elif idx == 2:
            s7a += f"Worth monitoring closely is {name} from {club}. "
        elif idx == 3:
            s7a += f"Further down the list, {name} from {club} also needs attention. "
        else:
            s7a += f"And the final case to flag is {name} from {club}. "

        if tier == 'suspended':
            s7a += f"He is suspended for Gameweek {NEXT_GW}, so he offers nothing this round. At {own_spoken} percent ownership, the priority is to move him off the active bench and decide whether to sell before any price movement. "
        elif tier == 'ruled_out':
            s7a += f"He is ruled out for this fixture window. {own_spoken} percent of managers are still holding, but {src} is clear on the situation. "
            if clean_news:
                s7a += f"The latest update is {clean_news.lower()}. "
            s7a += "If you own him, the question is whether to sell now or wait for more information. The longer you wait, the greater the price risk becomes. "
        elif tier in ('major_doubt', 'real_doubt'):
            s7a += f"The current chance of playing is {chance} percent, which is not a number you can trust for a starting selection. At {own_spoken} percent ownership, holding him into the weekend without clarity is a real risk. "
            if clean_news:
                s7a += f"Information from {src} points to {clean_news.lower()}. "
            s7a += "The Friday press conferences are the key moment here. Hold until then, but have a plan ready if the news turns negative. "
        elif tier == 'minor_doubt':
            s7a += f"He is listed at {chance} percent to play, which places him in a grey area. With {own_spoken} percent ownership, he is not an urgent sell at this stage, but he is someone you want clarity on before starting. "
            if clean_news:
                s7a += f"The latest from {src} is {clean_news.lower()}. "
            s7a += "Wait for the pre-match updates before making any decisions here. "
        elif tier == 'returning':
            s7a += f"Good news here. The {role} appears to have come through training without any concerns and looks set to play. At {own_spoken} percent ownership, that is a straightforward situation resolved. "

    s7a += (
        "The general rule is simple. Do not make transfer decisions based on midweek speculation. "
        "The manager press conferences on Friday are where the useful information comes from. "
        "With the availability picture clearer now, let's move on to the full twenty-team breakdown."
    )

    script_data["Slide_7a_Injury_Risk"] = s7a
    print(f"✅ Slide 7a — {len(top_flags[:5])} players narrated")
except Exception as e:
    print(f"❌ Slide 7a: {e}")

# ── SLIDE 7: ALL 20 TEAMS ─────────────────────────────────────────────────
try:
    def _format_val(val):
        # Converts floats to spoken text so Edge-TTS reads decimals perfectly
        return str(round(float(val), 1)).replace(".", " point ")

    # Separate teams into PLAYING and BLANK groups
    playing_xg = [t for t in xg_ranking if t.get('matches')]
    blank_xg   = [t for t in xg_ranking if not t.get('matches')]

    playing_cs = [t for t in cs_ranking if t.get('matches')]

    # Build BGW name list
    bgw_team_names = sorted(set(t['name'] for t in blank_xg))

    # ── INTRO ────────────────────────────────────────────────────────────
    s7 = (
        f'This slide gives you the full twenty-team picture for Gameweek {NEXT_GW}. Two tables side by side, attacking projections on the left and clean sheet probability on the right. '
        f'Everything here is driven by this week\'s fixtures and the expected goals model, so it reflects the matchups in front of us rather than broader seasonal trends. '
    )

    if len(bgw_team_names) > 0:
        if len(bgw_team_names) == 1:
            names_joined = bgw_team_names[0]
        elif len(bgw_team_names) == 2:
            names_joined = f'{bgw_team_names[0]} and {bgw_team_names[1]}'
        else:
            names_joined = ', '.join(bgw_team_names[:-1]) + f', and {bgw_team_names[-1]}'
        s7 += f'Worth noting before anything else, {names_joined} have no fixture this round. Any players from those clubs need to come off your active starting eleven. '

    s7 += 'Starting with the attacking projections. '

    # ── TOP ATTACKS (PLAYING ONLY) ─────────────────────────────
    if len(playing_xg) >= 5:
        top_atk = playing_xg[:5]
        xg_0 = _format_val(top_atk[0]['xg'])
        xg_1 = _format_val(top_atk[1]['xg'])
    s7 += (
            f'At the top of the attacking table this week are {top_atk[0]["name"]} and {top_atk[1]["name"]}, projecting {xg_0} and {xg_1} expected goals respectively. '
            f'Both are producing strong shot volume against their specific opponents this round, and their forward and midfield assets deserve serious starting consideration. '
            f'{top_atk[2]["name"]}, {top_atk[3]["name"]}, and {top_atk[4]["name"]} complete the top five. '
            f'All three bring real upside this week, and any attacking picks from those squads are well supported by the underlying numbers. '
        )

    # ── BOTTOM ATTACKS (PLAYING ONLY) ──────────────────────
    if len(playing_xg) >= 3:
        bot_atk = playing_xg[-3:]
    s7 += (
            f'At the bottom of the attacking table sit {bot_atk[0]["name"]}, {bot_atk[1]["name"]}, and {bot_atk[2]["name"]}. '
            f'Their projected output this week is limited, and if you have a benching decision to make, the data points pretty clearly to where the lowest-ceiling options are. '
        )

    # ── TOP DEFENSES (PLAYING ONLY) ─────────────────────────────
    s7 += 'On the defensive side, the clean sheet probabilities for this specific gameweek tell a fairly clear story. '

    if len(playing_cs) >= 5:
        top_def = playing_cs[:5]
        cs_0 = _format_val(top_def[0]['cs'])
        cs_1 = _format_val(top_def[1]['cs'])
        cs_2 = _format_val(top_def[2]['cs'])
    s7 += (
            f'Leading the defensive table this week is {top_def[0]["name"]} at {cs_0} percent shutout probability. '
            f'{top_def[1]["name"]} and {top_def[2]["name"]} follow at {cs_1} and {cs_2} percent respectively. '
            f'These three offer the best defensive floor for this round, and their backs and goalkeepers are reliable starting options. '
            f'{top_def[3]["name"]} and {top_def[4]["name"]} also sit in a reasonable defensive position this week, which helps resolve any close calls in your backline selection. '
        )

    # ── BOTTOM DEFENSES (PLAYING ONLY) ──────────────────────
    if len(playing_cs) >= 3:
        bot_def = playing_cs[-3:]
    s7 += (
            f'At the bottom of the clean sheet table this week are {bot_def[0]["name"]}, {bot_def[1]["name"]}, and {bot_def[2]["name"]}. '
            f'Across this run, their shutout outlook is weak enough that using their defenders as your main clean sheet route is difficult to defend when stronger picks are available. '
        )

    # ── OUTRO ────────────────────────────────────────────────────────────
    s7 += (
        'That leaves you with a simple read on the table. The teams at the top of both columns are the ones to lean on at the start of the round, while the sides at the bottom are the ones you question when finalising your lineup. '
        f'{_get_transition("all_20_teams")}'
    )

    script_data['Slide_7_All_20_Teams'] = s7
    print(f'✅ Slide 7   Top atk: {[t["name"] for t in playing_xg[:5]]}   Bottom atk: {[t["name"] for t in playing_xg[-3:]]}   BGW: {bgw_team_names}')
except Exception as e:
    print(f'❌ Slide 7: {e}')

# ── SLIDE 8: TARGETS & AVOIDS (DGW/BGW aware) ───────────────────────────
try:
    import re
    TEAM_EXPANDER = globals().get('TEAM_EXPANDER', {})

    def _clean_name_s8(name):
        s = re.sub(r"^[A-Z]\.\s*", "", str(name).strip())
        s = re.sub(r"\s+[A-Z]\.?$", "", s)
        return s.strip()

    def _team_name_s8(p):
        tid = p.get('team_id', 0)
        team = next((t for t in boot_data['teams'] if t['id'] == tid), None)
        if not team: return "their club"
        m = {"Brighton & Hove Albion":"Brighton","Manchester United":"Man United",
             "Manchester City":"Man City","Tottenham Hotspur":"Spurs"}
        return m.get(team['name'], team['name'])

    def _fix_s8(p):
        raw_fix = p.get('next_opp', '').replace('[H]',' at home').replace('[A]',' away').replace(' + ',' and ')
        for _s, _f in TEAM_EXPANDER.items():
            raw_fix = re.sub(rf'\b{_s}\b', _f, raw_fix)
        return raw_fix.strip()

    def _format_val(val):
        # Ensures Edge-TTS perfectly pronounces decimals (e.g., "4 point 5")
        return str(round(float(val), 1)).replace(".", " point ")

    role_map = {"GK":"goalkeeper","DEF":"defender","MID":"midfielder","FWD":"forward"}


    # ── TARGET INTRO OPENERS (5 unique, no repetition) ───────────────────
    _tgt_openers = [
        "{name} from {club} moves straight into focus this week. The {role} is climbing rapidly in the transfer market, and the data behind that movement is difficult to ignore. ",
        "Attention also turns toward {name}, the {club} {role}. This is less about hype and more about a profile lining up perfectly with the fixture run ahead. ",
        "{name} deserves serious consideration here. The {club} {role} is still slipping under the radar for many managers, which is exactly where the upside sits. ",
        "Another player firmly catching the eye is {name} from {club}. The fixture landscape is shaping up well for this type of {role} profile right now. ",
        "And completing the target list is {name} from {club}. The underlying numbers continue to build a strong argument in his favour. ",
    ]

    # ── TARGET FIXTURE DESCRIPTORS (varied) ──────────────────────────────
    _tgt_fix_lines = [
        "Against {fix}, the model projects {xpts} expected points. The fixture setup is favourable, and the supporting metrics reinforce that projection across the board. ",
        "The upcoming meeting with {fix} carries an expected return of {xpts} points. The difficulty rating stays manageable, and the ceiling is clearly there if the trend continues. ",
        "There is genuine attacking potential in the {fix} fixture. The model lands at {xpts} expected points, and the recent data fully supports that number. ",
        "Heading into {fix}, everything in the projection model points positively. {xpts} expected points alongside an fdr of {fdr} makes this one of the more attractive fixtures on the slate. ",
        "This move is heavily driven by the {fix} matchup itself. With {xpts} expected points and a comfortable difficulty score, the case almost builds itself. ",
    ]

    # ── TARGET OWNERSHIP LINES (varied) ──────────────────────────────────
    _tgt_own_lines = [
        "Ownership currently sits at {own} percent, so moving early still creates room for meaningful rank gains before the wider market reacts. ",
        "At {own} percent ownership, this is still a position where a strong return can create immediate separation from the template squads. ",
        "With only {own} percent of managers currently invested, the punishment for missing out on a haul is beginning to grow significantly. ",
        "The {own} percent ownership figure is sitting in an ideal range. High enough to signal confidence, but still low enough to preserve the upside. ",
        "At {own} percent ownership, the profile becomes increasingly difficult to argue against. Fixture quality, expected output, and market positioning are all aligning. ",
    ]

    # ── DGW TARGET LINES (varied) ────────────────────────────────────────
    _dgw_lines = [
        "The double gameweek against {fix} changes the entire conversation around this pick. Across both fixtures the model projects {xpts} points, and at {own} percent ownership the upside becomes extremely difficult to overlook. ",
        "Everything here revolves around the double gameweek against {fix}. Two opportunities pushes the projection to {xpts} expected points, while {own} percent ownership still keeps the differential value intact. ",
        "Having two matches against {fix} immediately raises both floor and ceiling. The model lands at {xpts} expected points, and the ownership level at {own} percent still leaves substantial room for gains. ",
        "Playing twice against {fix} dramatically increases the appeal here. The projection climbs to {xpts} expected points, and with ownership sitting at {own} percent, a strong return could heavily influence rank movement. ",
        "The schedule is doing most of the heavy lifting here. Two fixtures against {fix} generates a projection of {xpts} expected points, while {own} percent ownership still keeps this from becoming fully template. ",
    ]

    # ── AVOID INTRO OPENERS (5 unique, no repetition) ────────────────────
    _avd_openers = [
        "On the opposite side of the board, {name} from {club} stands out as one of the clearest sell candidates heading into this week. ",
        "The numbers are also beginning to turn against {name} from {club}. The {role} now carries more risk than upside in the immediate schedule. ",
        "{name} from {club} is becoming increasingly difficult to justify holding. The underlying trends are no longer supporting the pick. ",
        "There is a growing argument for moving away from {name} before the deadline arrives. The fixture run is beginning to work against the {club} player quite heavily. ",
        "And the final avoid name is {name}, the {club} {role}. The short term outlook is becoming increasingly uncomfortable from an FPL perspective. ",
    ]

    # ── AVOID FIXTURE LINES (varied) ─────────────────────────────────────
    _avd_fix_lines = [
        "Against {fix}, the projection falls to just {xpts} expected points. That simply does not compare favourably against the stronger alternatives available this week. ",
        "The upcoming fixture against {fix} is where the value starts to disappear. The model projects only {xpts} points, and the difficulty score supports the concern. ",
        "Looking ahead to {fix}, the data remains unconvincing. {xpts} expected points is not the kind of projection capable of making a meaningful impact. ",
        "Facing {fix} with an fdr of {fdr}, the route to attacking returns becomes extremely narrow. The model settling at {xpts} expected points only strengthens the case to move away. ",
        "The core issue here is the matchup against {fix}. At the current difficulty level, {xpts} expected points suggests very limited room for meaningful output. ",
    ]

    # ── AVOID OWNERSHIP LINES (varied) ───────────────────────────────────
    _avd_own_lines = [
        "With {own} percent ownership still attached to him, selling now positions you ahead of what could become a much larger market shift after the deadline. ",
        "At {own} percent ownership, there is still enough flexibility to move away without immediately exposing your rank to major damage. ",
        "The {own} percent ownership can make this feel like a difficult decision emotionally, but the fixture data itself is far more decisive. ",
        "Do not let the {own} percent ownership cloud the situation. The stronger move is the one supported by the numbers, not by familiarity. ",
        "There are simply cleaner and more explosive routes available right now. At {own} percent ownership, this is beginning to look more like inertia than strategy. ",
    ]

    # ── BGW AVOID LINES (varied) ─────────────────────────────────────────
    _bgw_intros = [
        "{name} from {club} becomes an automatic problem this week due to the blank fixture. No match means no pathway to points, so the decision is largely made for you. ",
        "{club} do not play this round, which immediately pushes {name} into either bench territory or transfer consideration. Carrying inactive starters rarely ends well. ",
        "With {club} blanking this gameweek, {name} cannot contribute regardless of form. The {role} has to come out of the active lineup. ",
        "{name} from {club} blanks this week, creating a direct issue for starting eleven structure ahead of the deadline. ",
        "The blank fixture removes {name} from realistic consideration this round. Whether the solution is a bench spot or a transfer depends entirely on squad depth. ",
    ]

    _bgw_body = [
        "With {own} percent ownership still in place, many managers will simply bench him and reassess later. Acting early reduces the number of problems left to solve closer to deadline day. ",
        "At {own} percent ownership, squad depth becomes far more important than usual this week. Reliable bench cover suddenly matters a great deal more. ",
        "The {own} percent ownership figure does not change the reality of the situation. A player without a fixture cannot generate points, regardless of previous form. ",
        "Most of the managers within the {own} percent ownership group are likely to bench rather than sell. If your structure allows a proactive transfer, this is a strong opportunity to improve flexibility. ",
        "No fixture always simplifies the equation. The {own} percent ownership is irrelevant if the player is delivering zero minutes and zero returns this week. ",
    ]

    # ── BUILD NARRATION ───────────────────────────────────────────────────
    s8 = ("Now into the individual targets and avoids section. Injuries and suspensions have already been dealt with earlier, so the focus here shifts entirely toward fixture quality, projected output, and transfer direction. "
          "The left side highlights the players the data is actively encouraging managers to move toward. The right side focuses on the names beginning to drift away from the optimal path. ")

    s8 += ("At this stage of the campaign, short term scheduling starts carrying even more weight in transfer planning. "
           if globals().get('GW', 1) >= 36 else
           "The season is beginning to settle into shape now, so identifying fixture swings before the wider market reacts becomes increasingly valuable. "
           if globals().get('GW', 1) <= 3 else
           "Around this point of the season, squad structure becomes more refined and every transfer starts carrying greater strategic weight. "
           if globals().get('GW', 1) in [18, 19] else
           "")

    s8 += ("Starting with the targets. ")

    # 🔴 PRO FIX: Safely fetch from globals to prevent NameError crashes
    _target_players_pool = globals().get('target_players', globals().get('top_in', []))
    for i, p in enumerate(_target_players_pool[:5]):
        name   = _clean_name_s8(p.get('name',''))

        # Strip '%' before formatting so float() doesn't break
        raw_own = str(p.get('sel','0')).replace('%','')
        own    = _format_val(raw_own)
        xpts   = _format_val(p.get('locked_xpts', 0))
        fdr    = _format_val(p.get('next_fdr', 3.0))

        fix    = _fix_s8(p)
        role   = role_map.get(p.get('pos','').upper(), 'player')
        club   = _team_name_s8(p)
        is_dgw = p.get('is_dgw', False)

        if is_dgw:
            s8 += _tgt_openers[i % len(_tgt_openers)].format(name=name, club=club, role=role)
            s8 += _dgw_lines[i % len(_dgw_lines)].format(fix=fix, xpts=xpts, own=own)
        else:
            s8 += _tgt_openers[i % len(_tgt_openers)].format(name=name, club=club, role=role)
            s8 += _tgt_fix_lines[i % len(_tgt_fix_lines)].format(fix=fix, fdr=fdr, xpts=xpts)
            s8 += _tgt_own_lines[i % len(_tgt_own_lines)].format(own=own)

    s8 += ("Those are the names the projection model is clearly favouring this week. "
           "Now shifting across to the players where the data is moving in the opposite direction. ")

    # 🔴 PRO FIX: Safely fetch avoids
    _avoid_players_pool = globals().get('avoid_players', globals().get('top_out', []))
    for i, p in enumerate(_avoid_players_pool[:5]):
        name   = _clean_name_s8(p.get('name',''))

        raw_own = str(p.get('sel','0')).replace('%','')
        own    = _format_val(raw_own)
        xpts   = _format_val(p.get('locked_xpts', 0))
        fdr    = _format_val(p.get('next_fdr', 5.0))

        fix    = _fix_s8(p)
        role   = role_map.get(p.get('pos','').upper(), 'player')
        club   = _team_name_s8(p)
        is_bgw = p.get('is_bgw', False)

        if is_bgw:
            s8 += _bgw_intros[i % len(_bgw_intros)].format(name=name, club=club, role=role)
            s8 += _bgw_body[i % len(_bgw_body)].format(own=own)
        else:
            s8 += _avd_openers[i % len(_avd_openers)].format(name=name, club=club, role=role)
            s8 += _avd_fix_lines[i % len(_avd_fix_lines)].format(fix=fix, fdr=fdr, xpts=xpts)
            s8 += _avd_own_lines[i % len(_avd_own_lines)].format(own=own)

    s8 += ("That completes the targets and avoids picture for this gameweek. ")

    s8 += ("With the season now entering its final stretch, protecting rank becomes just as important as chasing upside. "
           if globals().get('GW', 1) >= 36 else
           "There is still a long runway left in the season, so the priority remains building strong foundations rather than reacting emotionally to single week swings. "
           if globals().get('GW', 1) <= 3 else
           "At this stage of the year, small efficiency gains in transfer decisions often create the biggest differences over the remaining weeks. "
           if globals().get('GW', 1) in [18, 19] else
           "")

    s8 += ("And this being the final gameweek of the season means there is very little value left in holding transfers back or protecting rank cautiously. "
           if globals().get('GW', 1) == 38 else
           "")

    s8 += ("The strongest decisions are usually the simplest ones. "
           "Trust the fixture quality, trust the projection data, and avoid forcing moves that need excessive justification. "
           f"{_get_transition('targets_avoids')}")

    script_data["Slide_8_Targets_Avoids"] = s8
    print(f"✅ Slide 8 (Refactored for Broadcast Flow & Safely Fetched Targets)")
except Exception as e:
    print(f"❌ Slide 8: {e}")

# ── LIVE FIXTURE STATUS CHECKER ──────────────────────────────────────────
def _team_has_played(team_id, gw):
    """
    Returns:
        True   – fixture already completed / officially recorded
        False  – fixture scheduled but still to be played or currently live
        None   – team does not feature this gameweek (BGW)

    Early season logic:
        Keeps track of opening fixtures while the campaign is still taking shape
        and sample sizes remain limited.

    Mid season logic:
        Useful around the halfway point when fixture timing, rotation,
        and scheduling pressure become far more important to monitor.

    Late season logic:
        Critical during the closing stretch when every completed fixture
        heavily impacts rank movement and chip strategy.

    Final gameweek logic:
        Particularly important on the final gameweek, where simultaneous
        kickoffs and live swings can rapidly reshape overall standings.
    """
    gw_fixes = [f for f in fix_data
                if f.get('event') == gw
                and (f['team_h'] == team_id or f['team_a'] == team_id)]

    if not gw_fixes:
        return None   # BGW

    return any(
        f.get('finished', False)
        or f.get('finished_provisional', False)
        for f in gw_fixes
    )

# ── SLIDE 9: GW REVIEW ────────────────────────────────────────────────────
try:
    def _clean_s9(web_name):
        s = str(web_name).strip()
        s = re.sub(r'^[A-Z]\.\s*', '', s)
        s = re.sub(r'\s+[A-Z]\.?$', '', s)
        return s.strip()

    def _full_team_s9(team_id):
        t = next((t for t in boot_data['teams'] if t['id'] == team_id), None)
        if not t: return "their club"
        _map = {
            "Brighton & Hove Albion":  "Brighton",
            "Manchester United":       "Manchester United",
            "Manchester City":         "Manchester City",
            "Tottenham Hotspur":       "Tottenham",
            "Nottingham Forest":       "Nottingham Forest",
            "Wolverhampton Wanderers": "Wolves",
            "West Ham United":         "West Ham",
            "Newcastle United":        "Newcastle",
            "Leicester City":          "Leicester",
            "Crystal Palace":          "Crystal Palace",
            "Ipswich Town":            "Ipswich",
            "Aston Villa":             "Aston Villa",
        }
        return _map.get(t['name'], t['name'])

    def _clean_opp_s9(opp_str):
        if not opp_str or opp_str == 'BLANK': return 'no fixture'
        _code_map = {
            "ARS":"Arsenal","AVL":"Aston Villa","BOU":"Bournemouth",
            "BRE":"Brentford","BHA":"Brighton","CHE":"Chelsea",
            "CRY":"Crystal Palace","EVE":"Everton","FUL":"Fulham",
            "IPS":"Ipswich","LEI":"Leicester","LIV":"Liverpool",
            "MCI":"Manchester City","MUN":"Manchester United",
            "NEW":"Newcastle","NFO":"Nottingham Forest",
            "SOU":"Southampton","TOT":"Tottenham","WHU":"West Ham",
            "WOL":"Wolves","SUN":"Sunderland","BUR":"Burnley",
        }
        result = (opp_str
                  .replace('(H)', 'at home').replace('(A)', 'away')
                  .replace('[H]', 'at home').replace('[A]', 'away')
                  .replace(' + ', ' and '))
        for code, full in _code_map.items():
            result = re.sub(rf'\b{code}\b', full, result)
        return result.strip()

    # ── PULL LIVE SQUAD DATA ─────────────────────────────────────────
    _s9_all      = s9_starters + s9_bench
    _active_chip = globals().get('s9_payload', {}).get('active_chip')

    # ── LIVE GAMEWEEK CONTEXT (Single, Double, Blank) ─────────────────
    _curr_fixes = [f for f in fix_data if f.get('event') == CURR_GW]
    _teams_in_gw = {}
    for f in _curr_fixes:
        _teams_in_gw[f['team_h']] = _teams_in_gw.get(f['team_h'], 0) + 1
        _teams_in_gw[f['team_a']] = _teams_in_gw.get(f['team_a'], 0) + 1

    _is_global_bgw = len(_curr_fixes) == 0
    _is_global_dgw = any(count > 1 for count in _teams_in_gw.values())

    # Find starters who have fixtures that are not yet finished
    _yet_to_play = []
    for p in s9_starters:
        _tid = p.get('team_id', p.get('team', 0))
        _player_fixes = [f for f in _curr_fixes if f['team_h'] == _tid or f['team_a'] == _tid]

        _has_remaining = False
        for f in _player_fixes:
            _finished = f.get('finished', False) or f.get('finished_provisional', False)
            if not _finished:
                _has_remaining = True
                break

        if _has_remaining:
            _yet_to_play.append(p)

    # ── DYNAMIC GW INSIGHT NARRATION ───────────────────────────────────
    _gw_insight = ""
    if _is_global_bgw:
        _gw_insight = f"Gameweek {CURR_GW} is confirmed as a Blank Gameweek, so there are no official fixtures to track right now. "
    elif _yet_to_play:
        _by_team = {}
        for p in _yet_to_play:
            _tname = _full_team_s9(p.get('team_id', p.get('team', 0)))
            _pname = _clean_s9(p['web_name'])
            _by_team.setdefault(_tname, []).append(_pname)

        _remaining_statements = []
        for _tname, _pnames in _by_team.items():
            if len(_pnames) == 1:
                _remaining_statements.append(f"{_pnames[0]} from {_tname}")
            elif len(_pnames) == 2:
                _remaining_statements.append(f"{_pnames[0]} and {_pnames[1]} from {_tname}")
            else:
                _joined = ", ".join(_pnames[:-1]) + f", and {_pnames[-1]}"
                _remaining_statements.append(f"{_joined} from {_tname}")

        _joined_statements = ", ".join(_remaining_statements[:-1]) + f", and {_remaining_statements[-1]}" if len(_remaining_statements) > 1 else _remaining_statements[0]

        if _is_global_dgw:
            _gw_insight = f"This is a Double Gameweek, and the picture is still unfinished. We are still waiting on active minutes from {_joined_statements}. "
        else:
            _gw_insight = f"The gameweek is still live, with remaining fixtures still to be completed by {_joined_statements}. "

    # ── LIVE CAPTAIN DETECTION BY PLAYER ID ──────────────────────────
    _s9_cap  = next((p for p in _s9_all if p.get('is_cap')), None)

    # Hero/Villain selection (excluding players who haven't played yet)
    _s9_played_all = [p for p in _s9_all if p not in _yet_to_play]
    _s9_hero = max(_s9_played_all, key=lambda p: p.get('pts', 0) * p.get('multiplier', 1), default=None) if _s9_played_all else None

    _s9_starters_played = [p for p in s9_starters if p.get('multiplier', 1) > 0 and p not in _yet_to_play]
    _s9_villain = min(_s9_starters_played, key=lambda p: p.get('pts', 0), default=None) if _s9_starters_played else None

    # ── LIVE POINTS CALCULATION ──────────────────────────────────────
    _s9_gw_pts = sum(
        p.get('pts', 0) * p.get('multiplier', 1)
        for p in s9_starters
        if p.get('multiplier', 0) > 0
    )

    def _pos_int_s9(p):
        pt = p.get('pos', p.get('element_type', 3))
        if isinstance(pt, str):
            return {"GK":1,"DEF":2,"MID":3,"FWD":4}.get(pt.upper(), 3)
        return int(pt)

    _s9_gk  = [_clean_s9(p['web_name']) for p in s9_starters if _pos_int_s9(p) == 1]
    _s9_def = [_clean_s9(p['web_name']) for p in s9_starters if _pos_int_s9(p) == 2]
    _s9_mid = [_clean_s9(p['web_name']) for p in s9_starters if _pos_int_s9(p) == 3]
    _s9_fwd = [_clean_s9(p['web_name']) for p in s9_starters if _pos_int_s9(p) == 4]
    _s9_bnch = [_clean_s9(p['web_name']) for p in s9_bench]

    def _s9_join(names):
        if not names: return "nobody"
        if len(names) == 1: return names[0]
        return ", ".join(names[:-1]) + f" and {names[-1]}"

    # ── DYNAMIC CHIP INTRO ───────────────────────────────────────────
    if _active_chip == 'freehit':
        _intro_line = (
            f"Gameweek {CURR_GW} is in the books. "
            f"This was a deliberate Free Hit setup, built for a one week strike at the best fixture upside. "
            f"Here is how the move played out. "
        )
    elif _active_chip == 'wildcard':
        _intro_line = (
            f"Gameweek {CURR_GW} is in the books. "
            f"The Wildcard was activated to reshape the squad from the ground up. "
            f"Here is how the new structure performed. "
        )
    elif _active_chip == 'bboost':
        _intro_line = (
            f"Gameweek {CURR_GW} is in the books. "
            f"The Bench Boost was used to squeeze value from the full fifteen man squad. "
            f"Here is how the depth responded. "
        )
    elif _active_chip == '3xc':
        _tc_cap_name = _clean_s9(_s9_cap['web_name']) if _s9_cap else "our captain"
        _tc_cap_team = _full_team_s9(_s9_cap.get('team_id', _s9_cap.get('team', 0))) if _s9_cap else ""
        _tc_cap_pts  = (_s9_cap.get('pts', 0) * _s9_cap.get('multiplier', 3)) if _s9_cap else 0
        _tc_base_pts = _s9_cap.get('pts', 0) if _s9_cap else 0
        _intro_line = (
            f"Gameweek {CURR_GW} is in the books. "
            f"This was a Triple Captain week, and the chip was placed on {_tc_cap_name} from {_tc_cap_team}. "
            f"He returned {_tc_base_pts} base points, which rose to {_tc_cap_pts} with the multiplier. "
            f"Here is the full breakdown of the squad's return. "
        )
    else:
        _intro_line = (
            f"Gameweek {CURR_GW} is in the books. "
            f"Let us break down how the locked squad handled the swing of the weekend. "
        )

    # ── DYNAMIC CAPTAIN LINE ─────────────────────────────────────────
    _cap_line = ""
    if _s9_cap and _s9_cap not in _yet_to_play:
        _cap_pts   = _s9_cap.get('pts', 0) * _s9_cap.get('multiplier', 1)
        _cap_name  = _clean_s9(_s9_cap['web_name'])
        _cap_team  = _full_team_s9(_s9_cap.get('team_id', _s9_cap.get('team', 0)))
        _cap_opp   = _clean_opp_s9(_s9_cap.get('next_opp', ''))
        _cap_mult  = _s9_cap.get('multiplier', 1)
        _cap_base  = _s9_cap.get('pts', 0)

        if _cap_mult == 3:
            if _cap_pts >= 30:
                _cap_line = (f"The Triple Captain armband delivered exactly as hoped. {_cap_name} from {_cap_team} scored {_cap_base} base points against {_cap_opp}, "
                             f"which became a huge {_cap_pts} points after the multiplier. A premium chip call. ")
            elif _cap_pts >= 18:
                _cap_line = (f"The Triple Captain on {_cap_name} from {_cap_team} produced a healthy {_cap_base} base points against {_cap_opp}, "
                             f"turning into {_cap_pts} points overall. A strong return from the chip. ")
            else:
                _cap_line = (f"The Triple Captain on {_cap_name} did not quite click. He managed {_cap_base} base points against {_cap_opp}, "
                             f"leaving us with {_cap_pts} points after the multiplier. A frustrating outcome. ")
        elif _cap_mult == 2:
            if _cap_pts >= 24:
                _cap_line = f"The captaincy paid off emphatically. {_cap_name} from {_cap_team} delivered {_cap_pts} points against {_cap_opp}. "
            elif _cap_pts >= 12:
                _cap_line = f"{_cap_name} from {_cap_team} carried the armband well and returned {_cap_pts} points against {_cap_opp}. "
            else:
                _cap_line = f"The captaincy did not land this week. {_cap_name} managed only {_cap_pts} points against {_cap_opp}. "
        else:
            _cap_line = f"{_cap_name} added {_cap_pts} points against {_cap_opp}. "

    # ── HERO ──────────────────────────────────────────────────────────
    _hero_line = ""
    if _s9_hero:
        _hero_name   = _clean_s9(_s9_hero['web_name'])
        _hero_pts    = _s9_hero.get('pts', 0) * _s9_hero.get('multiplier', 1)
        _hero_team   = _full_team_s9(_s9_hero.get('team_id', _s9_hero.get('team', 0)))

        if _hero_pts >= 6:
            _hero_line = f"The standout performer this gameweek was {_hero_name} from {_hero_team}, who delivered {_hero_pts} points and drove the haul. "
        elif _hero_pts >= 2:
            _hero_line = f"{_hero_name} from {_hero_team} added a steady {_hero_pts} points to the total this week. "

    # ── VILLAIN ───────────────────────────────────────────────────────
    _villain_line = ""
    if _s9_villain:
        _villain_name   = _clean_s9(_s9_villain['web_name'])
        _villain_team   = _full_team_s9(_s9_villain.get('team_id', _s9_villain.get('team', 0)))
        _villain_pts    = _s9_villain.get('pts', 0)

        if _villain_pts == 0:
            _villain_line = f"On the other side of it, {_villain_name} from {_villain_team} blanked completely. That is one to watch heading into the next round. "
        elif _villain_pts <= 2:
            _villain_line = f"{_villain_name} from {_villain_team} posted the lowest return of the week with just {_villain_pts} points. Worth a second look before the next deadline. "

    # ── DYNAMIC RANK ARROWS ──────────────────────────────────────────
    _rank_diff_text = ""
    try:
        _hist_r  = requests.get(
            f"https://fantasy.premierleague.com/api/entry/{globals().get('MANAGER_ID', 664987)}/history/",
            timeout=5
        ).json()
        _c_rank  = next(x['overall_rank'] for x in _hist_r['current'] if x['event'] == CURR_GW)
        _p_rank  = next(x['overall_rank'] for x in _hist_r['current'] if x['event'] == CURR_GW - 1)
        _rank_delta = _p_rank - _c_rank
        if _c_rank < _p_rank:
            _rank_diff_text = f"Importantly, that score delivered a green arrow, climbing {abs(_rank_delta):,} places to an overall rank of {_c_rank:,}. "
        elif _c_rank > _p_rank:
            _rank_diff_text = f"Unfortunately, the week produced a red arrow, slipping {abs(_rank_delta):,} places to an overall rank of {_c_rank:,}. "
        else:
            _rank_diff_text = f"Our overall rank stayed exactly where it was at {_c_rank:,}. "
    except:
        pass

    # ── DYNAMIC VERDICT ───────────────────────────────────────────────
    if _s9_gw_pts >= 100:
        _verdict = "A huge gameweek. The chip call and the structure combined for a truly elite haul."
    elif _s9_gw_pts >= 80:
        _verdict = "A very strong return, comfortably above the weekly standard. The model got plenty right."
    elif _s9_gw_pts >= 60:
        _verdict = "A respectable return above average. The squad stayed on course."
    elif _s9_gw_pts >= 45:
        _verdict = "A middling week, not disastrous but not decisive either. The numbers will steer the response."
    else:
        _verdict = "A poor gameweek. The data now needs a hard reset before the next deadline is locked in."

    # ── ASSEMBLE TENSE LOGIC ──────────────────────────────────────────
    if _yet_to_play:
        _points_wrap = f"So far, this version has already reached {_s9_gw_pts} points. "
        _closing_line = f"The gameweek is still live, but the next deadline is already coming into view. {_get_transition('gw_review')}"
    else:
        _points_wrap = f"Once the dust settled, this version finished on {_s9_gw_pts} points. "
        _closing_line = f"With the round complete and the squad resetting, it is time to look ahead. {_get_transition('gw_review')}"

    # ── FINAL ASSEMBLED NARRATION ─────────────────────────────────────
    s9 = (
        f"{_intro_line}"
        f"Anchoring things in goal, {_s9_join(_s9_gk)} carried the defensive foundation. "
        f"The back line was built around {_s9_join(_s9_def)} as the structure at the back. "
        f"In midfield, {_s9_join(_s9_mid)} were the main creative force driving the attack. "
        f"Up front, {_s9_join(_s9_fwd)} were tasked with converting the chance volume into points. "
        f"On the bench, {_s9_join(_s9_bnch)} provided the necessary cover. "
        f"{_cap_line}"
        f"{_hero_line}"
        f"{_villain_line}"
        f"{_gw_insight}"
        f"{_points_wrap}"
        f"{_rank_diff_text}"
        f"{_verdict} "
        f"{_closing_line}"
    )

    script_data["Slide_9_Base"] = s9
    print(f"✅ Slide 9 (GW{CURR_GW} Review — {_s9_gw_pts}pts — Chip: {_active_chip or 'none'} — Fully Dynamic)")

except Exception as e:
    print(f"❌ Slide 9: {e}")

# ── SLIDE 10: NEXT GW PLAN ────────────────────────────────────────────────
try:
    def _clean_s10(web_name):
        s = str(web_name).strip()
        s = re.sub(r'^[A-Z]\.\s*', '', s)
        s = re.sub(r'\s+[A-Z]\.?$', '', s)
        return s.strip()

    def _full_team_s10(team_id):
        t = next((t for t in boot_data['teams'] if t['id'] == team_id), None)
        if not t: return "their club"
        _map = {
            "Brighton & Hove Albion":  "Brighton",
            "Manchester United":       "Manchester United",
            "Manchester City":         "Manchester City",
            "Tottenham Hotspur":       "Tottenham",
            "Nottingham Forest":       "Nottingham Forest",
            "Wolverhampton Wanderers": "Wolves",
            "West Ham United":         "West Ham",
            "Newcastle United":        "Newcastle",
        }
        return _map.get(t['name'], t['name'])

    def _clean_opp_s10(opp_str):
        if not opp_str or opp_str == 'BLANK': return 'no fixture this gameweek'
        _code_map = {
            "ARS":"Arsenal","AVL":"Aston Villa","BOU":"Bournemouth",
            "BRE":"Brentford","BHA":"Brighton","CHE":"Chelsea",
            "CRY":"Crystal Palace","EVE":"Everton","FUL":"Fulham",
            "IPS":"Ipswich","LEI":"Leicester","LIV":"Liverpool",
            "MCI":"Manchester City","MUN":"Manchester United",
            "NEW":"Newcastle","NFO":"Nottingham Forest",
            "SOU":"Southampton","TOT":"Tottenham","WHU":"West Ham",
            "WOL":"Wolves"
        }
        result = (opp_str
                  .replace('(H)', 'at home').replace('(A)', 'away')
                  .replace('[H]', 'at home').replace('[A]', 'away')
                  .replace(' + ', ' and ').replace('/', ' and '))
        for code, full in _code_map.items():
            result = re.sub(rf'\b{code}\b', full, result)
        return result.strip()

    def _format_val(val):
        return str(round(float(val), 1)).replace(".", " point ")

    def _pos_int_s10(p):
        pt = p.get('pos', p.get('element_type', 3))
        if isinstance(pt, str):
            return {"GK":1,"DEF":2,"MID":3,"FWD":4}.get(pt.upper(), 3)
        return int(pt)

    # ── LIVE FINANCIALS FROM API PAYLOAD ─────────────────────────────
    _hist      = globals().get('s9_payload', {}).get('entry_history', {})
    _bank      = _hist.get('bank', 0) / 10.0
    _value     = _hist.get('value', 1000) / 10.0
    _free_xfers = _hist.get('event_transfers', 0)

    # ── LIVE CHIP DETECTION FOR NEXT GW PLAN ─────────────────────────
    # Detect remaining chips from event history
    _used_chips = set()
    try:
        _hist_full = requests.get(
            f"https://fantasy.premierleague.com/api/entry/{globals().get('MANAGER_ID', 664987)}/history/",
            timeout=5
        ).json()
        for _ev in _hist_full.get('chips', []):
            _used_chips.add(_ev.get('name'))
    except:
        pass

    # Remaining chips the manager still has available
    _all_chips  = {'wildcard', 'freehit', 'bboost', '3xc'}
    _remaining  = _all_chips - _used_chips
    _chip_names = {
        'wildcard': 'Wildcard',
        'freehit':  'Free Hit',
        'bboost':   'Bench Boost',
        '3xc':      'Triple Captain'
    }

    # ── LIVE CAPTAIN DETECTION BY PLAYER ID ──────────────────────────
    _s10_cap = next((p for p in s10_starters if p.get('is_cap')), None)
    _s10_vc  = next((p for p in s10_starters if p.get('is_vc')),  None)

    _cap_name  = _clean_s10(_s10_cap['web_name']) if _s10_cap else "our captain"
    _cap_team  = _full_team_s10(_s10_cap.get('team_id', _s10_cap.get('team', 0))) if _s10_cap else ""
    _cap_xp    = _format_val(_s10_cap.get('ep_next', 0)) if _s10_cap else "0"
    _cap_fix   = _clean_opp_s10(_s10_cap.get('next_opp', '')) if _s10_cap else "their fixture"
    _cap_own   = str(_s10_cap.get('selected_by_percent', '0')) if _s10_cap else "0"

    _vc_name   = _clean_s10(_s10_vc['web_name']) if _s10_vc else "our vice captain"
    _vc_team   = _full_team_s10(_s10_vc.get('team_id', _s10_vc.get('team', 0))) if _s10_vc else ""
    _vc_fix    = _clean_opp_s10(_s10_vc.get('next_opp', '')) if _s10_vc else "their fixture"
    _vc_xp     = _format_val(_s10_vc.get('ep_next', 0)) if _s10_vc else "0"

    # ── CHECK IF CAP HAS DGW (live fixture count) ─────────────────────
    _cap_team_id   = _s10_cap.get('team_id', _s10_cap.get('team', 0)) if _s10_cap else 0
    _cap_fixtures  = [f for f in fix_data
                      if f.get('event') == NEXT_GW
                      and (_cap_team_id == f['team_h'] or _cap_team_id == f['team_a'])]
    _cap_is_dgw    = len(_cap_fixtures) > 1

    # ── DYNAMIC CAPTAIN CONTEXT (no hardcoded player) ─────────────────
    if _cap_is_dgw:
        _cap_context = (
            f"The captaincy is locked on {_cap_name} from {_cap_team}. "
            f"He carries a confirmed double gameweek against {_cap_fix}, "
            f"projecting {_cap_xp} expected points across both appearances. "
            f"At {_cap_own} percent ownership, the risk-to-reward ratio here is heavily in our favour. "
        )
    else:
        _cap_context = (
            f"The captaincy is mathematically locked on {_cap_name} from {_cap_team}. "
            f"Facing {_cap_fix}, the model projects a baseline of {_cap_xp} expected points. "
            f"With {_cap_own} percent of the game backing him, this is the highest-confidence armband decision this round. "
        )

    _vc_context = (
        f"{_vc_name} from {_vc_team} holds the vice captaincy, "
        f"projecting {_vc_xp} expected points against {_vc_fix}. "
        f"If the captain misses out, the vice captain is perfectly positioned to absorb the damage. "
    )

    # ── DYNAMIC REMAINING CHIP ADVICE (based on actual remaining chips) ──
    _chip_advice = ""
    if _remaining:
        _rem_names = [_chip_names.get(c, c) for c in _remaining]
        if len(_rem_names) == 1:
            _chips_str = _rem_names[0]
        elif len(_rem_names) == 2:
            _chips_str = f"{_rem_names[0]} and {_rem_names[1]}"
        else:
            _chips_str = ", ".join(_rem_names[:-1]) + f", and {_rem_names[-1]}"

        # Detect upcoming blank/double gameweeks from fixture data
        _gw_team_counts = {}
        for _future_gw in range(NEXT_GW + 1, min(NEXT_GW + 5, MAX_GW + 1)):
            for _fx in fix_data:
                if _fx.get('event') != _future_gw:
                    continue
                for _tid in (_fx['team_h'], _fx['team_a']):
                    _gw_team_counts.setdefault(_future_gw, {})
                    _gw_team_counts[_future_gw][_tid] = _gw_team_counts[_future_gw].get(_tid, 0) + 1

        _dgw_gws  = [gw for gw, teams in _gw_team_counts.items()
                     if any(v > 1 for v in teams.values())]
        _blank_gws = [gw for gw, teams in _gw_team_counts.items()
                      if len(teams) < 20]

        if _dgw_gws:
            _next_dgw = min(_dgw_gws)
            _chip_advice = (
                f"With the {_chips_str} still available, the scheduling points heavily toward Gameweek {_next_dgw} "
                f"where a confirmed Double Gameweek creates major upside potential. "
                f"Well timed chip usage in these windows can completely reshape rank trajectory, particularly once the season reaches its decisive stages. "
            )
        elif _blank_gws:
            _next_bgw = min(_blank_gws)
            _chip_advice = (
                f"With the {_chips_str} still in reserve, the Free Hit should be strongly considered for Gameweek {_next_bgw} "
                f"where blank fixtures are expected to disrupt squad structure across the template. "
                f"Patience matters here. Hold those chips until the fixture calendar fully reveals the strongest entry point. "
            )
        else:
            _chip_advice = (
                f"The {_chips_str} remain available for deployment. "
                f"Track the fixture announcements closely and position those chips around the next major Double or Blank Gameweek swing. "
            )

            if globals().get('CURR_GW', 1) <= 3:
                _chip_advice += (
                    f"At this early stage of the season, preserving flexibility is often more valuable than forcing aggressive chip usage too soon. "
                )
            elif globals().get('CURR_GW', 1) in [18, 19]:
                _chip_advice += (
                    f"With the campaign approaching the halfway mark, chip strategy becomes increasingly tied to long term squad planning rather than isolated one week punts. "
                )
            elif globals().get('CURR_GW', 1) >= 36:
                _chip_advice += (
                    f"The season is now entering the closing stretch, so remaining chips carry enormous importance in deciding final rank outcomes. "
                )

            if globals().get('CURR_GW', 1) == 38:
                _chip_advice += (
                    f"And with this being the final gameweek of the campaign, there is no longer any value in holding resources back. "
                )

    else:
        _chip_advice = (
            f"Every chip has already been used this season. "
            f"From this point forward, success depends entirely on transfer precision, squad balance, and navigating the fixture swings cleanly. "
        )

        if globals().get('CURR_GW', 1) >= 36:
            _chip_advice += (
                f"With the season moving into its final weeks, even minor transfer decisions can now create massive rank swings. "
            )

        if globals().get('CURR_GW', 1) == 38:
            _chip_advice += (
                f"This being the final gameweek means there is nothing left to preserve. Every move now is purely about maximizing the final scoreline. "
            )

    # ── LIVE SQUAD POSITION GROUPS ────────────────────────────────────
    _s10_gk   = [_clean_s10(p['web_name']) for p in s10_starters if _pos_int_s10(p) == 1]
    _s10_def  = [_clean_s10(p['web_name']) for p in s10_starters if _pos_int_s10(p) == 2]
    _s10_mid  = [_clean_s10(p['web_name']) for p in s10_starters if _pos_int_s10(p) == 3]
    _s10_fwd  = [_clean_s10(p['web_name']) for p in s10_starters if _pos_int_s10(p) == 4]
    _s10_bnch = [_clean_s10(p['web_name']) for p in s10_bench]

    def _s10_join(names):
        if not names: return "nobody confirmed"
        if len(names) == 1: return names[0]
        return ", ".join(names[:-1]) + f" and {names[-1]}"

    # ── LIVE PLAYER HIGHLIGHTS (by position, ranked by ep_next) ───────
    _best_def = max(
        [p for p in s10_starters if _pos_int_s10(p) == 2],
        key=lambda x: float(x.get('ep_next', 0)), default=None
    )
    _best_mid = max(
        [p for p in s10_starters if _pos_int_s10(p) == 3],
        key=lambda x: float(x.get('ep_next', 0)), default=None
    )
    _best_fwd = max(
        [p for p in s10_starters if _pos_int_s10(p) == 4],
        key=lambda x: float(x.get('ep_next', 0)), default=None
    )

    def _highlight(p, role):
        if not p: return ""
        name = _clean_s10(p['web_name'])
        team = _full_team_s10(p.get('team_id', p.get('team', 0)))
        xpts = _format_val(p.get('ep_next', 0))
        opp  = _clean_opp_s10(p.get('next_opp', ''))
        own  = str(p.get('selected_by_percent', '0'))

        if role == 'def':
            return (
                f"At the back, the model leans heavily toward {name} from {team}. "
                f"The defensive projection lands at {xpts} expected points against {opp}, "
                f"making him one of the standout structural picks in the squad. "
            )

        elif role == 'mid':
            return (
                f"In midfield, {name} from {team} sits at the centre of the attacking projections this week. "
                f"At {own} percent ownership, the forecasted {xpts} expected points against {opp} "
                f"makes him one of the defining pieces of this build. "
            )

        elif role == 'fwd':
            return (
                f"Leading the line, {name} from {team} remains the central attacking reference point. "
                f"The model currently projects {xpts} expected points against {opp}, "
                f"placing him firmly among the premium forward targets for the round. "
            )

        return ""

    # ── LIVE WATCHLIST TARGETS ─────────────────────────────────────────
    _target_lines = []
    for _tp in top_targets[:4]:
        _tn    = _clean_s10(_tp['web_name'])
        _tteam = _full_team_s10(_tp.get('team_id', 0))
        _txp   = _format_val(_tp.get('ep_next', 0))
        _tfix  = _clean_opp_s10(_tp.get('next_opp', ''))

        _target_lines.append(
            f"{_tn} from {_tteam}, carrying a projection of {_txp} expected points against {_tfix}"
        )

    _target_block = ". ".join(_target_lines) + "." if _target_lines else ""

    # ── FINAL ASSEMBLED NARRATION ──────────────────────────────────────
    s10 = (
        f"Here is the fully locked structure for Gameweek {NEXT_GW}. "
        f"Every decision inside this draft is driven directly by the Vortex fixture model, "
        f"stripping away emotion and focusing purely on projected output and fixture strength. "
        f"Between the posts, {_s10_join(_s10_gk)} has been trusted to provide the strongest defensive platform available this week. "
        f"The defensive unit is built around {_s10_join(_s10_def)}, "
        f"selected specifically for stability, clean sheet probability, and fixture alignment heading into the round. "
        f"{_highlight(_best_def, 'def')}"
        f"In midfield, {_s10_join(_s10_mid)} carries the creative burden and most of the attacking projection. "
        f"{_highlight(_best_mid, 'mid')}"
        f"Leading the attack, {_s10_join(_s10_fwd)} has been positioned to capitalize on the strongest goal involvement projections in the model. "
        f"{_highlight(_best_fwd, 'fwd')}"
        f"Providing cover from the bench are {_s10_join(_s10_bnch)}, "
        f"giving the squad the flexibility needed to absorb unexpected rotation or late availability issues. "
        f"{_cap_context}"
        f"{_vc_context}"
    )

    if globals().get('NEXT_GW', 1) <= 3:
        s10 += (
            f"At this stage of the season, the focus is still on building long term structure rather than chasing isolated one week swings. "
        )
    elif globals().get('NEXT_GW', 1) in [18, 19]:
        s10 += (
            f"With the campaign approaching the halfway stage, squad balance and transfer efficiency become increasingly important heading into the second half of the season. "
        )
    elif globals().get('NEXT_GW', 1) >= 36:
        s10 += (
            f"The season is now entering the closing stretch, where every transfer, captaincy call, and bench decision can dramatically influence final rank outcomes. "
        )

    if globals().get('NEXT_GW', 1) == 38:
        s10 += (
            f"This is the final gameweek of the campaign, so there is no longer any value in holding back aggressive decisions or protecting unused resources. "
        )

    if _target_block:
        s10 += (
            f"Turning toward the transfer watchlist ahead of the deadline, "
            f"these are the players the model is currently flagging as priority additions. "
            f"{_target_block} "
            f"These names deserve close attention before transfers are finalized. "
        )


    # ── DYNAMIC CHIP FORWARD-PLANNING (no hardcoded GW numbers) ───────
    s10 += f"{_chip_advice}"

    s10 += (
        f"Make sure the squad is fully confirmed and locked before the Gameweek {NEXT_GW} deadline arrives. "
        f"{_get_transition('master_plan')}"
    )

    script_data["Slide_10_Plan_Base"] = s10
    print(f"✅ Slide 10 (GW{NEXT_GW} Plan — Cap: {_cap_name} — Remaining chips: {_remaining or 'none'})")

except Exception as e:
    print(f"❌ Slide 10: {e}")


# ── SLIDE 11: CHIP STRATEGY ──────────────────────────────
try:
    CURR_GW = globals().get('CURR_GW', 38)
    _h1_c = {"wildcard": 0, "freehit": 0, "bboost": 0, "3xc": 0}
    _h2_c = {"wildcard": 0, "freehit": 0, "bboost": 0, "3xc": 0}
    for ev in boot_data.get('events', []):
        _gw = ev['id']
        for cp in ev.get('chip_plays', []):
            _cname = cp['chip_name']
            if _cname in _h1_c:
                if _gw <= 19: _h1_c[_cname] += cp['num_played']
                else:         _h2_c[_cname] += cp['num_played']

    _tm = boot_data.get('total_players', 10000000) or 10000000

    def _pct(num):
        return str(round((num / _tm) * 100, 1)).replace(".", " point ")

    wc_h1 = _pct(_h1_c['wildcard'])
    wc_h2 = _pct(_h2_c['wildcard'])
    fh_h1 = _pct(_h1_c['freehit'])
    fh_h2 = _pct(_h2_c['freehit'])
    bb_h1 = _pct(_h1_c['bboost'])
    bb_h2 = _pct(_h2_c['bboost'])
    tc_h1 = _pct(_h1_c['3xc'])
    tc_h2 = _pct(_h2_c['3xc'])

    h1_tot = sum(_h1_c.values())
    h2_tot = sum(_h2_c.values())

    s11 = "This is the chip strategy overview. "
    s11 += f"In the opening phase of the season, {wc_h1} percent of managers activated their Wildcard, while {tc_h1} percent committed early to the Triple Captain. "

    if CURR_GW <= 3:
        s11 += (
            "At this very early stage of the campaign, the long season still stretches ahead, "
            "and the smartest approach is usually structural patience rather than aggressive chip usage. "
            "Early decisions tend to be about fixing balance, not chasing short term spikes. "
        )
    elif CURR_GW <= 19:
        s11 += (
            "As we move through the middle phase of the season, the focus shifts toward refinement rather than reconstruction. "
            "Most elite squads are now stable, and chips are generally being preserved for clearer opportunities later in the run. "
        )
    elif CURR_GW < 36:
        s11 += f"Now entering the second half of the season, {fh_h2} percent of managers have already deployed the Free Hit, and {wc_h2} percent have activated their second Wildcard. "
        if h2_tot > h1_tot:
            s11 += "Chip usage in the second half has now overtaken early season deployment, reflecting the increased fixture volatility as the season progresses. "
        else:
            s11 += "Interestingly, early season chip usage still holds a higher total than the second half, showing how cautious many managers were with later opportunities. "
        s11 += (
            "At this stage, timing becomes everything. Holding Bench Boost and Free Hit for confirmed doubles and blanks is often the difference between steady rank gains and explosive jumps. "
        )
    else:
        s11 += f"As we enter the final stretch of the season, {fh_h2} percent have used the Free Hit, while {bb_h2} percent have already activated the Bench Boost. "
        if h2_tot > h1_tot:
            s11 += "Late season chip deployment has clearly accelerated past early season usage, driven by high impact double gameweeks. "
        else:
            s11 += "Surprisingly, early season chip usage has remained competitive with late season activation, highlighting inconsistent timing across the template. "
        s11 += (
            "At this point in the campaign, precision timing separates rank gain from rank stagnation. "
            "The managers who aligned their chips with confirmed fixture swings are the ones who maximized their overall season outcome. "
        )

    script_data['Slide_11_Chip_Strategy'] = s11
    print('✅ Slide 11 (Dynamic Season-Aware Narration)')
except Exception as e:
    print(f'❌ Slide 11: {e}')

# ── SLIDE 12: OUTRO ───────────────────────────────────────────────────────
try:

    # ── 1. Dynamic market movers (net transfer leaders this GW) ──────────
    try:
        _players = [p for p in boot_data.get("elements", []) if p.get("status") != "u"]
        _top_in  = sorted(
            _players,
            key=lambda x: x.get("transfers_in_event", 0) - x.get("transfers_out_event", 0),
            reverse=True
        )[0]
        _top_out = sorted(
            _players,
            key=lambda x: x.get("transfers_out_event", 0) - x.get("transfers_in_event", 0),
            reverse=True
        )[0]
        dyn_in  = _top_in.get('web_name', 'our premium targets')
        dyn_out = _top_out.get('web_name', 'our red-flagged assets')
    except Exception:
        dyn_in  = "our premium targets"
        dyn_out = "the red-flagged assets"

    # ── 2. Captaincy extraction ───────────────────────────────────────────
    # FIX: This block is now at the correct 4-space scope inside the outer
    # try, NOT inside the dyn_in except clause. Previously it was indented
    # at 8 spaces, making Python place it inside except — which never fired
    # when dyn_in succeeded, leaving c_name undefined for the s12 string.
    def _clean_name_s12(raw):
        s = str(raw).strip()
        s = re.sub(r"^[A-Z]\.\s*", "", s)
        s = re.sub(r"\s+[A-Z]\.?$", "", s)
        return s.strip()

    try:
        _s12_pool  = globals().get('s10_starters', []) + globals().get('s10_bench', [])
        _cap_data  = next((p for p in _s12_pool if p.get("is_cap")), None)
        _vc_data   = next((p for p in _s12_pool if p.get("is_vc")),  None)

        c_name  = _clean_name_s12(_cap_data['web_name']) if _cap_data else "our premium captain"
        vc_name = _clean_name_s12(_vc_data['web_name'])  if _vc_data  else "our vice captain"

        # Prepend full club name for natural broadcast phrasing
        if _cap_data:
            _c_team = next(
                (t['name'] for t in boot_data['teams'] if t['id'] == _cap_data.get('team', 0)),
                ""
            )
            if _c_team:
                c_name = f"{_c_team}'s {c_name}"

        if _vc_data:
            _vc_team = next(
                (t['name'] for t in boot_data['teams'] if t['id'] == _vc_data.get('team', 0)),
                ""
            )
            if _vc_team:
                vc_name = f"{_vc_team}'s {vc_name}"

    except Exception as _cap_err:
        print(f"   ⚠️ Captaincy extraction fallback triggered: {_cap_err}")
        c_name  = "our premium captain"
        vc_name = "our vice captain"


    # ── 3. Assemble outro narration ───────────────────────────────────────
    s12 = (
        f"That wraps up Gameweek {NEXT_GW}. Let's pull the whole picture together. "
        f"From the opening stretch to the midpoint grind and all the way into the run-in, the same principle holds: attack the fixtures that are screaming for returns. "
        f"This video has already shown you where the high expected goals zones are, so trust the edge and lean into them. "
        f"Next, get the moves done with intent. "
        f"Players like {dyn_in} are drawing serious attention for a reason, and every hour closer to deadline makes that edge harder to capture. "
        f"Then cut loose the weak links. "
        f"Assets like {dyn_out} are staring at a rough setup, and letting poor fixtures drag on is exactly how rank slips away over time. "
        f"If your squad is already in good shape, there is real value in sitting tight and banking the transfer. "
        f"Going into the next round with an extra move can be a major weapon, especially when rotation starts to bite and the season tightens up. "
        f"And finally, captaincy. The model is backing {c_name} this week, "
        f"with {vc_name} ready as the safety net if anything shifts before deadline. "
        f"If you have a transfer dilemma or a squad call you want pressure-tested, drop it in the comments. "
        f"I go through them before the deadline and either reply directly or feature the best ones in the next video. "
        f"Good luck with the gameweek."
    )

    script_data['Slide_12_Outro'] = s12
    print(f"✅ Slide 12 — Cap: {c_name} | VC: {vc_name}")

except Exception as e:
    print(f"❌ Slide 12: {e}")
# ═══════════════════════════════════════════════════════════════════
# TTS SAFETY SWEEP 1 — Strip "X." prefixes (e.g., J. Timber -> Timber)
# ═══════════════════════════════════════════════════════════════════
for _slide_key in list(script_data.keys()):
    _txt = script_data[_slide_key]
    # The \s* ensures it catches names with a space after the dot
    _txt = re.sub(r'\b([A-Z])\.\s*([A-Za-z]+)', r'\2', _txt)
    script_data[_slide_key] = _txt
print("✅ TTS sweep 1 applied — initial-dot prefixes stripped")

# ═══════════════════════════════════════════════════════════════════
# TTS SAFETY SWEEP 2 — Expand team short codes to full names (e.g., BUR -> Burnley)
# ═══════════════════════════════════════════════════════════════════
_TEAM_CODE_TO_FULL = {t["short_name"]: t["name"] for t in boot_data["teams"]}

_TEAM_CODE_TO_FULL["BHA"] = "Brighton"
_TEAM_CODE_TO_FULL["WHU"] = "West Ham"
_TEAM_CODE_TO_FULL["TOT"] = "Spurs"
_TEAM_CODE_TO_FULL["MUN"] = "Manchester United"
_TEAM_CODE_TO_FULL["MCI"] = "Manchester City"
_TEAM_CODE_TO_FULL["NEW"] = "Newcastle"
_TEAM_CODE_TO_FULL["NFO"] = "Forest"
_TEAM_CODE_TO_FULL["WOL"] = "Wolves"
_TEAM_CODE_TO_FULL["LEI"] = "Leicester"
_TEAM_CODE_TO_FULL["LEE"] = "Leeds"
_TEAM_CODE_TO_FULL["BOU"] = "Bournemouth"
_TEAM_CODE_TO_FULL["BUR"] = "Burnley"
_TEAM_CODE_TO_FULL["IPS"] = "Ipswich"
_TEAM_CODE_TO_FULL["SOU"] = "Southampton"
_TEAM_CODE_TO_FULL["SUN"] = "Sunderland"
_TEAM_CODE_TO_FULL["FUL"] = "Fulham"
_TEAM_CODE_TO_FULL["EVE"] = "Everton"
_TEAM_CODE_TO_FULL["CHE"] = "Chelsea"
_TEAM_CODE_TO_FULL["ARS"] = "Arsenal"
_TEAM_CODE_TO_FULL["LIV"] = "Liverpool"
_TEAM_CODE_TO_FULL["BRE"] = "Brentford"
_TEAM_CODE_TO_FULL["CRY"] = "Crystal Palace"
_TEAM_CODE_TO_FULL["AVL"] = "Aston Villa"

_sorted_codes = sorted(_TEAM_CODE_TO_FULL.keys(), key=len, reverse=True)

for _slide_key in list(script_data.keys()):
    _txt = script_data[_slide_key]
    for _code in _sorted_codes:
        _full = _TEAM_CODE_TO_FULL[_code]  # <-- Restored the missing variable to fix short codes!
        _txt = re.sub(rf'\b{_code}\b', _full, _txt)
    script_data[_slide_key] = _txt
print("✅ TTS sweep 2 applied — team short codes expanded to full names")

# ── GENERATE ALL AUDIO + VTT ───────────────────────────────────────────────
print(f"\n🎙️ Generating audio for {len(script_data)} slides...")

async def generate_all():
    import asyncio
    import subprocess

    # Helper: Retry logic for TTS to prevent random network crashes
    async def _generate_with_retry(text, audio_out, vtt_out, words_out, retries=3):
        for attempt in range(retries):
            try:
                comm = edge_tts.Communicate(text, VOICE, rate=RATE, pitch=PITCH, boundary="WordBoundary")
                sub_maker = edge_tts.SubMaker()
                word_list = []

                with open(audio_out, "wb") as f:
                    async for chunk in comm.stream():
                        if chunk["type"] == "audio":
                            f.write(chunk["data"])
                        elif chunk["type"] == "WordBoundary":
                            off_s = chunk["offset"] / 10_000_000
                            dur_s = chunk["duration"] / 10_000_000
                            word_list.append({
                                "text":  chunk["text"],
                                "start": round(off_s, 3),
                                "end":   round(off_s + dur_s, 3),
                            })
                            try: sub_maker.feed(chunk)
                            except Exception: pass
                        elif chunk["type"] == "SentenceBoundary":
                            try: sub_maker.feed(chunk)
                            except Exception: pass

                with open(vtt_out, "w", encoding="utf-8") as f:
                    f.write(sub_maker.get_srt().replace(',', '.'))

                with open(words_out, "w", encoding="utf-8") as f:
                    json.dump(word_list, f, ensure_ascii=False)

                return True # Success
            except Exception as e:
                if attempt == retries - 1:
                    print(f" ❌ Failed after {retries} attempts: {e}")
                    return False
                await asyncio.sleep(2) # Wait before retry

    # Step 1 — generate all slides EXCEPT slide 1 first
    # ── MODULE-AWARE FILTER ──────────────────────────────────────────────
    _active_mods = globals().get('final_modules', None)
    if _active_mods:
        _mod_reg = globals().get('MODULE_REGISTRY', {})
        _active_keys = {_mod_reg[m]['slide_key'] for m in _active_mods if m in _mod_reg}
        print(f"   🔎 Module-aware filter active — {len(_active_keys)} slides queued")
    else:
        _active_keys = None  # No filter — generate all

    other_slides = {
        k: v for k, v in script_data.items()
        if k != "Slide_1_Intro" and (_active_keys is None or k in _active_keys)
    }

    for slide_name, text in other_slides.items():
        clean = expand(
            text.replace("[MUSIC CUE: intro.mp3] ","")
                .replace("[MUSIC CUE: mid.mp3] ","")
                .replace("[MUSIC CUE: outro.mp3] ","")
        )
        audio_path = os.path.join(AUDIO_DIR, f"{slide_name}.mp3")
        vtt_path   = os.path.join(AUDIO_DIR, f"{slide_name}.vtt")
        words_path = os.path.join(AUDIO_DIR, f"{slide_name}.words.json")

        print(f"  🎙️ {slide_name}...", end="", flush=True)
        success = await _generate_with_retry(clean, audio_path, vtt_path, words_path)
        if success:
            size = os.path.getsize(audio_path) / 1024
            print(f" ✅ ({size:.0f} KB)")

    # Step 2 — measure real total duration
    total_secs = 0
    for _f in os.listdir(AUDIO_DIR):
        if not _f.endswith('.mp3') or _f in ['intro.mp3','mid.mp3','outro.mp3']: continue
        _r = subprocess.run(
            ["ffprobe","-v","error","-show_entries","format=duration",
             "-of","default=noprint_wrappers=1:nokey=1",
             os.path.join(AUDIO_DIR,_f)],
            capture_output=True, text=True)
        try: total_secs += float(_r.stdout.strip())
        except: pass

    total_mins = int(total_secs / 60)
    real_dur_str = (
        "around ten minutes"          if total_mins<=10 else
        "around fifteen minutes"      if total_mins<=15 else
        "around twenty minutes"       if total_mins<=20 else
        "around twenty five minutes"  if total_mins<=25 else
        "around thirty minutes"       if total_mins<=30 else
        "around thirty five minutes"  if total_mins<=35 else
        "around forty minutes"
    )
    print(f"\n⏱️ Real total duration: ~{total_mins} min → '{real_dur_str}'")

    # Step 3 — update Slide 1 with real duration using safer Regex
    slide1_text = script_data["Slide_1_Intro"]

    # Safer Regex replacement: Catches any previous duration string safely
    slide1_text = re.sub(
        r'around (ten|fifteen|twenty|twenty five|thirty|thirty five|forty) minutes',
        real_dur_str,
        slide1_text
    )

    clean1 = expand(slide1_text)
    audio_path1 = os.path.join(AUDIO_DIR, "Slide_1_Intro.mp3")
    vtt_path1   = os.path.join(AUDIO_DIR, "Slide_1_Intro.vtt")
    words_path1 = os.path.join(AUDIO_DIR, "Slide_1_Intro.words.json")

    print(f"  🎙️ Slide_1_Intro...", end="", flush=True)
    await asyncio.sleep(2.5) # Network buffer
    success = await _generate_with_retry(clean1, audio_path1, vtt_path1, words_path1)

    if success:
        size = os.path.getsize(audio_path1) / 1024
        print(f" ✅ ({size:.0f} KB)")
        print(f"   → Narrator will say: '{real_dur_str}'")


asyncio.run(generate_all())

# Save master script
json_path = os.path.join(AUDIO_DIR,"locked_script_data.json")
with open(json_path, 'w') as f:
    json.dump(script_data, f, indent=4)

print(f"\n{'='*60}")
print(f" 🟣 CELL 16 COMPLETE — ALL AUDIO GENERATED")
print(f" Scripts: {len(script_data)} slides")
print(f" Estimated video length: ~18-22 minutes")
print(f"{'='*60}")

✅ Previous duration: ~15 min → 'around fifteen minutes'
✅ Old audio files cleared
✅ Slide 1 (Dynamic Agenda Script Built — 5 items)
✅ Slide 2 — Timestamp injected: At the time of recording, on Sunday 5 July 2026 at 9:00 PM Eastern, this is the ...
✅ Slide 3   Shields: ['Haaland', 'Gabriel', 'Raya']   Swords: ['Dunk', 'Ampadu', 'Palhinha']
✅ Slide 4 (Refined for Broadcast Flow & TTS Limits)
✅ Slide 5 (Refined for Broadcast Flow & TTS Limits)
✅ Slide 6 (Refined for Broadcast Flow & TTS Sync)
✅ Slide 7a — 5 players narrated
✅ Slide 7   Top atk: ['Spurs', 'Man City', 'Chelsea', 'Leeds', 'Liverpool']   Bottom atk: ['Burnley', 'Bournemouth', 'Wolves']   BGW: []
✅ Slide 8 (Refactored for Broadcast Flow & Safely Fetched Targets)
✅ Slide 9 (GW38 Review — 75pts — Chip: none — Fully Dynamic)
✅ Slide 10 (GW39 Plan — Cap: Gabriel — Remaining chips: none)
✅ Slide 11 (Dynamic Season-Aware Narration)
✅ Slide 12 — Cap: Gabriel | VC: Calvert-Lewin
✅ TTS sweep 1 applied — initial-dot prefixes stripped
✅ 

In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 FPL VORTEX NEWSROOM OS v2
Cell 0c: Assembly Engine — Verifies generated assets and locks final DYNAMIC_MANIFEST
Run ORDER: Aåfter Cell 16 (audio), before Cell Y (card generator) and Cells 17a–d.
"""

import os, json

print("🟣 Newsroom OS v2 — Assembly Engine initializing...")

try: GD_ROOT
except NameError: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

PNG_DIR   = os.path.join(GD_ROOT, "png")
AUDIO_DIR = os.path.join(GD_ROOT, "audio")

# ═══════════════════════════════════════════════════════════════════════
# 1. PULL MODULE REGISTRY + PLANNED MODULE LIST FROM CELL 0b
# ═══════════════════════════════════════════════════════════════════════
_registry = globals().get('MODULE_REGISTRY')
if not _registry:
    raise RuntimeError("❌ MODULE_REGISTRY not found. Re-run Cell 0b first.")

_planned_modules = globals().get('final_modules')
if not _planned_modules:
    print("⚠️  final_modules not set — defaulting to all registered modules.")
    _planned_modules = list(_registry.keys())

# ═══════════════════════════════════════════════════════════════════════
# 2. VERIFY PNG + AUDIO EXIST FOR EACH MODULE
# ═══════════════════════════════════════════════════════════════════════
_verified = []
_dropped  = []

for mod in _planned_modules:
    cfg = _registry.get(mod, {})
    png_path   = os.path.join(PNG_DIR,   cfg.get("png_file",  ""))
    audio_path = os.path.join(AUDIO_DIR, f"{cfg.get('slide_key', '')}.mp3")

    png_ok   = os.path.exists(png_path)
    audio_ok = os.path.exists(audio_path)

    if png_ok and audio_ok:
        _verified.append(mod)
    else:
        reasons = []
        if not png_ok:   reasons.append(f"PNG missing  → {cfg.get('png_file', '?')}")
        if not audio_ok: reasons.append(f"Audio missing → {cfg.get('slide_key','?')}.mp3")
        _dropped.append((mod, reasons))

if _dropped:
    print(f"\n⚠️  {len(_dropped)} module(s) dropped — assets incomplete:")
    for mod, reasons in _dropped:
        for r in reasons:
            print(f"     • [{mod}]  {r}")

# ═══════════════════════════════════════════════════════════════════════
# 3. BUILD VERIFIED DYNAMIC_MANIFEST
# ═══════════════════════════════════════════════════════════════════════
DYNAMIC_MANIFEST = [
    (
        i + 1,
        _registry[mod]["png_file"],
        _registry[mod]["slide_key"],
        _registry[mod]["slide_type"],
    )
    for i, mod in enumerate(_verified)
]

globals()["DYNAMIC_MANIFEST"] = DYNAMIC_MANIFEST
globals()["final_modules"]    = _verified

# ═══════════════════════════════════════════════════════════════════════
# 4. REPORT
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f" 🟢 ASSEMBLY ENGINE — MANIFEST LOCKED")
print(f"   Planned modules  : {len(_planned_modules)}")
print(f"   Verified modules : {len(_verified)}")
print(f"   Dropped modules  : {len(_dropped)}")
for i, mod in enumerate(_verified):
    cfg = _registry[mod]
    print(f"     {i+1:>2}. [{cfg['slide_type']:<10}]  {mod}")
print(f"{'='*60}\n")

# Save final manifest log for inspection
_log_path = os.path.join(AUDIO_DIR, "final_manifest.json")
try:
    with open(_log_path, "w") as f:
        json.dump({
            "verified_modules": _verified,
            "dropped_modules":  [m for m, _ in _dropped],
            "manifest":         DYNAMIC_MANIFEST,
        }, f, indent=2, default=str)
    print(f"✅ Final manifest saved → {_log_path}")
except Exception as e:
    print(f"⚠️  Could not save manifest log: {e}")

🟣 Newsroom OS v2 — Assembly Engine initializing...

 🟢 ASSEMBLY ENGINE — MANIFEST LOCKED
   Planned modules  : 12
   Verified modules : 12
   Dropped modules  : 0
      1. [agenda_s1 ]  intro
      2. [squad_s9  ]  gw_review
      3. [two_stage ]  market_trends
      4. [two_stage ]  sword_shield
      5. [none      ]  fdr_map
      6. [none      ]  cs_odds
      7. [two_stage ]  injury_report
      8. [none      ]  all_20_teams
      9. [two_stage ]  targets_avoids
     10. [two_stage ]  chip_strategy
     11. [plan_s10  ]  master_plan
     12. [two_stage ]  outro

✅ Final manifest saved → /content/drive/MyDrive/Vortex_Final/audio/final_manifest.json


In [ ]:

# -*- coding: utf-8 -*-
"""
🟣 Cell X — Delete all existing player cards
Wipes /player_cards/ root only. Subfolders (squad_frames, plan_frames)
are managed independently by Cell 11+12 and are NOT touched here.
"""
import os

try:    GD_ROOT
except: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

CARDS_DIR = os.path.join(GD_ROOT, "player_cards")

if not os.path.isdir(CARDS_DIR):
    print(f"⚠️ {CARDS_DIR} does not exist — nothing to delete.")
else:
    deleted = 0
    skipped_dirs = 0
    for entry in os.listdir(CARDS_DIR):
        path = os.path.join(CARDS_DIR, entry)
        # Only delete loose .png files at the root level
        if os.path.isfile(path) and entry.lower().endswith(".png"):
            try:
                os.remove(path)
                deleted += 1
            except Exception as e:
                print(f"   ⚠️ Failed to delete {entry}: {e}")
        elif os.path.isdir(path):
            skipped_dirs += 1   # leave squad_frames/ and plan_frames/ alone

    print(f"✅ Deleted {deleted} player cards from /player_cards/")
    if skipped_dirs > 0:
        print(f"   (Preserved {skipped_dirs} subfolder(s) — managed by Cell 11+12)")

✅ Deleted 0 player cards from /player_cards/
   (Preserved 2 subfolder(s) — managed by Cell 11+12)


In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 Cell Y — Universal Player Card Generator (Dynamic Pro Edition)
Club color backgrounds, strict API image fetching, and dynamic position-based stat tables.
"""

import os, io, re, unicodedata, requests
from PIL import Image, ImageDraw, ImageEnhance, ImageFont

try:    GD_ROOT
except: GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

CARDS_DIR    = os.path.join(GD_ROOT, "player_cards")
ELEMENTS_DIR = os.path.join(GD_ROOT, "elements")
PHOTOS_DIR   = os.path.join(GD_ROOT, "players")
os.makedirs(CARDS_DIR, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0"}

# Guard: ensure required globals exist
for _required in ['boot_data', 'fix_data', 'NEXT_GW', 'get_vortex_fdr']:
    if _required not in globals():
        raise RuntimeError(f"❌ Run Cell 2 first — '{_required}' is missing.")

# ── Club color map ─────────────────────────────────────────────────────────
CLUB_COLORS = {
    "ARS": ("#EF0107", "#FFFFFF"),
    "AVL": ("#670E36", "#95BFE5"),
    "BOU": ("#DA291C", "#000000"),
    "BRE": ("#C8102E", "#FFFFFF"),
    "BHA": ("#0057B8", "#FFFFFF"),
    "CHE": ("#034694", "#FFFFFF"),
    "CRY": ("#1B458F", "#C4122E"),
    "EVE": ("#003399", "#FFFFFF"),
    "FUL": ("#CC0000", "#FFFFFF"),
    "IPS": ("#003090", "#FFFFFF"),
    "LEI": ("#003090", "#FDBE11"),
    "LIV": ("#C8102E", "#F6EB61"),
    "MCI": ("#6CABDD", "#1C2C5B"),
    "MUN": ("#DA291C", "#FBE122"),
    "NEW": ("#241F20", "#41B6E6"),
    "NFO": ("#DD0000", "#FFFFFF"),
    "SOU": ("#D71920", "#FFFFFF"),
    "TOT": ("#132257", "#FFFFFF"),
    "WHU": ("#7A263A", "#1BB1E7"),
    "WOL": ("#FDB913", "#231F20"),
    "SUN": ("#EB172B", "#FFFFFF"),
    "BUR": ("#6C1D45", "#99D6EA"),
}

def _nc(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ASCII","ignore").decode()
    return re.sub(r"[^a-z0-9]", "", s.lower())

def fnt(style, size):
    try:
        return ImageFont.truetype(os.path.join(ELEMENTS_DIR, f"Roboto-{style}.ttf"), size)
    except:
        return ImageFont.load_default()

def get_photo(code, web_name, size=(280,340)):
    fb = Image.new("RGBA", size, (0,0,0,0))

    for ext in ['.png', '.jpg', '.jpeg']:
        local_path = os.path.join(PHOTOS_DIR, f"{code}{ext}")
        if os.path.exists(local_path):
            try:
                img = Image.open(local_path).convert("RGBA")
                fb.paste(img.resize(size, Image.LANCZOS), (0,0), img.resize(size, Image.LANCZOS))
                return fb
            except Exception as e:
                print(f"⚠️ Local cache error for {code}{ext}: {e}")

    try:
        url = f"https://resources.premierleague.com/premierleague/photos/players/250x250/p{code}.png"
        r = requests.get(url, headers=HEADERS, timeout=6)
        if r.status_code == 200:
            if len(r.content) > 3000:
                img = Image.open(io.BytesIO(r.content)).convert("RGBA")
                fb.paste(img.resize(size, Image.LANCZOS), (0,0), img.resize(size, Image.LANCZOS))
                return fb
            else:
                print(f"⚠️ API placeholder detected for {code}")
        else:
            print(f"⚠️ API error for {code}: HTTP {r.status_code}")
    except Exception as e:
        print(f"❌ API Request failed for {code}: {e}")

    p_data = next((p for p in boot_data.get('elements', []) if p.get('code') == code), None)
    if p_data:
        team_id = p_data.get('team')
        team_data = next((t for t in boot_data.get('teams', []) if t['id'] == team_id), None)
        if team_data:
            badge = get_logo(team_data['code'], size=(int(size[0]*0.7), int(size[0]*0.7)))
            bx = (size[0] - badge.width) // 2
            by = (size[1] - badge.height) // 2
            fb.paste(badge, (bx, by), badge)
            return fb

    d = ImageDraw.Draw(fb)
    cx, cy = size[0]//2, size[1]//2
    d.ellipse([cx-60,cy-120,cx+60,cy], fill="#555555")
    d.ellipse([cx-90,cy,cx+90,cy+160], fill="#555555")
    return fb

def get_logo(team_code, size=(120,120)):
    fb = Image.new("RGBA", size, (0,0,0,0))
    try:
        url = f"https://resources.premierleague.com/premierleague/badges/70/t{team_code}.png"
        r   = requests.get(url, headers=HEADERS, timeout=5)
        if r.status_code == 200:
            img = Image.open(io.BytesIO(r.content)).convert("RGBA")
            fb.paste(img.resize(size,Image.LANCZOS),(0,0),img.resize(size,Image.LANCZOS))
    except: pass
    return fb

def luminance(hex_color):
    r,g,b = int(hex_color[1:3],16), int(hex_color[3:5],16), int(hex_color[5:7],16)
    return 0.299*r + 0.587*g + 0.114*b

def hex_to_rgba(hex_color, alpha=255):
    r,g,b = int(hex_color[1:3],16), int(hex_color[3:5],16), int(hex_color[5:7],16)
    return (r,g,b,alpha)

def get_upcoming_fixtures(team_id, next_gw, count=7):
    season_end = max((e["id"] for e in boot_data.get("events", [])), default=next_gw)
    team_fixes = [f for f in fix_data
             if (f['team_h']==team_id or f['team_a']==team_id)
             and f.get('event')
             and f['event'] >= next_gw
             and f['event'] <= season_end
             and not f.get('finished', False)]

    result = []
    gw_cursor = next_gw

    while len(result) < count and gw_cursor <= season_end:
        gw_fixtures = [f for f in team_fixes if f['event'] == gw_cursor]
        if not gw_fixtures:
            result.append({'gw': gw_cursor, 'opponent': 'BLANK', 'is_home': True, 'fdr': 5.0, 'is_blank': True})
        else:
            for fx in gw_fixtures:
                if len(result) >= count: break
                is_home = fx['team_h'] == team_id
                opp_id  = fx['team_a'] if is_home else fx['team_h']
                opp     = next((t for t in boot_data['teams'] if t['id']==opp_id), None)
                fdr     = get_vortex_fdr(team_id, opp_id, is_home)
                result.append({
                    'gw': fx['event'], 'opponent': opp['short_name'] if opp else '???',
                    'is_home': is_home, 'fdr': fdr, 'is_blank': False
                })
        gw_cursor += 1
    return result

def get_fdr_color(fdr):
    if fdr <= 2.0: return "#00FF87", "#000000"
    if fdr <= 2.5: return "#76FF03", "#000000"
    if fdr <= 3.5: return "#FFD700", "#000000"
    if fdr <= 4.5: return "#FF1744", "#FFFFFF"
    return "#8B0000", "#FFFFFF"

# ── Dynamic Stats Extraction by Position ─────────────────────────────────────
def get_stats_for_pos(p, pos_int):
    mins = max(1, p.get('minutes', 0))
    g = p.get('goals_scored', 0)
    a = p.get('assists', 0)
    saves = p.get('saves', 0)

    def sf(k):
        try: return float(p.get(k, 0.0))
        except: return 0.0

    xg = sf('expected_goals')
    xa = sf('expected_assists')
    xgi = sf('expected_goal_involvements')
    xgc = sf('expected_goals_conceded')
    npxg = sf('expected_goals_without_penalties') or xg
    bps = p.get('bps', 0)
    pts = p.get('total_points', 0)
    xpts = sf('ep_next')
    cs = p.get('clean_sheets', 0)

    saves_90 = (saves / mins) * 90 if mins > 0 else 0
    ga_90 = ((g + a) / mins) * 90 if mins > 0 else 0
    g_diff = g - xg

    if pos_int == 1:   # GKP
        hdrs = ["xGC", "SAV/90", "BPS", "CS", "MINS", "xPTS", "PTS"]
        vals = [f"{xgc:.2f}", f"{saves_90:.1f}", str(bps), str(cs), str(mins), f"{xpts:.1f}", str(pts)]
    elif pos_int == 2: # DEF
        hdrs = ["xGC", "xGI", "CS", "BPS", "MINS", "xPTS", "PTS"]
        vals = [f"{xgc:.2f}", f"{xgi:.2f}", str(cs), str(bps), str(mins), f"{xpts:.1f}", str(pts)]
    elif pos_int == 3: # MID
        hdrs = ["xG", "xA", "xGI", "G+A/90", "npxG", "BPS", "xPTS"]
        vals = [f"{xg:.2f}", f"{xa:.2f}", f"{xgi:.2f}", f"{ga_90:.2f}", f"{npxg:.2f}", str(bps), f"{xpts:.1f}"]
    else:              # FWD
        hdrs = ["xG", "npxG", "xA", "G-xG", "MINS", "BPS", "xPTS"]
        vals = [f"{xg:.2f}", f"{npxg:.2f}", f"{xa:.2f}", f"{g_diff:.2f}", str(mins), str(bps), f"{xpts:.1f}"]

    return hdrs, vals

# ── MASTER CARD GENERATOR ──────────────────────────────────────────────────
def generate_card(player_data, team_data, fixtures, next_gw):
    SHORT  = team_data['short_name']
    CLUB_BG, CLUB_ACC = CLUB_COLORS.get(SHORT, ("#0A0515","#FFD700"))

    CW, CH = 1800, 2400
    card   = Image.new("RGBA", (CW, CH), hex_to_rgba(CLUB_BG, 255))
    draw   = ImageDraw.Draw(card)

    # ── Foreground player photo ─────────────────────────────────────────────
    # 🔴 FIX: Route through Cell 2 global cache
    photo = get_player_photo(player_data['code'], player_data['web_name'], size=(1200, 1400))
    card.paste(photo, ((CW - 1200)//2, 100), photo)

    # ── Bottom Dark Panel for Data ─────────────────────────────────────────
    panel_y = 1450
    draw.rectangle([0, panel_y, CW, CH], fill="#11151C")
    draw.line([(0, panel_y), (CW, panel_y)], fill=CLUB_ACC, width=15)

    # ── Top Left Badges ─────────────────────────────────────────────────────
    badge_x = 80
    badge_y = 80
    badge_w = 400
    badge_h = 160
    spacing = 40

    def draw_badge(title, value, y_off):
        draw.rounded_rectangle([badge_x, y_off, badge_x+badge_w, y_off+badge_h], radius=25, fill="#11151C")
        draw.text((badge_x + badge_w//2, y_off + 40), title, font=fnt("Bold", 45), fill="#AAAAAA", anchor="mm")
        draw.text((badge_x + badge_w//2, y_off + 105), value, font=fnt("Bold", 65), fill="#FFFFFF", anchor="mm")

    draw_badge("PRICE", f"£{player_data.get('now_cost',0)/10.0}m", badge_y)
    draw_badge("SELECTED BY", f"{player_data.get('selected_by_percent','0.0')}%", badge_y + badge_h + spacing)

    pos_map = {1:"GKP", 2:"DEF", 3:"MID", 4:"FWD"}
    pos_str = pos_map.get(player_data.get('element_type', 3), "MID")
    draw_badge("POSITION", pos_str, badge_y + 2*(badge_h + spacing))

    # ── Team logo ───────────────────────────────────────────────────────────
    # 🔴 FIX: Route through Cell 2 global cache
    logo = get_team_logo(team_data['code'], size=(300,300))
    card.paste(logo, (CW - 380, 80), logo)

    # ── Player name ─────────────────────────────────────────────────────────
    name = player_data.get('web_name', '').upper()
    name_font = fnt("Bold", 180)
    name_bbox = draw.textbbox((0,0), name, font=name_font)
    while (name_bbox[2]-name_bbox[0]) > (CW - 100) and name_font.size > 80:
        name_font = fnt("Bold", name_font.size-5)
        name_bbox = draw.textbbox((0,0), name, font=name_font)
    draw.text((CW//2, panel_y + 120), name, font=name_font, fill="#FFFFFF", anchor="mm")

    # ── Upcoming Fixtures ───────────────────────────────────────────────
    n_slots = max(1, len(fixtures))
    fix_y = panel_y + 250
    box_w = (CW - 100) // n_slots
    box_h = 240
    start_x = 50

    for slot in range(n_slots):
        bx = start_x + slot*box_w
        if slot < len(fixtures):
            fx = fixtures[slot]
            if fx.get('is_blank'):
                bg_col, fg_col = "#222222", "#FF1744"
                opp_str = "BLANK"
                fdr_str = "-"
                gw_str  = f"GW{fx['gw']}"
            else:
                bg_col, fg_col = get_fdr_color(fx['fdr'])
                opp_str = f"{fx['opponent']} ({'H' if fx['is_home'] else 'A'})"
                fdr_str = f"{fx['fdr']:.1f}"
                gw_str  = f"GW{fx['gw']}"
        else:
            bg_col, fg_col = "#333333", "#888888"
            opp_str, fdr_str, gw_str = "BLANK", "-", ""

        # GW Top Bar
        draw.rectangle([bx+5, fix_y, bx+box_w-5, fix_y+60], fill="#222831")
        if gw_str:
            draw.text((bx + box_w//2, fix_y + 30), gw_str, font=fnt("Bold", 40), fill="#FFFFFF", anchor="mm")

        # Main FDR Box
        draw.rectangle([bx+5, fix_y+60, bx+box_w-5, fix_y+box_h], fill=bg_col)

        opp_font = fnt("Bold", 65)
        ob = draw.textbbox((0,0), opp_str, font=opp_font)
        while (ob[2]-ob[0]) > box_w-15 and opp_font.size > 28:
            opp_font = fnt("Bold", opp_font.size-2)
            ob = draw.textbbox((0,0), opp_str, font=opp_font)

        draw.text((bx + box_w//2, fix_y + 120), opp_str, font=opp_font, fill=fg_col, anchor="mm")
        draw.text((bx + box_w//2, fix_y + 195), fdr_str, font=fnt("Bold", 60), fill=fg_col, anchor="mm")

    # ── Dynamic Stats table ─────────────────────────────────────────────────
    pos_int = player_data.get('element_type', 3)
    hdrs, vals = get_stats_for_pos(player_data, pos_int)

    tbl_y = fix_y + box_h + 80
    tbl_h = 240
    col_w = (CW - 100) / len(hdrs)

    draw.rounded_rectangle([50, tbl_y, CW-50, tbl_y+tbl_h], radius=20, fill="#1B202B")
    draw.rounded_rectangle([50, tbl_y, CW-50, tbl_y+80], radius=20, fill="#222831")
    draw.rectangle([50, tbl_y+40, CW-50, tbl_y+80], fill="#222831")

    for col in range(len(hdrs)):
        cx = 50 + (col*col_w) + (col_w/2)
        if col > 0:
            draw.line([(50+col*col_w, tbl_y), (50+col*col_w, tbl_y+tbl_h)], fill="#333A45", width=4)
        draw.text((cx, tbl_y + 40), hdrs[col], font=fnt("Bold", 45), fill="#00FF87", anchor="mm")
        draw.text((cx, tbl_y + 160), vals[col], font=fnt("Bold", 75), fill="#FFFFFF", anchor="mm")

    return card

# ── COLLECT ALL PLAYERS DIRECTLY FROM API ─────────────────────────────────
print("Collecting all players from FPL API...")

all_players = {}
priority_codes = set()

priority_globals = (
    'swords', 'shields', 'target_players', 'avoid_players', 's7a_locked_players',
    's9_starters', 's9_bench', 's10_starters', 's10_bench', 'top_targets',
    'top_in', 'top_out', 'fh_targets',
)
for global_name in priority_globals:
    pool = globals().get(global_name, [])
    for p in pool:
        code = p.get('code')
        if code: priority_codes.add(code)

for global_name in ('s10_squad_pool',):
    pool = globals().get(global_name, [])
    for p in pool:
        if p.get('is_cap') or p.get('is_vc'):
            code = p.get('code')
            if code: priority_codes.add(code)

for tc in globals().get('tc_options', []):
    code = tc.get('code')
    if code: priority_codes.add(code)

print(f"   Priority players (always get cards): {len(priority_codes)}")

for el in boot_data['elements']:
    try:
        p_code = el.get('code')
        if not p_code: continue

        is_priority = p_code in priority_codes
        if not is_priority and float(el.get('selected_by_percent', 0)) < 1.0:
            continue
        if el.get('status') not in ['a', 'd', 'i', 's']:
            continue

        # STRICT ID KEY: Prevents players with the same name from overwriting each other
        if p_code not in all_players:
            all_players[p_code] = el
    except:
        continue

print(f"✅ Total unique players to generate cards for: {len(all_players)}")

# ── GENERATE ALL CARDS ─────────────────────────────────────────────────────
print("\nGenerating cards...\n")

generated = 0
failed    = 0

for p_code, player in all_players.items():
    web_name = player.get('web_name', str(p_code))
    try:
        team_id   = player['team']
        team_data = next(t for t in boot_data['teams'] if t['id']==team_id)
        fixtures  = get_upcoming_fixtures(team_id, NEXT_GW)

        card = generate_card(player, team_data, fixtures, NEXT_GW)

        save_path = os.path.join(CARDS_DIR, f"{player['code']}.png")

        if os.path.exists(save_path):
            os.remove(save_path)
        card.save(save_path, "PNG")
        print(f"  ✅ {web_name} → {player['code']}.png")

        generated += 1
        del card

    except Exception as e:
        print(f"  ❌ {web_name} failed: {e}")
        failed += 1

print(f"\n{'='*50}")
print(f"✅ Generated : {generated} cards")
print(f"❌ Failed    : {failed} cards")
print(f"📁 Saved to  : {CARDS_DIR}")
print(f"{'='*50}")

   Priority players (always get cards): 56
✅ Total unique players to generate cards for: 204

Generating cards...

  ✅ Raya → 154561.png
  ✅ Arrizabalaga → 109745.png
⚠️ API 404 error for 551221: HTTP 403
  ✅ Setford → 551221.png
  ✅ Gabriel → 226597.png
  ✅ Saliba → 462424.png
  ✅ Calafiori → 466075.png
  ✅ J.Timber → 445122.png
  ✅ Lewis-Skelly → 499169.png
  ✅ Saka → 223340.png
  ✅ Ødegaard → 184029.png
  ✅ Trossard → 116216.png
  ✅ Rice → 204480.png
⚠️ API 404 error for 481655: HTTP 403
  ✅ Zubimendi → 481655.png
  ✅ Eze → 232413.png
⚠️ API 404 error for 224117: HTTP 403
  ✅ Gyökeres → 224117.png
  ✅ Martinez → 98980.png
  ✅ Cash → 199796.png
  ✅ Digne → 101188.png
  ✅ Konsa → 199798.png
  ✅ Rogers → 244850.png
  ✅ Watkins → 178301.png
  ✅ Walker → 58621.png
  ✅ Estève → 477717.png
  ✅ Anthony → 444180.png
  ✅ Barnes → 44699.png
  ✅ Dúbravka → 67089.png
  ✅ Petrović → 457569.png
  ✅ Senesi → 221466.png
⚠️ API 404 error for 494521: HTTP 403
  ✅ Truffert → 494521.png
  ✅ Hill → 46398

In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 CELL 17a — Constants, Utilities, Image Helpers
"""

import os, re, sys, subprocess, unicodedata, shutil, math, io, requests
from datetime import datetime, timedelta
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

print("🟣 Cell 17a — Constants + Utilities loaded")

try:
    GD_ROOT
except NameError:
    GD_ROOT = "/content/drive/MyDrive/Vortex_Final"

PNG_DIR    = os.path.join(GD_ROOT, "png")
AUDIO_DIR  = os.path.join(GD_ROOT, "audio")
CARDS_DIR  = os.path.join(GD_ROOT, "player_cards")
PHOTOS_DIR = os.path.join(GD_ROOT, "players")
ELEM_DIR   = os.path.join(GD_ROOT, "elements")
OUTPUT_DIR = os.path.join(GD_ROOT, "output")
TEMP_DIR   = "/tmp/vortex_master"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TEMP_DIR,   exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0"}

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  TUNABLE CONSTANTS                                                   ║
# ╚══════════════════════════════════════════════════════════════════════╝

VW, VH         = 1920, 1080
# Pulls from Master Control Panel in Cell 0
FPS            = 5 if "Draft" in globals().get('RENDER_QUALITY', "Draft") else 30
MUSIC_VOL      = 0.099
INTRO_DUR      = 7.0
OUTRO_DUR      = 7.0
AUDIO_FADE     = 0.50
FADE_DUR       = 0.35

BIG_CARD_X        = 10
BIG_CARD_Y        = 200
BIG_CARD_W        = 650
BIG_CARD_H        = 870

INFO_ROW_W        = 580
INFO_ROW_H        = 130
INFO_HEADSHOT_R   = 55
INFO_PADDING_X    = 20

_S12_CAP_LOCK_Y_8K    = ((4320 // 2) + 100) + 950
_S12_CAP_BELOW_Y_1080 = int(_S12_CAP_LOCK_Y_8K / 4320 * VH) + 40

# 🔴 FIX: Spacing and box sizes updated to support 10 players beautifully


SLIDE_LAYOUTS = {
    "Slide_2_Market_Trends": {
        "COL_1_X":      680,   # Shifted left to center perfectly in the new wider column
        "COL_2_X":      1300,  # Shifted right to center perfectly in the new wider column
        "ROW_START_Y":  220,
        "ROW_GAP":      160,   # Increased gap to fit taller cards
        "INFO_W":       560,   # Wider rectangular size
        "INFO_H":       145,   # Taller rectangular size to stack text cleanly
    },
    "Slide_3_Sword_Shield": {
        "COL_1_X":      698,   # Aligned to Shield container left edge (2790/7680×1920)
        "COL_2_X":      1262,  # Aligned to Sword container left edge (5040/7680×1920)
        "ROW_START_Y":  165,   # Just below subheadings — fits inside top border
        "ROW_GAP":      132,   # 6 cards: 165 + 5×132 + 130 = 955 ≤ 974 container bottom ✓
        "INFO_W":       520,   # 698+520=1218 < container right 1234 ✓  1262+520=1782 < 1797 ✓
        "INFO_H":       130,   # Preserves readable font sizes (h//5=26, h//6=21)
    },
    "Slide_8_Targets_Avoids": {
        "COL_1_X":      620,
        "COL_2_X":      1240,
        "ROW_START_Y":  162,
        "ROW_GAP":      160,
        "INFO_W":       640,
        "INFO_H":       145,
    },
    "Slide_7a_Injury_Risk": {
        "COL_1_X":      680,
        "COL_2_X":      1300,
        "ROW_START_Y":  220,
        "ROW_GAP":      160,
        "INFO_W":       560,
        "INFO_H":       145,
    },
    "Slide_12_Outro": {
        "COL_1_X":      1050,
        "COL_2_X":      1450,
        "ROW_START_Y":  _S12_CAP_BELOW_Y_1080,
        "ROW_GAP":      0,
        "INFO_W":       380,
        "INFO_H":       85,
    },
}


S9_SIDEBAR_X      = 60
S9_SIDEBAR_Y      = 280
S9_SIDEBAR_W      = 550
S9_SIDEBAR_H      = 780

S10_HEADING_X     = 1500
S10_HEADING_Y     = 90

PITCH_S10 = {
    "gk":      [(1380, 780)],
    "def":     [(1080, 600), (1380, 600), (1680, 600)],
    "mid":     [(1020, 400), (1200, 400), (1380, 400), (1560, 400), (1740, 400)],
    "fwd":     [(1230, 200), (1530, 200)],
    "bench":   [(880, 980), (1180, 980), (1480, 980), (1780, 980)],
    "targets": [(160, 940), (330, 940), (500, 940), (670, 940)],
}

CARD_H        = 175
TARGET_CARD_H = 185

# ── MANIFEST ─────────────────────────────────────────────────────────────
_dynamic = globals().get("DYNAMIC_MANIFEST", None)
if _dynamic:
    MANIFEST = _dynamic
    print(f"✅ MANIFEST loaded from Newsroom OS — {len(MANIFEST)} modules")
else:
    print("⚠️  DYNAMIC_MANIFEST not found — Cell 0c may not have run.")
    print("   Falling back to full 13-slide static manifest.")
    MANIFEST = [
        ( 1, "Slide_1_Intro_Empty.png",         "Slide_1_Intro",           "agenda_s1"),
        ( 2, "Slide_2_Empty.png",               "Slide_2_Market_Trends",   "two_stage"),
        ( 3, "Slide_3_Empty.png",               "Slide_3_Sword_Shield",    "two_stage"),
        ( 4, "Slide_4_fdr_map.png",             "Slide_4_fdr_map",         "none"     ),
        ( 5, "Slide_5_Projected_Goals.png",     "Slide_5_Projected_Goals", "none"     ),
        ( 6, "Slide_6_CS_Odds.png",             "Slide_6_CS_Odds",         "none"     ),
        ( 7, "Slide_7a_Injury_MinuteRisk.png",  "Slide_7a_Injury_Risk",    "two_stage"),
        ( 8, "Slide_7_All_20_Teams.png",        "Slide_7_All_20_Teams",    "none"     ),
        ( 9, "Slide_8_Empty.png",               "Slide_8_Targets_Avoids",  "two_stage"),
        (10, "Slide_9_Base.png",                "Slide_9_Base",            "squad_s9" ),
        (11, "Slide_10_Plan_Base.png",          "Slide_10_Plan_Base",      "plan_s10" ),
        (12, "Slide_11_Chip_Strategy.png",      "Slide_11_Chip_Strategy",  "none"     ),
        (13, "Slide_12_Outro_Empty.png",        "Slide_12_Outro",          "two_stage"),
    ]


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  UTILITIES                                                           ║
# ╚══════════════════════════════════════════════════════════════════════╝

def ts2sec(ts):
    ts = ts.strip().split()[0].replace(",", ".")
    p = ts.split(":")
    try:
        if len(p)==3: return int(p[0])*3600 + int(p[1])*60 + float(p[2])
        if len(p)==2: return int(p[0])*60 + float(p[1])
    except: pass
    return 0.0

def get_dur(path):
    r = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration",
                        "-of","default=noprint_wrappers=1:nokey=1", path],
                       capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return 0.0

def _get_font(size, weight="Bold"):
    try: return ImageFont.truetype(os.path.join(ELEM_DIR, f"Roboto-{weight}.ttf"), size)
    except: return ImageFont.load_default()

def _normalize_word(w):
    clean = re.sub(r'^[A-Z]\.', '', w)
    clean = re.sub(r'[^\w\-]', '', clean)
    norm  = unicodedata.normalize("NFKD", clean).encode("ASCII","ignore").decode()
    return re.sub(r"[^a-z0-9]", "", norm.lower())

def _clean_audio_name(name):
    clean = re.sub(r"^[A-Z]\.\s*", "", str(name).strip())
    clean = re.sub(r"\s+[A-Z]\.?$", "", clean)
    return clean.strip()

def _get_strict_targets(player):
    targets = []
    phonetic_map = {
        "wan-bissaka": ["wonn bissaka", "bissaka"],
        "wan bissaka": ["wonn bissaka", "bissaka"],
        "bruno fernandes": ["broo no fer nan desh", "broo no", "fer nan desh"],
        "fernandes": ["fer nan desh"]
    }

    for field in ['web_name', 'second_name', 'name']:
        val = str(player.get(field, ''))
        if val:
            norm_words = [_normalize_word(w) for w in val.split()]
            norm_words = [w for w in norm_words if w]
            if norm_words:
                targets.append(" ".join(norm_words))

            val_spaced = val.replace("-", " ")
            norm_spaced = [_normalize_word(w) for w in val_spaced.split()]
            norm_spaced = [w for w in norm_spaced if w]
            if norm_spaced:
                targets.append(" ".join(norm_spaced))

            clean_val = val.lower().strip()
            if clean_val in phonetic_map:
                for phonetic_target in phonetic_map[clean_val]:
                    norm_phonetic = [_normalize_word(w) for w in phonetic_target.split()]
                    norm_phonetic = [w for w in norm_phonetic if w]
                    if norm_phonetic: targets.append(" ".join(norm_phonetic))

    first = str(player.get('first_name', ''))
    second = str(player.get('second_name', ''))
    if first and second:
        full_name = f"{first} {second}"
        norm_words = [_normalize_word(w) for w in full_name.split()]
        norm_words = [w for w in norm_words if w]
        if norm_words:
            targets.append(" ".join(norm_words))

        clean_full = full_name.lower().strip()
        if clean_full in phonetic_map:
            for phonetic_target in phonetic_map[clean_full]:
                norm_phonetic = [_normalize_word(w) for w in phonetic_target.split()]
                norm_phonetic = [w for w in norm_phonetic if w]
                if norm_phonetic: targets.append(" ".join(norm_phonetic))

    unique_targets = list(set(targets))
    unique_targets.sort(key=lambda x: len(x.split()), reverse=True)
    return unique_targets


def _find_sequential_trigger(player, wt, start_idx, delay=1.5):
    targets = _get_strict_targets(player)
    for i in range(start_idx, len(wt)):
        for target in targets:
            target_words = target.split()
            if not target_words: continue
            if i + len(target_words) <= len(wt):
                chunk = " ".join([wt[i+k][0] for k in range(len(target_words))])
                if chunk == target:
                    return wt[i + len(target_words) - 1][2] + delay, i + len(target_words)
    return None, start_idx


def _find_exact_trigger(player, wt, delay=0.5):
    targets = _get_strict_targets(player)
    for i in range(len(wt)):
        for target in targets:
            target_words = target.split()
            if not target_words: continue
            if i + len(target_words) <= len(wt):
                chunk = " ".join([wt[i+k][0] for k in range(len(target_words))])
                if chunk == target:
                    return wt[i + len(target_words) - 1][2] + delay
    return None

def build_word_timeline(cues, audio_stem=None):
    if audio_stem:
        import json as _json
        jp = os.path.join(AUDIO_DIR, f"{audio_stem}.words.json")
        if os.path.exists(jp):
            try:
                data = _json.load(open(jp, encoding="utf-8"))
                wt = []
                for entry in data:
                    nw = _normalize_word(entry.get("text",""))
                    if nw: wt.append((nw, float(entry["start"]), float(entry["end"])))
                if wt: return wt
            except Exception: pass
    wt = []
    for (s, e, text) in cues:
        words = text.split()
        if not words: continue
        step = (e-s) / len(words)
        for i, w in enumerate(words):
            nw = _normalize_word(w)
            if nw: wt.append((nw, s + i*step, s + (i+1)*step))
    return wt

def parse_vtt(path):
    if not os.path.exists(path): return []
    raw = open(path, encoding="utf-8", errors="replace").read()
    raw = re.sub(r"^WEBVTT[^\n]*\n", "", raw.replace("\r\n", "\n"))
    cues = []
    for blk in re.split(r"\n{2,}", raw.strip()):
        lines = [l.strip() for l in blk.split("\n") if l.strip()]
        ti = next((i for i, l in enumerate(lines) if "-->" in l), None)
        if ti is None: continue
        parts = lines[ti].split("-->")
        if len(parts) < 2: continue
        text = " ".join(lines[ti+1:]).strip()
        if text: cues.append((ts2sec(parts[0]), ts2sec(parts[1]), text))
    return cues

def _find_strict_trigger(clean_name, wt, used_idx):
    if not clean_name or len(clean_name) < 3: return None
    for i, (w, s, e) in enumerate(wt):
        if i in used_idx: continue
        if w == clean_name:
            used_idx.add(i); return max(0.0, s + 0.05)
    for i in range(len(wt)):
        if i in used_idx: continue
        for span in range(2, 4):
            if i + span > len(wt): break
            combo = "".join(w for w,_,_ in wt[i:i+span])
            if combo == clean_name:
                for k in range(i, i+span): used_idx.add(k)
                return max(0.0, wt[i][1] + 0.05)
    for i, (w, s, e) in enumerate(wt):
        if i in used_idx: continue
        if len(w) >= 4 and len(clean_name) >= 4:
            if clean_name in w or w in clean_name:
                used_idx.add(i); return max(0.0, s + 0.05)
    if len(clean_name) >= 5:
        prefix = clean_name[:5]
        for i, (w, s, e) in enumerate(wt):
            if i in used_idx: continue
            if w.startswith(prefix) or prefix in w:
                used_idx.add(i); return max(0.0, s + 0.05)
    return None

def _clean_fn(s):
    if not s: return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ASCII","ignore").decode()
    return re.sub(r'[^a-z0-9]', '', s.lower())

# <<< START REPLACEMENT (Cell 17a - Cached Fetchers) <<<
def fetch_team_logo(team_code, size=60):
    cache_key = f"circ_logo_{team_code}_{size}"
    _cache = globals().setdefault('_IMAGE_CACHE', {})
    if cache_key in _cache: return _cache[cache_key].copy()

    canvas = Image.new("RGBA", (size, size), (0,0,0,0))
    try:
        url = f"https://resources.premierleague.com/premierleague/badges/70/t{team_code}.png"
        r = requests.get(url, headers=HEADERS, timeout=5)
        if r.status_code == 200:
            img = Image.open(io.BytesIO(r.content)).convert("RGBA").resize((size, size), Image.LANCZOS)
            canvas.paste(img, (0, 0), img)
            _cache[cache_key] = canvas
    except: pass
    return canvas.copy()

def fetch_circular_headshot(opta_code, web_name, radius=55):
    diam = radius * 2
    cache_key = f"circ_head_{opta_code}_{diam}"
    _cache = globals().setdefault('_IMAGE_CACHE', {})
    if cache_key in _cache: return _cache[cache_key].copy()

    canvas = Image.new("RGBA", (diam, diam), (0,0,0,0))
    raw = None
    for ext in ['.png', '.jpg', '.jpeg']:
        local_path = os.path.join(PHOTOS_DIR, f"{opta_code}{ext}")
        if os.path.exists(local_path):
            try:
                im = Image.open(local_path).convert("RGBA")
                canvas.paste(im.resize((diam, diam), Image.LANCZOS), (0, 0), im.resize((diam, diam), Image.LANCZOS))
                _cache[cache_key] = canvas
                return canvas.copy()
            except Exception as e:
                print(f"⚠️ Local cache read error for {opta_code}{ext}: {e}")
    if raw is None:
        try:
            url = f"https://resources.premierleague.com/premierleague/photos/players/250x250/p{opta_code}.png"
            r = requests.get(url, headers=HEADERS, timeout=6)
            if r.status_code == 200:
                if len(r.content) > 3000:
                    raw = Image.open(io.BytesIO(r.content)).convert("RGBA")
                else:
                    print(f"⚠️ API placeholder detected for {opta_code}")
            else:
                print(f"⚠️ API error for {opta_code}: HTTP {r.status_code}")
        except Exception as e:
            print(f"❌ API Request failed for {opta_code}: {e}")
    if raw is None:
        p_data = next((p for p in boot_data.get('elements', []) if p.get('code') == opta_code), None)
        if p_data:
            team_id = p_data.get('team')
            team_data = next((t for t in boot_data.get('teams', []) if t['id'] == team_id), None)
            if team_data:
                return fetch_team_logo(team_data['code'], size=diam)
        d = ImageDraw.Draw(canvas)
        d.ellipse([0, 0, diam, diam], fill="#444444")
        d.ellipse([radius//2, radius//4, radius*3//2, radius*5//4], fill="#666666")
        _cache[cache_key] = canvas
        return canvas.copy()
    w, h = raw.size
    side = min(w, h)
    crop_top = max(0, (h - side) // 4)
    raw = raw.crop(((w-side)//2, crop_top, (w+side)//2, crop_top + side))
    raw = raw.resize((diam, diam), Image.LANCZOS)
    mask = Image.new("L", (diam, diam), 0)
    ImageDraw.Draw(mask).ellipse([0, 0, diam, diam], fill=255)
    canvas.paste(raw, (0, 0), mask)
    _cache[cache_key] = canvas
    return canvas.copy()

# 🔴 FIX: Increased font scaling and name spacing for visibility
def make_info_row(player_dict, accent_color, idx, slide_key):
    layout  = SLIDE_LAYOUTS.get(slide_key, SLIDE_LAYOUTS["Slide_2_Market_Trends"])
    w       = layout.get("INFO_W", INFO_ROW_W)
    h       = layout.get("INFO_H", INFO_ROW_H)
    img     = Image.new("RGBA", (w, h), (0, 0, 0, 0))
    d       = ImageDraw.Draw(img)

    bdr = max(2, h // 40)
    d.rounded_rectangle([0, 0, w, h], radius=12, fill=(20, 12, 35, 230),
                        outline=accent_color, width=bdr)

    head_r   = max(28, (h - 8) // 2)
    head_r   = min(head_r, 48)
    pad_x    = max(6, h // 14)
    code     = player_dict.get('code', 0)
    web_name = player_dict.get('web_name') or player_dict.get('name', '')
    head     = fetch_circular_headshot(code, web_name, radius=head_r)
    img.paste(head, (pad_x, (h - head_r * 2) // 2), head)

    team_id    = player_dict.get('team') or player_dict.get('team_id') or 0
    team_obj   = next((t for t in boot_data['teams'] if t['id'] == team_id), None)
    team_short = team_obj['short_name'] if team_obj else '???'
    team_code  = team_obj['code']       if team_obj else 0
    logo_sz    = max(28, h // 4)
    logo       = fetch_team_logo(team_code, size=logo_sz)
    logo_x     = pad_x + head_r * 2 + 8
    img.paste(logo, (logo_x, 8), logo)

    fs_team = max(14, h // 6)
    fs_name = max(16, h // 5)
    fs_stat = max(13, h // 6)

    d.text((logo_x + logo_sz + 6, 10),
           team_short, font=_get_font(fs_team, "Bold"), fill="#FFFFFF")

    # ── Resolve shared stats values ───────────────────────────────────────────
    full_p = next(
        (p for p in boot_data.get('elements', [])
         if p.get('code') == player_dict.get('code')), {})
    eo     = player_dict.get('selected_by_percent',
                             player_dict.get('sel',
                             full_p.get('selected_by_percent', '0')))
    eo_str = str(eo).replace('%', '').strip()

    price  = player_dict.get('now_cost') or full_p.get('now_cost')
    if price is None:
        alt_cost = player_dict.get('cost')
        price_str = (f"£{float(alt_cost):.1f}m" if alt_cost is not None
                     else player_dict.get('price', '£?.?m'))
    else:
        price_str = f"£{price / 10:.1f}m"

    stats_x = w - 185  # right panel left edge

    # ── Player name ───────────────────────────────────────────────────────────
    name_clean = re.sub(r'^[A-Z]\.\s*', '', web_name).upper()

    if slide_key == "Slide_3_Sword_Shield":
        max_name_w = stats_x - logo_x - 4
        name_y     = int(h * 0.52)          # ≈ 58 when h=112
    elif slide_key == "Slide_8_Targets_Avoids":
        max_name_w = w - logo_x - logo_sz - 185
        name_y     = fs_team + 14
    else:
        max_name_w = w - logo_x - logo_sz - 185
        name_y     = h // 2 + 4

    name_font = _get_font(fs_name, "Bold")
    name_bbox = d.textbbox((0, 0), name_clean, font=name_font)
    while (name_bbox[2] - name_bbox[0]) > max_name_w and name_font.size > 12:
        name_font = _get_font(name_font.size - 1, "Bold")
        name_bbox = d.textbbox((0, 0), name_clean, font=name_font)
    d.text((logo_x, name_y), name_clean, font=name_font, fill=accent_color)

    # ── Common right stats panel — skipped for Slide_3 (handled below) ────────
    if slide_key != "Slide_3_Sword_Shield":
        d.text((stats_x, 8),  f"EO: {eo_str}%",
               font=_get_font(fs_stat - 1, "Bold"), fill="#00E5FF")
        d.text((stats_x, 8 + fs_stat + 4), price_str,
               font=_get_font(fs_stat - 1, "Bold"), fill="#FFD700")

    # ════════════════════════════════════════════════════════════════════════════
    # SLIDE-SPECIFIC STAT ROWS
    # ════════════════════════════════════════════════════════════════════════════

    if slide_key == "Slide_2_Market_Trends":
        trans = player_dict.get('transfer_str', '')

        # ── Next-GW FDR chips (up to 2 fixtures) ─────────────────────────
        _team_id_s2 = player_dict.get('team') or player_dict.get('team_id') or 0
        _s2_chips   = []
        try:
            _gw_s2 = globals().get('NEXT_GW', 0)
            for _fx in globals().get('fix_data', []):
                if _fx.get('event') != _gw_s2:
                    continue
                if _fx['team_h'] != _team_id_s2 and _fx['team_a'] != _team_id_s2:
                    continue
                _ih  = (_fx['team_h'] == _team_id_s2)
                _oid = _fx['team_a'] if _ih else _fx['team_h']
                _opp = next((t['short_name'] for t in boot_data['teams']
                             if t['id'] == _oid), '?')
                _fdr = round(get_vortex_fdr(_team_id_s2, _oid, _ih), 1)
                _s2_chips.append({'opp': _opp, 'home': _ih, 'fdr': _fdr})
        except Exception:
            pass

        _cx0 = stats_x + 2
        _cw0 = (w - stats_x) - 6           # 179 px — full right panel minus padding
        _ch0 = 18                           # chip height
        _cy0 = 72 if len(_s2_chips) <= 1 else 62   # vertical anchor

        for _fi, _fxc in enumerate(_s2_chips[:2]):
            _fv  = _fxc['fdr']
            _fbg = ("#00FF87" if _fv <= 2.0 else
                    "#76FF03" if _fv <= 2.5 else
                    "#FFD700" if _fv <= 3.5 else
                    "#FF1744" if _fv <= 4.5 else "#8B0000")
            _lr, _lg_, _lb = int(_fbg[1:3], 16), int(_fbg[3:5], 16), int(_fbg[5:7], 16)
            _tc  = "#000000" if (0.299*_lr + 0.587*_lg_ + 0.114*_lb) > 140 else "#FFFFFF"
            _lbl = f"{_fxc['opp']} {'H' if _fxc['home'] else 'A'}  {_fv}"
            _cy  = _cy0 + _fi * (_ch0 + 3)
            d.rounded_rectangle([_cx0, _cy, _cx0 + _cw0, _cy + _ch0],
                                 radius=4, fill=_fbg)
            _cf  = _get_font(12, "Bold")
            _cb  = d.textbbox((0, 0), _lbl, font=_cf)
            d.text((_cx0 + (_cw0 - (_cb[2] - _cb[0])) // 2,
                    _cy  + (_ch0 - (_cb[3] - _cb[1])) // 2 - 1),
                   _lbl, font=_cf, fill=_tc)

        d.text((stats_x, h - fs_stat - 12), trans,
               font=_get_font(fs_stat + 2, "Bold"), fill=accent_color)

    # ─────────────────────────────────────────────────────────────────────────
    elif slide_key == "Slide_3_Sword_Shield":
        stats_x = w - 175      # Override: tighter left/right split — closes internal gap
        panel_w = w - stats_x  # = 175; player name max-width = 345-112-4 = 229px

        # Fixed y anchors (calibrated to INFO_H=112)
        _y_price  = logo_sz + 10          # ≈ 38  (left col, below logo)
        _y_xpts   = int(h * 0.04)         # ≈  4  (right panel row 1)
        _y_eo     = int(h * 0.32)         # ≈ 36  (right panel row 2)
        _y_pill   = int(h * 0.56)         # ≈ 63  (right panel row 3)
        _pill_h   = h - _y_pill - 4       # ≈ 45

        # ── Left col: Price below team name ──────────────────────────────────
        price_font_s3 = _get_font(max(11, fs_stat - 4), "Bold")
        d.text((logo_x, _y_price), price_str,
               font=price_font_s3, fill="#FFD700")

        xpts = player_dict.get('locked_xpts', player_dict.get('ep_next', 0))

        # ── Right panel Row 1: xPts (accent, large) ──────────────────────────
        xpts_text = f"xPts  {xpts:.1f}"
        xpts_font = _get_font(min(fs_stat + 4, 22), "Bold")
        xpts_bb   = d.textbbox((0, 0), xpts_text, font=xpts_font)
        xpts_tx   = stats_x + (panel_w - (xpts_bb[2] - xpts_bb[0])) // 2
        d.text((xpts_tx, _y_xpts), xpts_text, font=xpts_font, fill=accent_color)

        # ── Right panel Row 2: EO% (cyan) ────────────────────────────────────
        eo_text = f"EO  {eo_str}%"
        eo_font = _get_font(max(12, fs_stat - 2), "Bold")
        eo_bb   = d.textbbox((0, 0), eo_text, font=eo_font)
        eo_tx   = stats_x + (panel_w - (eo_bb[2] - eo_bb[0])) // 2
        d.text((eo_tx, _y_eo), eo_text, font=eo_font, fill="#00E5FF")

        # ── Right panel Row 3: rank_str filled pill ───────────────────────────
        if 'rank_str' in player_dict and _pill_h >= 16:
            r_str = player_dict['rank_str']
            r_col = player_dict.get('rank_color', '#00FF87')

            pill_x = stats_x + 4
            pill_w = panel_w - 8

            d.rounded_rectangle(
                [pill_x, _y_pill, pill_x + pill_w, _y_pill + _pill_h],
                radius=8, fill=r_col
            )

            # Auto-shrink font to fit pill width
            pill_font = _get_font(19, "Bold")
            pill_bb   = d.textbbox((0, 0), r_str, font=pill_font)
            while (pill_bb[2] - pill_bb[0]) > (pill_w - 10) and pill_font.size > 10:
                pill_font = _get_font(pill_font.size - 1, "Bold")
                pill_bb   = d.textbbox((0, 0), r_str, font=pill_font)

            # Luminance-based contrast text color
            ri = int(r_col[1:3], 16)
            gi = int(r_col[3:5], 16)
            bi = int(r_col[5:7], 16)
            txt_col = "#000000" if (0.299 * ri + 0.587 * gi + 0.114 * bi) > 128 else "#FFFFFF"

            pill_tx = pill_x + (pill_w - (pill_bb[2] - pill_bb[0])) // 2
            pill_ty = _y_pill + (_pill_h - (pill_bb[3] - pill_bb[1])) // 2
            d.text((pill_tx, pill_ty), r_str, font=pill_font, fill=txt_col)

    # ─────────────────────────────────────────────────────────────────────────
    elif slide_key == "Slide_8_Targets_Avoids":
        _pos_map9   = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}
        _pos_colors = {"GK": "#FFD700", "DEF": "#00E5FF",
                       "MID": "#00FF87", "FWD": "#FF6B35"}
        _et = player_dict.get('element_type') or {
            "GK": 1, "DEF": 2, "MID": 3, "FWD": 4
        }.get(player_dict.get('pos', 'MID'), 3)
        _pos_str = _pos_map9.get(_et, player_dict.get('pos', 'MID'))
        _pos_col = _pos_colors.get(_pos_str, "#FFFFFF")

        _head_bottom = (h - head_r * 2) // 2 + head_r * 2
        _pb_w, _pb_h = 44, 20
        _pb_x = pad_x + (head_r * 2 - _pb_w) // 2
        _pb_y = _head_bottom + 3
        d.rounded_rectangle([_pb_x, _pb_y, _pb_x + _pb_w, _pb_y + _pb_h],
                             radius=6, fill=_pos_col)
        d.text((_pb_x + _pb_w // 2, _pb_y + _pb_h // 2),
               _pos_str, font=_get_font(13, "Bold"),
               fill="#000000", anchor="mm")

        _xpts   = player_dict.get('locked_xpts', player_dict.get('ep_next', 0))
        _fdr    = player_dict.get('next_fdr', 3.0)
        _is_dgw = player_dict.get('is_dgw', False)
        _is_bgw = player_dict.get('is_bgw', False)

        if _is_dgw:
            _stat_str = f"DGW  xPts {_xpts:.1f}"
            _stat_col = "#FFD700"
        elif _is_bgw:
            _stat_str = "BLANK GW"
            _stat_col = "#FF1744"
        else:
            _fdr_col  = ("#00FF87" if _fdr <= 2.5
                         else "#FFD700" if _fdr <= 3.5 else "#FF4444")
            _stat_str = f"xPts {_xpts:.1f}  FDR {_fdr:.1f}"
            _stat_col = _fdr_col

        d.text((stats_x, h - fs_stat - 12), _stat_str,
               font=_get_font(fs_stat + 1, "Bold"), fill=_stat_col)

        _opp = player_dict.get('next_opp', '')
        if _opp and _opp != 'BLANK':
            _parts  = [p.strip() for p in _opp.split('+')]
            _chip_h = 20
            _chip_y = h - fs_stat - 46
            _chip_x = logo_x
            for _part in _parts[:2]:
                _fdr_match = re.search(r'-(\d+(?:\.\d+)?)$', _part.strip())
                _fdr_val   = float(_fdr_match.group(1)) if _fdr_match else 3.0
                _chip_lbl  = re.sub(r'\s+\[', '[', _part.strip())
                _chip_bg   = ("#00FF87" if _fdr_val <= 2.5
                              else "#FFD700" if _fdr_val <= 3.5
                              else "#FF4444")
                _cf  = _get_font(12, "Bold")
                _cb  = d.textbbox((0, 0), _chip_lbl, font=_cf)
                _chip_w = (_cb[2] - _cb[0]) + 12
                d.rounded_rectangle([_chip_x, _chip_y,
                                     _chip_x + _chip_w, _chip_y + _chip_h],
                                    radius=5, fill=_chip_bg)
                _ri = int(_chip_bg[1:3], 16)
                _gi = int(_chip_bg[3:5], 16)
                _bi = int(_chip_bg[5:7], 16)
                _tc = "#000000" if (0.299*_ri + 0.587*_gi + 0.114*_bi) > 128 else "#FFFFFF"
                d.text((_chip_x + _chip_w // 2, _chip_y + _chip_h // 2),
                       _chip_lbl, font=_cf, fill=_tc, anchor="mm")
                _chip_x += _chip_w + 6

    elif slide_key == "Slide_12_Outro":
        d.text((stats_x, h - fs_stat - 10), "★ CAPTAIN",
               font=_get_font(22, "Bold"), fill="#FFD700")

    out = os.path.join(TEMP_DIR, f"info_{slide_key}_{idx:02d}.png")
    img.save(out, "PNG")
    return out

print("✅ Cell 17a complete — all constants, utilities, and image helpers ready")

🟣 Cell 17a — Constants + Utilities loaded
✅ MANIFEST loaded from Newsroom OS — 12 modules
✅ Cell 17a complete — all constants, utilities, and image helpers ready


In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 CELL 17b — Teaser Generators (Slides 1 / 2 / 3 / 8 )
Depends on: Cell 17a
"""

print("🟣 Cell 17b — Teaser Generators loading...")


def make_slide1_teaser():
    img = Image.new("RGBA", (VW, VH), (0, 0, 0, 0))
    d   = ImageDraw.Draw(img)
    C_BORDER_GOLD = "#FFD700"
    C_GREEN       = "#00FF87"
    C_YELLOW      = "#FFEA00"
    C_CYAN        = "#00E5FF"
    C_WHITE       = "#FFFFFF"
    C_GREY        = "#AAAAAA"
    C_BG_SOLID    = (15, 10, 30, 255)
    HEAD_R        = 50
    curr_gw  = globals().get('CURR_GW', '?')
    start_x  = 1100
    end_x    = 1750
    mid_x    = (start_x + end_x) // 2
    panel_y1 = 200
    panel_y2 = 880
    d.rounded_rectangle([start_x - 30, panel_y1, end_x + 30, panel_y2],
                        radius=24, fill=C_BG_SOLID, outline=C_BORDER_GOLD, width=4)
    d.text((mid_x, 260), f"GW{curr_gw} TOP CAPTAINS",
           font=_get_font(42, "Bold"), fill=C_BORDER_GOLD, anchor="mm")
    d.line([(start_x + 50, 290), (end_x - 50, 290)], fill=C_BORDER_GOLD, width=3)
    _all_els  = boot_data.get("elements", [])
    _top_caps = sorted(
        [p for p in _all_els
         if float(p.get("selected_by_percent") or 0) > 0
         and p.get("event_points", 0) != 0],
        key=lambda p: float(p.get("selected_by_percent") or 0),
        reverse=True
    )[:3]
    colors     = [C_GREEN, C_YELLOW, C_CYAN]
    _row_y     = 350
    _row_step  = 180
    for i, _p in enumerate(_top_caps):
        _name     = _clean_audio_name(_p.get("web_name", "")).upper()
        _own      = float(_p.get("selected_by_percent") or 0)
        _base_pts = _p.get("event_points", 0)
        _cap_pts  = _base_pts * 2
        _col      = colors[i]
        head = fetch_circular_headshot(_p.get("code", 0), _p.get("web_name", ""), radius=HEAD_R)
        img.paste(head, (start_x, _row_y), head)
        tx = start_x + 125
        d.text((tx, _row_y + 10), _name, font=_get_font(30, "Bold"), fill=C_WHITE)
        d.text((tx, _row_y + 50), f"CHOSEN BY {_own:.1f}% OF MANAGERS",
               font=_get_font(20, "Bold"), fill=C_GREY)
        bar_w = int(400 * (_own / 100.0)) + 80
        d.rounded_rectangle([tx, _row_y + 80, tx + bar_w, _row_y + 104], radius=12, fill=_col)
        d.text((tx + bar_w + 20, _row_y + 92), f"{_cap_pts} PTS",
               font=_get_font(26, "Bold"), fill=_col, anchor="lm")
        _row_y += _row_step
    out = os.path.join(TEMP_DIR, "slide1_teaser.png")
    img.save(out, "PNG")
    return out


def make_slide2_teaser():
    TEASER_W, TEASER_H = 1034, 500
    PANEL_GAP           = 90
    HALF_W              = (TEASER_W - PANEL_GAP) // 2
    C_BG                = (15, 10, 35, 235)
    C_BORDER_GOLD       = "#FFD700"
    C_BORDER_CYAN       = "#00E5FF"
    C_GREEN             = "#00FF87"
    C_RED               = "#FF1744"
    C_WHITE             = "#FFFFFF"
    C_GREY              = "#888888"
    HEAD_R              = 36
    img = Image.new("RGBA", (TEASER_W, TEASER_H), (0, 0, 0, 0))
    d   = ImageDraw.Draw(img)
    lx1, ly1 = 0, 0
    lx2, ly2 = HALF_W, TEASER_H
    d.rounded_rectangle([lx1, ly1, lx2, ly2], radius=24, fill=C_BG,
                        outline=C_BORDER_GOLD, width=4)
    d.text((HALF_W // 2, 36), "GW AT A GLANCE",
           font=_get_font(28, "Bold"), fill=C_BORDER_GOLD, anchor="mm")
    d.line([(20, 62), (HALF_W - 20, 62)], fill=C_BORDER_GOLD, width=2)
    _next_evt = next((e for e in boot_data.get("events", []) if e.get("is_next")), {})
    _curr_evt = next((e for e in boot_data.get("events", []) if e.get("is_current")), {})
    try:
        _utc = datetime.strptime(_next_evt.get("deadline_time", ""), "%Y-%m-%dT%H:%M:%SZ")
        _est = _utc - timedelta(hours=4)
        _deadline_str = _est.strftime("%a %d %b  %H:%M EST")
    except Exception:
        _deadline_str = _next_evt.get("deadline_time", "---")
    _avg       = _curr_evt.get("average_entry_score", "---")
    _high      = _curr_evt.get("highest_score", "---")
    _total     = boot_data.get("total_players", 0)
    _total_str = (f"{_total / 1_000_000:.1f}M" if _total >= 1_000_000 else str(_total))
    _xfers     = sum(int(p.get("transfers_in_event") or 0)
                     for p in boot_data.get("elements", []))
    _xfers_str = (f"{_xfers / 1_000_000:.1f}M" if _xfers >= 1_000_000
                  else f"{_xfers // 1000}K")
    _rows = [
        ("DEADLINE",   _deadline_str, "#FFFFFF"),
        ("AVG SCORE",  f"{_avg} pts",  C_BORDER_GOLD),
        ("TOP SCORE",  f"{_high} pts", "#00FF87"),
        ("TOTAL MGRS", _total_str,     "#FFFFFF"),
        ("TRANSFERS",  _xfers_str,     C_BORDER_CYAN),
    ]
    _row_y = 82
    for label, value, vcol in _rows:
        d.text((24, _row_y), label, font=_get_font(24, "Bold"), fill="#FFFFFF")
        d.text((HALF_W - 20, _row_y), value, font=_get_font(26, "Bold"),
               fill=vcol, anchor="ra")
        _row_y += 74
    rx1, ry1 = HALF_W + PANEL_GAP, 0
    rx2, ry2 = TEASER_W, TEASER_H
    d.rounded_rectangle([rx1, ry1, rx2, ry2], radius=24, fill=C_BG,
                        outline=C_BORDER_CYAN, width=4)
    RX_OFF = rx1
    d.text((RX_OFF + HALF_W // 2, 36), "PRICE CHANGE ALERTS",
           font=_get_font(28, "Bold"), fill=C_BORDER_CYAN, anchor="mm")
    d.line([(RX_OFF + 20, 62), (RX_OFF + HALF_W - 20, 62)], fill=C_BORDER_CYAN, width=2)
    COL_MID = HALF_W // 2
    arr1_x = RX_OFF + COL_MID // 2 - 35
    arr1_y = 86
    d.polygon([(arr1_x, arr1_y - 7), (arr1_x - 7, arr1_y + 5), (arr1_x + 7, arr1_y + 5)],
              fill=C_GREEN)
    d.text((RX_OFF + COL_MID // 2 + 10, 86), "RISING",
           font=_get_font(20, "Bold"), fill=C_GREEN, anchor="mm")
    arr2_x = RX_OFF + COL_MID + COL_MID // 2 - 40
    arr2_y = 86
    d.polygon([(arr2_x - 7, arr2_y - 5), (arr2_x + 7, arr2_y - 5), (arr2_x, arr2_y + 7)],
              fill=C_RED)
    d.text((RX_OFF + COL_MID + COL_MID // 2 + 10, 86), "FALLING",
           font=_get_font(20, "Bold"), fill=C_RED, anchor="mm")
    d.line([(RX_OFF + COL_MID, 66), (RX_OFF + COL_MID, TEASER_H - 16)],
           fill="#333355", width=2)
    _all_els = boot_data.get("elements", [])
    _rising  = sorted(
        [p for p in _all_els if (p.get("cost_change_event") or 0) > 0],
        key=lambda p: (p.get("cost_change_event", 0),
                       float(p.get("selected_by_percent") or 0)),
        reverse=True
    )[:3]
    _falling = sorted(
        [p for p in _all_els if (p.get("cost_change_event") or 0) < 0],
        key=lambda p: (p.get("cost_change_event", 0),
                       float(p.get("selected_by_percent") or 0))
    )[:3]

    def _draw_price_row(player, col_x, row_y, accent):
        code     = player.get("code", 0)
        web_name = player.get("web_name", "")
        change   = player.get("cost_change_event", 0)
        own      = player.get("selected_by_percent", "0")
        head = fetch_circular_headshot(code, web_name, radius=HEAD_R)
        img.paste(head, (col_x, row_y), head)
        name_clean = re.sub(r'^[A-Z]\.\s*', '', web_name).upper()
        name_font  = _get_font(21, "Bold")
        name_bbox  = d.textbbox((0, 0), name_clean, font=name_font)
        while (name_bbox[2] - name_bbox[0]) > (COL_MID - HEAD_R * 2 - 28) \
                and name_font.size > 14:
            name_font = _get_font(name_font.size - 1, "Bold")
            name_bbox = d.textbbox((0, 0), name_clean, font=name_font)
        tx = col_x + HEAD_R * 2 + 10
        d.text((tx, row_y + 2),  name_clean, font=name_font, fill="#FFFFFF")
        d.text((tx, row_y + 30), f"{'+' if change > 0 else ''}{change / 10:.1f}m",
               font=_get_font(20, "Bold"), fill=accent)
        d.text((tx, row_y + 54), f"EO {own}%", font=_get_font(16, "Bold"), fill="#888888")

    _entry_y  = 108
    _row_step = 122
    for _p in _rising:
        _draw_price_row(_p, RX_OFF + 14, _entry_y, C_GREEN)
        _entry_y += _row_step
    _entry_y = 108
    for _p in _falling:
        _draw_price_row(_p, RX_OFF + COL_MID + 14, _entry_y, C_RED)
        _entry_y += _row_step
    out = os.path.join(TEMP_DIR, "slide2_teaser.png")
    img.save(out, "PNG")
    return out


def make_slide3_teaser():
    TEASER_W, TEASER_H = 1034, 500
    PANEL_GAP           = 90
    HALF_W              = (TEASER_W - PANEL_GAP) // 2
    C_BG                = (15, 10, 35, 235)
    C_BORDER_BLUE       = "#1E90FF"
    C_BORDER_GOLD       = "#FFD700"
    C_GREEN             = "#00FF87"
    C_RED               = "#FF1744"
    C_WHITE             = "#FFFFFF"
    C_GREY              = "#888888"
    HEAD_R              = 32
    LOGO_SZ             = 28
    img = Image.new("RGBA", (TEASER_W, TEASER_H), (0, 0, 0, 0))
    d   = ImageDraw.Draw(img)
    _all_els   = boot_data.get("elements", [])
    _all_teams = boot_data.get("teams", [])

    def _team_obj(team_id):
        return next((t for t in _all_teams if t["id"] == team_id), {})

    def _team_short_name(team_id):
        t = _team_obj(team_id)
        n = t.get("name", "")
        _short_map = {
            "Brighton & Hove Albion":   "Brighton",
            "Manchester United":        "Man Utd",
            "Manchester City":          "Man City",
            "Tottenham Hotspur":        "Spurs",
            "Nottingham Forest":        "Forest",
            "Wolverhampton Wanderers":  "Wolves",
            "West Ham United":          "West Ham",
            "Newcastle United":         "Newcastle",
        }
        return _short_map.get(n, n)

    def _draw_player_row(player, row_x, row_y, stat_label, stat_value, stat_color):
        code      = player.get("code", 0)
        web_name  = player.get("web_name", "")
        team_id   = player.get("team", 0)
        team_obj  = _team_obj(team_id)
        team_code = team_obj.get("code", 0)
        team_name = _team_short_name(team_id)
        head = fetch_circular_headshot(code, web_name, radius=HEAD_R)
        img.paste(head, (row_x, row_y), head)
        logo = fetch_team_logo(team_code, size=LOGO_SZ)
        img.paste(logo,
                  (row_x + HEAD_R*2 - LOGO_SZ + 4, row_y + HEAD_R*2 - LOGO_SZ + 4),
                  logo)
        tx = row_x + HEAD_R * 2 + 12
        name_clean = re.sub(r'^[A-Z]\.\s*', '', web_name).upper()
        name_font  = _get_font(21, "Bold")
        name_bbox  = d.textbbox((0, 0), name_clean, font=name_font)
        while (name_bbox[2] - name_bbox[0]) > (HALF_W - HEAD_R*2 - 30) \
                and name_font.size > 13:
            name_font = _get_font(name_font.size - 1, "Bold")
            name_bbox = d.textbbox((0, 0), name_clean, font=name_font)
        d.text((tx, row_y + 2),  name_clean, font=name_font, fill=C_WHITE)
        d.text((tx, row_y + 26), team_name, font=_get_font(16, "Bold"), fill=C_GREY)
        pill_txt  = f"{stat_label}  {stat_value}"
        pill_font = _get_font(18, "Bold")
        pill_bb   = d.textbbox((0, 0), pill_txt, font=pill_font)
        pill_w    = (pill_bb[2] - pill_bb[0]) + 18
        pill_x    = tx
        pill_y    = row_y + 46
        d.rounded_rectangle([pill_x, pill_y, pill_x + pill_w, pill_y + 24],
                            radius=7, fill=(20, 20, 40, 200), outline=stat_color, width=2)
        d.text((pill_x + 9, pill_y + 3), pill_txt, font=pill_font, fill=stat_color)

    d.rounded_rectangle([0, 0, HALF_W, TEASER_H], radius=24, fill=C_BG,
                        outline=C_BORDER_BLUE, width=4)
    d.text((HALF_W // 2, 30), "TOP CAPTAINCY PICKS",
           font=_get_font(24, "Bold"), fill=C_BORDER_BLUE, anchor="mm")
    d.text((HALF_W // 2, 56), f"GW{NEXT_GW} CAPTAIN OWNERSHIP",
           font=_get_font(16, "Bold"), fill=C_GREY, anchor="mm")
    d.line([(20, 70), (HALF_W - 20, 70)], fill=C_BORDER_BLUE, width=2)
    _cap_pool = sorted(
        [p for p in _all_els
         if float(p.get("selected_by_percent") or 0) >= 5.0
         and p.get("status") == "a"
         and float(p.get("ep_next") or 0) >= 3.0],
        key=lambda p: (float(p.get("selected_by_percent") or 0)
                       * float(p.get("ep_next") or 0)),
        reverse=True
    )[:5]
    _row_y, _row_step = 82, 82
    for _p in _cap_pool:
        _own  = float(_p.get("selected_by_percent") or 0)
        _xpts = float(_p.get("ep_next") or 0)
        _draw_player_row(_p, row_x=14, row_y=_row_y,
                         stat_label="★",
                         stat_value=f"{_own:.1f}%  xPts {_xpts:.1f}",
                         stat_color=C_BORDER_BLUE)
        _row_y += _row_step
    RX_OFF = HALF_W + PANEL_GAP
    d.rounded_rectangle([RX_OFF, 0, TEASER_W, TEASER_H], radius=24, fill=C_BG,
                        outline=C_BORDER_GOLD, width=4)
    d.text((RX_OFF + HALF_W // 2, 30), "TRANSFER MOMENTUM",
           font=_get_font(24, "Bold"), fill=C_BORDER_GOLD, anchor="mm")
    d.text((RX_OFF + HALF_W // 2, 56), "BIGGEST NET RISERS THIS GW",
           font=_get_font(16, "Bold"), fill=C_GREY, anchor="mm")
    d.line([(RX_OFF + 20, 70), (RX_OFF + HALF_W - 20, 70)], fill=C_BORDER_GOLD, width=2)
    _risers = sorted(
        [p for p in _all_els
         if (int(p.get("transfers_in_event") or 0)
             - int(p.get("transfers_out_event") or 0)) > 0
         and p.get("status") == "a"],
        key=lambda p: (int(p.get("transfers_in_event") or 0)
                       - int(p.get("transfers_out_event") or 0)),
        reverse=True
    )[:5]
    _row_y = 82
    for _p in _risers:
        _net = (int(_p.get("transfers_in_event") or 0)
                - int(_p.get("transfers_out_event") or 0))
        _net_str = (f"+{_net // 1000}K" if _net >= 1000 else f"+{_net}")
        _own = float(_p.get("selected_by_percent") or 0)
        _draw_player_row(_p, row_x=RX_OFF + 14, row_y=_row_y,
                         stat_label="▲",
                         stat_value=f"{_net_str}  EO {_own:.1f}%",
                         stat_color=C_BORDER_GOLD)
        _row_y += _row_step
    out = os.path.join(TEMP_DIR, "slide3_teaser.png")
    img.save(out, "PNG")
    return out


def make_slide8_teaser():
    TEASER_W, TEASER_H = 1034, 500
    PANEL_GAP           = 90
    HALF_W              = (TEASER_W - PANEL_GAP) // 2
    C_BG                = (15, 10, 35, 235)
    C_BORDER_GREEN      = "#00FF87"
    C_BORDER_CYAN       = "#00E5FF"
    C_BORDER_GOLD       = "#FFD700"
    C_BORDER_PURPLE     = "#B366FF"
    C_WHITE             = "#FFFFFF"
    C_GREY              = "#888888"
    HEAD_R              = 32
    LOGO_SZ             = 28
    img = Image.new("RGBA", (TEASER_W, TEASER_H), (0, 0, 0, 0))
    d   = ImageDraw.Draw(img)
    _all_els   = boot_data.get("elements", [])
    _all_teams = boot_data.get("teams", [])

    def _team_obj(team_id):
        return next((t for t in _all_teams if t["id"] == team_id), {})

    def _team_short_name(team_id):
        t = _team_obj(team_id)
        n = t.get("name", "")
        _short_map = {
            "Brighton & Hove Albion":   "Brighton",
            "Manchester United":        "Man Utd",
            "Manchester City":          "Man City",
            "Tottenham Hotspur":        "Spurs",
            "Nottingham Forest":        "Forest",
            "Wolverhampton Wanderers":  "Wolves",
            "West Ham United":          "West Ham",
            "Newcastle United":         "Newcastle",
        }
        return _short_map.get(n, n)

    _next_gw_local  = next((e["id"] for e in boot_data.get("events", [])
                            if e.get("is_next")), 1)
    _gw_team_counts = {}
    for _f in fix_data:
        if _f.get("event") != _next_gw_local:
            continue
        for _tid in (_f["team_h"], _f["team_a"]):
            _gw_team_counts[_tid] = _gw_team_counts.get(_tid, 0) + 1
    _playing_team_ids = set(_gw_team_counts.keys())

    def _draw_player_row(player, row_x, row_y,
                         left_label, left_value, left_color,
                         right_label, right_value, right_color):
        code      = player.get("code", 0)
        web_name  = player.get("web_name", "")
        team_id   = player.get("team", 0)
        team_obj  = _team_obj(team_id)
        team_code = team_obj.get("code", 0)
        team_name = _team_short_name(team_id)
        head = fetch_circular_headshot(code, web_name, radius=HEAD_R)
        img.paste(head, (row_x, row_y), head)
        logo = fetch_team_logo(team_code, size=LOGO_SZ)
        img.paste(logo,
                  (row_x + HEAD_R*2 - LOGO_SZ + 4, row_y + HEAD_R*2 - LOGO_SZ + 4),
                  logo)
        tx = row_x + HEAD_R * 2 + 12
        name_clean = re.sub(r'^[A-Z]\.\s*', '', web_name).upper()
        name_font  = _get_font(20, "Bold")
        name_bbox  = d.textbbox((0, 0), name_clean, font=name_font)
        while (name_bbox[2] - name_bbox[0]) > (HALF_W - HEAD_R*2 - 30) \
                and name_font.size > 13:
            name_font = _get_font(name_font.size - 1, "Bold")
            name_bbox = d.textbbox((0, 0), name_clean, font=name_font)
        d.text((tx, row_y + 2),  name_clean, font=name_font, fill=C_WHITE)
        d.text((tx, row_y + 24), team_name, font=_get_font(15, "Bold"), fill=C_GREY)
        _left_txt = f"{left_label}  {left_value}"
        _lf       = _get_font(17, "Bold")
        _lb       = d.textbbox((0, 0), _left_txt, font=_lf)
        _lw       = (_lb[2] - _lb[0]) + 16
        _ly       = row_y + 44
        d.rounded_rectangle([tx, _ly, tx + _lw, _ly + 22],
                            radius=6, fill=(20, 20, 40, 200),
                            outline=left_color, width=2)
        d.text((tx + 8, _ly + 2), _left_txt, font=_lf, fill=left_color)
        _right_txt = f"{right_label}  {right_value}"
        _rf        = _get_font(17, "Bold")
        _rb        = d.textbbox((0, 0), _right_txt, font=_rf)
        _rw        = (_rb[2] - _rb[0]) + 16
        _rx        = tx + _lw + 10
        d.rounded_rectangle([_rx, _ly, _rx + _rw, _ly + 22],
                            radius=6, fill=(20, 20, 40, 200),
                            outline=right_color, width=2)
        d.text((_rx + 8, _ly + 2), _right_txt, font=_rf, fill=right_color)

    d.rounded_rectangle([0, 0, HALF_W, TEASER_H], radius=24, fill=C_BG,
                        outline=C_BORDER_GREEN, width=4)
    d.text((HALF_W // 2, 30), "FORM TABLE",
           font=_get_font(24, "Bold"), fill=C_BORDER_GREEN, anchor="mm")
    d.text((HALF_W // 2, 56), "TOP FORM  +  NEXT FIXTURE",
           font=_get_font(16, "Bold"), fill=C_GREY, anchor="mm")
    d.line([(20, 70), (HALF_W - 20, 70)], fill=C_BORDER_GREEN, width=2)
    _form_pool = sorted(
        [p for p in _all_els
         if float(p.get("form") or 0) >= 2.0
         and p.get("status") == "a"
         and float(p.get("ep_next") or 0) >= 3.0
         and p.get("team") in _playing_team_ids],
        key=lambda p: float(p.get("form") or 0),
        reverse=True
    )[:5]
    _row_y, _row_step = 82, 80
    for _p in _form_pool:
        _form    = float(_p.get("form") or 0)
        _xpts    = float(_p.get("ep_next") or 0)
        _tid     = _p.get("team", 0)
        _p_fixes = [f for f in fix_data
                    if f.get("event") == _next_gw_local
                    and (_tid == f["team_h"] or _tid == f["team_a"])]
        _fix_parts = []
        for _fx in _p_fixes:
            _is_h   = _fx["team_h"] == _tid
            _opp_id = _fx["team_a"] if _is_h else _fx["team_h"]
            _opp    = next((t for t in _all_teams if t["id"] == _opp_id), {})
            _fix_parts.append(f"{_opp.get('short_name', '???')} {'H' if _is_h else 'A'}")
        _fix_str = " + ".join(_fix_parts) if _fix_parts else "BGW"
        _draw_player_row(_p, row_x=14, row_y=_row_y,
                         left_label="FORM", left_value=f"{_form:.1f}",
                         left_color=C_BORDER_GREEN,
                         right_label="vs", right_value=_fix_str,
                         right_color=C_BORDER_GOLD)
        _row_y += _row_step
    RX_OFF = HALF_W + PANEL_GAP
    d.rounded_rectangle([RX_OFF, 0, TEASER_W, TEASER_H], radius=24, fill=C_BG,
                        outline=C_BORDER_CYAN, width=4)
    d.text((RX_OFF + HALF_W // 2, 30), "ICT + THREAT MATRIX",
           font=_get_font(24, "Bold"), fill=C_BORDER_CYAN, anchor="mm")
    d.text((RX_OFF + HALF_W // 2, 56), "ELITE UNDERLYING METRICS",
           font=_get_font(16, "Bold"), fill=C_GREY, anchor="mm")
    d.line([(RX_OFF + 20, 70), (RX_OFF + HALF_W - 20, 70)], fill=C_BORDER_CYAN, width=2)
    _ict_pool = sorted(
        [p for p in _all_els
         if float(p.get("ict_index") or 0) >= 50.0
         and p.get("status") == "a"
         and p.get("team") in _playing_team_ids],
        key=lambda p: float(p.get("ict_index") or 0),
        reverse=True
    )[:5]
    _row_y = 82
    for _p in _ict_pool:
        _ict    = float(_p.get("ict_index") or 0)
        _threat = float(_p.get("threat") or 0)
        _draw_player_row(_p, row_x=RX_OFF + 14, row_y=_row_y,
                         left_label="ICT",    left_value=f"{_ict:.0f}",
                         left_color=C_BORDER_CYAN,
                         right_label="THREAT", right_value=f"{_threat:.0f}",
                         right_color=C_BORDER_PURPLE)
        _row_y += _row_step
    out = os.path.join(TEMP_DIR, "slide8_teaser.png")
    img.save(out, "PNG")
    return out

print("✅ Cell 17b complete — all teaser generators ready")

🟣 Cell 17b — Teaser Generators loading...
✅ Cell 17b complete — all teaser generators ready


In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 CELL 17c — Trigger Finders + Squad Engine
Depends on: Cell 17a, Cell 17b
"""

import os, re, requests
from PIL import Image, ImageDraw

print("🟣 Cell 17c — Trigger Finders + Squad Engine loading...")


def find_agenda_triggers(cues, dur, slide_key):
    overlays_dir = os.path.join(PNG_DIR, "slide1_agenda")
    if not os.path.isdir(overlays_dir): return []
    files = sorted(f for f in os.listdir(overlays_dir) if f.lower().endswith(".png"))
    if not files: return []


    # ── DYNAMIC KEYWORDS from actual agenda items ─────────────────────
    # 🔴 FIX: Fallback to the exact default agenda items if the global isn't set, ensuring triggers match narration
    _agenda_items = globals().get("agenda_items", [
        ("FIXTURE TICKER",     None),
        ("TRANSFER TARGETS",   None),
        ("INJURY ALERT",       None),
        ("CAPTAINCY MATRIX",   None),
        ("TEAM REVEAL",        None)
    ])
    _agenda_labels = [item for item, _ in _agenda_items]

    def _extract_keywords(label):
        """Pull 1–2 most meaningful words from each agenda label."""
        stopwords = {"the","and","or","of","to","a","an","is","in","vs","&",
                     "our","this","your","with","for","from","on","at","by"}
        words = re.sub(r"[^a-zA-Z\s]", "", label).lower().split()
        meaningful = [w for w in words if w not in stopwords and len(w) >= 3]
        return meaningful[:2] if meaningful else [words[0]] if words else []

    KEYWORDS = [_extract_keywords(lbl) for lbl in _agenda_labels]

    # Pad to cover all files if more overlays than agenda items
    while len(KEYWORDS) < len(files):
        KEYWORDS.append([])

    wt = build_word_timeline(cues, slide_key)
    used_idx = set()
    triggers = []
    last_t   = 0.5
    n_files  = len(files)

    for idx, fname in enumerate(files):
        full    = os.path.join(overlays_dir, fname)
        kws     = KEYWORDS[idx] if idx < len(KEYWORDS) else []
        t_start = None

        if kws and wt:
            target_words  = [_normalize_word(w) for w in kws]
            joined_target = "".join(target_words)

            for i in range(len(wt)):
                if i in used_idx: continue

                # Multi-word match
                if (len(target_words) > 1
                        and i <= len(wt) - len(target_words)
                        and all(wt[i+j][0] == target_words[j]
                                for j in range(len(target_words)))):
                    for j in range(len(target_words)):
                        used_idx.add(i + j)
                    t_start = max(0.0, wt[i][1] + 0.05)
                    break

                # Single joined match
                if wt[i][0] == joined_target:
                    used_idx.add(i)
                    t_start = max(0.0, wt[i][1] + 0.05)
                    break

                # First keyword only (partial fallback)
                if wt[i][0] == target_words[0] and i not in used_idx:
                    used_idx.add(i)
                    t_start = max(0.0, wt[i][1] + 0.05)
                    break

        # ── EVEN DISTRIBUTION fallback across full audio duration ─────
        if t_start is None or t_start < last_t:
            gap     = (dur - 1.0 - last_t) / max(1, n_files - idx)
            # 🔴 FIX: Increased minimum gap to 2.5s so popups never fire too rapidly if text matching fails
            t_start = min(dur - 1.0, last_t + max(gap, 2.5))

        last_t = t_start

        img = Image.open(full).convert("RGBA")
        if img.size != (VW, VH):
            img = img.resize((VW, VH), Image.LANCZOS)
        dst = os.path.join(TEMP_DIR, f"agenda_{idx:02d}.png")
        img.save(dst, "PNG")

        triggers.append({
            'kind': 'agenda', 'path': dst,
            'cw': VW, 'ch': VH, 'x': 0, 'y': 0,
            't_start': t_start, 't_end': dur - 0.05,
        })

    return triggers


def _resolve_card_path_for_player(player_dict, all_files, used_paths=None):
    if not isinstance(player_dict, dict): return None
    code = player_dict.get('code')
    if not code: return None
    target_filename = f"{code}.png"
    if target_filename in all_files:
        full_path = os.path.join(CARDS_DIR, target_filename)
        if used_paths is not None:
            used_paths.add(full_path)
        return target_filename
    print(f"⚠️ Card missing for code {code}")
    return None


def make_big_card(card_path, idx):
    """Scales the high-res player card into the cinematic overlay dimensions."""
    out_path = os.path.join(TEMP_DIR, f"big_card_{idx}.png")
    if os.path.exists(card_path):
        try:
            img = Image.open(card_path).convert("RGBA")
            img = img.resize((BIG_CARD_W, BIG_CARD_H), Image.LANCZOS)
            img.save(out_path, "PNG")
            return out_path
        except Exception as e:
            print(f"⚠️ Failed to process big card {idx}: {e}")
    Image.new("RGBA", (BIG_CARD_W, BIG_CARD_H), (0, 0, 0, 0)).save(out_path, "PNG")
    return out_path


def find_two_stage_triggers(cues, dur, slide_key):
    wt         = build_word_timeline(cues, slide_key)
    layout     = SLIDE_LAYOUTS.get(slide_key, SLIDE_LAYOUTS["Slide_2_Market_Trends"])
    used_paths = set()
    if not os.path.isdir(CARDS_DIR): return []
    all_files  = os.listdir(CARDS_DIR)

    locked = []
    if slide_key == "Slide_2_Market_Trends":
        for p in globals().get('top_in', [])[:6]:  locked.append((p, "left", "#00FF87"))
        for p in globals().get('top_out', [])[:6]: locked.append((p, "right", "#FF1744"))
    elif slide_key == "Slide_3_Sword_Shield":
        for p in globals().get('shields', [])[:6]: locked.append((p, "left", "#1E90FF"))
        for p in globals().get('swords', [])[:6]:  locked.append((p, "right", "#FFD700"))
    elif slide_key == "Slide_8_Targets_Avoids":
        def _pos_sort(p):
            _et = p.get('element_type') or {"GK": 1, "DEF": 2, "MID": 3, "FWD": 4}.get(p.get('pos', 'MID'), 3)
            return {1: 0, 2: 1, 3: 2, 4: 3}.get(_et, 2)
        _tgts = sorted(globals().get('target_players', []), key=_pos_sort)
        _avds = sorted(globals().get('avoid_players', []), key=_pos_sort)
        for _p in _tgts: locked.append((_p, "left",  "#00FF87"))
        for _p in _avds: locked.append((_p, "right", "#FF1744"))


    elif slide_key == "Slide_12_Outro":
        pool = globals().get('s10_squad_pool', [])
        cap = next((p for p in pool if p.get('is_cap')), None)
        vc  = next((p for p in pool if p.get('is_vc')),  None)
        if cap: locked.append((cap, "left", "#FFD700"))
        if vc:  locked.append((vc,  "right", "#00E5FF"))

    player_cards = []
    all_files_set = set(all_files)
    for p, side, color in locked:
        code = p.get('code')
        fpath = None
        if code:
            target_file = f"{code}.png"
            if target_file in all_files_set and target_file not in used_paths:
                fpath = target_file
                used_paths.add(target_file)
        if fpath:
            player_cards.append((p, side, color, os.path.join(CARDS_DIR, fpath)))

    wt_cursor = 0
    fired = set()
    matches = []

    for (p, side, color, full_path) in player_cards:
        t_start, new_cursor = _find_sequential_trigger(p, wt, wt_cursor, delay=1.5)
        if t_start is not None:
            wt_cursor = new_cursor
        else:
            # 🔴 FIX: Only advance by 1 on miss so we don't accidentally jump over the next player's name
            wt_cursor = min(wt_cursor + 1, len(wt) - 1)

        if full_path in fired: continue
        fired.add(full_path)
        slot = sum(1 for m in matches if m[2] == side)
        matches.append([full_path, t_start, side, color, slot, p])

    i = 0
    while i < len(matches):
        if matches[i][1] is None:
            block_count = 0
            for j in range(i, len(matches)):
                if matches[j][1] is None: block_count += 1
                else: break
            next_t = matches[i + block_count][1] if (i + block_count < len(matches)) else dur - 1.5
            prev_t = matches[i-1][1] if i > 0 else 0.0
            gap = next_t - prev_t
            stagger = min(2.5, gap / (block_count + 1))
            for k in range(block_count):
                matches[i + k][1] = next_t - ((block_count - k) * stagger)
            i += block_count
        else:
            i += 1

    triggers = []
    _big_dur   = 2.0 if slide_key == "Slide_12_Outro" else 3.0
    _end_clamp = dur - 5.0 if slide_key == "Slide_12_Outro" else dur - 1.5

    for idx, (path, ts, side, color, slot, player) in enumerate(matches):
        if idx + 1 < len(matches):
            t_big_end = matches[idx + 1][1] - 0.05
        else:
            t_big_end = ts + _big_dur

        t_big_end    = max(ts + _big_dur, min(t_big_end, _end_clamp))
        t_info_start = t_big_end + FADE_DUR
        t_info_end   = dur - 0.05
        info_w  = layout.get("INFO_W", INFO_ROW_W)
        info_h  = layout.get("INFO_H", INFO_ROW_H)
        row_gap = layout.get("ROW_GAP", info_h + 8)
        col_x = layout["COL_1_X"] if side == "left" else layout["COL_2_X"]
        row_y = layout["ROW_START_Y"] + slot * row_gap

        big_path  = make_big_card(path, idx)
        info_path = make_info_row(player, color, idx, slide_key)

        triggers.append({
            'kind': 'big', 'path': big_path,
            'cw': BIG_CARD_W, 'ch': BIG_CARD_H,
            'x': BIG_CARD_X, 'y': BIG_CARD_Y,
            't_start': ts, 't_end': t_big_end,
        })
        triggers.append({
            'kind': 'info', 'path': info_path,
            'cw': info_w, 'ch': info_h,
            'x': col_x, 'y': row_y,
            't_start': t_info_start, 't_end': t_info_end,
        })

        # <<< START REPLACEMENT (Cell 17c - Add Slide 11 Triggers) <<<
    if slide_key == "Slide_11_Chip_Strategy":
        chip_dir = os.path.join(PNG_DIR, "slide11_agenda")
        if os.path.isdir(chip_dir):
            chip_files = sorted(f for f in os.listdir(chip_dir) if f.lower().endswith(".png"))
            n_chip = len(chip_files)
            for idx, fname in enumerate(chip_files):
                full = os.path.join(chip_dir, fname)
                frac    = (idx + 1) / (n_chip + 1)
                t_start = dur * frac
                img = Image.open(full).convert("RGBA")
                if img.size != (VW, VH): img = img.resize((VW, VH), Image.LANCZOS)
                dst = os.path.join(TEMP_DIR, f"chip_agenda_{idx:02d}.png")
                img.save(dst, "PNG")
                triggers.append({'kind': 'agenda', 'path': dst, 'cw': VW, 'ch': VH,
                                  'x': 0, 'y': 0, 't_start': t_start, 't_end': dur - 0.05})
# >>> END REPLACEMENT >>>

    if slide_key == "Slide_12_Outro":
        outro_dir = os.path.join(PNG_DIR, "slide12_agenda")
        if os.path.isdir(outro_dir):
            outro_files = sorted(f for f in os.listdir(outro_dir) if f.lower().endswith(".png"))
            n_outro = len(outro_files)
            for idx, fname in enumerate(outro_files):
                full = os.path.join(outro_dir, fname)
                frac    = (idx + 1) / (n_outro + 1)
                t_start = dur * frac
                img = Image.open(full).convert("RGBA")
                if img.size != (VW, VH): img = img.resize((VW, VH), Image.LANCZOS)
                dst = os.path.join(TEMP_DIR, f"outro_agenda_{idx:02d}.png")
                img.save(dst, "PNG")
                triggers.append({'kind': 'agenda', 'path': dst, 'cw': VW, 'ch': VH,
                                  'x': 0, 'y': 0, 't_start': t_start, 't_end': dur - 0.05})

    for s_key, t_fn, t_name, tx, ty in [
    ("Slide_2_Market_Trends",  make_slide2_teaser, "slide2_teaser", 725, 250),
    ("Slide_3_Sword_Shield",   make_slide3_teaser, "slide3_teaser", 725, 250),
    ("Slide_8_Targets_Avoids", make_slide8_teaser, "slide8_teaser", 728, 165),
]:
        if slide_key == s_key:
            first_big_t = min((t['t_start'] for t in triggers if t.get('kind') == 'big'), default=None)
            if first_big_t is not None and first_big_t > (FADE_DUR * 2):
                teaser_path = t_fn()
                _tw, _th = Image.open(teaser_path).size
                triggers.insert(0, {'kind': 'teaser', 'path': teaser_path,
                                     'cw': _tw, 'ch': _th, 'x': tx, 'y': ty,
                                     't_start': FADE_DUR,
                                     't_end': max(FADE_DUR, first_big_t - 0.05)})
    return triggers


def make_s9_sidebar():
    w, h = S9_SIDEBAR_W, S9_SIDEBAR_H
    img = Image.new("RGBA", (w, h), (0, 0, 0, 0))
    d   = ImageDraw.Draw(img)
    d.rounded_rectangle([0, 0, w, h], radius=25, fill=(15, 10, 30, 230), outline="#FFD700", width=4)
    curr_gw   = globals().get('CURR_GW', '?')
    gw_pts    = globals().get('current_gw_pts', 0)
    overall   = globals().get('overall_pts', '---')
    rank_now  = globals().get('current_rank', 0)
    try:
        MANAGER_ID = globals().get('MANAGER_ID', 664987)
        hist_url = f"https://fantasy.premierleague.com/api/entry/{MANAGER_ID}/history/"
        hist_r = requests.get(hist_url, headers=HEADERS, timeout=5).json()
        c_hist = next(x for x in hist_r['current'] if x['event'] == curr_gw)
        p_hist = next(x for x in hist_r['current'] if x['event'] == curr_gw - 1)
        rank_now = c_hist['overall_rank']
        rank_prev = p_hist['overall_rank']
    except:
        rank_prev = globals().get('prev_rank', 0)
    rank_diff = rank_prev - rank_now if (rank_prev and rank_now) else 0
    title_txt = f"GW{curr_gw} REVIEW"
    t_font = _get_font(40, "Bold")
    t_bb = d.textbbox((0,0), title_txt, font=t_font)
    t_w = t_bb[2] - t_bb[0]
    fh_size = 65
    gap = 15
    total_w = fh_size + gap + t_w
    start_x = (w - total_w) // 2
    s9_payload = globals().get('s9_payload', {})
    active_chip = s9_payload.get('active_chip')
    chip_drawn = False
    if active_chip:
        chip_map = {"freehit": "FH.png", "wildcard": "WC.png", "bbench": "BB.png", "3xc": "TC.png"}
        chip_file = chip_map.get(active_chip)
        if chip_file:
            chip_path = os.path.join(ELEM_DIR, chip_file)
            if not os.path.exists(chip_path):
                chip_path = os.path.join(ELEM_DIR, chip_file.lower())
            if os.path.exists(chip_path):
                try:
                    chip_img = Image.open(chip_path).convert("RGBA").resize((fh_size, fh_size), Image.LANCZOS)
                    img.paste(chip_img, (start_x, 22), chip_img)
                    d.text((start_x + fh_size + gap, 50), title_txt, font=t_font, fill="#FFD700", anchor="lm")
                    chip_drawn = True
                except Exception: pass
    if not chip_drawn:
        start_x = (w - t_w) // 2
        d.text((start_x, 50), title_txt, font=t_font, fill="#FFD700", anchor="lm")
    d.line([(30, 90), (w-30, 90)], fill="#FFD700", width=3)
    d.text((w//2, 190), str(gw_pts), font=_get_font(140, "Bold"), fill="#FFFFFF", anchor="mm")
    d.text((w//2, 290), "POINTS THIS WEEK", font=_get_font(24, "Bold"), fill="#00E5FF", anchor="mm")
    d.line([(40, 350), (w-40, 350)], fill="#444444", width=2)
    d.text((w//2, 420), f"{overall}", font=_get_font(65, "Bold"), fill="#FFD700", anchor="mm")
    d.text((w//2, 480), "OVERALL TOTAL", font=_get_font(22, "Bold"), fill="#00E5FF", anchor="mm")
    d.line([(40, 530), (w-40, 530)], fill="#444444", width=2)
    rank_str = f"{rank_now:,}" if rank_now else "---"
    r_font = _get_font(50, "Bold")
    r_bb = d.textbbox((0, 0), rank_str, font=r_font)
    r_w = r_bb[2] - r_bb[0]
    r_x = (w - r_w) // 2
    r_y = 590
    d.text((w//2, r_y), rank_str, font=r_font, fill="#FFFFFF", anchor="mm")
    arr_x = r_x - 35
    if rank_diff > 0:
        d.polygon([(arr_x, r_y - 12), (arr_x - 12, r_y + 8), (arr_x + 12, r_y + 8)], fill="#00FF87")
        d.rectangle([arr_x - 5, r_y + 8, arr_x + 5, r_y + 28], fill="#00FF87")
    elif rank_diff < 0:
        d.polygon([(arr_x, r_y + 28), (arr_x - 12, r_y + 8), (arr_x + 12, r_y + 8)], fill="#FF1744")
        d.rectangle([arr_x - 5, r_y - 12, arr_x + 5, r_y + 8], fill="#FF1744")
    else:
        d.rectangle([arr_x - 10, r_y + 6, arr_x + 10, r_y + 12], fill="#888888")
    d.text((w//2, 640), "OVERALL RANK", font=_get_font(22, "Bold"), fill="#888888", anchor="mm")
    d.text((w//2, 700), "INSIDE 10K", font=_get_font(40, "Bold"), fill="#00FF87", anchor="mm")
    d.text((w//2, 740), "TARGET RANK", font=_get_font(18, "Bold"), fill="#00FF87", anchor="mm")
    out = os.path.join(TEMP_DIR, "s9_sidebar.png")
    img.save(out, "PNG")
    return out


def make_s10_dynamic_overlays(t_out, t_in, total_xpts, bank_val, squad_val, target_dict=None):
    tw, th = 540, 240
    t_img = Image.new("RGBA", (tw, th), (0,0,0,0))
    d = ImageDraw.Draw(t_img)
    if t_out and t_in:
        p_out = t_out[0]
        p_in = t_in[0]
        h_out = fetch_circular_headshot(p_out.get('code',0), p_out.get('web_name',''), radius=50)
        h_in = fetch_circular_headshot(p_in.get('code',0), p_in.get('web_name',''), radius=50)
        t_img.paste(h_out, (60, 10), h_out)
        t_img.paste(h_in, (380, 10), h_in)
        arr_x, arr_y = 270, 60
        d.line([(arr_x - 30, arr_y), (arr_x + 10, arr_y)], fill="#00FF87", width=6)
        d.polygon([(arr_x + 10, arr_y - 12), (arr_x + 10, arr_y + 12), (arr_x + 35, arr_y)], fill="#00FF87")
        def _draw_ptext(px, p_dict):
            name = re.sub(r'^[A-Z]\.\s*', '', p_dict.get('web_name', '')).upper()
            opp  = p_dict.get('next_opp', 'BLANK').upper()
            xpts = f"xPts: {p_dict.get('ep_next', 0):.1f}"
            d.text((px, 130), name, font=_get_font(24, "Bold"), fill="#FFFFFF", anchor="mm")
            d.text((px, 165), opp, font=_get_font(18, "Bold"), fill="#AAAAAA", anchor="mm")
            d.text((px, 195), xpts, font=_get_font(20, "Bold"), fill="#00FF87", anchor="mm")
        _draw_ptext(110, p_out)
        _draw_ptext(430, p_in)
    out_transfer = os.path.join(TEMP_DIR, "s10_dyn_transfer.png")
    t_img.save(out_transfer, "PNG")

    xw, xh = 540, 80
    x_img = Image.new("RGBA", (xw, xh), (0,0,0,0))
    dx = ImageDraw.Draw(x_img)
    dx.text((xw//2, xh//2), f"PREDICTED STARTING XI: {total_xpts:.1f} XPTS", font=_get_font(28, "Bold"), fill="#FFFFFF", anchor="mm")
    out_xi = os.path.join(TEMP_DIR, "s10_dyn_xi.png")
    x_img.save(out_xi, "PNG")

    ww, wh = 540, 60
    w_img = Image.new("RGBA", (ww, wh), (0,0,0,0))
    dw = ImageDraw.Draw(w_img)
    dw.text((ww//2, wh//2), "TRANSFER WATCH LIST", font=_get_font(32, "Bold"), fill="#FFD700", anchor="mm")
    out_watch = os.path.join(TEMP_DIR, "s10_dyn_watch.png")
    w_img.save(out_watch, "PNG")

    bw, bh = 540, 80
    b_img = Image.new("RGBA", (bw, bh), (0,0,0,0))
    db = ImageDraw.Draw(b_img)
    db.rounded_rectangle([0, 0, bw, bh], radius=15, fill=(20,15,40,230), outline="#00E5FF", width=3)
    db.text((135, 25), "IN THE BANK", font=_get_font(18, "Bold"), fill="#AAAAAA", anchor="mm")
    db.text((135, 55), f"£{bank_val:.1f}m", font=_get_font(28, "Bold"), fill="#00E5FF", anchor="mm")
    db.text((405, 25), "SQUAD VALUE", font=_get_font(18, "Bold"), fill="#AAAAAA", anchor="mm")
    db.text((405, 55), f"£{squad_val:.1f}m", font=_get_font(28, "Bold"), fill="#FFD700", anchor="mm")
    out_bank = os.path.join(TEMP_DIR, "s10_dyn_bank.png")
    b_img.save(out_bank, "PNG")

    out_target = None
    if target_dict:
        tgw, tgh = 540, 140
        tg_img = Image.new("RGBA", (tgw, tgh), (0,0,0,0))
        dtg = ImageDraw.Draw(tg_img)
        dtg.rounded_rectangle([0, 0, tgw, tgh], radius=15, fill=(15,10,30,240), outline="#00FF87", width=3)
        t_head = fetch_circular_headshot(target_dict.get('code',0), target_dict.get('web_name',''), radius=50)
        tg_img.paste(t_head, (20, 20), t_head)
        t_name = re.sub(r'^[A-Z]\.\s*', '', target_dict.get('web_name', '')).upper()
        _raw_cost = target_dict.get('now_cost')
        t_price = (_raw_cost / 10.0) if _raw_cost else target_dict.get('cost', 0.0)
        t_xpts = float(target_dict.get('ep_next', 0))
        dtg.text((140, 35), t_name, font=_get_font(30, "Bold"), fill="#FFFFFF", anchor="lm")

        # 🔴 FIX: Auto-detect GW type — no hardcoding
        _t_team = target_dict.get('team') or target_dict.get('team_id', 0)
        _t_gw   = globals().get('NEXT_GW', 0)
        try:
            _t_fix_count = sum(
                1 for _f in globals().get('fix_data', [])
                if _f.get('event') == _t_gw and
                (_f['team_h'] == _t_team or _f['team_a'] == _t_team)
            )
        except Exception:
            _t_fix_count = 1
        if _t_fix_count > 1:
            _priority_label = "PRIORITY DGW TARGET"
        elif _t_fix_count == 0:
            _priority_label = "PRIORITY BGW TARGET"
        else:
            _priority_label = "PRIORITY TARGET"
        dtg.text((140, 75), _priority_label, font=_get_font(20, "Bold"), fill="#00FF87", anchor="lm")

        dtg.text((140, 105), f"Price: £{t_price:.1f}m  |  xPts: {t_xpts:.1f}", font=_get_font(20, "Bold"), fill="#FFD700", anchor="lm")
        out_target = os.path.join(TEMP_DIR, "s10_dyn_target.png")
        tg_img.save(out_target, "PNG")
    return out_transfer, out_xi, out_watch, out_bank, out_target


def get_dynamic_coords(count, y_pos):
    if count == 1: return [(1380, y_pos)]
    if count == 2: return [(1230, y_pos), (1530, y_pos)]
    if count == 3: return [(1080, y_pos), (1380, y_pos), (1680, y_pos)]
    if count == 4: return [(1110, y_pos), (1290, y_pos), (1470, y_pos), (1650, y_pos)]
    if count == 5: return [(1020, y_pos), (1200, y_pos), (1380, y_pos), (1560, y_pos), (1740, y_pos)]
    return []


def find_squad_triggers(cues, dur, slide_key):
    frame_folder = "squad_frames" if slide_key == "Slide_9_Base" else "plan_frames"
    folder = os.path.join(CARDS_DIR, frame_folder)
    if not os.path.exists(folder): return []
    files = sorted(f for f in os.listdir(folder) if f.endswith(".png"))
    wt    = build_word_timeline(cues, slide_key)

    if slide_key == "Slide_9_Base":
        starters = globals().get('s9_starters', [])
        bench    = globals().get('s9_bench', [])
        targets  = []
    else:
        starters = globals().get('s10_starters', [])
        bench    = globals().get('s10_bench', [])
        targets  = globals().get('top_targets', [])

    all_players = starters + bench + targets

    def _pos_int(p):
        pt = p.get('element_type') or p.get('pos', 3)
        if isinstance(pt, str):
            return {"GK":1,"DEF":2,"MID":3,"FWD":4}.get(pt.upper(), 3)
        try: return int(pt)
        except (TypeError, ValueError): return 3

    def _resolve(name_str):
        name_clean = _clean_fn(name_str)
        if not name_clean: return None
        for pl in all_players:
            web = _clean_fn(pl.get('web_name', pl.get('name', '')))
            sec = _clean_fn(pl.get('second_name', ''))
            if name_clean == web or name_clean == sec: return pl
            if len(name_clean) >= 4 and (name_clean in web or web in name_clean): return pl
        return None

    gks  = sum(1 for p in starters if _pos_int(p) == 1)
    defs = sum(1 for p in starters if _pos_int(p) == 2)
    mids = sum(1 for p in starters if _pos_int(p) == 3)
    fwds = sum(1 for p in starters if _pos_int(p) == 4)

    coords = {1: list(get_dynamic_coords(gks, 780)),
              2: list(get_dynamic_coords(defs, 600)),
              3: list(get_dynamic_coords(mids, 400)),
              4: list(get_dynamic_coords(fwds, 200))}

    bench_coords  = list(PITCH_S10["bench"])
    target_coords = list(PITCH_S10["targets"])
    starter_codes = {p.get('code') or p.get('id') for p in starters}
    bench_codes   = {p.get('code') or p.get('id') for p in bench}
    used_idx, used_paths = set(), set()
    raw = []

    for f in files:
        path = os.path.join(folder, f)
        if path in used_paths: continue
        base = os.path.splitext(f)[0]
        parts = base.split("_", 2)
        if len(parts) < 3: continue
        p = _resolve(parts[2])
        if p is None:
            print(f"⚠️ Could not resolve squad player from filename: {f}")
            continue
        code = p.get('code') or p.get('id')
        matched_player = next((pl for pl in all_players if (pl.get('code') or pl.get('id')) == code), p)
        p = matched_player
        code = p.get('code') or p.get('id')

        is_st = code in starter_codes
        is_bn = code in bench_codes
        is_tg = (not is_st) and (not is_bn)
        pt = _pos_int(p)
        cx, cy = 0, 0

        if is_st and pt in coords and coords[pt]: cx, cy = coords[pt].pop(0)
        elif is_bn and bench_coords:              cx, cy = bench_coords.pop(0)
        elif is_tg and target_coords:             cx, cy = target_coords.pop(0)
        else: continue

        ch = TARGET_CARD_H if is_tg else CARD_H
        img = Image.open(path).convert("RGBA")
        ratio = ch / img.height
        cw = int(img.width * ratio)
        img = img.resize((cw, ch), Image.LANCZOS)
        dst = os.path.join(TEMP_DIR, f"squad_{slide_key}_{len(raw):02d}.png")
        img.save(dst, "PNG")

        t_start = _find_exact_trigger(p, wt, delay=0.5)
        used_paths.add(path)
        raw.append({'kind': 'squad', 'path': dst,
                    'x': cx - cw // 2, 'y': cy - ch // 2,
                    'cw': cw, 'ch': ch, 't_start': t_start})

    matched = [r['t_start'] for r in raw if r['t_start'] is not None]
    last_t  = max(matched) if matched else max(1.0, dur - 10.0)
    defer   = last_t + 0.8
    for r in raw:
        if r['t_start'] is None:
            r['t_start'] = min(defer, dur - 0.5); defer += 0.6
    raw.sort(key=lambda r: r['t_start'])

    cur = 0.0
    for r in raw:
        if r['t_start'] < cur + 0.30: r['t_start'] = cur + 0.30
        cur = r['t_start']

    triggers = [{'kind':'squad', 'path':r['path'], 'cw':r['cw'], 'ch':r['ch'],
                 'x':r['x'], 'y':r['y'],
                 't_start':r['t_start'], 't_end': dur - 0.05} for r in raw]

    if slide_key == "Slide_10_Plan_Base":
        s9_pool  = globals().get('s9_squad_pool', [])
        s10_pool = globals().get('s10_squad_pool', [])
        s9_ids   = {p['id'] for p in s9_pool}
        s10_ids  = {p['id'] for p in s10_pool}

        t_out_all = [p for p in s9_pool if p['id'] not in s10_ids]
        t_in_all  = [p for p in s10_pool if p['id'] not in s9_ids]
        t_out_all.sort(key=lambda x: int(x.get('element_type', 1)), reverse=True)
        t_in_all.sort(key=lambda x: int(x.get('element_type', 1)), reverse=True)

        matched_out, matched_in = None, None
        for po in t_out_all:
            for pi in t_in_all:
                if int(po.get('element_type', 1)) == int(pi.get('element_type', 1)):
                    matched_out, matched_in = po, pi
                    break
            if matched_out: break

        t_out = [matched_out] if matched_out else t_out_all
        t_in  = [matched_in] if matched_in else t_in_all

        total_xpts = sum(p.get('ep_next', 0) for p in globals().get('s10_starters', []))
        cap = next((p for p in globals().get('s10_starters', []) if p.get('is_cap')), None)
        if cap: total_xpts += cap.get('ep_next', 0)

        _hist  = globals().get('s9_payload', {}).get('entry_history', {})
        _bank  = _hist.get('bank', 0) / 10.0
        _value = _hist.get('value', 1000) / 10.0
        _best_cp_def = next((p for p in globals().get('top_targets', []) if p.get('element_type') == 2), None)

        transfer_png, xi_png, watch_png, bank_png, target_png = make_s10_dynamic_overlays(
            t_out, t_in, total_xpts, _bank, _value, _best_cp_def
        )

        _wt_total = len(wt)
        _step     = max(1, _wt_total // 6)

        t_bank     = wt[_step * 1][1] if _wt_total > _step * 1 else 2.0
        t_transfer = wt[_step * 2][1] if _wt_total > _step * 2 else 6.0
        t_target   = wt[_step * 3][1] if _wt_total > _step * 3 else 8.0
        t_xi       = wt[_step * 4][1] if _wt_total > _step * 4 else 10.0
        t_watch    = wt[_step * 5][1] if _wt_total > _step * 5 else 12.0

        # 🔴 FIX: Dynamic y positions — computed top-down, xi sits just above target card
        _bank_y     = 225
        _transfer_y = _bank_y + 80 + 10          # 315, ends 555
        _xi_y       = _transfer_y + 240 + 7      # 562, ends 642
        _target_y   = _xi_y + 80 + 8             # 650, ends 790
        _watch_y    = _target_y + 140 + 8        # 798, ends 858

        triggers.append({'kind': 'info', 'path': bank_png,     'cw': 540, 'ch': 80,  'x': 60, 'y': _bank_y,     't_start': t_bank,     't_end': dur - 0.05})
        triggers.append({'kind': 'info', 'path': transfer_png, 'cw': 540, 'ch': 240, 'x': 60, 'y': _transfer_y, 't_start': t_transfer, 't_end': dur - 0.05})
        triggers.append({'kind': 'info', 'path': xi_png,       'cw': 540, 'ch': 80,  'x': 60, 'y': _xi_y,       't_start': t_xi,       't_end': dur - 0.05})

        if target_png:
            triggers.append({'kind': 'info', 'path': target_png, 'cw': 540, 'ch': 140, 'x': 60, 'y': _target_y, 't_start': t_target, 't_end': dur - 0.05})

        triggers.append({'kind': 'info', 'path': watch_png, 'cw': 540, 'ch': 60, 'x': 60, 'y': _watch_y, 't_start': t_watch, 't_end': dur - 0.05})

    return triggers


print("✅ Cell 17c complete — all trigger finders and squad engine ready")

🟣 Cell 17c — Trigger Finders + Squad Engine loading...
✅ Cell 17c complete — all trigger finders and squad engine ready


In [ ]:
# -*- coding: utf-8 -*-
"""
🟣 CELL 17d — FFmpeg Builder + Slide Builder + Main Execution
Depends on: Cell 17a, 17b, 17c
"""

print("🟣 Cell 17d — FFmpeg + Build Engine loading...")

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FFMPEG BUILDER                                                      ║
# ╚══════════════════════════════════════════════════════════════════════╝

def build_ffmpeg_cmd(bg_path, audio_path, triggers, dur, out_path,
                     extra_static_overlays=None):
    inputs = ["-loop","1","-framerate",str(FPS),"-i", bg_path, "-i", audio_path]

    # 🔴 ELITE UPGRADE: Hollywood Camera Lens (Dolly Push + Depth of Field)
    # Adds a subtle 3% push-in combined with a Gaussian blur (sigma=4) to simulate
    # a cinematic wide-aperture lens. This separates the sharp UI foreground from the studio background.
    fc = [
        f"[0:v]scale={VW*2}:{VH*2},zoompan=z='min(1.03, zoom+0.00015)':d=1:x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s={VW}x{VH},fps={FPS},format=yuv420p[bg_base]"
    ]

    prev = "bg_base"
    next_inp = 2

    if extra_static_overlays:
        for ovr_path, ovr_x, ovr_y in extra_static_overlays:
            inputs += ["-loop","1","-framerate",str(FPS),"-i", ovr_path]
            cur = f"vs{next_inp}"
            fc.append(f"[{next_inp}:v]format=rgba,colorchannelmixer=aa=1[ovr{next_inp}]")
            fc.append(f"[{prev}][ovr{next_inp}]overlay=x={ovr_x}:y={ovr_y}:format=auto[{cur}]")
            prev = cur
            next_inp += 1

    for trig in triggers:
        inputs += ["-loop","1","-framerate",str(FPS),"-i", trig['path']]

    for i, trig in enumerate(triggers):
        cw, ch = trig['cw'], trig['ch']
        x      = trig['x']
        y      = trig['y']
        ts     = max(0.0, float(trig['t_start']))
        te     = max(ts + 0.1, float(trig['t_end']))
        kind   = trig['kind']
        inp    = next_inp + i
        cur    = f"sl{next_inp}v{i}"
        card_l = f"sl{next_inp}c{i}"

        if kind in ('big', 'teaser'):
            fos = max(ts + 0.1, te - FADE_DUR)
            fc.append(
                f"[{inp}:v]scale={cw}:{ch},format=rgba,colorchannelmixer=aa=1,"
                f"fade=t=in:st={ts:.3f}:d={FADE_DUR}:alpha=1,"
                f"fade=t=out:st={fos:.3f}:d={FADE_DUR}:alpha=1[{card_l}]"
            )
            enable_expr = f"between(t,{ts:.3f},{te:.3f})"
        else:
            fc.append(
                f"[{inp}:v]scale={cw}:{ch},format=rgba,colorchannelmixer=aa=1,"
                f"fade=t=in:st={ts:.3f}:d={FADE_DUR}:alpha=1[{card_l}]"
            )
            enable_expr = f"gte(t,{ts:.3f})"

        # 🔴 ELITE UPGRADE: Cinematic Cubic Ease-Out Motion
        # Replaces linear sliding with professional cubic bezier easing (fast start, smooth settle).
        # The cards now drift up 60 pixels using an exponential deceleration curve.
        dynamic_y = f"{y} + 60*pow(1 - min(1, (t-{ts:.3f})/{FADE_DUR}), 3)"

        fc.append(
            f"[{prev}][{card_l}]overlay=x={x}:y='{dynamic_y}':format=auto:"
            f"enable='{enable_expr}'[{cur}]"
        )
        prev = cur

    fc.append(
        f"[1:a]afade=t=in:st=0:d={AUDIO_FADE},"
        f"afade=t=out:st={max(0,dur-AUDIO_FADE):.3f}:d={AUDIO_FADE}[aout]"
    )

    cmd = ["ffmpeg","-y", *inputs,
           "-filter_complex", "; ".join(fc),
           "-map", f"[{prev}]", "-map", "[aout]",
           "-c:v", "libx264", "-preset", "ultrafast", "-crf", "23",
           "-pix_fmt", "yuv420p",
           "-c:a", "aac", "-b:a", "192k",
           "-t", str(dur), "-r", str(FPS), out_path]
    return cmd

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SLIDE BUILDER                                                       ║
# ╚══════════════════════════════════════════════════════════════════════╝
def build_slide(n, png_file, slide_key, slide_type):
    png_path   = os.path.join(PNG_DIR,   png_file)
    audio_path = os.path.join(AUDIO_DIR, f"{slide_key}.mp3")
    vtt_path   = os.path.join(AUDIO_DIR, f"{slide_key}.vtt")
    out_path   = os.path.join(TEMP_DIR,  f"slide_{n:02d}.mp4")
    bg_path    = os.path.join(TEMP_DIR,  f"slide_{n:02d}_bg.png")

    print(f"  Slide {n:02d} │ {png_file[:38]:<38}", end="", flush=True)

    if not os.path.exists(png_path) or not os.path.exists(audio_path):
        print("  ✗ Assets missing"); return None
    dur = get_dur(audio_path)
    if dur <= 0: print("  ✗ Bad audio"); return None
    print(f"  {dur:.1f}s", flush=True)

    img = Image.open(png_path).convert("RGB")
    if img.size != (VW, VH): img = img.resize((VW, VH), Image.LANCZOS)
    img.save(bg_path)

    extra_static = []

    if slide_type == "none":
        cmd = ["ffmpeg","-y","-loop","1","-framerate",str(FPS),"-i",bg_path,
               "-i",audio_path,"-vf",f"fps={FPS},format=yuv420p",
               "-c:v","libx264","-preset","ultrafast","-crf","23",
               "-c:a","aac","-b:a","192k","-t",str(dur),"-r",str(FPS),
               "-af",f"afade=t=in:st=0:d={AUDIO_FADE},"
                     f"afade=t=out:st={max(0,dur-AUDIO_FADE):.3f}:d={AUDIO_FADE}",
               out_path]
        r = subprocess.run(cmd, capture_output=True, text=True)
        return out_path if r.returncode == 0 else None

    cues = parse_vtt(vtt_path)

    if slide_type == "agenda_s1":
        triggers = find_agenda_triggers(cues, dur, slide_key)
        first_agenda_t = min(
            (t['t_start'] for t in triggers if t.get('kind') == 'agenda'),
            default=None
        )
        if first_agenda_t is not None and first_agenda_t > (FADE_DUR * 2):
            teaser_path = make_slide1_teaser()
            triggers.insert(0, {
                'kind': 'teaser', 'path': teaser_path,
                'cw': VW, 'ch': VH, 'x': 0, 'y': 0,
                't_start': FADE_DUR,
                't_end': max(FADE_DUR, first_agenda_t - 0.05),
            })
    elif slide_type == "two_stage":
        triggers = find_two_stage_triggers(cues, dur, slide_key)


    elif slide_type == "squad_s9":
        triggers = find_squad_triggers(cues, dur, slide_key)
        sidebar_path = make_s9_sidebar()
        extra_static.append((sidebar_path, S9_SIDEBAR_X, S9_SIDEBAR_Y))
    elif slide_type == "plan_s10":
        triggers = find_squad_triggers(cues, dur, slide_key)
    else:
        triggers = []

    if not triggers and not extra_static:
        cmd = ["ffmpeg","-y","-loop","1","-framerate",str(FPS),"-i",bg_path,
               "-i",audio_path,"-vf",f"fps={FPS},format=yuv420p",
               "-c:v","libx264","-preset","ultrafast","-crf","23",
               "-c:a","aac","-b:a","192k","-t",str(dur),"-r",str(FPS),
               "-af",f"afade=t=in:st=0:d={AUDIO_FADE},"
                     f"afade=t=out:st={max(0,dur-AUDIO_FADE):.3f}:d={AUDIO_FADE}",
               out_path]
    else:
        cmd = build_ffmpeg_cmd(bg_path, audio_path, triggers, dur, out_path,
                               extra_static_overlays=extra_static)

    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"   ✗ FFmpeg error:\n{r.stderr[-2000:]}")
        return None

    drive_path = os.path.join(OUTPUT_DIR, f"slide_{n:02d}_{slide_key}.mp4")
    shutil.copy(out_path, drive_path)
    return out_path


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  MUSIC + CONCAT                                                      ║
# ╚══════════════════════════════════════════════════════════════════════╝

def build_music_mix(video_path, out_path):
    ip = os.path.join(ELEM_DIR,"intro.mp3")
    mp = os.path.join(ELEM_DIR,"mid.mp3")
    op = os.path.join(ELEM_DIR,"outro.mp3")
    dur = get_dur(video_path)
    available = {k:p for k,p in [("intro",ip),("mid",mp),("outro",op)] if os.path.exists(p)}
    if not available:
        shutil.copy(video_path, out_path); return out_path
    inputs = ["-i", video_path]
    fc = ["[0:a]volume=1.0[narr]"]
    srcs = ["[narr]"]; idx = 1; intro_end = 0.0
    if "intro" in available and dur > INTRO_DUR:
        inputs += ["-i", available["intro"]]
        fc.append(f"[{idx}:a]afade=t=out:st={INTRO_DUR-0.5:.1f}:d=0.5,volume={MUSIC_VOL}[imu]")
        srcs.append("[imu]"); idx+=1; intro_end=INTRO_DUR
    outro_start = max(0, dur-OUTRO_DUR)
    if "outro" in available and dur > OUTRO_DUR:
        inputs += ["-i", available["outro"]]
        fc.append(f"[{idx}:a]adelay={int(outro_start*1000)}|{int(outro_start*1000)},"
                  f"afade=t=in:st={outro_start:.1f}:d=0.5,volume={MUSIC_VOL}[omu]")
        srcs.append("[omu]"); idx+=1
    mid_dur = outro_start - intro_end
    if "mid" in available and mid_dur > 1.0:
        inputs += ["-i", available["mid"]]
        fc.append(f"[{idx}:a]aloop=loop=-1:size=2e+09,atrim=duration={mid_dur:.3f},"
                  f"adelay={int(intro_end*1000)}|{int(intro_end*1000)},"
                  f"afade=t=in:st={intro_end:.1f}:d=0.5,"
                  f"afade=t=out:st={outro_start-0.5:.1f}:d=0.5,volume={MUSIC_VOL*0.3}[mmu]")
        srcs.append("[mmu]"); idx+=1
    fc.append(f"{''.join(srcs)}amix=inputs={len(srcs)}:duration=first:normalize=0[aout]")
    cmd = ["ffmpeg","-y",*inputs,"-filter_complex","; ".join(fc),
           "-map","0:v","-map","[aout]","-c:v","copy","-c:a","aac","-b:a","192k", out_path]
    print("🎵 Mixing music...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0: shutil.copy(video_path, out_path)
    return out_path

def concatenate_slides(slide_paths, out_path):
    if len(slide_paths) == 1:
        shutil.copy(slide_paths[0], out_path); return out_path
    print(f"\n🔗 Concatenating {len(slide_paths)} slides...")
    list_file = os.path.join(TEMP_DIR,"concat_list.txt")
    with open(list_file,"w") as f:
        for p in slide_paths: f.write(f"file '{p}'\n")
    cmd = ["ffmpeg","-y","-f","concat","-safe","0","-i",list_file,
           "-c:v","libx264","-preset","ultrafast","-crf","23",
           "-c:a","aac","-b:a","192k","-pix_fmt","yuv420p", out_path]
    subprocess.run(cmd, capture_output=True, text=True)
    return out_path

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  MAIN                                                                ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("\n📽️  Building broadcast video...\n")
slide_paths = []
for row in MANIFEST:
    p = build_slide(*row)
    if p: slide_paths.append(p)

if not slide_paths: raise RuntimeError("❌ Zero slides built.")

concat_path = os.path.join(TEMP_DIR, "concat_raw.mp4")
concatenate_slides(slide_paths, concat_path)

final_path = os.path.join(OUTPUT_DIR, "final_video.mp4")
build_music_mix(concat_path, final_path)

size = os.path.getsize(final_path)/1024/1024
dur  = get_dur(final_path)
print(f"\n{'='*60}")
print(f" 🟣 VORTEX BROADCAST EDITION COMPLETE")
print(f"    → {final_path}")
print(f"    Duration : {dur/60:.1f} min")
print(f"    Size     : {size:.0f} MB")
print(f"{'='*60}")

🟣 Cell 17d — FFmpeg + Build Engine loading...

📽️  Building broadcast video...

  Slide 01 │ Slide_1_Intro_Empty.png                 60.0s
  Slide 02 │ Slide_9_Base.png                        79.1s
  Slide 03 │ Slide_2_Empty.png                       180.6s
⚠️ API error for 551221: HTTP 403
  Slide 04 │ Slide_3_Empty.png                       172.9s
⚠️ API error for 434752: HTTP 403
  Slide 05 │ Slide_4_fdr_map.png                     111.7s
  Slide 06 │ Slide_6_CS_Odds.png                     96.6s
  Slide 07 │ Slide_7a_Injury_MinuteRisk.png          152.7s
  Slide 08 │ Slide_7_All_20_Teams.png                117.5s
  Slide 09 │ Slide_8_Empty.png                       156.0s
⚠️ API error for 424876: HTTP 403
  Slide 10 │ Slide_11_Chip_Strategy.png              39.2s
  Slide 11 │ Slide_10_Plan_Base.png                  158.2s
  Slide 12 │ Slide_12_Outro_Empty.png                69.9s

🔗 Concatenating 12 slides...

 🟣 VORTEX BROADCAST EDITION COMPLETE
    → /content/drive/MyDrive/Vortex